# ALMA target explorer (Julia)

Pick one of **100 protoplanetary-disk targets** (listed alphabetically, each tagged by how TW Hya-like it is: T Tauri analogue, cavity, young/embedded, Herbig, non-axisymmetric, multiple system, edge-on, outbursting), or any custom source, query the ALMA archive, download a continuum product, and run the deprojected radial-profile and ring/gap analysis — generalised from the TW Hya notebook.

**Workflow:** menu → `select_target` → ALMA TAP query → observation table (finest resolution first) → size-guarded download → FITS image → centre fit → radial profile → Savitzky–Golay baseline subtraction → rings/gaps → literature comparison (where the catalogue has one).

**Physical analysis (optional, after a run):** `phys = physical_analysis(out)` fits Gaussian profiles to the rings (radius, width, amplitude with errors), fits the disk geometry from the ring ridges, tests for a lifted (flared) dust layer from the ring-centre offsets, measures azimuthal asymmetry (m = 1, 2 amplitudes per ring, beam and thickness checks, candidate arcs/vortices) and derives brightness temperature and optical depth; see the cell after *re-tune*.

**Quick start:** run all cells down to *Select and run*. Leave `TARGET_CHOICE = nothing` to be prompted with the menu, or set a number/name (`"MWC 480"`, or a menu number; numbers are alphabetical) or `custom_target("name"; distance_pc = ...)`.

**Caveat:** only the first-listed 25 well-studied disks (TW Hya, AS 209, HD 163296, ...) have stored coordinates, inclination and PA; the other 75 are resolved by name (CDS Sesame) and have approximate distances only. The catalogue's distances, inclinations and position angles are approximate literature values — verify them before quoting results. Unknown inclination/PA (`?`) means the disk is treated as face-on. See the changelog in the setup cell for what changed from the TW Hya notebook (notably a deprojection fix).

In [ ]:
# alma_target_explorer_julia.ipynb
#
# Generic ALMA continuum explorer: pick one of 100 protoplanetary-disk targets
# (or any custom target), query the ALMA archive, download a continuum
# product, and run the radial-profile / ring-gap analysis.
#
# Changelog
#   v1  Generalised from alma_twhya_query_julia (v1):
#       - 25-target menu + custom targets (CDS Sesame name resolution)
#       - observation chooser (finest spatial resolution first, optional band)
#       - tarball size guard before downloading
#       - FIXED deprojection geometry (the TW Hya v1 code stretched the major
#         axis instead of the minor axis; negligible at i = 7 deg, wrong for
#         inclined disks). PA is now east of north on the sky.
#       - star centre refined by minimising within-annulus scatter instead of
#         using the brightest pixel (which is wrong for ring/cavity disks)
#       - beam-based radial bin width, BUNIT-aware labels
#   v2  - menu expanded from 25 to 100 disks, listed alphabetically (print_target_menu(by = :group)
#         restores the TW Hya-likeness grouping); new groups: edge-on, outbursting
#       - the 75 added targets have no stored coordinates: CDS Sesame resolves them when selected
#       - the TW Hya default for the prompt is kept (Enter = TW Hya)
#       - archive top list: 100 targets, name variants merged, OFF/Sun dropped, curated entries kept
#   v3  - new group 'Multiple system' (AS 205, GW Ori, HK Tau, IRS 43, T Tau, UZ Tau, XZ Tau); added GW Ori,
#         IRS 43, RY Tau (dropped Sz 65, Sz 100, FP Tau to stay at 100)
#       - archive_url(target): ALMA Science Archive web-portal link, printed in the run header
#       - long-form names also match ("TW Hydrae", "V4046 Sagittarii", "HK Tauri A", "GW Orionis")
#   v4  - FIXED centre fit for off-centre disks: IM Lup's star is ~1" from the image centre, but the
#         symmetry search only covered +-0.15" and stuck at its corner (profile was garbage). The fit now
#         starts from the emission centroid and keeps walking while the optimum sits on the window edge
#   v5  - size guard: when an observation's package exceeds MAX_TAR_GB (HL Tau's finest-resolution one is
#         34.3 GB) the pipeline now tries the next observations automatically (auto_fallback); refused
#         packages are listed by print_too_large(); print_target_menu() sits next to 'Select and run'
#   v6  - geometry_mode = :auto: when the catalogue has no inclination/PA (most of the 75 added disks) they are
#         fitted from the image (radial-symmetry criterion, +-~5 deg); :fit re-measures even catalogue values
#       - warnings: coarser-resolution fallback, disk spanning < 8 beams, exclusion zone covering much of the disk
#       - min_radius_beams is now a run_alma_pipeline option; archive list drops planets/comets, curated_only = true
#       - <target>_target_summary.txt is written next to the PNGs: ALMA portal link, target description, data used,
#         radial-profile and ring/gap analysis, literature comparison (write_target_summary)
#       - menu grows to 500: the 100 curated disks + the most-observed archive disk targets (use_extended_catalogue!, MENU_SIZE);
#         archive entries get a Gaia distance (Bailer-Jones+21, SIMBAD parallax as fallback) and i/PA from the image fit
#       - show_archive_link(): the ALMA portal link in its own cell after 'Select and run' (clickable where Markdown renders)
#   v7  - physical_analysis(out): Gaussian ring fits (radius, width, amplitude with uncertainties; tapered-power-law disk + one
#         Gaussian per ring, own Levenberg-Marquardt), azimuthal asymmetry (axisymmetric-model residual image, m = 1 / m = 2
#         amplitudes per ring with false-alarm probabilities, candidate arcs / vortices), and brightness temperature / optical
#         depth (Planck T_b, passive-disk T_d, tau = -ln(1 - T_b/T_d)); writes <slug>_physical_analysis.txt and three PNGs
#       - geometry check: fit_ring_geometry fits one shared centre offset, inclination and PA to the ridges of the bright rings
#         (physical_analysis(out; use_fitted_geometry = true) adopts them; the asymmetry is always compared before/after)
#       - asymmetry fixes: the elliptical-beam m = 2 expectation now uses the same window-mean estimator as the measurement (the earlier peak-based
#         expectation was ~1.65x too large); the sub-pixel centre shift no longer clashes with the arc accumulators
#       - flaring test (section 1c): core_centroid + fit_flaring compare each ring centre with the central core (offset along the minor axis
#         = z sin i, none along the major axis) and give z/R of the emitting dust layer; a vertical-thickness scan of the m = 2 pattern (Doi & Kataoka 2021)
#       - round-beam test (step 2b): the image is smoothed to circular beams of 1.0, 1.5 and 2.0 x the beam major axis, the rings are refitted and the
#         m=2 excess is re-measured; an amplitude-type modulation survives the smoothing, a width / resolution-type one is diluted (do_roundbeam)
#       - ring widths and dust trapping (step 3c, do_trapping): alpha / St limits from the deconvolved ring width against the gas scale height (Dullemond+18), St ranges, gap-spacing period ratios
#       - menu: every archive disk target (MENU_SIZE = 100_000), printed in four cells of 500 (print_target_menu(from =, to =)); catalogue designations opt-in
#       - gap-opening planet masses (step 3b, do_planets): gap depth and width from the profile, Kanagawa+15/16 relations, mass ranges for alpha = 1e-3 and 1e-4
#       - the flaring, round-beam and thickness diagnostics are now opt-in (do_flare, do_roundbeam, do_thickness = false by default)
#       - load_fits_image returns freq_hz (CTYPEn = FREQ / RESTFRQ); STELLAR_PARAMS holds approximate L*, M* for ten disks
#
# Packages go into a local project environment (./alma_env); the first run
# downloads and precompiles everything (CairoMakie takes a few minutes).

import Pkg
Pkg.activate(joinpath(pwd(), "alma_env"))

let needed = ["HTTP", "EzXML", "CSV", "DataFrames", "FITSIO", "CairoMakie"]
    installed = keys(Pkg.project().dependencies)
    to_add = filter(p -> !(p in installed), needed)
    if !isempty(to_add)
        println("Installing: ", join(to_add, ", "))
        Pkg.add(to_add)
    end
end

using HTTP, EzXML, CSV, DataFrames, FITSIO, CairoMakie
using Downloads, Tar                      # stdlib
using Statistics, LinearAlgebra, Printf   # stdlib

println("Julia ", VERSION, " | environment: ", dirname(Base.active_project()))

## Configuration and plot theme

In [ ]:
const CACHE_DIR  = "./alma_cache"
const OUTPUT_DIR = "./alma_plots"
mkpath(CACHE_DIR)
mkpath(OUTPUT_DIR)

# Step-by-step checkpoints inside the radial-profile functions.
const DEBUG = Ref(true)
# Also write every figure to OUTPUT_DIR as a PNG.
const SAVE_PLOTS = Ref(true)
# Refuse to download a product tarball larger than this (GB) unless forced.
const MAX_TAR_GB = Ref(15.0)

_dbg(args...) = DEBUG[] && println("[DEBUG] ", join(string.(args), " "))

# ALMA's TAP/datalink services intermittently return transient 502/503
# errors; the regional mirrors serve the same archive, so we retry with
# backoff and rotate mirrors.
const ALMA_MIRRORS = [
    "https://almascience.nrao.edu",
    "https://almascience.eso.org",
    "https://almascience.nao.ac.jp",
]

# Dark, publication-style theme (#111111 background, #EEEEEE text).
const BG_COLOR = "#111111"
const FG_COLOR = "#EEEEEE"
const C_FLUX   = "#FF9F43"   # azimuthal mean
const C_RING   = "#FFD43B"   # rings
const C_GAP    = "#5DADE2"   # gaps
const C_RESID  = "#4EA1FF"   # residual curve
const C_LIT    = "#FF4D6D"   # literature gap markers

function alma_dark_theme!()
    set_theme!(theme_dark())
    update_theme!(
        backgroundcolor = BG_COLOR,
        Axis = (
            backgroundcolor = BG_COLOR,
            xgridcolor = (FG_COLOR, 0.10), ygridcolor = (FG_COLOR, 0.10),
            xlabelcolor = FG_COLOR, ylabelcolor = FG_COLOR, titlecolor = FG_COLOR,
            xticklabelcolor = FG_COLOR, yticklabelcolor = FG_COLOR,
            xtickcolor = FG_COLOR, ytickcolor = FG_COLOR,
            leftspinecolor = FG_COLOR, rightspinecolor = FG_COLOR,
            topspinecolor = FG_COLOR, bottomspinecolor = FG_COLOR,
        ),
        Legend = (backgroundcolor = BG_COLOR, framecolor = (FG_COLOR, 0.4), labelcolor = FG_COLOR),
        Colorbar = (labelcolor = FG_COLOR, ticklabelcolor = FG_COLOR, tickcolor = FG_COLOR),
    )
end
alma_dark_theme!()

function _save_fig(fig, name)
    SAVE_PLOTS[] || return nothing
    path = joinpath(OUTPUT_DIR, name)
    save(path, fig; px_per_unit = 2)
    println("Saved figure: ", path)
    return path
end

## Target catalogue and menu

In [ ]:
# ---------------------------------------------------------------------------
# Target catalogue: 100 widely studied ALMA disks, listed alphabetically
# ---------------------------------------------------------------------------
# IMPORTANT: distances, inclinations, position angles, spectral-type groupings
# and literature gap radii are approximate values entered from memory (DSHARP,
# MAPS, ALMA Partnership 2015, Gaia). Distances/inclinations/PAs only drive the
# au scaling and deprojection; VERIFY them before quoting results and override
# per run with `incl_deg=`, `pa_deg=`, `distance_pc=`. `?` in the menu =
# unknown, in which case the disk is treated as face-on. Literature gap radii
# (`lit`) exist only for TW Hya, HL Tau and HD 163296.
#
# Groups (how well the TW Hya-style azimuthally averaged analysis applies):
#   :analogue  T Tauri stars (K-M), mostly axisymmetric rings and gaps
#   :cavity    T Tauri stars with large inner cavities (transition disks)
#   :young     younger / embedded (envelope remnants)
#   :herbig    Herbig (A/B/F) stars: more luminous, more massive
#   :asym      strongly non-axisymmetric dust (horseshoes, arcs, shadows)
#   :edge      edge-on disks (cannot be deprojected; radial profile is not a ring/gap plot)
#   :outburst  outbursting (FU Ori / EXor) disks
#   :multiple  multiple-star systems (binary/triple; several disks in one field)
#
# The first 25 entries (TW Hya ... HD 143006) carry coordinates, inclination and PA.
# The 75 added entries are lighter: approximate distance only; their coordinates
# are resolved with CDS Sesame when you select them (needs network), and inclination/PA
# are unknown (treated face-on) -- pass incl_deg= / pa_deg= from the literature.

_norm(s::AbstractString) = replace(lowercase(s), r"[^a-z0-9]" => "")

const GROUP_LABEL = Dict(:analogue => "T Tauri analogue", :cavity => "T Tauri + cavity",
                         :young => "Young/embedded", :herbig => "Herbig",
                         :asym => "Non-axisymmetric", :edge => "Edge-on",
                         :outburst => "Outbursting", :multiple => "Multiple system",
                         :archive => "Archive disk", :custom => "Custom")
const GROUP_DIFFERS = Dict(
    :analogue => "Closest analogue: axisymmetric rings/gaps",
    :cavity   => "Ring, not centrally peaked; gaps = cavity physics",
    :young    => "Class I/II, remnant envelope; younger; inclined",
    :herbig   => "Higher-mass star; warmer, larger disk",
    :asym     => "Azimuthal average misleading (horseshoe/arc)",
    :edge     => "Edge-on: cannot deproject; no ring/gap profile",
    :outburst => "Outbursting FU Ori/EXor: hot, bright, young",
    :multiple => "Several stars in one field: centre fit lands on one disk",
    :archive  => "Not curated: i/PA fitted from the image, d from Gaia",
    :custom   => "",
)
const GROUP_ORDER = [:analogue, :cavity, :young, :herbig, :asym, :multiple, :edge, :outburst, :archive]

# (radius_au, width_au, depth, kind, narrower_than_beam): published ring/gap widths and gap depths (Huang+18 DSHARP Table 1).
# width = outer minus inner edge (beam not removed); depth = I_gap / I_ring (gap rows only, NaN for rings); kind = :gap or :ring.
const LitProp = Tuple{Float64,Float64,Float64,Symbol,Bool}

Base.@kwdef struct AlmaTarget
    name::String
    slug::String = replace(name, r"[^A-Za-z0-9]+" => "_")
    keys::Vector{String} = String[]                # normalised name aliases for target_name matching
    ra_deg::Float64 = NaN                          # NaN -> resolved by Sesame on selection
    dec_deg::Float64 = NaN
    distance_pc::Float64 = 100.0
    distance_known::Bool = true
    incl_deg::Union{Nothing,Float64} = nothing
    pa_deg::Union{Nothing,Float64} = nothing       # east of north, major axis
    rmax_arcsec::Float64 = 2.0                     # outer radius of the radial profile
    proposal_id::Union{Nothing,String} = nothing
    lit::Vector{Tuple{Float64,Symbol}} = Tuple{Float64,Symbol}[]   # (radius_au, kind)
    lit_distance_pc::Union{Nothing,Float64} = nothing
    lit_props::Vector{LitProp} = LitProp[]          # published widths and depths (see LitProp)
    note::String = ""
    group::Symbol = :custom
    differs::String = ""                           # how it differs from TW Hya (menu column)
    simbad::String = ""                           # name(s) for Sesame, '|'-separated (default: name)
    name_match::Bool = true                       # false: the name is not an archive target_name (catalogue-only entry), so the archive rows and FITS OBJECT are not filtered by name
end

_T(name, ra, dec, d; aliases = String[], group = :custom, differs = nothing, kw...) =
    AlmaTarget(; name = name, keys = unique([_norm(name); _norm.(aliases)]),
               ra_deg = ra, dec_deg = dec, distance_pc = d, group = group,
               differs = differs === nothing ? GROUP_DIFFERS[group] : differs, kw...)

# coordinate-less entry: Sesame resolves it when selected
_N(name, d; kw...) = _T(name, NaN, NaN, d; kw...)

# natural alphabetical sort key: "HD 97048" before "HD 100546"
_sortkey(s::AbstractString) = replace(lowercase(s), r"\d+" => m -> lpad(m, 6, '0'))

const ALMA_TARGETS = [
    # ---- T Tauri analogues ------------------------------------------------
    _T("TW Hya", 165.466273, -34.704731, 60.1; group = :analogue, aliases = ["TW Hydrae"],
       differs = "Reference: nearest, face-on, ~8-10 Myr, compact",
       incl_deg = 7.0, pa_deg = 155.0, rmax_arcsec = 2.5,
       proposal_id = "2015.1.00686.S", lit_distance_pc = 54.0,
       lit = Tuple{Float64,Symbol}[(1.0, :hole), (22.0, :deep), (37.0, :deep), (6.0, :shallow), (28.0, :shallow), (44.0, :shallow)],
       note = "Closest disk; multi-ring (Andrews+16, 2015.1.00686.S)"),
    _T("AS 209", 252.313700, -14.369100, 121.0; group = :analogue,
       incl_deg = 35.0, pa_deg = 85.8, rmax_arcsec = 2.0, lit_props = LitProp[(8.69, 4.7, 0.45, :gap, false), (14.2, 8.9, NaN, :ring, false), (23.84, 3.4, 0.895, :gap, true), (27.8, 4.7, NaN, :ring, false), (35.04, 3.0, 0.97, :gap, true), (38.7, 3.4, NaN, :ring, true), (60.8, 15.5, 0.014, :gap, false), (74.15, 9.3, NaN, :ring, false), (89.9, 4.5, 0.48, :gap, false), (96.7, 8.1, NaN, :ring, false), (105.5, 14.7, 0.016, :gap, false), (120.14, 11.2, NaN, :ring, false), (137.0, 4.2, 0.9, :gap, true), (141.0, 2.8, NaN, :ring, true)], lit_distance_pc = 121.0,
       lit = Tuple{Float64,Symbol}[(8.7, :gap), (14.2, :ring), (23.8, :gap), (27.8, :ring), (35.0, :gap), (38.7, :ring),
                                   (60.8, :gap), (74.2, :ring), (89.9, :gap), (96.7, :ring), (105.5, :gap), (120.1, :ring), (137.0, :gap), (141.0, :ring)],
       note = "Multi-ring disk (DSHARP 2016.1.00484.L; radii from Huang+18 Table 1)"),
    _T("IM Lup", 239.038300, -37.935000, 158.0; lit_props = LitProp[(117.4, 15.8, 0.66, :gap, false), (133.5, 18.4, NaN, :ring, false)], lit_distance_pc = 158.0, lit = Tuple{Float64,Symbol}[(117.4, :gap), (133.5, :ring)], group = :analogue,
       incl_deg = 47.5, pa_deg = 144.5, rmax_arcsec = 2.5,
       note = "Large disk with spirals (DSHARP, MAPS)"),
    _T("GW Lup", 236.686300, -34.510000, 155.0; lit_props = LitProp[(74.3, 12.1, 0.31, :gap, false), (85.4, 11.3, NaN, :ring, false), (103.0, 4.3, 0.85, :gap, true), (108.0, 5.5, NaN, :ring, true)], lit_distance_pc = 155.0, lit = Tuple{Float64,Symbol}[(74.3, :gap), (85.4, :ring), (103.0, :gap), (108.0, :ring)], group = :analogue,
       incl_deg = 38.7, pa_deg = 37.7, rmax_arcsec = 1.5,
       note = "Compact ring + gap (DSHARP)"),
    _T("RU Lup", 239.176300, -37.821000, 159.0; lit_props = LitProp[(29.1, 4.5, 0.784, :gap, false), (34.0, 5.5, NaN, :ring, false)], lit_distance_pc = 159.0, lit = Tuple{Float64,Symbol}[(21.0, :gap), (24.0, :ring), (29.1, :gap), (34.0, :ring)], group = :analogue,
       incl_deg = 18.8, pa_deg = 121.0, rmax_arcsec = 1.5,
       note = "Multi-ring disk (DSHARP)"),
    _T("DM Tau", 68.453100, 18.169400, 145.0; group = :analogue,
       incl_deg = 35.0, pa_deg = 157.0, rmax_arcsec = 2.0,
       note = "Disk with inner cavity (approximate parameters)"),
    _T("V4046 Sgr", 273.543700, -32.792900, 72.0; group = :analogue, aliases = ["V4046 Sagittarii"],
       differs = "Binary star; circumbinary disk",
       incl_deg = 34.0, pa_deg = 76.0, rmax_arcsec = 1.5,
       note = "Circumbinary disk"),
    _T("Elias 24", 246.600300, -24.270400, 136.0; lit_props = LitProp[(56.8, 22.8, 0.03, :gap, false), (76.71, 12.2, NaN, :ring, false)], lit_distance_pc = 136.0, lit = Tuple{Float64,Symbol}[(56.8, :gap), (76.71, :ring), (89.0, :gap), (123.0, :ring)], group = :analogue, incl_deg = 29.0, pa_deg = 45.7, rmax_arcsec = 1.8,
       note = "Wide gap (DSHARP 2016.1.00484.L)"),
    _T("Elias 27", 246.687600, -24.385500, 116.0; lit_props = LitProp[(69.1, 14.3, 0.73, :gap, false), (86.0, 21.2, NaN, :ring, false)], lit_distance_pc = 116.0, lit = Tuple{Float64,Symbol}[(69.1, :gap), (86.0, :ring)], group = :analogue,
       differs = "Spirals; less suited to an azimuthal average",
       incl_deg = 56.2, pa_deg = 118.8, rmax_arcsec = 3.0,
       note = "Grand-design spirals (DSHARP)"),
    _T("DoAr 25", 246.598710, -24.720520, 138.0; lit_props = LitProp[(98.0, 15.5, 0.22, :gap, false), (111.3, 14.3, NaN, :ring, false), (125.0, 10.0, 0.63, :gap, false), (137.0, 12.8, NaN, :ring, false)], lit_distance_pc = 138.0, lit = Tuple{Float64,Symbol}[(98.0, :gap), (111.3, :ring), (125.0, :gap), (137.0, :ring)], group = :analogue,
       differs = "Highly inclined; less suited to deprojection",
       incl_deg = 67.4, pa_deg = 110.6, rmax_arcsec = 1.5, note = "Multi-ring, highly inclined (DSHARP)"),
    # ---- T Tauri with large cavities --------------------------------------
    _T("GM Aur", 73.795800, 30.366500, 159.0; group = :cavity,
       incl_deg = 53.2, pa_deg = 57.2, rmax_arcsec = 2.0,
       note = "Large cavity (MAPS)"),
    _T("LkCa 15", 69.824200, 22.351000, 157.0; group = :cavity,
       incl_deg = 50.0, pa_deg = 60.0, rmax_arcsec = 1.8,
       note = "Wide cavity"),
    _T("PDS 70", 212.042300, -41.397900, 112.0; group = :cavity,
       incl_deg = 49.7, pa_deg = 158.6, rmax_arcsec = 1.5,
       note = "Cavity with two embedded protoplanets"),
    # ---- Younger / embedded -----------------------------------------------
    _T("HL Tau", 67.910167, 18.232694, 140.0; group = :young,
       incl_deg = 46.7, pa_deg = 138.0, rmax_arcsec = 1.6, lit_distance_pc = 140.0,
       lit = Tuple{Float64,Symbol}[(13.0, :gap), (32.0, :gap), (42.0, :gap), (50.0, :gap), (63.0, :gap), (74.0, :gap)],
       note = "Iconic multi-gap disk (2014 Long Baseline SV; 140 pc adopted)"),
    # ---- Herbig stars -----------------------------------------------------
    _T("HD 163296", 269.088700, -21.956100, 101.0; group = :herbig,
       incl_deg = 46.7, pa_deg = 133.3, rmax_arcsec = 3.0, lit_props = LitProp[(10.0, 3.2, 0.93, :gap, true), (14.0, 3.6, NaN, :ring, true), (48.0, 20.2, 0.033, :gap, false), (67.04, 15.8, NaN, :ring, false), (86.4, 16.2, 0.151, :gap, false), (99.69, 12.0, NaN, :ring, false), (145.0, 13.4, 0.76, :gap, false), (155.0, 14.6, NaN, :ring, false)], lit_distance_pc = 101.0,
       lit = Tuple{Float64,Symbol}[(10.0, :gap), (14.0, :ring), (48.0, :gap), (67.04, :ring), (86.4, :gap),
                                   (99.69, :ring), (145.0, :gap), (155.0, :ring)],
       note = "Rings and gaps (DSHARP 2016.1.00484.L, radii from Huang+18; MAPS 2018.1.01055.L)"),
    _T("MWC 480", 74.692800, 29.843600, 162.0; lit_distance_pc = 161.0, lit = Tuple{Float64,Symbol}[(73.43, :gap), (97.58, :ring)], group = :herbig,
       incl_deg = 37.0, pa_deg = 148.0, rmax_arcsec = 2.0, note = "Herbig Ae disk (MAPS)"),
    _T("HD 169142", 276.124100, -29.780400, 114.0; group = :herbig,
       differs = "Herbig, but near face-on with rings: closest of its group",
       incl_deg = 13.0, pa_deg = 5.0, rmax_arcsec = 1.5, note = "Near face-on ring + gap system"),
    _T("HD 97048", 167.013800, -77.654900, 185.0; group = :herbig,
       incl_deg = 41.0, pa_deg = 3.0, rmax_arcsec = 3.0, aliases = ["CU Cha"],
       note = "Large multi-ring Herbig disk"),
    _T("HD 100546", 173.356000, -70.194800, 108.0; group = :herbig, rmax_arcsec = 2.0,
       note = "Embedded planet candidates"),
    _T("HD 135344B", 228.951900, -37.154400, 135.0; group = :herbig, rmax_arcsec = 1.5,
       aliases = ["SAO 206462"], note = "Spiral arms + cavity"),
    _T("MWC 758", 82.614700, 25.332500, 160.0; group = :herbig, rmax_arcsec = 2.0,
       note = "Cavity + spirals"),
    _T("AB Aur", 73.941000, 30.551200, 156.0; group = :herbig, rmax_arcsec = 3.0,
       incl_deg = 23.2, pa_deg = 54.0, lit_distance_pc = 140.0,
       lit = Tuple{Float64,Symbol}[(120.0, :ring)],
       note = "Dust ring ~120 au, gas spirals; strongly non-axisymmetric (Tang+17, project 2015.1.00889.S; i = 23 deg)"),
    # ---- Strongly non-axisymmetric ----------------------------------------
    _T("Oph IRS 48", 246.904900, -24.509700, 134.0; group = :asym, rmax_arcsec = 1.2,
       aliases = ["IRS 48", "WLY 2-48"], note = "Strongly asymmetric dust trap"),
    _T("HD 142527", 239.174500, -42.323100, 157.0; group = :asym, rmax_arcsec = 3.0,
       note = "Horseshoe dust trap (not axisymmetric)"),
    _T("HD 143006", 239.653800, -22.954200, 165.0; lit_props = LitProp[(6.0, 5.0, NaN, :ring, true), (22.0, 21.7, 0.04, :gap, false), (40.8, 12.2, NaN, :ring, false), (51.0, 12.8, 0.53, :gap, false), (64.9, 11.5, NaN, :ring, false)], lit_distance_pc = 165.0, lit = Tuple{Float64,Symbol}[(6.0, :ring), (22.0, :gap), (40.8, :ring), (51.0, :gap), (64.9, :ring)], group = :asym,
       differs = "Partly asymmetric (rings + shadows)", incl_deg = 18.6, pa_deg = 169.0, rmax_arcsec = 1.5,
       note = "Rings, asymmetries, shadows (DSHARP)"),
    # ---- 75 more disks (coordinates resolved by CDS Sesame at selection;
    #      distances approximate; inclination/PA unknown -> treated face-on) ----
    _N("DoAr 33", 140.0; incl_deg = 41.8, pa_deg = 81.1, group = :analogue),
    _N("Elias 20", 138.0; lit_props = LitProp[(25.07, 3.5, 0.75, :gap, false), (29.0, 5.2, NaN, :ring, false), (33.0, 2.5, 0.95, :gap, true), (36.0, 1.9, NaN, :ring, true)], lit_distance_pc = 138.0, lit = Tuple{Float64,Symbol}[(25.07, :gap), (29.0, :ring), (33.0, :gap), (36.0, :ring)], incl_deg = 49.0, pa_deg = 153.2, group = :analogue, simbad = "Elias 2-20|[EC92] 20", aliases = ["Elias 2-20"]),
    _N("SR 4", 134.0; lit_props = LitProp[(11.0, 6.3, 0.23, :gap, false), (18.0, 13.3, NaN, :ring, false)], lit_distance_pc = 134.0, lit = Tuple{Float64,Symbol}[(11.0, :gap), (18.0, :ring)], incl_deg = 22.0, pa_deg = 18.0, group = :analogue, simbad = "SR 4|EM* SR 4"),
    _N("Sz 98", 156.0; group = :analogue, differs = "Large, warped/inclined; otherwise analogue", simbad = "Sz 98|HK Lup", aliases = ["HK Lup"]),
    _N("Sz 111", 158.0; group = :analogue),
    _N("Sz 114", 162.0; lit_props = LitProp[(38.6, 4.3, 0.94, :gap, true), (45.4, 5.4, NaN, :ring, false)], lit_distance_pc = 162.0, lit = Tuple{Float64,Symbol}[(38.6, :gap), (45.4, :ring)], incl_deg = 21.3, pa_deg = 165.0, group = :analogue),
    _N("Sz 129", 160.0; lit_props = LitProp[(10.0, 17.6, NaN, :ring, false), (41.0, 4.1, 0.98, :gap, true), (46.1, 3.5, NaN, :ring, true)], lit_distance_pc = 161.0, lit = Tuple{Float64,Symbol}[(10.0, :ring), (41.0, :gap), (46.1, :ring)], incl_deg = 34.1, pa_deg = 151.0, group = :analogue),
    _N("V1094 Sco", 153.0; group = :analogue, differs = "Very large disk with wide rings"),
    _N("WaOph 6", 123.0; lit_props = LitProp[(79.0, 6.7, 0.98, :gap, false), (87.5, 3.6, NaN, :ring, true)], lit_distance_pc = 123.0, lit = Tuple{Float64,Symbol}[(79.0, :gap), (87.5, :ring)], incl_deg = 47.3, pa_deg = 174.2, group = :analogue, simbad = "WaOph 6|V* V2062 Oph"),
    _N("WSB 52", 135.0; incl_deg = 54.4, pa_deg = 138.4, group = :analogue),
    _N("AA Tau", 136.0; group = :analogue, differs = "Moderately inclined; warped inner disk",
       incl_deg = 59.1, pa_deg = 93.0, rmax_arcsec = 2.0, lit_distance_pc = 145.0,
       lit = Tuple{Float64,Symbol}[(48.7, :ring), (94.9, :ring), (142.6, :ring)],
       note = "Three rings at 0.34/0.66/0.99 arcsec, i = 59.1 deg (Loomis+17, project 2013.1.01070.S)"),
    _N("BP Tau", 128.0; group = :analogue),
    _N("CI Tau", 158.0; lit_distance_pc = 158.0, lit = Tuple{Float64,Symbol}[(13.92, :gap), (27.67, :ring), (48.36, :gap), (61.95, :ring), (99.22, :ring), (118.99, :gap), (152.8, :ring)], incl_deg = 49.99, pa_deg = 11.22, group = :analogue),
    _N("CY Tau", 128.0; group = :analogue),
    _N("DG Tau", 121.0; group = :analogue, differs = "Jet-driving, younger Class II; envelope traces"),
    _N("DH Tau", 135.0; group = :analogue, differs = "Wide companion; small disk"),
    _N("DL Tau", 159.0; lit_distance_pc = 159.0, lit = Tuple{Float64,Symbol}[(39.29, :gap), (46.44, :ring), (66.95, :gap), (78.08, :ring), (88.9, :gap), (112.27, :ring)], incl_deg = 44.95, pa_deg = 52.14, group = :analogue),
    _N("DN Tau", 128.0; lit_distance_pc = 128.0, lit = Tuple{Float64,Symbol}[(15.36, :ring), (49.29, :gap), (53.39, :ring)], incl_deg = 35.18, pa_deg = 79.19, group = :analogue),
    _N("DR Tau", 195.0; group = :analogue, differs = "Younger; strong accretion; more distant"),
    _N("DS Tau", 159.0; lit_distance_pc = 159.0, lit = Tuple{Float64,Symbol}[(32.93, :gap), (56.78, :ring)], incl_deg = 65.19, pa_deg = 159.62, group = :analogue),
    _N("FT Tau", 127.0; lit_distance_pc = 127.0, lit = Tuple{Float64,Symbol}[(24.78, :gap), (32.14, :ring)], incl_deg = 35.55, pa_deg = 121.8, group = :analogue),
    _N("GO Tau", 144.0; lit_distance_pc = 144.0, lit = Tuple{Float64,Symbol}[(58.91, :gap), (73.02, :ring), (86.99, :gap), (109.45, :ring)], incl_deg = 53.91, pa_deg = 20.89, group = :analogue),
    _N("HP Tau", 161.0; group = :analogue),
    _N("IP Tau", 130.0; lit_distance_pc = 130.0, lit = Tuple{Float64,Symbol}[(27.05, :ring)], incl_deg = 45.24, pa_deg = 173.0, group = :analogue),
    _N("IQ Tau", 131.0; lit_distance_pc = 131.0, lit = Tuple{Float64,Symbol}[(41.15, :gap), (48.22, :ring), (82.79, :ring)], incl_deg = 62.12, pa_deg = 42.38, group = :analogue),
    _N("SU Aur", 158.0; group = :analogue, differs = "G-type, ~2 Msun; envelope/tail structure"),
    _N("TWA 7", 34.0; group = :analogue, differs = "Nearer than TW Hya but faint M dwarf disk"),
    _N("PDS 66", 98.0; group = :analogue, differs = "Older (~10-20 Myr), near face-on", simbad = "PDS 66|MP Mus", aliases = ["MP Mus"]),
    _N("CS Cha", 160.0; group = :cavity),
    _N("GG Tau A", 150.0; group = :cavity, differs = "Binary; circumbinary ring", simbad = "GG Tau A|GG Tau", aliases = ["GG Tau"]),
    _N("RX J1604.3-2130A", 150.0; group = :cavity, differs = "Upper Sco; ~10 Myr, big cavity", simbad = "RX J1604.3-2130A|RX J1604.3-2130", aliases = ["RXJ1604"]),
    _N("RX J1615.3-3255", 156.0; group = :cavity, simbad = "RX J1615.3-3255|RX J1615.3-3255", aliases = ["RXJ1615"]),
    _N("RX J1852.3-3700", 146.0; group = :cavity, aliases = ["RXJ1852"]),
    _N("SR 21", 138.0; group = :cavity, simbad = "SR 21|EM* SR 21"),
    _N("SR 24S", 114.0; group = :cavity, differs = "Ring, not centrally peaked; companion SR 24N", simbad = "SR 24S|EM* SR 24", aliases = ["SR 24"]),
    _N("Sz 91", 159.0; group = :cavity),
    _N("T Cha", 107.0; group = :cavity, differs = "Transition disk; ~7 Myr; small ring"),
    _N("UX Tau A", 140.0; group = :cavity, differs = "Multiple star; cavity ~ 25 au", simbad = "UX Tau A|UX Tau", aliases = ["UX Tau"]),
    _N("WSB 60", 135.0; group = :cavity),
    _N("CQ Tau", 162.0; group = :herbig, differs = "Herbig F star; ring and cavity"),
    _N("HD 100453", 104.0; group = :herbig, differs = "Herbig; spiral arms, companion"),
    _N("HD 104237", 107.0; group = :herbig, differs = "Herbig; compact disk"),
    _N("HD 139614", 135.0; group = :herbig, differs = "Herbig; ring and gap"),
    _N("HD 141569", 111.0; group = :herbig, differs = "Herbig; gas-poor, nearly debris-like rings"),
    _N("HD 142666", 148.0; lit_props = LitProp[(6.0, 5.3, NaN, :ring, false), (16.0, 3.5, 0.73, :gap, false), (20.0, 7.8, NaN, :ring, false), (55.0, 2.2, 0.96, :gap, true), (58.0, 2.1, NaN, :ring, true)], lit_distance_pc = 148.0, lit = Tuple{Float64,Symbol}[(6.0, :ring), (16.0, :gap), (20.0, :ring), (37.0, :gap), (40.0, :ring), (55.0, :gap), (58.0, :ring)], incl_deg = 62.22, pa_deg = 162.11, group = :herbig),
    _N("HD 145718", 152.0; group = :herbig),
    _N("HD 34282", 309.0; group = :herbig, differs = "Herbig; more distant, large cavity"),
    _N("V1247 Ori", 400.0; group = :herbig, differs = "Herbig; distant, ring + spiral",
       note = "F0V; Kraus+17: ring i = 30 deg, PA of the minor axis 25.4 deg, d = 320 pc (Gaia DR1); L*, M* not found. Verify"),
    _N("V892 Tau", 134.0; group = :herbig, differs = "Herbig Ae binary; circumbinary disk", simbad = "V892 Tau|Elias 1", aliases = ["Elias 1"]),
    _N("Elias 29", 137.0; group = :young, simbad = "Elias 29|[EC92] 29", aliases = ["Elias 2-29"]),
    _N("GY 91", 140.0; group = :young, differs = "Class I; embedded (eDisk)", simbad = "GY 91|[GY92] 91"),
    _N("IRAS 04302+2247", 160.0; group = :young, differs = "Edge-ish Class I; 'butterfly' nebula", aliases = ["IRAS04302"]),
    _N("IRAS 16253-2429", 140.0; group = :young, differs = "Class 0; embedded (eDisk)"),
    _N("IRS 63", 144.0; group = :young, differs = "Class I; embedded, rings (eDisk)", simbad = "IRS 63|Oph IRS 63|[GY92] 197", aliases = ["Oph IRS 63"]),
    _N("L1489 IRS", 140.0; group = :young, simbad = "IRAS 04016+2610|L1489 IRS", aliases = ["IRAS 04016+2610"]),
    _N("L1527 IRS", 140.0; group = :young, differs = "Class 0/I; near edge-on, embedded", simbad = "IRAS 04368+2557|L1527 IRS", aliases = ["IRAS 04368+2557", "L1527"]),
    _N("L1551 IRS 5", 140.0; group = :young, differs = "Class I binary; circumbinary disk"),
    _N("TMC1A", 140.0; group = :young, differs = "Class I; embedded", simbad = "IRAS 04365+2535|TMC1A", aliases = ["IRAS 04365+2535"]),
    _N("WL 17", 137.0; group = :young, differs = "Class I; ring + cavity, embedded", simbad = "WL 17|[WL78] 17"),
    _N("Flying Saucer", 138.0; group = :edge, simbad = "2MASS J16281370-2431391", aliases = ["2MASS J16281370-2431391"]),
    _N("HH 30", 140.0; group = :edge, differs = "Edge-on; jet-driving"),
    _N("Oph 163131", 147.0; group = :edge, differs = "Edge-on; large, gas-rich", simbad = "2MASS J16313124-2426281", aliases = ["2MASS J16313124-2426281"]),
    _N("Sz 133", 153.0; group = :edge),
    _N("EX Lup", 155.0; group = :outburst, differs = "EXor prototype; accretion-burst disk"),
    _N("FU Ori", 408.0; group = :outburst, differs = "FU Ori prototype; distant, bright"),
    _N("V883 Ori", 388.0; group = :outburst, differs = "FU Ori-type; water snowline resolved"),
    _N("V960 Mon", 1100.0; group = :outburst, differs = "FU Ori-type; very distant, fragmenting"),
    # ---- multiple-star systems (several stars/disks in one field) ----
    _N("AS 205", 128.0; group = :multiple, differs = "Close binary (N/S), separate disks; spirals", simbad = "AS 205", aliases = ["AS 205 A", "AS 205 B", "AS 205 N", "AS 205 S"]),
    _N("GW Ori", 402.0; group = :multiple, differs = "Triple star; broken, misaligned rings at ~46/188/338 au; 400 pc", simbad = "GW Ori", aliases = ["GW Orionis", "GW Orionis A", "GW Orionis B", "GW Orionis C"], rmax_arcsec = 1.5,
       note = "Circumtriple disk (Bi+20; Kraus+20); inclination ~21-38 deg by ring"),
    _N("HK Tau", 133.0; group = :multiple, differs = "Binary; A and B disks misaligned, B edge-on", simbad = "HK Tau", aliases = ["HK Tauri", "HK Tauri A", "HK Tauri B", "HK Tau A", "HK Tau B"]),
    _N("IRS 43", 137.3; group = :multiple, differs = "Embedded Class I binary (YLW 15); compact disks, younger", simbad = "YLW 15|IRS 43|Oph IRS 43", aliases = ["Oph IRS 43", "IRS 43 A", "IRS 43 B", "YLW 15"],
       note = "eDisk X; VLA1 compact disk (~6 au), VLA2 smaller"),
    _N("T Tau", 144.0; group = :multiple, differs = "Triple system; disk not isolated", simbad = "T Tau"),
    _N("UZ Tau", 131.0; lit_distance_pc = 131.0, lit = Tuple{Float64,Symbol}[(11.02, :ring), (17.42, :ring), (69.05, :gap), (77.04, :ring)], incl_deg = 56.15, pa_deg = 90.39, group = :multiple, differs = "Multiple star; circumbinary disk", simbad = "UZ Tau"),
    _N("XZ Tau", 140.0; group = :multiple, differs = "Binary with interacting disks", simbad = "XZ Tau"),
    _N("RY Tau", 138.0; lit_distance_pc = 128.0, lit = Tuple{Float64,Symbol}[(18.19, :ring), (43.41, :gap), (49.04, :ring)], incl_deg = 65.0, pa_deg = 23.06, group = :analogue, differs = "Intermediate-mass (~2 Msun), jet source; bright, cavity", simbad = "RY Tau"),
]
sort!(ALMA_TARGETS; by = t -> _sortkey(t.name))

# the menu lists this vector; use_extended_catalogue!(n) swaps in the archive-based list
const ACTIVE_TARGETS = Ref{Vector{AlmaTarget}}(ALMA_TARGETS)

_fmt(x) = x === nothing ? "?" : @sprintf("%.1f", x)

"""
    print_target_menu(targets=ACTIVE_TARGETS[]; group=nothing, by=:name, from=1, to=typemax(Int), legend=true)

Print the menu: alphabetical by default (`by = :name`), or grouped by how TW Hya-like
a target is (`by = :group`). `group = :analogue` (or :cavity, :young, :herbig, :asym,
:edge, :outburst) shows one group only. Menu numbers are always the global numbers
(alphabetical), so they do not change with `by` or `group`. `from` / `to` show only menu numbers from..to (the long menu is split into
ranges in separate cells), and `legend = false` leaves out the notes under the table.
"""
function print_target_menu(targets = ACTIVE_TARGETS[]; group = nothing, by::Symbol = :name,
                           from::Integer = 1, to::Integer = typemax(Int), legend::Bool = true)
    w = 124
    lo = max(Int(from), 1)
    hi = min(Int(to), length(targets))
    if lo > hi
        println("Menu numbers $lo-$(to == typemax(Int) ? "end" : string(to)): none (the menu has $(length(targets)) entries).")
        return nothing
    end
    order = collect(eachindex(targets))
    if by === :group
        gi(t) = something(findfirst(==(t.group), GROUP_ORDER), length(GROUP_ORDER) + 1)
        sort!(order; by = k -> (gi(targets[k]), k))
    end
    println("="^w)
    println("ALMA target menu ($(length(targets)) disks; showing numbers $lo-$hi), ",
            by === :group ? "grouped by how TW Hya-like they are" : "alphabetical")
    println("="^w)
    @printf("%3s  %-24s %-17s %6s %6s %7s  %s\n", "#", "Target", "Group", "d[pc]", "i[deg]", "PA[deg]",
            "How it differs from TW Hya")
    println("-"^w)
    prev = nothing
    for k in order
        t = targets[k]
        (k < lo || k > hi) && continue
        group !== nothing && t.group != group && continue
        by === :group && t.group != prev && (prev !== nothing && println("-"^w); prev = t.group)
        @printf("%3d  %-24s %-17s %6s %6s %7s  %s\n", k, t.name, get(GROUP_LABEL, t.group, string(t.group)),
                _fmt(t.distance_known ? t.distance_pc : nothing), _fmt(t.incl_deg), _fmt(t.pa_deg), t.differs)
    end
    println("-"^w)
    legend || return nothing
    println("Groups: T Tauri analogue = K-M stars, mostly axisymmetric | T Tauri + cavity = transition disks | Young/embedded = envelope")
    println("remnants | Herbig = A/B/F stars | Non-axisymmetric = horseshoes/arcs | Multiple = several stars in the field | Edge-on = cannot deproject | Outbursting = FU Ori/EXor.")
    println("d / i / PA are approximate literature values -- verify ('?' = unknown, treated face-on). Targets with '?' for both and no")
    println("coordinates in the catalogue are resolved by CDS Sesame when selected. print_target_menu(by = :group) groups the list.")
    println("Archive disk = picked from the ALMA archive, not curated: d is looked up (Bailer-Jones+21 Gaia distance, SIMBAD parallax as fallback) and i/PA are fitted from the image when you run it.")
    println("Use custom_target(\"name\"; distance_pc=...) for anything not listed. archive_url(target) gives the ALMA portal link.")
    return nothing
end

"""
    resolve_name(name) -> (ra_deg, dec_deg)

Resolve an object name to ICRS coordinates with CDS Sesame (SIMBAD/NED/VizieR).
"""
function resolve_name(name::AbstractString)
    url = "https://cds.unistra.fr/cgi-bin/nph-sesame/-oI/A?" * HTTP.URIs.escapeuri(name)
    resp = HTTP.get(url; status_exception = false, read_idle_timeout = 30, retry = false)
    text = String(resp.body)
    m = match(r"^%J\s+([-+]?\d+\.?\d*)\s+([-+]?\d+\.?\d*)"m, text)
    m === nothing && error("Could not resolve \"$name\" via Sesame (HTTP $(resp.status)). " *
                           "Pass ra_deg= and dec_deg= explicitly.")
    return parse(Float64, m[1]), parse(Float64, m[2])
end

"""
    custom_target(name; ra_deg=nothing, dec_deg=nothing, distance_pc=nothing, incl_deg=nothing,
                  pa_deg=nothing, rmax_arcsec=2.0, proposal_id=nothing) -> AlmaTarget

Any ALMA target not in the menu. If `ra_deg`/`dec_deg` are omitted the name
is resolved with Sesame. Without `distance_pc`, radii in au assume 100 pc
(arcsec values are unaffected).
"""
function custom_target(name::AbstractString; ra_deg = nothing, dec_deg = nothing,
                       distance_pc = nothing, kwargs...)
    if ra_deg === nothing || dec_deg === nothing
        ra_deg, dec_deg = resolve_name(name)
        @printf("Resolved %s -> RA = %.5f deg, Dec = %.5f deg (CDS Sesame)\n", name, ra_deg, dec_deg)
    end
    known = distance_pc !== nothing
    return AlmaTarget(; name = String(name), keys = [_norm(name)], ra_deg = Float64(ra_deg),
                      dec_deg = Float64(dec_deg), distance_pc = known ? Float64(distance_pc) : 100.0,
                      distance_known = known, kwargs...)
end

"""
    _ensure_coords(t) -> AlmaTarget

Entries added without coordinates get RA/Dec from CDS Sesame (tries each name in
`t.simbad`, '|'-separated, then `t.name`). Entries that already have coordinates
are returned unchanged.
"""
function _ensure_coords(t::AlmaTarget)
    (isnan(t.ra_deg) || isnan(t.dec_deg)) || return t
    names = unique(String.(strip.(vcat(split(t.simbad, '|'), [t.name]))))
    filter!(!isempty, names)
    for nm in names
        try
            ra, dec = resolve_name(nm)
            @printf("Resolved %s (as \"%s\") -> RA = %.5f deg, Dec = %+.5f deg (CDS Sesame)\n", t.name, nm, ra, dec)
            vals = Any[getfield(t, f) for f in fieldnames(AlmaTarget)]
            vals[findfirst(==(:ra_deg), fieldnames(AlmaTarget))] = ra
            vals[findfirst(==(:dec_deg), fieldnames(AlmaTarget))] = dec
            return AlmaTarget(vals...)
        catch
        end
    end
    error("Could not resolve coordinates for \"$(t.name)\" (tried: $(join(names, ", "))). " *
          "Use custom_target(\"$(t.name)\"; ra_deg = ..., dec_deg = ..., distance_pc = $(t.distance_pc)).")
end

_default_index(targets) = something(findfirst(t -> t.name == "TW Hya", targets), 1)

"""
    archive_url(t; sorted=true) -> String

Link to the ALMA Science Archive web portal for this target (the portal resolves the
name itself): handy for browsing every observation, previews and proposal details.
"""
function archive_url(t::AlmaTarget; sorted::Bool = true)
    sort_part = sorted ? "observationsSortProp=sourceName&observationsSortDir=asc&" : ""
    return "https://almascience.nrao.edu/aq/?" * sort_part * "sourceNameResolver=" *
           HTTP.URIs.escapeuri(t.name) * "&skyViewHeight=48.99&skyViewWidth=50.95"
end

"""
    select_target(choice=nothing) -> AlmaTarget

`choice` may be a menu number, a name (any spelling: "hd163296", "HD 163296"),
an `AlmaTarget` (e.g. from `custom_target`), or `nothing` to print the menu and
prompt for a choice (an input box appears in Jupyter/VS Code).
"""
function select_target(choice = nothing; targets = ACTIVE_TARGETS[])
    if choice === nothing
        if length(targets) > 500
            println("The menu has $(length(targets)) entries: print the part you want with print_target_menu(from = 1, to = 500) (or run the menu cells).")
        else
            print_target_menu(targets)
        end
        print("\nEnter a target number (1-$(length(targets))) or name [default $(_default_index(targets)) = $(targets[_default_index(targets)].name)]: ")
        flush(stdout)
        s = ""
        try
            s = strip(readline())
        catch
            println("(no input available; using the default)")
        end
        choice = isempty(s) ? _default_index(targets) : s
    end
    choice isa AlmaTarget && return _prepare(choice)
    if choice isa Integer
        1 <= choice <= length(targets) || error("Target number must be between 1 and $(length(targets)).")
        return _prepare(targets[choice])
    end
    s = String(choice)
    n = tryparse(Int, s)
    if n !== nothing
        1 <= n <= length(targets) || error("Target number must be between 1 and $(length(targets)).")
        return _prepare(targets[n])
    end
    key = _norm(s)
    i = findfirst(t -> key in t.keys, targets)
    i === nothing && (i = findfirst(t -> any(k -> occursin(key, k), t.keys), targets))
    i === nothing && error("No menu target matches \"$s\". Use custom_target(\"$s\"; distance_pc=...).")
    return _prepare(targets[i])
end

"""
    top_archive_targets(n=100; public_only=true, science=true, sort_by=:name, exclude=...) -> Vector{AlmaTarget}

OPTIONAL: rank target names by number of science rows (one per spectral window)
in the ALMA archive, via an ADQL GROUP BY on ivoa.obscore.
  * Name variants (HD_163296 / HD163296 / hd 163296 ...) are merged and their
    row counts summed.
  * `sort_by = :name` (default) lists the top n alphabetically; `:rows` lists by archive rows.
  * Names matching `exclude` (OFF-source pointings, the Sun, planets/moons, comets by default) are dropped.
  * `curated_only = true` keeps only targets that match a catalogue entry (the disks).
  * Names that match a curated catalogue entry (ALMA_TARGETS) return that entry, so
    distance, inclination, PA and literature features are kept. Others get a
    placeholder 100 pc -- pass distance_pc to run_alma_pipeline for those.
"""
function top_archive_targets(n::Integer = 100; public_only::Bool = true, science::Bool = true,
                             sort_by::Symbol = :name, curated_only::Bool = false,
                             exclude::Regex = r"(_OFF_?\d*$|^Sun(_\d+)?$|^(Mars|Venus|Mercury|Jupiter|Saturn|Uranus|Neptune|Pluto|Moon|Io|Europa|Ganymede|Callisto|Titan|Enceladus|Ceres|Vesta)(_\d+)?$|ATLAS|PANSTARRS|^\d+P|(^|_)C\d{4}_[A-Z]\d)"i)
    conds = String[]
    science && push!(conds, "science_observation = 'T'")
    public_only && push!(conds, "data_rights = 'Public'")
    where_clause = isempty(conds) ? "" : "WHERE " * join(conds, " AND ") * " "
    fetch_n = curated_only ? 5000 : max(10n, 300)      # over-fetch: merging and filtering shrink the list
    adql = "SELECT TOP $fetch_n target_name, COUNT(*) AS n_rows, AVG(s_ra) AS ra_deg, AVG(s_dec) AS dec_deg " *
           "FROM ivoa.obscore $(where_clause)GROUP BY target_name ORDER BY n_rows DESC"
    df = tap_query(adql)

    # merge name variants on the normalised key
    merged = Dict{String,NamedTuple}()
    order = String[]
    for row in eachrow(df)
        name = String(row.target_name)
        occursin(exclude, name) && continue
        cnt = tryparse(Int, string(row.n_rows)); cnt === nothing && continue
        ra, dec = tryparse(Float64, string(row.ra_deg)), tryparse(Float64, string(row.dec_deg))
        (ra === nothing || dec === nothing) && continue
        key = _norm(name)
        if haskey(merged, key)
            m = merged[key]
            merged[key] = (name = m.name, n = m.n + cnt, ra = m.ra, dec = m.dec)   # keep biggest variant's name/coords
        else
            merged[key] = (name = name, n = cnt, ra = ra, dec = dec)
            push!(order, key)
        end
    end
    curated_only && (order = filter(k -> any(t -> k in t.keys, ALMA_TARGETS), order))
    ranked = sort(order; by = k -> -merged[k].n)[1:min(n, length(order))]   # the n most-observed
    # display order: :name = alphabetical (easy to scan), :rows = most-observed first
    sort_by === :name && sort!(ranked; by = k -> lowercase(merged[k].name))

    targets = AlmaTarget[]
    println("="^80)
    println("Top $n most-observed science targets in the ALMA archive, sorted by $(sort_by === :name ? "name" : "rows") (rows = spectral-window entries)")
    println("  name variants merged; OFF positions, Sun, planets and comets excluded; * = curated catalogue entry")
    println("="^80)
    for key in ranked
        m = merged[key]
        j = findfirst(t -> key in t.keys, ALMA_TARGETS)
        if j !== nothing
            push!(targets, ALMA_TARGETS[j])
            @printf("%3d  %-28s rows = %-5d * %s, d = %.0f pc\n", length(targets), m.name, m.n,
                    ALMA_TARGETS[j].name, ALMA_TARGETS[j].distance_pc)
        else
            push!(targets, custom_target(m.name; ra_deg = m.ra, dec_deg = m.dec,
                                         note = "archive rows: $(m.n)"))
            @printf("%3d  %-28s rows = %d\n", length(targets), m.name, m.n)
        end
    end
    return targets
end


## ALMA archive access (TAP query, datalink, mirror retry)

In [ ]:
# ---------------------------------------------------------------------------
# HTTP helpers: mirror rotation + retry/backoff, TAP queries, datalink
# ---------------------------------------------------------------------------

const TRANSIENT_STATUS = (502, 503, 504)

"""
    robust_get(path; query, max_attempts=3, base_delay=5) -> (response, mirror)

GET `<mirror><path>` with retry/backoff on transient errors (502/503/504,
network exceptions), rotating through `ALMA_MIRRORS`. Non-transient
responses (including 4xx) are returned to the caller to interpret.
"""
function robust_get(path::AbstractString; query = Dict{String,String}(),
                    max_attempts::Int = 3, base_delay::Real = 5)
    last_err = nothing
    for mirror in ALMA_MIRRORS
        for attempt in 1:max_attempts
            println("  [mirror=$mirror, attempt $attempt/$max_attempts] GET $path ...")
            try
                resp = HTTP.get(mirror * path; query = query, status_exception = false,
                                retry = false, read_idle_timeout = 180, connect_timeout = 30)
                if resp.status in TRANSIENT_STATUS
                    last_err = "HTTP $(resp.status)"
                    if attempt < max_attempts
                        delay = base_delay * attempt
                        println("  Transient error ($(last_err)); retrying in $(delay)s ...")
                        sleep(delay)
                        continue
                    end
                    break
                end
                return resp, mirror
            catch err
                last_err = sprint(showerror, err)
                if attempt < max_attempts
                    delay = base_delay * attempt
                    println("  Transient error ($(typeof(err))); retrying in $(delay)s ...")
                    sleep(delay)
                end
            end
        end
        println("  Mirror $mirror did not succeed, trying next mirror ...")
    end
    error("GET $path failed across all mirrors $(ALMA_MIRRORS). Last error: $last_err")
end

# ---- minimal VOTable (TABLEDATA) reader -----------------------------------

"""
    parse_votable(text) -> DataFrame

Parse the first TABLE of a VOTable using the TABLEDATA serialization into an
all-`String` DataFrame. Raises on a server-reported error or on BINARY/BINARY2
serializations (request CSV or a TABLEDATA VOTable instead).
"""
function parse_votable(text::AbstractString)
    # Strip XML namespace declarations so plain XPath works.
    clean = replace(text, r"\s(xmlns(:\w+)?|xsi:\w+)=\"[^\"]*\"" => "")
    doc = EzXML.parsexml(clean)

    err_node = findfirst("//INFO[@name='QUERY_STATUS'][@value='ERROR']", doc)
    if err_node !== nothing
        error("Server reported a query error: ", strip(nodecontent(err_node)))
    end

    table = findfirst("//RESOURCE[@type='results']/TABLE", doc)
    table === nothing && (table = findfirst("//TABLE", doc))
    table === nothing && error("No <TABLE> found in VOTable response.")

    if findfirst("./DATA/BINARY", table) !== nothing || findfirst("./DATA/BINARY2", table) !== nothing
        error("VOTable uses BINARY serialization; only TABLEDATA/CSV are supported here.")
    end

    names = [field["name"] for field in findall("./FIELD", table)]
    rows = findall("./DATA/TABLEDATA/TR", table)
    cols = [String[] for _ in names]
    for tr in rows
        tds = findall("./TD", tr)
        for (k, col) in enumerate(cols)
            push!(col, k <= length(tds) ? String(strip(nodecontent(tds[k]))) : "")
        end
    end
    return DataFrame([Symbol(n) => c for (n, c) in zip(names, cols)])
end

"""
    tap_query(adql) -> DataFrame

Run a synchronous ADQL query against the ALMA TAP service (mirror rotation
and retry included). Requests a VOTable first (verified against the ALMA
service) and falls back to CSV. All columns are returned as `String`.
"""
function tap_query(adql::AbstractString)
    last_msg = ""
    for fmt in ("votable", "csv")
        query = Dict("REQUEST" => "doQuery", "LANG" => "ADQL",
                     "FORMAT" => fmt, "QUERY" => adql)
        resp, mirror = robust_get("/tap/sync"; query = query)
        text = String(resp.body)
        _dbg("tap_query: format =", fmt, "| HTTP status =", resp.status,
             "| bytes =", length(text), "| mirror =", mirror)
        try
            resp.status == 200 || error("HTTP $(resp.status): ", first(text, 400))
            if startswith(lstrip(text), "<")
                return parse_votable(text)
            else
                df = CSV.read(IOBuffer(text), DataFrame; types = String)
                return mapcols(c -> coalesce.(c, ""), df)
            end
        catch err
            last_msg = sprint(showerror, err)
            println("  FORMAT=$fmt failed: ", first(last_msg, 300))
        end
    end
    error("TAP query failed in both CSV and VOTable formats. Last error: $last_msg")
end

"""
    robust_get_data_info(member_ous_uid) -> DataFrame

Datalink listing of the downloadable products for one member OUS (the
equivalent of astroquery's `Alma.get_data_info`). Returns the datalink
table with empty-URL rows removed and URLs percent-decoded.

The ALMA datalink service only exposes whole `.tar` packages for this kind
of delivery, so astroquery's `expand_tarfiles=True` returns the same
listing and no separate option is needed here.
"""
function robust_get_data_info(member_ous_uid::AbstractString)
    println("  fetching data info for $member_ous_uid ...")
    resp, mirror = robust_get("/datalink/sync"; query = Dict("ID" => String(member_ous_uid)))
    resp.status == 200 || error("datalink returned HTTP $(resp.status): ", first(String(resp.body), 400))
    df = parse_votable(String(resp.body))
    _dbg("robust_get_data_info: columns =", names(df), "| rows =", nrow(df), "| mirror =", mirror)
    "access_url" in names(df) || error("datalink response has no access_url column; columns = $(names(df))")
    df = df[.!isempty.(df.access_url), :]
    df.access_url = [HTTP.URIs.unescapeuri(u) for u in df.access_url]
    return df
end

## Query → choose observation → inspect → pick product tarball

In [ ]:
# ---------------------------------------------------------------------------
# Query -> choose observation -> inspect -> pick product tarball
# ---------------------------------------------------------------------------

_col(df, name) = name in names(df) ? df[!, name] : fill("", nrow(df))

"""
    query_target(t::AlmaTarget; public_only=true, science=true, proposal_id=t.proposal_id,
                 radius_deg=0.05, name_filter=true) -> DataFrame

Query ivoa.obscore with a cone around the target position (server side), then
keep rows whose `target_name` matches one of the target's name aliases
(spaces/underscores/case ignored). `name_filter=false` keeps everything the cone
returns. `proposal_id` restricts to one project.
"""
function query_target(t::AlmaTarget; public_only::Bool = true, science::Bool = true,
                      proposal_id = t.proposal_id, radius_deg::Real = 0.05, name_filter::Bool = true)
    conds = ["INTERSECTS(CIRCLE('ICRS', $(t.ra_deg), $(t.dec_deg), $radius_deg), s_region) = 1"]
    public_only && push!(conds, "data_rights = 'Public'")
    science && push!(conds, "science_observation = 'T'")
    proposal_id === nothing || push!(conds, "proposal_id = '$(strip(proposal_id))'")

    select = "SELECT obs_id, target_name, member_ous_uid, band_list, frequency, " *
             "spatial_resolution, velocity_resolution, t_exptime, proposal_id, " *
             "obs_release_date, science_keyword FROM ivoa.obscore WHERE "
    # spatial_scale_max = the largest angular scale the observation recovers (arcsec); an archive without it gets the plain query
    select_las = replace(select, ", science_keyword FROM" => ", science_keyword, spatial_scale_max FROM")
    println("Querying ALMA archive for $(t.name) ...")
    _dbg("query_target: ADQL =", select * join(conds, " AND "))

    results = try
        try
            tap_query(select_las * join(conds, " AND "))
        catch
            tap_query(select * join(conds, " AND "))
        end
    catch err
        println("Query with public/science filters failed (", first(sprint(showerror, err), 200),
                "); retrying with position", proposal_id === nothing ? "" : " + proposal", " only ...")
        base = [conds[1]]
        proposal_id === nothing || push!(base, last(conds))
        tap_query(select * join(base, " AND "))
    end
    println("Found $(nrow(results)) rows in the cone.")

    if name_filter && nrow(results) > 0
        name_col = "target_name" in names(results) ? :target_name : :obs_id
        keep = [any(k -> occursin(k, _norm(string(v))), t.keys) for v in results[!, name_col]]
        if !any(keep)
            seen = unique(string.(results[!, name_col]))
            println("No target_name matched $(t.keys). Distinct names in the cone: ",
                    join(first(seen, 10), ", "), length(seen) > 10 ? ", ..." : "")
            println("Re-run with name_filter=false to keep all rows in the cone.")
        end
        results = results[keep, :]
        println("Filtered to $(nrow(results)) rows matching $(t.name).")
    end
    if nrow(results) == 0
        println("No rows. Try public_only=false (proprietary data shows up but can't be downloaded), " *
                "a larger radius_deg, or proposal_id=nothing.")
    end
    return results
end

"""
    summarize_observations(results) -> DataFrame

One row per member OUS (an observing block set; several spectral-window rows
collapse into one), sorted by best (finest) spatial resolution in arcsec.
"""
function summarize_observations(results::DataFrame)
    uids = String.(_col(results, "member_ous_uid"))
    res  = String.(_col(results, "spatial_resolution"))
    band = String.(_col(results, "band_list"))
    prop = String.(_col(results, "proposal_id"))
    las  = String.(_col(results, "spatial_scale_max"))
    frq  = String.(_col(results, "frequency"))
    texp = String.(_col(results, "t_exptime"))
    rel  = String.(_col(results, "obs_release_date"))
    rows = NamedTuple[]
    for u in unique(uids)
        idx = findall(==(u), uids)
        rv = Float64[x for x in (tryparse(Float64, res[i]) for i in idx) if x !== nothing]
        tv = Float64[x for x in (tryparse(Float64, texp[i]) for i in idx) if x !== nothing]
        lv = Float64[x for x in (tryparse(Float64, las[i]) for i in idx) if x !== nothing]
        fv = Float64[x for x in (tryparse(Float64, frq[i]) for i in idx) if x !== nothing]
        push!(rows, (member_ous_uid = u, proposal_id = prop[idx[1]], band = band[idx[1]],
                     best_res_arcsec = isempty(rv) ? NaN : minimum(rv),
                     t_exp_s = isempty(tv) ? NaN : maximum(tv),
                     release = first(rel[idx[1]], 10), n_rows = length(idx),
                     las_arcsec = isempty(lv) ? NaN : sort(lv)[cld(length(lv), 2)],      # median over the rows of the observation
                     freq_ghz = isempty(fv) ? NaN : (m = sort(fv)[cld(length(fv), 2)]; m > 1e6 ? m / 1e9 : m)))
    end
    df = DataFrame(rows)
    nrow(df) > 0 && sort!(df, :best_res_arcsec)        # NaN sorts last
    return df
end

"""
    select_observation(obs; band=nothing, choice=1) -> Union{DataFrameRow,Nothing}

Print the (up to 15) finest-resolution observations, optionally restricted to
a band (e.g. `band=7`), and return row number `choice` of that list (`quiet` skips the table).
"""
function select_observation(obs::DataFrame; band = nothing, choice::Integer = 1, quiet::Bool = false)
    sel = obs
    if band !== nothing
        sel = obs[[occursin(string(band), b) for b in obs.band], :]
        println("Restricted to band $band: $(nrow(sel)) of $(nrow(obs)) observations.")
    end
    if nrow(sel) == 0
        println("No observations left after filtering.")
        return nothing
    end
    if !quiet
        println("\nObservations (member OUS), finest spatial resolution first:")
        show(stdout, MIME("text/plain"), first(sel, 15); allcols = true, allrows = true, truncate = 40)
        println()
    end
    1 <= choice <= nrow(sel) || error("obs_choice must be between 1 and $(nrow(sel)).")
    row = sel[choice, :]
    @printf("\nUsing observation #%d: %s (proposal %s, band %s, best resolution %.4f\")\n",
            choice, row.member_ous_uid, row.proposal_id, row.band, row.best_res_arcsec)
    return row
end

function _to_gb(s)
    x = tryparse(Float64, s)
    return x === nothing ? missing : round(x / 1e9; digits = 3)
end

"""
    inspect_products(member_ous_uid) -> DataFrame

List the downloadable packages (tar files) for one member OUS, with sizes in GB.
"""
function inspect_products(member_ous_uid::AbstractString; verbose::Bool = true)
    println("\nFetching file listing for $member_ous_uid ...")
    data_info = robust_get_data_info(member_ous_uid)
    if "content_length" in names(data_info)
        data_info.size_GB = [_to_gb(s) for s in data_info.content_length]
    end
    cols = filter(c -> c in names(data_info), ["access_url", "content_type", "size_GB"])
    if verbose
        show(stdout, MIME("text/plain"), data_info[:, cols]; allcols = true, allrows = true, truncate = 110)
        println()
    end
    return data_info
end

"""
    pick_product_tarball(data_info) -> Union{String,Nothing}

ALMA deliveries often expose only whole `.tar` packages. Preference: (1) the
plain main product tar (pipeline images), (2) the `external_ari_l` tar
(reprocessed images). Raw ASDM tars and the auxiliary tar are skipped.
"""
function pick_product_tarball(data_info::DataFrame)
    urls = String.(data_info.access_url)

    is_main_product_tar(u) = let lu = lowercase(u)
        endswith(lu, ".tar") && !occursin("auxiliary", lu) && !occursin("asdm", lu) &&
            !occursin("external_ari_l", lu) && !occursin("readme", lu)
    end
    is_ari_l_tar(u) = let lu = lowercase(u)
        endswith(lu, ".tar") && occursin("external_ari_l", lu)
    end

    main_candidates = filter(is_main_product_tar, urls)
    isempty(main_candidates) || return first(main_candidates)

    ari_l_candidates = filter(is_ari_l_tar, urls)
    if !isempty(ari_l_candidates)
        println("No plain main-product tar found; falling back to the external_ari_l (reprocessed) tarball.")
        return first(ari_l_candidates)
    end
    println("No suitable product tarball found -- only auxiliary/raw ASDM tars are available for this UID.")
    return nothing
end

"Size in GB of `url` according to the datalink listing, or `nothing`."
function tarball_size_gb(data_info::DataFrame, url::AbstractString)
    "content_length" in names(data_info) || return nothing
    i = findfirst(==(url), String.(data_info.access_url))
    i === nothing && return nothing
    x = tryparse(Float64, String(data_info.content_length[i]))
    return x === nothing ? nothing : x / 1e9
end

# Candidate FITS ordering: continuum + pbcor first; PSFs, residuals, models,
# masks, primary-beam maps, cubes and moment maps last.
const _JUNK_TAGS = ("psf", "residual", "model", "mask", "flux", "alpha", "sigma", "rms", "cube", "mom", ".pb.", "_pb.")

# Spectral-line products: named after the line (target_co32b7, ..._13co.cube..., hcn, n2hp, cs, so, sio, ch3oh, ...) or
# any FITS image with more than one channel/Stokes plane (a cube). Neither is a continuum image.
const _LINE_PATTERN = r"(^|[._])(12co|13co|c18o|c17o|co\d{1,2}[a-z\d]*|hcn|hcop|hco\+|n2hp|n2dp|dcop|dcn|cs\d*|so\d*|sio|h2co|ch3[a-z\d]*|cube|line)([._]|$)"i
const _CUBE_CACHE = Dict{String,Bool}()
"""
    _is_cube_file(path) -> Bool

True if the first image HDU has more than one plane along any axis beyond the first two (a spectral cube or
multi-Stokes image). Reads only the header; `false` if the file cannot be read. Cached per path.
"""
function _is_cube_file(f::AbstractString)
    get!(_CUBE_CACHE, String(f)) do
        try
            h = FITS(f)
            try
                for i in 1:length(h)
                    if h[i] isa ImageHDU && ndims(h[i]) >= 2
                        sz = size(h[i])
                        return length(sz) > 2 && any(>(1), sz[3:end])
                    end
                end
                return false
            finally
                close(h)
            end
        catch
            return false
        end
    end
end
_is_line_or_cube_file(f) = occursin(_LINE_PATTERN, basename(f)) || _is_cube_file(f)

# beams wider than this (au, major-axis FWHM) cannot resolve a typical protoplanetary disk (rings are tens of au)
const BEAM_RESOLVED_LIMIT_AU = 40.0

# Calibrator images ship in the same tarballs (bandpass / phase / flux / check sources, named like
# J1427-4206 or ..._bp.spw19...). They are bright point sources, not the target.
const _CAL_PATTERN = r"(^|[._])j\d{4}[+-]\d{4}|_bp[._]|_ph[._]|_phase|_flux[._]|_check|_pol[._]|bandpass|calibrator|_cal[._]"i
_is_calibrator_file(f) = occursin(_CAL_PATTERN, basename(f))

# The FITS header OBJECT of an image (first image HDU); "" if absent or unreadable. Cached per path.
const _OBJECT_CACHE = Dict{String,String}()
function _fits_object(f::AbstractString)
    get!(_OBJECT_CACHE, String(f)) do
        try
            h = FITS(f)
            try
                for i in 1:length(h)
                    if h[i] isa ImageHDU && ndims(h[i]) >= 2
                        hd = read_header(h[i])
                        return haskey(hd, "OBJECT") ? String(strip(string(hd["OBJECT"]))) : ""
                    end
                end
                return ""
            finally
                close(h)
            end
        catch
            return ""
        end
    end
end

# true if the image's OBJECT is empty/unreadable or matches one of the target's name keys (normalised, substring either way)
function _object_matches(f::AbstractString, keys)
    obj = _norm(_fits_object(f))
    (isempty(obj) || isempty(keys)) && return true
    return any(k -> occursin(k, obj) || occursin(obj, k), keys)
end

function _fits_score(f; keys = nothing)
    lf = lowercase(basename(f))
    s = occursin("cont", lf) && occursin("pbcor", lf) ? 0 :
        occursin("cont", lf) ? 1 : occursin("pbcor", lf) ? 2 : 3
    (any(j -> occursin(j, lf), _JUNK_TAGS) || _is_line_or_cube_file(f)) && (s += 10)
    _is_calibrator_file(f) && (s += 20)
    keys === nothing || _object_matches(f, keys) || (s += 4)       # labelled as another source: ranked below any matching image
    return s
end

rank_fits_files(files; keys = nothing) = sort(files; by = f -> _fits_score(f; keys = keys), alg = MergeSort)

# ---- packages refused by the size guard (this session) ----------------------
@isdefined(TOO_LARGE) || (global TOO_LARGE = NamedTuple[])

function _log_too_large(target, row, size_gb)
    uid = String(row.member_ous_uid)
    any(r -> r.member_ous_uid == uid, TOO_LARGE) && return
    push!(TOO_LARGE, (target = target.name, member_ous_uid = uid, proposal_id = String(row.proposal_id),
                      band = String(row.band), size_GB = round(Float64(size_gb); digits = 1)))
    return
end

"""
    print_too_large()

List the packages refused by the size guard (`MAX_TAR_GB`) in this session, e.g.
`HL Tau  2011.0.00015.SV  band 6  34.3 GB`. Download one anyway with
`run_alma_pipeline(target; obs_choice = N, force_download = true)`.
"""
function print_too_large()
    if isempty(TOO_LARGE)
        println("No packages have been refused for size in this session.")
        return nothing
    end
    @printf("Packages refused by the size guard (MAX_TAR_GB = %.1f GB):\n", MAX_TAR_GB[])
    for r in sort(TOO_LARGE; by = r -> r.target)
        @printf("  %-16s %-16s band %-3s %7.1f GB  %s\n", r.target, r.proposal_id, r.band, r.size_GB, r.member_ous_uid)
    end
    return nothing
end


## Download tarball and extract FITS members

In [ ]:
# ---------------------------------------------------------------------------
# Download a product tarball and extract only the FITS members
# ---------------------------------------------------------------------------

_find_fits(dir; pattern = r"\.fits$"i) =
    isdir(dir) ? [joinpath(root, f) for (root, _, files) in walkdir(dir)
                  for f in files if occursin(pattern, f)] : String[]

"""
    download_and_extract_fits(tarball_url; dest_dir=CACHE_DIR, fits_regex, delete_tar=true) -> Vector{String}

Download a single ALMA product tarball and extract only the FITS members
from it, then delete the tarball (ALMA only exposes whole `.tar` packages,
which can be multiple GB, so we don't keep them). Skips everything if FITS
files were already extracted on a previous run.
"""
function download_and_extract_fits(tarball_url::AbstractString; dest_dir::AbstractString = CACHE_DIR,
                                   fits_regex::Regex = r"\.fits$"i, delete_tar::Bool = true)
    tarball_name = basename(tarball_url)
    tar_local = joinpath(dest_dir, tarball_name)
    extract_dir = joinpath(dest_dir, splitext(tarball_name)[1] * "_extracted")

    existing = _find_fits(extract_dir; pattern = fits_regex)
    if !isempty(existing)
        println("Already have $(length(existing)) extracted FITS file(s), skipping download.")
        return existing
    end

    if !isfile(tar_local)
        println("\nDownloading $tarball_url ...")
        part = tar_local * ".part"          # never leave a truncated .tar that looks complete
        last_pct = Ref(-10)
        # try the URL as given, then the same file on the other ALMA mirrors; two attempts each (a "500 Proxy Error"
        # is usually a transient server or network-proxy problem)
        host_of(u) = match(r"^(https?://[^/]+)", u).captures[1]
        urls = unique([String(tarball_url);
                       [replace(String(tarball_url), host_of(tarball_url) => m) for m in ALMA_MIRRORS]])
        done = false
        last_err = ""
        for url in urls, attempt in 1:2
            try
                last_pct[] = -10
                isfile(part) && rm(part; force = true)
                url == tarball_url || println("  trying mirror: ", host_of(url))
                Downloads.download(url, part; progress = (total, now) -> begin
                    total > 0 || return
                    pct = floor(Int, 100 * now / total)
                    if pct >= last_pct[] + 10
                        last_pct[] = pct
                        @printf("  %3d%%  (%.2f / %.2f GB)\n", pct, now / 1e9, total / 1e9)
                    end
                end)
                done = true
                break
            catch err
                last_err = sprint(showerror, err)
                println("  download failed (", host_of(url), ", attempt ", attempt, "/2): ", first(last_err, 160))
                isfile(part) && rm(part; force = true)
                sleep(2 * attempt)
            end
        end
        if !done
            println("Download failed on every mirror. Last error: ", first(last_err, 200))
            return String[]
        end
        mv(part, tar_local; force = true)
        println("Download complete.")
    else
        println("Found cached tarball at $tar_local.")
    end

    println("Extracting FITS members ...")
    headers = Tar.list(tar_local; strict = false)
    members = String[h.path for h in headers if h.type == :file && occursin(fits_regex, h.path)]
    if isempty(members)
        println("No FITS members found inside this tarball.")
    else
        wanted = Set(members)
        isdir(extract_dir) && rm(extract_dir; recursive = true, force = true)   # Tar.extract needs an empty target
        try
            Tar.extract(h -> h.path in wanted, tar_local, extract_dir)
        catch err
            println("Tar.extract failed: ", sprint(showerror, err))
        end
        if !all(m -> isfile(joinpath(extract_dir, m)), members)
            println("Pure-Julia extraction incomplete; falling back to system `tar` ...")
            mkpath(extract_dir)
            run(`tar -xf $tar_local -C $extract_dir $members`)
        end
    end
    extracted_paths = [joinpath(extract_dir, m) for m in members if isfile(joinpath(extract_dir, m))]

    # Only delete the (large) tarball once we actually have the FITS files.
    if delete_tar && !isempty(extracted_paths) && isfile(tar_local)
        rm(tar_local)
        println("Deleted $tarball_name after extraction to save space.")
    end
    return extracted_paths
end

# ---- Individual continuum FITS files, for packages whose whole tarball is too large ------------------------------
# A multi-target project can ship one 100+ GB tarball, yet the archive also lists each target's continuum image as a
# separate small file. These helpers find those files and download them one by one.

"Download `url` to `dest` (same file on the other ALMA mirrors if it fails; two attempts each). Returns true on success."
function _download_file(url::AbstractString, dest::AbstractString)
    host_of(u) = match(r"^(https?://[^/]+)", u).captures[1]
    urls = unique([String(url); [replace(String(url), host_of(url) => m) for m in ALMA_MIRRORS]])
    part = dest * ".part"
    mkpath(dirname(dest))
    for u in urls, attempt in 1:2
        try
            isfile(part) && rm(part; force = true)
            Downloads.download(u, part)
            mv(part, dest; force = true)
            return true
        catch err
            println("  download failed (", host_of(u), ", attempt ", attempt, "/2): ", first(sprint(showerror, err), 160))
            isfile(part) && rm(part; force = true)
            sleep(2 * attempt)
        end
    end
    return false
end

"""
    direct_fits_urls(data_info, target; max_gb) -> Vector{String}

URLs of single `.fits` files in the datalink listing whose name contains the target's name, that are not cubes,
PSFs, residuals, models, masks or primary-beam maps, and are each smaller than `max_gb`.
"""
function direct_fits_urls(data_info::DataFrame, target; max_gb::Real = 15.0)
    urls = String.(data_info.access_url)
    out = String[]
    for (i, u) in enumerate(urls)
        lu = lowercase(basename(u))
        endswith(lu, ".fits") || continue
        any(k -> occursin(k, _norm(basename(u))), target.keys) || continue
        any(tag -> occursin(tag, lu), _JUNK_TAGS) && continue
        sz = tarball_size_gb(data_info, u)
        sz !== nothing && sz > max_gb && continue
        push!(out, u)
    end
    return out
end

"Download each URL into CACHE_DIR/direct_fits (skipping files already there). Returns the local paths that succeeded."
function download_direct_fits(urls::Vector{String}; dest_dir::AbstractString = joinpath(CACHE_DIR, "direct_fits"))
    paths = String[]
    for u in urls
        dest = joinpath(dest_dir, basename(u))
        if isfile(dest) && filesize(dest) > 0
            println("Already have ", basename(u))
            push!(paths, dest)
        else
            println("Downloading ", basename(u), " ...")
            _download_file(u, dest) && push!(paths, dest)
        end
    end
    return paths
end


## FITS loading and image display

In [ ]:
# ---------------------------------------------------------------------------
# FITS loading + 2D image display
# ---------------------------------------------------------------------------

_hget(h, key, default = nothing) = haskey(h, key) ? h[key] : default

"""
    _fits_freq_hz(header) -> Union{Float64,Nothing}

Observing frequency [Hz]: the reference value of a FREQ axis (CTYPEn = FREQ*), else RESTFRQ / RESTFREQ / REFFREQ.
"""
function _fits_freq_hz(h)
    for k in 1:5
        ct = uppercase(String(strip(string(_hget(h, "CTYPE$(k)", "")))))
        if startswith(ct, "FREQ")
            v = _hget(h, "CRVAL$(k)", nothing)
            v isa Real && v > 0 && return Float64(v)
        end
    end
    for key in ("RESTFRQ", "RESTFREQ", "REFFREQ")
        v = _hget(h, key, nothing)
        v isa Real && v > 0 && return Float64(v)
    end
    return nothing
end

"""
    pixel_scales_arcsec(header) -> (cdelt1_arcsec, cdelt2_arcsec)

Signed pixel scales along the two sky axes, from CDELT1/2 or the CD matrix.
RA normally increases to the left, so `cdelt1` is negative.
"""
function pixel_scales_arcsec(h)
    if haskey(h, "CDELT1") && haskey(h, "CDELT2")
        return 3600.0 * h["CDELT1"], 3600.0 * h["CDELT2"]
    elseif haskey(h, "CD1_1")
        cd11, cd12 = h["CD1_1"], _hget(h, "CD1_2", 0.0)
        cd21, cd22 = _hget(h, "CD2_1", 0.0), h["CD2_2"]
        s1, s2 = hypot(cd11, cd21), hypot(cd12, cd22)
        flip = (cd11 * cd22 - cd12 * cd21) < 0
        return 3600.0 * (flip ? -s1 : s1), 3600.0 * s2
    else
        error("FITS header has neither CDELT1/2 nor a CD matrix; can't determine pixel scale.")
    end
end

"""
    load_fits_image(filepath) -> NamedTuple

Read the first image HDU with >= 2 axes, collapse trailing axes
(frequency/Stokes) to their first plane (with a warning if they were longer
than 1, i.e. a cube), and return the 2D array `[x, y]` plus header values.
"""
function load_fits_image(filepath::AbstractString)
    f = FITS(filepath)
    try
        idx = findfirst(i -> f[i] isa ImageHDU && ndims(f[i]) >= 2, 1:length(f))
        idx === nothing && error("No image HDU with data found in $filepath")
        hdu = f[idx]
        raw = read(hdu)
        h = read_header(hdu)

        if ndims(raw) > 2 && any(>(1), size(raw)[3:end])
            println("WARNING: $(basename(filepath)) has image shape $(size(raw)); using only the first plane. ",
                    "This looks like a cube, not a continuum image.")
        end
        arr = raw
        while ndims(arr) > 2
            arr = selectdim(arr, ndims(arr), 1)
        end
        data = Array(arr)

        cdelt1, cdelt2 = pixel_scales_arcsec(h)
        return (
            path = String(filepath),
            data = data,
            cdelt1_arcsec = cdelt1,
            cdelt2_arcsec = cdelt2,
            pix_scale_arcsec = abs(cdelt1),
            bmaj_deg = _hget(h, "BMAJ"),
            bmin_deg = _hget(h, "BMIN"),
            bpa_deg  = _hget(h, "BPA"),
            bunit    = String(strip(string(_hget(h, "BUNIT", "Jy/beam")))),
            object   = String(strip(string(_hget(h, "OBJECT", "")))),
            freq_hz  = _fits_freq_hz(h),
        )
    finally
        close(f)
    end
end

"""
    display_fits_image(img; label="", center=nothing, cutout_arcsec=6.0, vmax_percentile=99.5,
                       asinh_a=0.01, filename="continuum_image.png") -> Figure

Display a 2D image with sky-offset axes (east to the left).

cutout_arcsec: square cutout size around `center` (1-based `(x, y)` pixel;
    default the image centre).
vmax_percentile: central percentile interval for the colour limits (astropy
    PercentileInterval semantics; 99.5 clips 0.25% at each end).
asinh_a: asinh stretch softening, y = asinh(x/a)/asinh(1/a) on the normalised
    image. Smaller values compress the bright core harder.
"""
function display_fits_image(img; label = "", center = nothing, cutout_arcsec = 6.0,
                            vmax_percentile = 99.5, asinh_a = 0.01, filename = "continuum_image.png")
    data = img.data
    nx, ny = size(data)
    cx, cy = center === nothing ? (nx ÷ 2 + 1, ny ÷ 2 + 1) : round.(Int, center)

    ilo, ihi, jlo, jhi = 1, nx, 1, ny
    if cutout_arcsec !== nothing
        size_pix = max(round(Int, cutout_arcsec / img.pix_scale_arcsec), 10)
        half = size_pix ÷ 2
        ilo = max(cx - half, 1);  ihi = min(ilo + size_pix - 1, nx)
        jlo = max(cy - half, 1);  jhi = min(jlo + size_pix - 1, ny)
    end
    cut = Float64.(data[ilo:ihi, jlo:jhi])

    vals = filter(isfinite, vec(cut))
    tail = (100 - vmax_percentile) / 200
    vmin, vmax = quantile(vals, [tail, 1 - tail])
    stretch(x) = asinh(x / asinh_a) / asinh(1 / asinh_a)
    normed = stretch.(clamp.((cut .- vmin) ./ (vmax - vmin), 0.0, 1.0))

    xs = range((ilo - cx) * img.cdelt1_arcsec, step = img.cdelt1_arcsec, length = size(cut, 1))
    ys = range((jlo - cy) * img.cdelt2_arcsec, step = img.cdelt2_arcsec, length = size(cut, 2))
    if img.cdelt1_arcsec < 0
        xs = reverse(xs);  normed = reverse(normed; dims = 1)
    end
    if img.cdelt2_arcsec < 0
        ys = reverse(ys);  normed = reverse(normed; dims = 2)
    end

    fig = Figure(size = (760, 700))
    ax = Axis(fig[1, 1]; aspect = DataAspect(), xreversed = true,
              xlabel = "ΔRA [arcsec]", ylabel = "ΔDec [arcsec]",
              title = "$(label) — $(basename(img.path))", titlesize = 12)
    hm = heatmap!(ax, xs, ys, normed; colormap = :inferno, colorrange = (0, 1))

    tick_vals = collect(range(vmin, vmax; length = 6))
    tick_pos = stretch.((tick_vals .- vmin) ./ (vmax - vmin))
    Colorbar(fig[1, 2], hm; label = "Flux density [$(img.bunit)]",
             ticks = (tick_pos, [@sprintf("%.1e", v) for v in tick_vals]))

    display(fig)
    _save_fig(fig, filename)
    return fig
end

## Disk geometry, centre fitting and radial profile

In [ ]:
# ---------------------------------------------------------------------------
# Disk geometry, centre fitting, deprojected radial profile
# ---------------------------------------------------------------------------

"""
Geometry tuple for `_dep_radius`. `pa_deg` is the position angle of the disk
major axis, east of north; `incl_deg` the inclination (0 = face-on).
"""
function _geometry(img, incl_deg, pa_deg)
    sin_pa, cos_pa = sincos(deg2rad(pa_deg))
    return (e = img.cdelt1_arcsec, n = img.cdelt2_arcsec,
            sin_pa = sin_pa, cos_pa = cos_pa, cos_i = cos(deg2rad(incl_deg)))
end

# Deprojected (face-on) radius in arcsec of pixel (i, j) about centre (cx, cy).
# Sky offsets: east = (i - cx) * cdelt1, north = (j - cy) * cdelt2. The major
# axis is at PA east of north, i.e. along (sin PA, cos PA) in (east, north); the
# minor-axis offset is foreshortened by cos(i) and is stretched back here.
@inline function _dep_radius(i, j, cx, cy, g)
    E = (i - cx) * g.e
    N = (j - cy) * g.n
    xmaj = E * g.sin_pa + N * g.cos_pa
    xmin = E * g.cos_pa - N * g.sin_pa
    return sqrt(xmaj * xmaj + (xmin / g.cos_i)^2)
end

"""
    _emission_centroid(img; half_width_arcsec, radius_arcsec, nsigma=5.0) -> (cx, cy, ok)

Starting guess for the disk centre: flux-weighted centroid of significant emission
(> nsigma x robust noise) in a block-averaged copy of the central
`half_width_arcsec`, restricted to `radius_arcsec` around the brightest cell.
Needed because ALMA images are phase-centred on the pointing, which can be ~1" from
the star (IM Lup is), far outside the symmetry fit's small search window.
`ok = false` means no significant emission was found (use the image centre).
"""
function _emission_centroid(img; half_width_arcsec::Real, radius_arcsec::Real, nsigma::Real = 5.0)
    data = img.data
    nx, ny = size(data)
    pix = img.pix_scale_arcsec
    cx0, cy0 = nx ÷ 2 + 1, ny ÷ 2 + 1
    bm = img.bmaj_deg === nothing ? 10 * pix : img.bmaj_deg * 3600.0
    f = max(1, round(Int, bm / 3 / pix))                     # ~3 cells per beam
    hw = min(floor(Int, half_width_arcsec / pix), min(nx, ny) ÷ 2 - f - 1)
    hw < 4f && return (cx0, cy0, false)
    i0, j0 = cx0 - hw, cy0 - hw
    nbk = (2hw) ÷ f
    blk = fill(NaN, nbk, nbk)
    @inbounds for bj in 1:nbk, bi in 1:nbk
        s = 0.0; c = 0
        for j in (j0 + (bj - 1) * f):(j0 + bj * f - 1), i in (i0 + (bi - 1) * f):(i0 + bi * f - 1)
            v = data[i, j]
            isnan(v) && continue
            s += v; c += 1
        end
        c >= max(1, (f * f) ÷ 2) && (blk[bi, bj] = s / c)
    end
    vals = filter(!isnan, vec(blk))
    length(vals) < 100 && return (cx0, cy0, false)
    med = median(vals)
    sig = 1.4826 * median(abs.(vals .- med))                 # robust (MAD) noise
    (isfinite(sig) && sig > 0) || return (cx0, cy0, false)
    _, ipk = findmax(v -> isnan(v) ? -Inf : v, blk)
    bi0, bj0 = Tuple(ipk)
    cell_i(bi) = i0 + (bi - 1) * f + (f - 1) / 2
    cell_j(bj) = j0 + (bj - 1) * f + (f - 1) / 2
    thr = med + nsigma * sig
    sw = 0.0; sx = 0.0; sy = 0.0; nsig = 0
    for bj in 1:nbk, bi in 1:nbk
        v = blk[bi, bj]
        (isnan(v) || v <= thr) && continue
        ci, cj = cell_i(bi), cell_j(bj)
        hypot((ci - cell_i(bi0)) * pix, (cj - cell_j(bj0)) * pix) <= radius_arcsec || continue
        w = v - thr
        sw += w; sx += w * ci; sy += w * cj; nsig += 1
    end
    nsig >= 3 || return (cx0, cy0, false)
    return (round(Int, sx / sw), round(Int, sy / sw), true)
end

"""
    fit_center(img, g; start, search_arcsec=0.15, fit_radius_arcsec=1.0, nb=50, max_walk=8) -> (cx, cy)

Refine the disk centre (1-based pixel) by minimising the total within-annulus
sum of squares of the image inside `fit_radius_arcsec` -- the centre that makes
the disk most axisymmetric. Coarse grid search (re-centred and repeated, up to `max_walk`
times, while the best point sits at the edge of the search window), then a 1-pixel refinement.
Works for ring/cavity disks where the brightest pixel is NOT the star.
"""
function fit_center(img, g; start, search_arcsec = 0.15, fit_radius_arcsec = 1.0, nb::Int = 50, max_walk::Int = 8)
    data = img.data
    nx, ny = size(data)
    pix = img.pix_scale_arcsec
    rpix = ceil(Int, fit_radius_arcsec / pix)

    function objective(cx, cy)
        ilo = max(1, cx - rpix); ihi = min(nx, cx + rpix)
        jlo = max(1, cy - rpix); jhi = min(ny, cy + rpix)
        sums = zeros(Float64, nb); sumsq = zeros(Float64, nb); cnt = zeros(Int, nb)
        @inbounds for j in jlo:jhi, i in ilo:ihi
            v = data[i, j]
            isnan(v) && continue
            r = _dep_radius(i, j, cx, cy, g)
            r < fit_radius_arcsec || continue
            b = floor(Int, r / fit_radius_arcsec * nb) + 1
            x = Float64(v)
            sums[b] += x; sumsq[b] += x * x; cnt[b] += 1
        end
        ss = 0.0
        for b in 1:nb
            cnt[b] > 1 && (ss += sumsq[b] - sums[b]^2 / cnt[b])
        end
        return ss
    end

    function search(c0, R, step)
        best = (Inf, c0[1], c0[2])
        for cx in (c0[1] - R):step:(c0[1] + R), cy in (c0[2] - R):step:(c0[2] + R)
            (1 <= cx <= nx && 1 <= cy <= ny) || continue
            o = objective(cx, cy)
            o < best[1] && (best = (o, cx, cy))
        end
        return best
    end

    R = max(1, round(Int, search_arcsec / pix))
    step = max(1, R ÷ 8)
    c = (Int(start[1]), Int(start[2]))
    walked = 0
    for it in 1:max_walk
        b1 = search(c, R, step)
        walked = it
        interior = abs(b1[2] - c[1]) <= R - 2step && abs(b1[3] - c[2]) <= R - 2step
        c = (b1[2], b1[3])
        interior && break
        it == max_walk && println("WARNING: centre fit still at the edge of its search window after $max_walk steps; ",
                                  "check the image, or pass center = (i, j).")
    end
    b2 = step > 1 ? search(c, step, 1) : (objective(c[1], c[2]), c[1], c[2])
    _dbg("fit_center: start =", start, "walk steps =", walked, "coarse =", c, "fine =", (b2[2], b2[3]))
    return (b2[2], b2[3])
end

"""
    fit_geometry(img; center, fit_radius_arcsec=1.0, nb=40, start=nothing) -> (score, incl_deg, pa_deg)

Estimate the disk inclination and position angle (east of north, 0-180) from the image
alone, for disks whose geometry is not in the catalogue. For each trial geometry it
computes the fraction of the pixel variance inside `fit_radius_arcsec` that a purely radial
(azimuthally symmetric) profile does NOT explain; the true geometry minimises it. Coarse
grid over (i, PA), then a shrinking pattern search. With `start = (incl, pa)` only the
local refinement runs. Accuracy is a few degrees (slightly low at high inclination when
the beam is large). Meaningless if the disk is barely resolved.
"""
function fit_geometry(img; center, fit_radius_arcsec::Real = 1.0, nb::Int = 40, start = nothing)
    data = img.data
    nx, ny = size(data)
    pix = img.pix_scale_arcsec
    cx, cy = center
    rpix = ceil(Int, fit_radius_arcsec / pix)
    ilo = max(1, cx - rpix); ihi = min(nx, cx + rpix)
    jlo = max(1, cy - rpix); jhi = min(ny, cy + rpix)

    function score(incl, pa)
        g = _geometry(img, incl, pa)
        sums = zeros(Float64, nb); sumsq = zeros(Float64, nb); cnt = zeros(Int, nb)
        tot = 0.0; totsq = 0.0; n = 0
        @inbounds for j in jlo:jhi, i in ilo:ihi
            v = data[i, j]
            isnan(v) && continue
            r = _dep_radius(i, j, cx, cy, g)
            r < fit_radius_arcsec || continue
            b = floor(Int, r / fit_radius_arcsec * nb) + 1
            x = Float64(v)
            sums[b] += x; sumsq[b] += x * x; cnt[b] += 1
            tot += x; totsq += x * x; n += 1
        end
        n > 10 || return Inf
        ssw = 0.0
        for b in 1:nb
            cnt[b] > 1 && (ssw += sumsq[b] - sums[b]^2 / cnt[b])
        end
        sst = totsq - tot^2 / n
        return sst > 0 ? ssw / sst : Inf
    end

    if start === nothing
        best = (score(0.0, 0.0), 0.0, 0.0)
        for incl in 10.0:10.0:80.0, pa in 0.0:15.0:165.0
            o = score(incl, pa)
            o < best[1] && (best = (o, incl, pa))
        end
    else
        best = (score(Float64(start[1]), Float64(start[2])), Float64(start[1]), Float64(start[2]))
    end
    step_i, step_p = 5.0, 7.5
    for _ in 1:3
        improved = true
        while improved
            improved = false
            for di in (-step_i, 0.0, step_i), dp in (-step_p, 0.0, step_p)
                (di == 0.0 && dp == 0.0) && continue
                i2 = clamp(best[2] + di, 0.0, 85.0)
                p2 = mod(best[3] + dp, 180.0)
                o = score(i2, p2)
                o < best[1] - 1e-9 && (best = (o, i2, p2); improved = true)
            end
        end
        step_i /= 2; step_p /= 2
    end
    sf = score(0.0, 0.0)                         # face-on score, for the fit-quality comparison
    _dbg("fit_geometry: score =", best[1], "incl =", best[2], "pa =", best[3], "face-on score =", sf)
    return (best[1], best[2], best[3], sf)
end

"""
    _disk_extent_au(profile; frac=0.9) -> Float64

Radius (au) enclosing `frac` of the positive azimuthally averaged flux (annulus area ~ r).
Only bins out to the last one that rises 3 sigma above the noise count (sigma = image noise / sqrt(beams in
the annulus); 2% of the peak when the noise is unknown). Without this cut, the tiny positive noise of the huge
outer annuli accumulates and inflates the radius of a compact disk several-fold.
"""
function _disk_extent_au(profile; frac::Real = 0.9)
    mf = profile.mean_flux
    ok = findall(!isnan, mf)
    isempty(ok) && return NaN
    pk = maximum(mf[ok])
    lastsig = 0
    for i in ok
        sg = _stat_sigma(profile, profile.radius_au[i])
        thr = isfinite(sg) ? 3 * sg : 0.02 * pk
        mf[i] >= thr && (lastsig = i)
    end
    lastsig == 0 && return NaN
    w = [(i <= lastsig && !isnan(mf[i])) ? max(mf[i], 0.0) : 0.0 for i in eachindex(mf)] .* profile.radius_au
    tot = sum(w)
    tot > 0 || return NaN
    i = findfirst(>=(frac), cumsum(w) ./ tot)
    return i === nothing ? profile.radius_au[end] : profile.radius_au[i]
end

"""
    extract_radial_profile(img; incl_deg=0.0, pa_deg=0.0, distance_pc=100.0, max_radius_arcsec=2.0,
                           n_bins=nothing, center=nothing, center_mode=:optimize, fit_geom=false) -> NamedTuple

Azimuthally averaged, deprojected radial profile.

center_mode (used when `center` is `nothing`):
  :optimize     emission-centroid start, then refine by symmetry (default; see `fit_center`)
  :image_center the image's central pixel (the pointing; can be ~1" from the star)
  :peak         the brightest pixel (only sensible for centrally peaked disks)
fit_geom: estimate inclination and PA from the image (see `fit_geometry`) instead of using
    `incl_deg`/`pa_deg`; the fitted values are returned in the result.
n_bins: number of radial annuli; `nothing` picks a bin width of half the beam
    major axis (40-400 bins). Adjacent bins are correlated at that sampling.
"""
function extract_radial_profile(img; incl_deg = 0.0, pa_deg = 0.0, distance_pc = 100.0,
                                max_radius_arcsec = 2.0, n_bins = nothing,
                                center = nothing, center_mode::Symbol = :optimize, fit_geom::Bool = false)
    data = img.data
    nx, ny = size(data)
    pix = img.pix_scale_arcsec
    _dbg("extract_radial_profile: image", img.path)
    _dbg("extract_radial_profile: data shape", size(data), "eltype", eltype(data),
         "nan count", count(isnan, data), "finite count", count(isfinite, data))

    bmaj_arcsec = bmin_arcsec = bmaj_au = bmin_au = nothing
    if img.bmaj_deg !== nothing && img.bmin_deg !== nothing
        bmaj_arcsec = img.bmaj_deg * 3600.0
        bmin_arcsec = img.bmin_deg * 3600.0
        bmaj_au = bmaj_arcsec * distance_pc
        bmin_au = bmin_arcsec * distance_pc
        @printf("Synthesized beam: %.4f\" x %.4f\" (PA=%s) = %.2f au x %.2f au at %.1f pc\n",
                bmaj_arcsec, bmin_arcsec, img.bpa_deg, bmaj_au, bmin_au, distance_pc)
    else
        println("WARNING: no BMAJ/BMIN in the FITS header -- beam size unknown.")
    end

    # Keep the profile inside the image.
    rimg = 0.5 * min(nx, ny) * pix
    if max_radius_arcsec > 0.95 * rimg
        println(@sprintf("max_radius_arcsec = %.2f exceeds the image half-width (%.2f\"); reducing to %.2f\".",
                         max_radius_arcsec, rimg, 0.95 * rimg))
        max_radius_arcsec = 0.95 * rimg
    end

    if n_bins === nothing
        binw = bmaj_arcsec === nothing ? 10 * pix : 0.5 * bmaj_arcsec
        n_bins = clamp(round(Int, max_radius_arcsec / binw), 40, 400)
    end
    n_bins < typemax(Int16) || error("n_bins too large")

    g = _geometry(img, incl_deg, pa_deg)
    c_img = (nx ÷ 2 + 1, ny ÷ 2 + 1)
    fit_r = min(1.0, 0.8 * max_radius_arcsec, 0.9 * rimg)

    # starting centre; for :optimize this is the emission centroid
    start = c_img
    if center !== nothing
        start = (round(Int, center[1]), round(Int, center[2]))
    elseif center_mode == :peak
        _, ci = findmax(v -> isnan(v) ? -Inf : v, data)
        start = Tuple(ci)
    elseif center_mode == :image_center
        start = c_img
    elseif center_mode == :optimize
        ccx, ccy, okc = _emission_centroid(img; half_width_arcsec = min(0.9 * rimg, max(2 * max_radius_arcsec, 4.0)),
                                           radius_arcsec = max_radius_arcsec)
        if okc
            start = (ccx, ccy)
            @printf("Emission centroid (start of the centre fit): pixel (%d, %d), dRA = %+.4f\", dDec = %+.4f\" from the image centre\n",
                    ccx, ccy, (ccx - c_img[1]) * img.cdelt1_arcsec, (ccy - c_img[2]) * img.cdelt2_arcsec)
        else
            println("No significant emission for a centroid start; the centre fit starts at the image centre.")
        end
    else
        error("center_mode must be :optimize, :image_center or :peak")
    end

    # inclination / PA from the image when they are not known (or when asked to re-measure them)
    geom_fitted = false
    geom_r2 = geom_r2_faceon = NaN
    if fit_geom
        if bmaj_arcsec !== nothing && fit_r < 2.5 * bmaj_arcsec
            @printf("Disk too small relative to the beam (fit radius %.2f\" < 2.5 beams) to fit its geometry; keeping i = %.1f, PA = %.1f deg.\n",
                    fit_r, incl_deg, pa_deg)
        else
            gs, incl_deg, pa_deg, gs0 = fit_geometry(img; center = start, fit_radius_arcsec = fit_r)
            geom_r2 = 1 - gs; geom_r2_faceon = 1 - gs0
            g = _geometry(img, incl_deg, pa_deg)
            geom_fitted = true
            @printf("Fitted disk geometry from the image: i = %.1f deg, PA = %.1f deg east of north (+-~5 deg).\n", incl_deg, pa_deg)
        end
    end

    if center === nothing && center_mode == :optimize
        cx, cy = fit_center(img, g; start = start, fit_radius_arcsec = fit_r)
        if geom_fitted
            gs, incl_deg, pa_deg, _ = fit_geometry(img; center = (cx, cy), fit_radius_arcsec = fit_r, start = (incl_deg, pa_deg))
            geom_r2 = 1 - gs
            g = _geometry(img, incl_deg, pa_deg)
            @printf("Refined disk geometry about the fitted centre: i = %.1f deg, PA = %.1f deg.\n", incl_deg, pa_deg)
        end
    else
        cx, cy = start
    end
    geom_fitted && @printf("Geometry fit quality: a purely radial profile explains %.0f%% of the pixel variance (face-on: %.0f%%).%s\n",
                           100 * geom_r2, 100 * geom_r2_faceon,
                           geom_r2 < 0.6 ? " LOW: the disk is probably non-axisymmetric or faint, so the fitted geometry is unreliable." : "")
    @printf("Centre pixel (1-based): (%d, %d); offset from image centre: dRA = %+.4f\", dDec = %+.4f\"\n",
            cx, cy, (cx - c_img[1]) * img.cdelt1_arcsec, (cy - c_img[2]) * img.cdelt2_arcsec)

    # robust image noise (MAD) in a box around the disk, for the detection check in the summary
    hw = round(Int, min(0.95 * rimg, max(3 * max_radius_arcsec, 4.0)) / pix)
    box = Float64[Float64(v) for v in @view(data[max(1, cx - hw):min(nx, cx + hw), max(1, cy - hw):min(ny, cy + hw)]) if isfinite(v)]
    image_rms = isempty(box) ? NaN : 1.4826 * median(abs.(box .- median(box)))
    binw = max_radius_arcsec / n_bins
    binidx = zeros(Int16, nx, ny)
    sums   = zeros(Float64, n_bins)
    counts = zeros(Int, n_bins)
    @inbounds for j in 1:ny, i in 1:nx
        v = data[i, j]
        isnan(v) && continue
        r = _dep_radius(i, j, cx, cy, g)
        r < max_radius_arcsec || continue
        b = floor(Int, r / binw) + 1
        b > n_bins && continue
        binidx[i, j] = b
        sums[b] += v
        counts[b] += 1
    end

    mean_flux = [counts[b] > 0 ? sums[b] / counts[b] : NaN for b in 1:n_bins]
    ssq = zeros(Float64, n_bins)
    @inbounds for j in 1:ny, i in 1:nx
        b = binidx[i, j]
        b == 0 && continue
        d = data[i, j] - mean_flux[b]
        ssq[b] += d * d
    end
    std_flux = [counts[b] > 0 ? sqrt(ssq[b] / counts[b]) : NaN for b in 1:n_bins]   # population std

    bin_edges = range(0, max_radius_arcsec; length = n_bins + 1)
    bin_centers = [0.5 * (bin_edges[k] + bin_edges[k + 1]) for k in 1:n_bins]
    @printf("Radial profile: %d bins of %.4f\" (%.2f au) out to %.2f\" (%.0f au)\n",
            n_bins, binw, binw * distance_pc, max_radius_arcsec, max_radius_arcsec * distance_pc)
    _dbg("extract_radial_profile: empty bins =", count(==(0), counts),
         "valid mean_flux entries =", count(!isnan, mean_flux))
    all(isnan, mean_flux) && println("WARNING: every radial bin is NaN -- check the centre, pixel scale and radius.")

    return (
        radius_arcsec = bin_centers, radius_au = bin_centers .* distance_pc,
        mean_flux = mean_flux, std_flux = std_flux,
        pix_scale_arcsec = pix, center_pix = (cx, cy), incl_deg = incl_deg, pa_deg = pa_deg,
        bmaj_arcsec = bmaj_arcsec, bmin_arcsec = bmin_arcsec, bmaj_au = bmaj_au, bmin_au = bmin_au,
        distance_pc = distance_pc, bunit = img.bunit,
        image_rms = image_rms, geom_r2 = geom_r2, geom_r2_faceon = geom_r2_faceon,
        pix_counts = counts,
    )
end

## Peak finding and ring/gap detection

In [ ]:
# ---------------------------------------------------------------------------
# Signal-processing helpers (ports of scipy.signal.savgol_filter / find_peaks)
# ---------------------------------------------------------------------------

"""
    savgol_filter(y, window, polyorder) -> Vector{Float64}

Savitzky-Golay smoothing equivalent to `scipy.signal.savgol_filter(y,
window, polyorder)` with its default `mode="interp"`: interior points use
the least-squares polynomial fit over the centred window; the first and last
`window÷2` points evaluate the polynomial fitted to the first/last `window`
samples.
"""
function savgol_filter(y::AbstractVector{<:Real}, window::Integer, polyorder::Integer)
    isodd(window) || throw(ArgumentError("window must be odd, got $window"))
    window > polyorder || throw(ArgumentError("window must exceed polyorder"))
    n = length(y)
    window <= n || throw(ArgumentError("window ($window) larger than data length ($n)"))
    h = window ÷ 2
    xs = collect(-h:h)
    V = [Float64(x)^k for x in xs, k in 0:polyorder]    # window × (polyorder+1)
    P = pinv(V)                                         # least-squares solver
    out = zeros(Float64, n)
    for i in 1:n
        s = clamp(i - h, 1, n - window + 1)            # window start (shifted at the edges)
        c = P * Float64.(y[s:(s + window - 1)])
        xi = Float64(i - (s + h))                       # position of i relative to the window centre
        out[i] = sum(c[k + 1] * xi^k for k in 0:polyorder)
    end
    return out
end

# Local maxima, handling flat plateaus by taking the middle sample
# (scipy.signal._peak_finding_utils._local_maxima_1d).
function _local_maxima(x::AbstractVector)
    n = length(x)
    mids = Int[]
    n < 3 && return mids
    i = 2
    i_max = n
    while i < i_max
        if x[i - 1] < x[i]
            i_ahead = i + 1
            while i_ahead < i_max && x[i_ahead] == x[i]
                i_ahead += 1
            end
            if x[i_ahead] < x[i]
                push!(mids, (i + (i_ahead - 1)) ÷ 2)
                i = i_ahead
            end
        end
        i += 1
    end
    return mids
end

# Topographic prominence of the peak at index p with no window limit
# (scipy.signal._peak_finding_utils._peak_prominences, wlen=None).
function _peak_prominence(x::AbstractVector, p::Int)
    n = length(x)
    left_min = x[p]
    i = p
    while i >= 1 && x[i] <= x[p]
        x[i] < left_min && (left_min = x[i])
        i -= 1
    end
    right_min = x[p]
    i = p
    while i <= n && x[i] <= x[p]
        x[i] < right_min && (right_min = x[i])
        i += 1
    end
    return x[p] - max(left_min, right_min)
end

"""
    find_peaks(x; prominence=nothing) -> (indices, prominences)

Indices (1-based) of local maxima of `x` whose prominence is at least
`prominence`, with their prominences -- the subset of
`scipy.signal.find_peaks` used by the original script.
"""
function find_peaks(x::AbstractVector{<:Real}; prominence = nothing)
    peaks = _local_maxima(x)
    proms = [_peak_prominence(x, p) for p in peaks]
    if prominence !== nothing
        keep = proms .>= prominence
        peaks, proms = peaks[keep], proms[keep]
    end
    return peaks, proms
end

# ---------------------------------------------------------------------------
# Ring / gap detection
# ---------------------------------------------------------------------------

"""
    find_rings_and_gaps(profile; ring_prominence=nothing, gap_prominence=nothing) -> (rings, gaps)

Locate local maxima (rings) and minima (gaps) in a radial flux profile with
`find_peaks`. `prominence=nothing` derives a threshold from the profile's own
scatter (3x the median `std_flux`); pass an explicit value to override if the
auto threshold is too strict/loose.
"""
function find_rings_and_gaps(profile; ring_prominence = nothing, gap_prominence = nothing)
    valid = .!isnan.(profile.mean_flux)
    flux_v = profile.mean_flux[valid]
    radius_v = profile.radius_au[valid]
    _dbg("find_rings_and_gaps: valid profile points =", length(flux_v), "/", length(profile.mean_flux))

    if ring_prominence === nothing
        ring_prominence = 3 * median(filter(!isnan, profile.std_flux))
    end
    gap_prominence === nothing && (gap_prominence = ring_prominence)
    _dbg("find_rings_and_gaps: ring_prominence =", ring_prominence, "gap_prominence =", gap_prominence)

    ring_idx, ring_proms = find_peaks(flux_v; prominence = ring_prominence)
    gap_idx, gap_proms = find_peaks(-flux_v; prominence = gap_prominence)
    _dbg("find_rings_and_gaps: raw ring_idx =", ring_idx, "raw gap_idx =", gap_idx)

    rings = [(radius_au = radius_v[i], flux = flux_v[i], prominence = p) for (i, p) in zip(ring_idx, ring_proms)]
    gaps  = [(radius_au = radius_v[i], flux = flux_v[i], prominence = p) for (i, p) in zip(gap_idx, gap_proms)]
    return rings, gaps
end

"""
    find_rings_and_gaps_residual(profile; window_frac=0.22, polyorder=3, ring_prominence=nothing,
                                 gap_prominence=nothing, prominence_sigma=1.2,
                                 min_radius_beams=2.5, min_abs_residual_frac=0.3)
        -> (rings, gaps, residual_data)

Locate ring/gap substructure using a smooth-baseline-subtraction approach
(as in e.g. Huang et al. 2018's DSHARP substructure analysis), rather than
`find_peaks` on the raw profile.

Why this is needed: `find_peaks` only flags genuine local maxima/minima.
When a disk's radial profile is dominated by a steep, monotonically
declining envelope (a bright unresolved central source falling off with
radius, as here), real ring/gap substructure often shows up only as
"shoulders" -- slope changes riding on top of the decline -- which never
turn over into a local max/min and so are invisible to plain `find_peaks`.
This function fits a smooth Savitzky-Golay baseline to the profile
(capturing just the broad monotonic falloff, not the substructure),
subtracts it to get a residual, and runs `find_peaks` on the residual.

window_frac: Savitzky-Golay window as a fraction of the number of valid
    profile bins (forced odd). Larger = smoother baseline = more sensitive
    to broad shoulders; smaller = baseline tracks the data more closely =
    less sensitive.
polyorder: polynomial order of the Savitzky-Golay fit (3 is a reasonable
    default for a smoothly declining profile with a few features).
prominence_sigma: multiplier on the residual's standard deviation used to
    derive ring/gap prominence thresholds when those are `nothing`. Lower
    values (e.g. 0.6-0.8) recover weaker, few-percent features (like the
    shallow gaps at ~6 au and ~44 au reported for TW Hya by Tsukagoshi et
    al. 2016 and Andrews et al. 2016) at the cost of more noise-driven false
    positives -- always sanity-check low-sigma detections against the plot
    and the literature rather than trusting them blindly.
min_radius_beams: exclude detections within this many synthesized-beam FWHMs
    (BMAJ) of the center. CLEAN deconvolution of a very bright, compact,
    high-dynamic-range core commonly leaves negative-bowl / restoration
    artifacts within the first few beam-widths of the peak, which can
    masquerade as a "gap". The baseline fit still uses the full profile;
    only the peak-finding *results* are filtered by radius. 0 disables.
min_abs_residual_frac: additionally require |residual| at the detected
    point to be at least this fraction of the residual's standard deviation.
    Prominence is relative (vs. neighbouring valleys/peaks), so a detection
    can be "prominent" while sitting right on the noise floor in absolute
    terms; this catches that case. 0 disables.
"""
function find_rings_and_gaps_residual(profile; window_frac = 0.22, polyorder = 3,
                                      ring_prominence = nothing, gap_prominence = nothing,
                                      prominence_sigma = 1.2, min_radius_beams = 2.5,
                                      min_abs_residual_frac = 0.3)
    valid = .!isnan.(profile.mean_flux)
    flux_v = profile.mean_flux[valid]
    radius_v = profile.radius_au[valid]

    n = length(flux_v)
    window = max(round(Int, window_frac * n), polyorder + 2)
    iseven(window) && (window += 1)
    window = min(window, isodd(n) ? n : n - 1)
    _dbg("find_rings_and_gaps_residual: n_valid =", n, "savgol window =", window, "polyorder =", polyorder)

    baseline = savgol_filter(flux_v, window, polyorder)
    residual = flux_v .- baseline
    bunit = profile.bunit
    residual_std = std(residual; corrected = false)          # population std, like np.std
    _dbg("find_rings_and_gaps_residual: residual std =", residual_std,
         "residual min/max =", minimum(residual), maximum(residual))

    ring_prominence === nothing && (ring_prominence = prominence_sigma * residual_std)
    gap_prominence === nothing && (gap_prominence = prominence_sigma * residual_std)
    _dbg("find_rings_and_gaps_residual: ring_prominence =", ring_prominence, "gap_prominence =", gap_prominence)

    ring_idx, ring_proms = find_peaks(residual; prominence = ring_prominence)
    gap_idx, gap_proms = find_peaks(-residual; prominence = gap_prominence)
    _dbg("find_rings_and_gaps_residual: raw ring_idx =", ring_idx, "raw gap_idx =", gap_idx)

    bmaj_au = profile.bmaj_au
    min_radius_au = (bmaj_au !== nothing && min_radius_beams > 0) ? min_radius_beams * bmaj_au : 0.0
    _dbg("find_rings_and_gaps_residual: bmaj_au =", bmaj_au, "min_radius_beams =", min_radius_beams,
         "-> min_radius_au =", min_radius_au)

    min_abs_residual = min_abs_residual_frac * residual_std
    _dbg("find_rings_and_gaps_residual: min_abs_residual_frac =", min_abs_residual_frac,
         "-> min_abs_residual =", min_abs_residual)

    keep(i) = radius_v[i] >= min_radius_au && abs(residual[i]) >= min_abs_residual

    all_idx = vcat(ring_idx, gap_idx)
    dropped_radius = [i for i in all_idx if radius_v[i] < min_radius_au]
    dropped_noise = [i for i in all_idx if radius_v[i] >= min_radius_au && abs(residual[i]) < min_abs_residual]
    if !isempty(dropped_radius)
        @printf("Excluded %d detection(s) within %.2f au (%sx beam FWHM) of center as likely beam/CLEAN artifacts:\n",
                length(dropped_radius), min_radius_au, min_radius_beams)
        for i in dropped_radius
            @printf("    r = %.1f au, residual = %.4e %s (%+.2f sigma)\n",
                    radius_v[i], residual[i], bunit, residual[i] / residual_std)
        end
    end
    if !isempty(dropped_noise)
        @printf("Excluded %d detection(s) with |residual| below %.2e %s (%sx residual std) as likely noise/boundary artifacts, despite passing the prominence test:\n",
                length(dropped_noise), min_abs_residual, bunit, min_abs_residual_frac)
        for i in dropped_noise
            @printf("    r = %.1f au, residual = %.4e %s (%+.2f sigma)\n",
                    radius_v[i], residual[i], bunit, residual[i] / residual_std)
        end
    end

    rings = [(radius_au = radius_v[i], flux = flux_v[i], residual = residual[i], prominence = p)
             for (i, p) in zip(ring_idx, ring_proms) if keep(i)]
    gaps = [(radius_au = radius_v[i], flux = flux_v[i], residual = residual[i], prominence = p)
            for (i, p) in zip(gap_idx, gap_proms) if keep(i)]
    residual_data = (radius_au = radius_v, flux = flux_v, baseline = baseline, residual = residual,
                     min_radius_au = min_radius_au, min_abs_residual = min_abs_residual,
                     distance_pc = profile.distance_pc, bunit = bunit)
    return rings, gaps, residual_data
end

"""
    report_residual_near(residual_data, target_radius_au; window_au=3.0)

Print the raw residual value(s) near a specific radius, independent of
whether `find_peaks` flagged anything there. Useful for checking a feature
that a prominence/noise-floor filter may have dropped (or that never
registered as a local extremum): "is there a real few-sigma dip near the
literature's 44 au gap even though nothing survived filtering there?" No
threshold is applied; it just reports what the residual actually does.
"""
function report_residual_near(residual_data, target_radius_au; window_au = 3.0)
    r_au = residual_data.radius_au
    res = residual_data.residual
    res_std = std(res; corrected = false)
    bunit = residual_data.bunit

    mask = abs.(r_au .- target_radius_au) .<= window_au
    if !any(mask)
        println("No profile bins found within $window_au au of $target_radius_au au.")
        return nothing
    end

    @printf("\nRaw residual near r = %s au (+/-%s au, residual std = %.4e %s):\n",
            target_radius_au, window_au, res_std, bunit)
    rm_, resm = r_au[mask], res[mask]
    dmin = minimum(abs.(rm_ .- target_radius_au))
    for (r, v) in zip(rm_, resm)
        marker = abs(r - target_radius_au) == dmin ? " <-- closest" : ""
        @printf("    r = %.2f au, residual = %.4e %s (%+.2f sigma)%s\n", r, v, bunit, v / res_std, marker)
    end

    ci = argmin(abs.(r_au .- target_radius_au))
    @printf("  Closest bin: r = %.2f au, residual = %.4e %s (%+.2f sigma)\n",
            r_au[ci], res[ci], bunit, res[ci] / res_std)
    return nothing
end

## Literature comparison and plots

In [ ]:
# ---------------------------------------------------------------------------
# Literature overlay + plots
# ---------------------------------------------------------------------------

const _LIT_LABEL = Dict(:hole => "central hole (lit.)", :deep => "deep gap (lit.)",
                        :shallow => "shallow gap (lit.)", :gap => "gap (lit.)", :ring => "ring (lit.)")
const _LIT_STYLE = Dict(:hole => (color = ("#AAAAAA", 0.7), linewidth = 1.2),
                        :deep => (color = (C_LIT, 0.85), linewidth = 1.6),
                        :shallow => (color = (C_LIT, 0.55), linewidth = 1.1),
                        :gap => (color = (C_LIT, 0.75), linewidth = 1.3),
                        :ring => (color = ("#FFB3C1", 0.85), linewidth = 1.4))

"""
    literature_features_au(t::AlmaTarget, distance_pc) -> Vector{(radius_au, label, kind)}

The target's literature gap radii rescaled from the distance the literature
assumed (`t.lit_distance_pc`) to the distance used in this run. Empty if the
catalogue has none for this target.
"""
function literature_features_au(t::AlmaTarget, distance_pc)
    isempty(t.lit) && return Tuple{Float64,String,Symbol}[]
    scale = distance_pc / (t.lit_distance_pc === nothing ? distance_pc : t.lit_distance_pc)
    return Tuple{Float64,String,Symbol}[(r * scale, _LIT_LABEL[kind], kind) for (r, kind) in t.lit]
end

"""
    literature_props_au(t::AlmaTarget, distance_pc) -> Vector{LitProp}

The published ring/gap radii and widths (Huang+18) rescaled from the literature distance to the one used in this run.
"""
function literature_props_au(t::AlmaTarget, distance_pc)
    isempty(t.lit_props) && return LitProp[]
    scale = distance_pc / (t.lit_distance_pc === nothing ? distance_pc : t.lit_distance_pc)
    return LitProp[(p[1] * scale, p[2] * scale, p[3], p[4], p[5]) for p in t.lit_props]
end

# the published feature of the given kind nearest to radius r (au), within tol (au), or nothing
function _nearest_prop(props, r::Real, kind::Symbol, tol::Real)
    best = nothing; bd = Inf
    for p in props
        p[4] === kind || continue
        d = abs(p[1] - r)
        if d <= tol && d < bd
            bd = d; best = p
        end
    end
    return best
end

"""
    compare_with_literature(lit, gaps, residual_data; bmaj_au=nothing)

Table of each literature feature against the nearest detected gap, in au and
in beams, plus the raw residual at the nearest bin when nothing was detected
within one beam.
"""
function compare_with_literature(lit, gaps, residual_data; bmaj_au = nothing, rings = [])
    isempty(lit) && return nothing
    r_au, res = residual_data.radius_au, residual_data.residual
    res_std = std(res; corrected = false)
    tol = bmaj_au === nothing ? 2.0 : bmaj_au
    println("\nLiterature comparison (radii rescaled to the adopted distance; tolerance = 1 beam = $(round(tol; digits = 2)) au):")
    @printf("  %-20s %9s   %-22s %s\n", "feature", "lit [au]", "nearest detected feature", "delta [au] (beams)")
    for (r, label, kind) in sort(lit; by = first)
        if r < residual_data.min_radius_au
            @printf("  %-20s %9.1f   %-22s %s\n", label, r, "-", "inside the excluded beam/CLEAN zone")
            continue
        end
        pool = kind == :ring ? rings : gaps       # published rings are matched to detected rings, gaps to gaps
        if !isempty(pool)
            gi = argmin([abs(g.radius_au - r) for g in pool])
            d = pool[gi].radius_au - r
            if abs(d) <= tol
                @printf("  %-20s %9.1f   %-22.1f %+.1f (%+.2f)\n", label, r, pool[gi].radius_au, d, d / tol)
                continue
            end
        end
        ci = argmin(abs.(r_au .- r))
        @printf("  %-20s %9.1f   %-22s raw residual %.2e at r = %.1f au (%+.2f sigma)\n",
                label, r, "none within 1 beam", res[ci], r_au[ci], res[ci] / res_std)
    end
    return nothing
end

"""
    plot_radial_profile(profile; rings=[], gaps=[], title="", filename=...) -> Figure

Azimuthally averaged flux vs. radius, with rings/gaps marked.
"""
function plot_radial_profile(profile; rings = [], gaps = [], title = "Radial profile",
                             filename = "radial_profile.png")
    _dbg("plot_radial_profile: called with", length(rings), "rings and", length(gaps), "gaps")
    r, f, s = profile.radius_au, profile.mean_flux, profile.std_flux

    fig = Figure(size = (820, 520))
    ax = Axis(fig[1, 1]; xlabel = "Deprojected radius [au]", ylabel = "Flux density [$(profile.bunit)]",
              title = title, titlesize = 13)
    band!(ax, r, f .- s, f .+ s; color = (C_FLUX, 0.2), label = "±1σ scatter")
    lines!(ax, r, f; color = C_FLUX, linewidth = 2, label = "Azimuthal mean")

    if !isempty(rings)
        rr = [x.radius_au for x in rings]
        vlines!(ax, rr; color = C_RING, linestyle = :dash, linewidth = 1)
        scatter!(ax, rr, [x.flux for x in rings]; color = C_RING, marker = :utriangle, markersize = 13, label = "Ring")
    end
    if !isempty(gaps)
        gr = [x.radius_au for x in gaps]
        vlines!(ax, gr; color = C_GAP, linestyle = :dash, linewidth = 1)
        scatter!(ax, gr, [x.flux for x in gaps]; color = C_GAP, marker = :dtriangle, markersize = 13, label = "Gap")
    end
    axislegend(ax; position = :rt)

    display(fig)
    _save_fig(fig, filename)
    return fig
end

"""
    plot_residual_profile(residual_data; rings=[], gaps=[], lit_features=[], title="", filename=...) -> Figure

Two panels: raw profile with its Savitzky-Golay baseline (top) and the
residual with detected rings (^) and gaps (v) (bottom). `lit_features` is the
output of `literature_features_au` (dotted vertical lines).
"""
function plot_residual_profile(residual_data; rings = [], gaps = [], lit_features = [],
                               title = "Substructure (baseline-subtracted)",
                               filename = "substructure_residual.png")
    _dbg("plot_residual_profile: called with", length(rings), "rings and", length(gaps), "gaps")
    r = residual_data.radius_au
    min_radius_au = residual_data.min_radius_au

    fig = Figure(size = (820, 820))
    ax1 = Axis(fig[1, 1]; ylabel = "Flux density [$(residual_data.bunit)]", title = title, titlesize = 13)
    ax2 = Axis(fig[2, 1]; xlabel = "Deprojected radius [au]", ylabel = "Residual [$(residual_data.bunit)]")
    linkxaxes!(ax1, ax2)
    hidexdecorations!(ax1; grid = false)
    rowsize!(fig.layout, 1, Auto(1.2))
    rowsize!(fig.layout, 2, Auto(1.0))
    rowgap!(fig.layout, 8)

    lines!(ax1, r, residual_data.flux; color = C_FLUX, linewidth = 2, label = "Azimuthal mean")
    lines!(ax1, r, residual_data.baseline; color = "#CCCCCC", linewidth = 1.4, linestyle = :dash, label = "Smooth baseline")
    if min_radius_au > 0
        vspan!(ax1, [0.0], [min_radius_au]; color = ("#888888", 0.22),
               label = @sprintf("Excluded (<%.1f au, beam/CLEAN artifacts)", min_radius_au))
        vspan!(ax2, [0.0], [min_radius_au]; color = ("#888888", 0.22))
    end

    hlines!(ax2, 0.0; color = "#888888", linewidth = 0.8)
    lines!(ax2, r, residual_data.residual; color = C_RESID, linewidth = 2, label = "Residual")

    for kind in (:hole, :deep, :shallow, :gap, :ring)
        sel = [x for x in lit_features if x[3] == kind]
        isempty(sel) && continue
        for ax in (ax1, ax2)
            vlines!(ax, [x[1] for x in sel]; linestyle = :dot, label = sel[1][2], _LIT_STYLE[kind]...)
        end
    end

    if !isempty(rings)
        rr = [x.radius_au for x in rings]
        vlines!(ax2, rr; color = C_RING, linestyle = :dash, linewidth = 1)
        scatter!(ax2, rr, [x.residual for x in rings]; color = C_RING, marker = :utriangle, markersize = 15, label = "Ring")
    end
    if !isempty(gaps)
        gr = [x.radius_au for x in gaps]
        vlines!(ax2, gr; color = C_GAP, linestyle = :dash, linewidth = 1)
        scatter!(ax2, gr, [x.residual for x in gaps]; color = C_GAP, marker = :dtriangle, markersize = 15, label = "Gap")
    end

    axislegend(ax1; position = :rt, labelsize = 11)
    axislegend(ax2; position = :rt, labelsize = 11)

    display(fig)
    _save_fig(fig, filename)
    return fig
end

## Target summary file

`write_target_summary(out)` writes `<target>_target_summary.txt` next to the PNG files: the ALMA portal link, a description of the target, the data used and an automatic analysis of the radial profile and the detected rings and gaps. `run_alma_pipeline` calls it at the end of every run.

In [ ]:
# ---------------------------------------------------------------------------
# Plain-text target summary, written next to the PNG files
# ---------------------------------------------------------------------------

const GROUP_DESC = Dict(
    :analogue => "A low-mass (K-M) T Tauri star surrounded by a gas-rich protoplanetary disk a few Myr old. ALMA continuum images of such disks very often show concentric rings and gaps, commonly interpreted as pressure traps or the imprint of forming planets.",
    :cavity   => "A T Tauri star whose disk has a large central cavity (a transition disk): the dust emission is a ring rather than a central peak. The inner hole is usually attributed to one or more companions or planets, or to photoevaporation.",
    :young    => "A young (Class 0/I/II) star still embedded in, or just emerging from, its natal envelope. The disk is compact and often inclined, and envelope emission can contaminate the continuum image.",
    :herbig   => "An intermediate-mass (A/B/F) Herbig star. Its disk is warmer and usually larger than a T Tauri disk, often with a large cavity and rings, spirals or shadows.",
    :asym     => "A disk with strongly non-axisymmetric dust emission (a horseshoe, vortex or arc). The azimuthal average used here mixes the bright and faint sides.",
    :multiple => "A multiple-star system: the field holds several stars and disks (circumstellar and/or circumbinary), so the fitted centre may sit on only one component.",
    :edge     => "An edge-on disk seen as a dark lane against its scattered light. Deprojection into face-on rings is not possible; radial structure is hidden by projection.",
    :outburst => "An FU Ori or EXor-type outbursting star: a young, hot, very bright accretion-burst disk, often with an envelope; usually farther away than the nearby T Tauri stars.",
    :custom   => "A user-defined target; no group description is available.",
)

const GROUP_CAVEAT = Dict(
    :analogue => "",
    :cavity   => "A central hole is expected: a low central flux and a peak at larger radius are physical, not artifacts.",
    :young    => "Envelope emission and a high inclination can produce spurious radial structure; treat weak features with caution.",
    :herbig   => "Large disks need long baselines and short spacings; the largest scales may be partly filtered out.",
    :asym     => "The azimuthal mean of a non-axisymmetric disk can produce rings and gaps that are not real; check the image.",
    :multiple => "Check the image for several sources; the radial profile describes only the component at the fitted centre.",
    :edge     => "The radial profile is not meaningful for an edge-on disk; use the image alone.",
    :outburst => "Very bright and often partly resolved by the envelope; dynamic range and CLEAN artifacts may dominate the weak features.",
    :custom   => "",
)

GROUP_DESC[:archive] = "A disk target picked from the ALMA archive (science category \"Disks and planet formation\"). It is not individually curated, so the type of star and of disk is not recorded here: it may be a T Tauri or Herbig disk, an embedded protostar, a multiple system or a debris disk."
GROUP_CAVEAT[:archive] = "Not curated: the distance is a Gaia distance (Bailer-Jones+21 or the SIMBAD parallax; or an assumed 100 pc), the geometry is fitted from the image, and the target may be embedded, multiple or only marginally resolved."

function _sexa(ra_deg::Real, dec_deg::Real)
    h = ra_deg / 15
    hh = floor(Int, h); mm = floor(Int, (h - hh) * 60); ss = ((h - hh) * 60 - mm) * 60
    d = abs(dec_deg); dd = floor(Int, d); dm = floor(Int, (d - dd) * 60); ds = ((d - dd) * 60 - dm) * 60
    return @sprintf("%02d:%02d:%05.2f  %s%02d:%02d:%04.1f", hh, mm, ss, dec_deg < 0 ? "-" : "+", dd, dm, ds)
end

_is_jybeam(b::AbstractString) = lowercase(strip(b)) == "jy/beam"

# observing frequency [GHz] of a FITS image from its header (NaN if absent)
function _image_freq_ghz(path)
    try
        f = FITS(path)
        try
            idx = findfirst(i -> f[i] isa ImageHDU && ndims(f[i]) >= 2, 1:length(f))
            idx === nothing && return NaN
            v = _fits_freq_hz(read_header(f[idx]))
            return v === nothing ? NaN : v / 1e9
        finally
            close(f)
        end
    catch
        return NaN
    end
end

function _fmt_flux(x::Real, bunit::AbstractString)
    isfinite(x) || return "n/a"
    return _is_jybeam(bunit) ? @sprintf("%.3g mJy/beam", 1e3 * x) : @sprintf("%.3g %s", x, bunit)
end

# first radius beyond the profile peak where the azimuthal mean falls below `frac` x peak
# first radius from the centre where the azimuthal mean reaches `frac` x its peak
function _radius_rising(profile, frac::Real)
    ok = findall(!isnan, profile.mean_flux)
    isempty(ok) && return NaN
    thr = frac * maximum(profile.mean_flux[ok])
    for i in ok
        profile.mean_flux[i] >= thr && return profile.radius_au[i]
    end
    return NaN
end

function _radius_below(profile, frac::Real)
    ok = findall(!isnan, profile.mean_flux)
    isempty(ok) && return NaN
    ipk = ok[argmax(profile.mean_flux[ok])]
    thr = frac * profile.mean_flux[ipk]
    for i in ipk:length(profile.mean_flux)
        v = profile.mean_flux[i]
        (!isnan(v) && v < thr) && return profile.radius_au[i]
    end
    return NaN
end

# last radius at which the azimuthal mean is still above `frac` x its peak (the outer edge, not the first dip)
function _radius_last_above(profile, frac::Real)
    ok = findall(!isnan, profile.mean_flux)
    isempty(ok) && return NaN
    thr = frac * maximum(profile.mean_flux[ok])
    for i in reverse(ok)
        profile.mean_flux[i] >= thr && return profile.radius_au[i]
    end
    return NaN
end

# statistical 1-sigma of the azimuthal mean at radius r_au: image noise / sqrt(independent beams in the annulus).
# (The scatter of the baseline-subtracted profile is NOT used: it contains the rings themselves.)
function _stat_sigma(profile, r_au::Real)
    (hasproperty(profile, :pix_counts) && hasproperty(profile, :image_rms)) || return NaN
    rmsn = profile.image_rms
    (isfinite(rmsn) && rmsn > 0 && profile.bmaj_arcsec !== nothing && profile.bmin_arcsec !== nothing) || return NaN
    beam_pix = (pi / (4 * log(2))) * Float64(profile.bmaj_arcsec) * Float64(profile.bmin_arcsec) / profile.pix_scale_arcsec^2
    (isfinite(beam_pix) && beam_pix > 0) || return NaN
    i = argmin(abs.(profile.radius_au .- r_au))
    nb = profile.pix_counts[i] / beam_pix
    return rmsn / sqrt(max(nb, 1.0))
end

# Detection significance of the azimuthal-mean profile: the largest bin value in units of its own statistical sigma
# (image noise / sqrt(independent beams in that annulus)). Unlike peak / image rms, this does not penalise faint but
# extended emission, whose annular mean is far below the per-beam noise yet significant. NaN if it cannot be computed.
function _detection_sigma(profile)
    best = NaN
    for i in eachindex(profile.mean_flux)
        f = profile.mean_flux[i]
        isfinite(f) || continue
        sg = _stat_sigma(profile, profile.radius_au[i])
        (isfinite(sg) && sg > 0) || continue
        z = f / sg
        (!isfinite(best) || z > best) && (best = z)
    end
    return best
end

# disk-integrated flux (mJy) out to the profile edge; deprojected annulus area x cos(i) = sky area
function _integrated_flux_mjy(profile)
    _is_jybeam(profile.bunit) || return NaN
    (profile.bmaj_arcsec === nothing || profile.bmin_arcsec === nothing) && return NaN
    length(profile.radius_arcsec) < 2 && return NaN
    omega = pi * profile.bmaj_arcsec * profile.bmin_arcsec / (4 * log(2))
    dr = profile.radius_arcsec[2] - profile.radius_arcsec[1]
    s = 0.0
    for (r, f) in zip(profile.radius_arcsec, profile.mean_flux)
        isnan(f) && continue
        s += f * 2pi * r * dr / omega
    end
    return 1e3 * s * cosd(profile.incl_deg)
end

"""
    write_target_summary(out; rings, gaps, residual_data, filename, note) -> path

Write `<slug>_target_summary.txt` to OUTPUT_DIR (next to the PNG files): the ALMA archive portal
link, a description of the target, the data used, and an automatic analysis of the radial
profile and the detected rings/gaps (literature comparison, caveats). Called at the end of
`run_alma_pipeline`; call it again with `rings =`, `gaps =`, `residual_data =` to document a
re-tuned detection. The text is generated from the measurements; it is a first-pass reading, not a
substitute for inspecting the images.
"""
function write_target_summary(out; rings = nothing, gaps = nothing, residual_data = nothing,
                              filename = nothing, note::AbstractString = "")
    if !(hasproperty(out, :profile) && out.profile !== nothing)
        println("No radial profile available; target summary not written.")
        return nothing
    end
    t, pr = out.target, out.profile
    rings = rings === nothing ? out.rings_residual : rings
    gaps = gaps === nothing ? out.gaps_residual : gaps
    rd = residual_data === nothing ? out.residual_data : residual_data
    (rings === nothing || gaps === nothing || rd === nothing) && (println("No substructure analysis available; target summary not written."); return nothing)

    L = String[]
    put(s::AbstractString = "") = push!(L, String(s))
    bm = pr.bmaj_au === nothing ? NaN : Float64(pr.bmaj_au)
    bunit = pr.bunit
    fscale = _is_jybeam(bunit) ? 1e3 : 1.0
    funit = _is_jybeam(bunit) ? "mJy/beam" : bunit
    rms = std(rd.residual[.!isnan.(rd.residual)]; corrected = false)
    nfirm = 0
    have_stat = isfinite(_stat_sigma(pr, pr.radius_au[max(1, length(pr.radius_au) ÷ 2)]))

    put("="^88)
    put("TARGET SUMMARY: " * t.name)
    put("="^88)
    put("Generated " * Libc.strftime("%Y-%m-%d %H:%M", time()) * " by alma_target_explorer_julia (v7)")
    isempty(note) || put("Note: " * note)
    put()
    put("ALMA archive web portal: " * archive_url(t))
    put()

    # ---------------------------------------------------------------- target
    put("1. THE TARGET")
    put("-"^88)
    put(@sprintf("  Name           : %s", t.name))
    put("  Coordinates    : " * _sexa(t.ra_deg, t.dec_deg) * @sprintf("  (RA %.5f, Dec %+.5f deg, J2000)", t.ra_deg, t.dec_deg))
    put(@sprintf("  Distance       : %.1f pc%s", pr.distance_pc, t.distance_known ? "" : "  (ASSUMED; radii in au are not reliable)"))
    put("  Class          : " * get(GROUP_LABEL, t.group, string(t.group)))
    isempty(t.note) || put("  Catalogue note : " * t.note)
    isempty(t.differs) || put("  Versus TW Hya  : " * t.differs)
    let dl = disc_info_line(t; r68_measured = _disk_extent_au(pr; frac = 0.68))
        isempty(dl) || put("  Published      : " * dl)
    end
    put("  Catalogue geometry: " * (t.incl_deg === nothing || t.pa_deg === nothing ? "inclination / PA not in the catalogue" :
        @sprintf("inclination %.1f deg, PA %.1f deg east of north", t.incl_deg, t.pa_deg)))
    put()
    put("  What kind of object this is:")
    put("    " * get(GROUP_DESC, t.group, ""))
    cav = get(GROUP_CAVEAT, t.group, "")
    isempty(cav) || put("    Analysis caveat for this class: " * cav)
    put()

    # ------------------------------------------------------------------ data
    put("2. ALMA DATA USED")
    put("-"^88)
    if hasproperty(out, :chosen) && out.chosen !== nothing
        c = out.chosen
        put(@sprintf("  Project        : %s", c.proposal_id))
        put(@sprintf("  Band           : %s", c.band))
        put(@sprintf("  Archive-reported resolution of the observation : %.4f arcsec (%.1f au; the beam of the image used is given below)", c.best_res_arcsec, c.best_res_arcsec * pr.distance_pc))
        put(@sprintf("  Exposure time  : %.0f s (%.2f h)", c.t_exp_s, c.t_exp_s / 3600))
        put(@sprintf("  Public release : %s", c.release))
        put(@sprintf("  Member OUS     : %s", c.member_ous_uid))
        hasproperty(c, :las_arcsec) && isfinite(c.las_arcsec) &&
            put(@sprintf("  Largest recoverable angular scale : %.2f arcsec (%.0f au); structure larger than this is filtered out", c.las_arcsec, c.las_arcsec * pr.distance_pc))
    end
    if hasproperty(out, :observations) && out.observations !== nothing
        obs = out.observations
        fin = obs.best_res_arcsec[1]
        put(@sprintf("  Observations in the archive query: %d; the finest has %.4f arcsec. Used: #%d of the resolution-sorted table.",
                     nrow(obs), fin, hasproperty(out, :used_choice) ? out.used_choice : 1))
    end
    if hasproperty(out, :used_choice) && out.used_choice != out.obs_choice && hasproperty(out, :chosen)
        put(@sprintf("  NOTE: the notebook fell back from observation #%d to #%d because the finer packages exceeded the size limit.", out.obs_choice, out.used_choice))
    end
    if hasproperty(out, :tarball_url) && out.tarball_url !== nothing
        sz = hasproperty(out, :size_gb) && out.size_gb !== nothing ? @sprintf(" (%.2f GB)", out.size_gb) : ""
        put("  Product package: " * basename(out.tarball_url) * sz)
    end
    hasproperty(out, :local_path) && put("  FITS image used: " * basename(out.local_path))
    put(@sprintf("  Pixel scale    : %.4f arcsec", pr.pix_scale_arcsec))
    if pr.bmaj_arcsec !== nothing
        put(@sprintf("  Synthesised beam: %.4f x %.4f arcsec = %.1f x %.1f au", pr.bmaj_arcsec, pr.bmin_arcsec, pr.bmaj_au, pr.bmin_au))
    else
        put("  Synthesised beam: not found in the FITS header (beam-based quantities unavailable)")
    end
    put("  Flux unit      : " * bunit)
    put()

    # -------------------------------------------------------------- geometry
    put("3. DISK GEOMETRY AND PROFILE SETUP")
    put("-"^88)
    src = hasproperty(out, :geometry_source) ? out.geometry_source : :catalogue
    put(@sprintf("  Inclination %.1f deg, position angle %.1f deg (east of north)", pr.incl_deg, pr.pa_deg))
    if src == :catalogue
        put("  Source: catalogue (literature) values.")
    elseif src == :fitted
        put("  Source: FITTED FROM THE IMAGE (radial-symmetry criterion), accuracy about +-5 deg.")
        if hasproperty(out, :cat_incl) && out.cat_incl !== nothing && out.cat_pa !== nothing
            put(@sprintf("  Catalogue values for comparison: i = %.1f, PA = %.1f deg.", out.cat_incl, out.cat_pa))
        end
    else
        put("  Source: UNKNOWN; the disk was treated as face-on, so an inclined disk is smeared in the profile.")
    end
    if src == :fitted && hasproperty(pr, :geom_r2) && isfinite(pr.geom_r2)
        put(@sprintf("  Fit quality: a purely radial profile explains %.0f%% of the pixel variance (face-on: %.0f%%).%s", 100 * pr.geom_r2, 100 * pr.geom_r2_faceon,
                     pr.geom_r2 < 0.6 ? " LOW: probably non-axisymmetric or faint; the fitted geometry is unreliable." : ""))
    end
    put(@sprintf("  Profile centre (pixel): (%d, %d); profile out to %.2f arcsec = %.0f au in %d bins of %.2f au",
                 pr.center_pix[1], pr.center_pix[2], pr.radius_arcsec[end] + 0.5 * (pr.radius_arcsec[2] - pr.radius_arcsec[1]),
                 (pr.radius_au[end] + 0.5 * (pr.radius_au[2] - pr.radius_au[1])), length(pr.radius_au), pr.radius_au[2] - pr.radius_au[1]))
    if hasproperty(out, :settings)
        put("  Run settings   : " * join(["$(k) = $(v)" for (k, v) in pairs(out.settings)], ", "))
    end
    put()

    # --------------------------------------------------------------- profile
    put("4. RADIAL PROFILE (azimuthal mean of the continuum)")
    put("-"^88)
    ok = findall(!isnan, pr.mean_flux)
    isempty(ok) && (put("  Every radial bin is NaN; no analysis possible."); return _write_summary_file(L, t, filename))
    ipk = ok[argmax(pr.mean_flux[ok])]
    peak, rpk = pr.mean_flux[ipk], pr.radius_au[ipk]
    inner = pr.mean_flux[ok[1]]
    r50 = _disk_extent_au(pr; frac = 0.5); r90 = _disk_extent_au(pr; frac = 0.9); r10 = _radius_last_above(pr, 0.1)
    totf = _integrated_flux_mjy(pr)
    put(@sprintf("  Peak of the azimuthal mean : %s at r = %.1f au (%.1f beams)", _fmt_flux(peak, bunit), rpk, rpk / bm))
    put(@sprintf("  Innermost bin              : %s (%.0f%% of the peak)", _fmt_flux(inner, bunit), 100 * inner / peak))
    put(@sprintf("  Radius holding 50%% / 90%% of the flux : %.0f au / %.0f au (%.1f / %.1f beams)", r50, r90, r50 / bm, r90 / bm))
    isfinite(r10) && put(@sprintf("  Outer edge (last radius with the mean above 10%% of the peak): %.0f au", r10))
    isfinite(totf) && put(@sprintf("  Integrated flux out to the profile edge: ~%.1f mJy", totf))
    put(@sprintf("  Residual scatter (1 sigma of the baseline-subtracted profile): %s", _fmt_flux(rms, bunit)))
    put(@sprintf("  Profile dynamic range (peak / residual sigma): %.0f", peak / rms))
    rmsimg = hasproperty(pr, :image_rms) ? pr.image_rms : NaN
    snr = (isfinite(rmsimg) && rmsimg > 0) ? peak / rmsimg : NaN
    isfinite(snr) && put(@sprintf("  Image noise (robust rms): %s; the peak of the azimuthal mean is %.1f x the noise", _fmt_flux(rmsimg, bunit), snr))
    put()

    # ---------------------------------------------------------- substructure
    put("5. SUBSTRUCTURE (baseline-subtracted profile)")
    put("-"^88)
    put(@sprintf("  Raw peak finder on the unsubtracted profile: %d ring(s), %d gap(s).", length(out.rings), length(out.gaps)))
    put(@sprintf("  After subtracting a smooth baseline: %d ring(s), %d gap(s).", length(rings), length(gaps)))
    put(@sprintf("  Detections inside %.0f au (the beam/CLEAN-artifact zone) are discarded; residuals below %s are ignored.",
                 rd.min_radius_au, _fmt_flux(rd.min_abs_residual, bunit)))
    put()
    feats = Tuple{Float64,String,Float64}[]
    if !isempty(rings) || !isempty(gaps)
        put(@sprintf("  %-5s %9s %8s %11s %16s %9s %9s", "type", "r [au]", "r [\"]", "r [beams]", "residual", "modul.", "signif."))
        for (kind, lst) in (("ring", rings), ("gap", gaps))
            for f in lst
                ci = argmin(abs.(rd.radius_au .- f.radius_au))
                base = rd.baseline[ci]
                modp = base > 0 ? 100 * f.residual / base : NaN      # n/a when the baseline is not positive
                sg = _stat_sigma(pr, f.radius_au)
                nsig = (isfinite(sg) && sg > 0) ? f.residual / sg : (rms > 0 ? f.residual / rms : NaN)
                (isfinite(nsig) && abs(nsig) >= 3 && (!isfinite(modp) || abs(modp) >= 10)) && (nfirm += 1)
                put(@sprintf("  %-5s %9.1f %8.3f %11.1f %12.3g %s %9s %+7.1f s", kind, f.radius_au, f.radius_au / pr.distance_pc,
                             f.radius_au / bm, fscale * f.residual, funit, isfinite(modp) ? @sprintf("%+.1f%%", modp) : "n/a", nsig))
                push!(feats, (f.radius_au, kind, nsig))
            end
        end
        if have_stat
            put("  (modulation = residual / local baseline; significance = residual / (image noise / sqrt(beams in the annulus)), 's' = sigma.")
            put("   Statistical only: it ignores CLEAN artifacts, baseline-filter ripples and non-axisymmetric structure.)")
        else
            put("  (modulation = residual / local baseline; significance = residual / scatter of the residual profile, 's' = sigma; image noise unavailable)")
        end
        sort!(feats; by = first)
        put()
        put("  Sequence from the centre: " * join([@sprintf("%s %.0f au", uppercase(first(k, 1)), r) for (r, k, _) in feats], "  ->  "))
        nstrong = count(f -> abs(f[3]) >= 3, feats)
        put(@sprintf("  %d of %d features are at least 3 sigma (%d of them also with modulation >= 10%%); weaker ones are tentative.", nstrong, length(feats), nfirm))
        for i in 2:length(feats)
            d = feats[i][1] - feats[i-1][1]
            isfinite(bm) && d < bm && put(@sprintf("  NOTE: %s at %.0f au and %s at %.0f au are %.1f au apart (< 1 beam): probably one blended feature or noise.",
                                                   feats[i-1][2], feats[i-1][1], feats[i][2], feats[i][1], d))
        end
        rr = sort([f.radius_au for f in rings]); gg = sort([f.radius_au for f in gaps])
        length(rr) >= 2 && put("  Ring spacing: " * join([@sprintf("%.0f au (x%.2f)", rr[i+1] - rr[i], rr[i+1] / rr[i]) for i in 1:length(rr)-1], ", "))
        length(gg) >= 2 && put("  Gap spacing : " * join([@sprintf("%.0f au (x%.2f)", gg[i+1] - gg[i], gg[i+1] / gg[i]) for i in 1:length(gg)-1], ", "))
    else
        put("  No significant rings or gaps were detected.")
    end
    put()

    # ------------------------------------------------------------ literature
    lit = hasproperty(out, :lit_features) ? out.lit_features : Tuple{Float64,String,Symbol}[]
    nrec = 0; nlit = 0; nrec_tent = 0; nlit_tent = 0; nweak = 0
    if !isempty(lit)
        put("6. COMPARISON WITH PUBLISHED FEATURES (radii rescaled to the adopted distance)")
        put("-"^88)
        tol = isfinite(bm) ? bm : 2.0
        put(@sprintf("  Tolerance = 1 beam = %.1f au; a feature counts as recovered if either the baseline-subtracted or the raw profile has one within it.", tol))
        for (r, label, kind) in sort(lit; by = first)
            rpool = kind == :ring ? rings : gaps
            xpool = kind == :ring ? out.rings : out.gaps
            kname = kind == :ring ? "ring" : "gap"
            if r < rd.min_radius_au
                nlit_tent += 1
                if !isempty(xpool)
                    gi = argmin([abs(g.radius_au - r) for g in xpool])
                    d = xpool[gi].radius_au - r
                    if abs(d) <= tol
                        nrec_tent += 1
                        put(@sprintf("  %-22s %7.1f au : inside the excluded zone; the raw profile has a %s at %.1f au (%+.1f au): consistent, not formally tested", label, r, kname, xpool[gi].radius_au, d))
                        continue
                    end
                end
                put(@sprintf("  %-22s %7.1f au : inside the excluded beam/CLEAN zone; cannot be tested", label, r))
                continue
            end
            nlit += 1
            best = nothing
            for (pool, tag) in ((rpool, "baseline-subtracted"), (xpool, "raw profile"))
                isempty(pool) && continue
                gi = argmin([abs(g.radius_au - r) for g in pool])
                d = pool[gi].radius_au - r
                if abs(d) <= tol && (best === nothing || abs(d) < abs(best[1]))
                    best = (d, pool[gi].radius_au, tag)
                end
            end
            if best !== nothing
                # a match on the baseline-subtracted profile only counts when that feature is itself at least 3 sigma
                nsg_m = NaN
                if best[3] == "baseline-subtracted"
                    sgn = kind == :ring ? 1.0 : -1.0
                    sigs = Float64[]
                    for j in findall(x -> abs(x - best[2]) <= 0.5 * tol, rd.radius_au)
                        sg = _stat_sigma(pr, rd.radius_au[j])
                        v = (isfinite(sg) && sg > 0) ? rd.residual[j] / sg : rd.residual[j] / rms
                        isfinite(v) && push!(sigs, sgn * v)
                    end
                    isempty(sigs) || (nsg_m = maximum(sigs))
                end
                if isfinite(nsg_m) && nsg_m < 3
                    nweak += 1
                    put(@sprintf("  %-22s %7.1f au : coincides with a %s at %.1f au (%+.1f au, %+.2f beams; %s) that is only %.1f sigma: consistent with noise, not counted as recovered",
                                 label, r, kname, best[2], best[1], best[1] / tol, best[3], nsg_m))
                    continue
                end
                nrec += 1
                put(@sprintf("  %-22s %7.1f au : RECOVERED, %s at %.1f au (%+.1f au, %+.2f beams; %s%s)", label, r, kname, best[2], best[1], best[1] / tol, best[3],
                             isfinite(nsg_m) ? @sprintf("; %.1f sigma", nsg_m) : ""))
                continue
            end
            ci = argmin(abs.(rd.radius_au .- r))
            sg = _stat_sigma(pr, rd.radius_au[ci])
            nsg = (isfinite(sg) && sg > 0) ? rd.residual[ci] / sg : rd.residual[ci] / rms
            put(@sprintf("  %-22s %7.1f au : not detected (residual %+.3g %s = %+.1f sigma at r = %.1f au)",
                         label, r, fscale * rd.residual[ci], funit, nsg, rd.radius_au[ci]))
        end
        put(@sprintf("  Recovered %d of %d testable published features%s%s.", nrec, nlit,
                     nweak > 0 ? @sprintf("; %d more coincide with features below 3 sigma (noise level)", nweak) : "",
                     nlit_tent > 0 ? @sprintf("; %d of %d inside the excluded zone also match the raw profile", nrec_tent, nlit_tent) : ""))
        put()
    else
        put("6. COMPARISON WITH PUBLISHED FEATURES")
        put("-"^88)
        put("  No published ring/gap radii are stored for this target in the catalogue.")
        put()
    end

    # -------------------------------------------------------- interpretation
    put("7. ANALYSIS")
    put("-"^88)
    n = 0
    bullet(s) = (n += 1; put(@sprintf("  (%d) %s", n, s)))
    dsig = _detection_sigma(pr)
    detected = isfinite(dsig) ? dsig >= 5 : !(isfinite(snr) && snr < 5)
    if !detected
        bullet(@sprintf("Detection: the strongest bin of the azimuthal mean is only %.1f sigma (peak %.1f x the per-beam image noise), so this target is not detected in this image (or is far fainter than the noise). The profile and any rings or gaps listed above are noise; try a deeper or different observation.", dsig, snr))
    end
    let ch = hasproperty(out, :chosen) ? out.chosen : nothing
        fghz = (ch !== nothing && hasproperty(ch, :freq_ghz)) ? ch.freq_ghz : NaN
        fhdr = hasproperty(out, :local_path) ? _image_freq_ghz(out.local_path) : NaN     # the image's own frequency, not the observation's median
        isfinite(fhdr) && (fghz = fhdr)
        frt = flux_recovery_text(flux_recovery(t, totf, fghz))
        isempty(frt) || bullet(frt)
        if ch !== nothing && hasproperty(ch, :las_arcsec) && isfinite(ch.las_arcsec)
            dinf = _disc_info(t)
            r68c = (dinf !== nothing && isfinite(dinf.r68_au)) ? dinf.r68_au : _disk_extent_au(pr; frac = 0.68)
            diam = 2 * r68c / pr.distance_pc
            if isfinite(diam) && ch.las_arcsec < 0.5 * diam
                bullet(@sprintf("Spatial filtering: this observation recovers angular scales up to %.2f arcsec (%.0f au), less than half of the disc's 68%%-flux diameter (%.2f arcsec = %.0f au). The extended emission is partly or wholly resolved out; choose an observation with a larger recoverable scale (obs_choice = N, a coarser configuration) before trusting sizes, contrasts or asymmetries.",
                                ch.las_arcsec, ch.las_arcsec * pr.distance_pc, diam, 2 * r68c))
            end
        end
    end
    if detected
    nb90 = r90 / bm
    if !isfinite(nb90)
        bullet("The beam size is not in the FITS header, so the resolution of the disk cannot be judged.")
    elseif nb90 < 1.5
        bullet(@sprintf("Resolution: 90%% of the flux lies within %.1f beams (%.1f au), so the source is unresolved (a point source). If the target is a protoplanetary disk this is probably not an image of it: it may be a calibrator or an unresolved companion. Check the FITS file name and header OBJECT; no ring or gap analysis is meaningful.", nb90, r90))
    elseif nb90 < 5
        bullet(@sprintf("Resolution: the disk (90%% of the flux within %.0f au) spans only %.1f beams of %.1f au. It is barely resolved: only the overall size and brightness trend can be trusted; rings or gaps narrower than ~%.0f au would not be seen.", r90, nb90, bm, 2 * bm))
    elseif nb90 < 10
        bullet(@sprintf("Resolution: the disk (90%% of the flux within %.0f au) spans %.1f beams of %.1f au. It is moderately resolved: only features wider than about %.0f au (2 beams) are reliable.", r90, nb90, bm, 2 * bm))
    else
        bullet(@sprintf("Resolution: the disk (90%% of the flux within %.0f au) spans %.1f beams of %.1f au, well resolved; features down to ~%.0f au (2 beams) can be recovered.", r90, nb90, bm, 2 * bm))
    end
    mr = rd.min_radius_au
    if isfinite(mr) && isfinite(bm) && (rpk < mr || mr > 0.4 * r90)
        what = rpk < mr ? @sprintf("the flux peak (%.0f au) and a large part of the disk", rpk) : "a large part of the disk"
        bullet(@sprintf("Excluded zone: detections inside %.0f au (%.1f beams) are dropped as beam/CLEAN artifacts. That zone holds %s (90%% of the flux lies within %.0f au), so structure there is not reported. Run with min_radius_beams = 1.0 to list candidates in that zone; treat them as tentative.", mr, mr / bm, what, r90))
    end
    if hasproperty(out, :used_choice) && out.used_choice != out.obs_choice && hasproperty(out, :observations)
        fin = out.observations.best_res_arcsec[min(out.obs_choice, nrow(out.observations))]
        bullet(@sprintf("Data choice: a package with finer resolution (%.4f arcsec) exists but exceeded the %.0f GB download limit, so a coarser observation was used. Finer substructure may exist.", fin, MAX_TAR_GB[]))
    end
    if src == :assumed_face_on
        if hasproperty(out, :settings) && out.settings.geometry_mode !== :given
            bullet("Geometry: the disk is too small compared with the beam for the image fit to measure its inclination and PA, so it was treated as face-on. If it is inclined, the azimuthal average mixes different radii and washes out rings; give incl_deg / pa_deg if you know them.")
        else
            bullet("Geometry: inclination and PA are unknown and the disk was treated as face-on. If it is inclined, the azimuthal average mixes different radii and washes out rings; use geometry_mode = :auto or give incl_deg / pa_deg.")
        end
    elseif src == :fitted
        bullet(@sprintf("Geometry: the image gives i = %.0f deg, PA = %.0f deg (uncertain by ~5 deg, a little more for disks that are barely resolved or near face-on). A wrong geometry would blur rings; check the deprojected profile for sharpness.", pr.incl_deg, pr.pa_deg))
    end
    if src == :fitted && hasproperty(pr, :geom_r2) && isfinite(pr.geom_r2) && pr.geom_r2 < 0.6
        bullet(@sprintf("Geometry quality: a purely radial profile explains only %.0f%% of the pixel variance at the fitted geometry, so the disk is probably non-axisymmetric (spirals, arcs, a dust trap) or faint and the fitted i and PA are unreliable. Give incl_deg / pa_deg from the literature if known.", 100 * pr.geom_r2))
    end
    if pr.incl_deg >= 70
        bullet("The disk is highly inclined (>= 70 deg); deprojection is stretched by 1/cos(i) and radial structure is poorly constrained.")
    end
    central = rpk <= max(1.5 * (isfinite(bm) ? bm : 0.0), 2 * (pr.radius_au[2] - pr.radius_au[1]))
    if central
        bullet(@sprintf("Shape: the emission is centrally peaked (the maximum lies at %.0f au, within ~1.5 beams of the star), like TW Hya: a smooth, steeply falling profile on which any rings are shoulders and dips.", rpk))
    elseif isfinite(rpk / bm) && rpk / bm < 2.5
        bullet(@sprintf("Shape: the azimuthal mean peaks at %.0f au (%.1f beams) and the innermost bin holds %.0f%% of the peak. A peak this close to the centre, with a dip of this size, is at the resolution limit: it is consistent with a centrally peaked disk smoothed by the %.0f au beam, and a cavity smaller than ~%.0f au cannot be established.", rpk, rpk / bm, 100 * inner / peak, bm, 2 * bm))
    elseif inner / peak < 0.7
        bullet(@sprintf("Shape: the profile is ring-like, not centrally peaked. The maximum is at %.0f au and the innermost bin has only %.0f%% of it, which points to a central cavity: the mean rises to half its peak at ~%.0f au.%s", rpk, 100 * inner / peak, _radius_rising(pr, 0.5),
                        inner < 0 ? " Negative central values are an artifact of missing short baselines (spatial filtering), not a real flux deficit." : ""))
    else
        bullet(@sprintf("Shape: the peak lies off-centre (%.0f au, %.1f beams) but the inner profile is not strongly depleted (%.0f%% of the peak); this may be a bright inner ring on a filled disk.", rpk, rpk / bm, 100 * inner / peak))
    end
    if isempty(rings) && isempty(gaps)
        bullet("Substructure: none was detected above the threshold. This says the profile is smooth at this resolution and sensitivity, not that the disk is smooth (rings narrower than the beam are averaged away). Re-tune with prominence_sigma = 0.6, min_abs_residual_frac = 0.1 to see weaker candidates.")
    else
        if count(f -> abs(f[3]) >= 3, feats) == 0
            bullet(@sprintf("Substructure: %d candidate ring(s)/gap(s) were found after baseline subtraction, but none reaches 3 sigma (the strongest is %.1f sigma), so they are consistent with noise and are not established. Candidate radii: %s.",
                            length(feats), maximum(filter(isfinite, [abs(f[3]) for f in feats]); init = 0.0),
                            join([@sprintf("%.0f au", f[1]) for f in feats], ", ")))
        else
            bullet(@sprintf("Substructure: %d ring(s) and %d gap(s) after baseline subtraction, with %d of %d at >= 3 sigma (%d also with modulation >= 10%%). Detected radii: %s.",
                            length(rings), length(gaps), count(f -> abs(f[3]) >= 3, feats), length(feats), nfirm,
                            join([@sprintf("%.0f au", f[1]) for f in feats], ", ")))
        end
        nshoulder = count(f -> !any(g -> abs(g.radius_au - f.radius_au) <= (isfinite(bm) ? bm : 2.0), vcat(out.rings, out.gaps)),
                          vcat(rings, gaps))
        nshoulder > 0 && bullet(@sprintf("%d of the %d baseline-subtracted features have no peak or dip within 1 beam in the raw profile: they are shoulders or inflections that only appear after the smooth baseline is removed, not isolated rings or gaps. Their modulation (residual / baseline) can be huge when the baseline is near zero.",
                                         nshoulder, length(rings) + length(gaps)))
    end
    end      # detected
    if nlit > 0
        bullet(@sprintf("Published features: %d of %d testable published rings/gaps were recovered within 1 beam (a baseline-subtracted match must be at least 3 sigma%s). Misses usually mean the published feature is narrower or shallower than this beam and sensitivity can show.",
                        nrec, nlit, nweak > 0 ? @sprintf("; %d more coincide with features below 3 sigma", nweak) : ""))
    end
    isempty(cav) || bullet("Class-specific note: " * cav)
    bullet("Detection method: the profile is an azimuthal mean over the deprojected disk; features are local maxima and minima of the residual after a Savitzky-Golay baseline is subtracted. Any structure that is not axisymmetric (spirals, arcs, companions) is averaged out and does not appear here.")
    put()

    # --------------------------------------------------------------- figures
    put("8. FIGURES IN THIS FOLDER")
    put("-"^88)
    figs = (("_image.png", "continuum image (asinh stretch, sky offsets, beam on the image)"),
            ("_radial_profile.png", "azimuthal mean flux versus deprojected radius, with the raw peak-finder detections"),
            ("_substructure.png", "baseline-subtracted profile: detected rings (yellow) and gaps (blue), literature features"),
            ("_substructure_retuned.png", "the same with the re-tuned detection settings"))
    for (suffix, desc) in figs
        isfile(joinpath(OUTPUT_DIR, t.slug * suffix)) && put("  " * t.slug * suffix * " : " * desc)
    end
    put()
    put("This summary is generated automatically from the measurements and the target catalogue; catalogue values")
    put("(distance, inclination, PA, literature gap radii) are approximate and should be verified before publication.")

    return _write_summary_file(L, t, filename)
end

function _write_summary_file(L::Vector{String}, t, filename)
    name = filename === nothing ? "$(t.slug)_target_summary.txt" : String(filename)
    path = joinpath(OUTPUT_DIR, name)
    open(path, "w") do io
        for l in L
            println(io, l)
        end
    end
    println("Saved target summary: ", path)
    return path
end


## Pipeline

In [ ]:
# ---------------------------------------------------------------------------
# Full pipeline for one target
# ---------------------------------------------------------------------------

"""
    run_alma_pipeline(target::AlmaTarget; kwargs...) -> NamedTuple

Query -> choose observation -> download product -> display -> radial profile ->
ring/gap detection -> literature comparison, for any `AlmaTarget`.

Selection / download
  fits_file      path to a continuum image you already have (e.g. a DSHARP release image): the archive is not queried and the
                 rest of the analysis (catalogue geometry, distance, published features) is used as for any other target
  proposal_id    restrict to one project (default: the catalogue's, usually none)
  band           e.g. 7 to keep only Band 7 observations
  obs_choice     row of the observation table to use (1 = finest resolution)
  public_only    only public data (default true)
  name_filter    match target_name against the target's aliases (default true)
  radius_deg     cone radius for the position query
  max_tar_gb     refuse tarballs larger than this unless force_download=true
  allow_object_mismatch  false (default): an image whose FITS OBJECT names another source (e.g. DG Tau B for MWC 480) is not used
                 while other observations are left to try; true uses it anyway (a warning is always printed)
  allow_cube     false (default): line-emission and spectral-cube FITS files (target_co32b7..., *.cube.*, any image with
                 more than one channel) are ignored and the next observation is tried when a package holds only those;
                 true keeps them (only the first plane is used, which is not a continuum image)
  auto_fallback  when an observation's package is too large (or holds no FITS images), try the next
                 observations in the table (up to max_fallback of them); refused packages are
                 listed by `print_too_large()`
Analysis (defaults come from the catalogue entry)
  distance_pc, incl_deg, pa_deg, max_radius_arcsec, n_bins
  geometry_mode  :auto (default) fits inclination/PA from the image when the catalogue lacks them;
                 :fit always fits (and compares with the catalogue); :given uses the values as they are
                 (face-on if unknown)
  min_radius_beams  detections closer than this many beams to the centre are dropped as artifacts (2.5)
  center / center_mode   see `extract_radial_profile`
  (a plain-text <target>_target_summary.txt is written next to the PNGs; see `write_target_summary`)
  prominence_sigma       ring/gap prominence threshold, in residual std (0.7)
  cutout_factor          image cutout = cutout_factor x max_radius (default 2.4)
"""
function run_alma_pipeline(target::AlmaTarget;
        proposal_id = target.proposal_id, band = nothing, obs_choice::Integer = 1,
        public_only::Bool = true, name_filter::Union{Nothing,Bool} = nothing, radius_deg::Real = 0.05,
        max_tar_gb::Real = MAX_TAR_GB[], force_download::Bool = false,
        auto_fallback::Bool = true, max_fallback::Int = 12, allow_cube::Bool = false,
        allow_object_mismatch::Bool = false, fits_file = nothing,
        geometry_mode::Symbol = :auto, min_radius_beams::Real = 2.5,
        distance_pc = target.distance_pc, incl_deg = target.incl_deg, pa_deg = target.pa_deg,
        max_radius_arcsec = target.rmax_arcsec, n_bins = nothing,
        center = nothing, center_mode::Symbol = :optimize,
        prominence_sigma::Real = 0.7, cutout_factor::Real = 2.4,
        asinh_a::Real = 0.01, vmax_percentile::Real = 99.5)

    target = _ensure_coords(target)
    # catalogue-only entries (name_match = false) carry a catalogue name, not an archive target_name: no name filtering
    name_filter = something(name_filter, target.name_match)
    allow_object_mismatch = allow_object_mismatch || !target.name_match
    println("="^80)
    @printf("Target: %s   (RA %.4f, Dec %+.4f; %s)\n", target.name, target.ra_deg, target.dec_deg, target.note)
    println("="^80)
    println("ALMA archive web portal: ", archive_url(target))
    target.distance_known || println("WARNING: no distance given; radii in au assume 100 pc (arcsec values are unaffected).")
    geometry_mode in (:auto, :fit, :given) || error("geometry_mode must be :auto, :fit or :given")
    geom_known = incl_deg !== nothing && pa_deg !== nothing
    fit_geom = geometry_mode === :fit || (geometry_mode === :auto && !geom_known)
    cat_incl, cat_pa = incl_deg, pa_deg
    if fit_geom
        println(geom_known ? "Disk geometry will be re-measured from the image (geometry_mode = :fit); the given values are only a reference." :
                             "Inclination/PA are not in the catalogue: they will be fitted from the image (geometry_mode = :auto).")
    elseif !geom_known
        println("NOTE: inclination/PA unknown -> treated as face-on (no deprojection). Pass incl_deg= and pa_deg=, or use geometry_mode = :auto.")
    end
    incl_deg = something(incl_deg, 0.0)
    pa_deg = something(pa_deg, 0.0)

    results = DataFrame(); obs = nothing; data_info = nothing; tarball_url = nothing; chosen = nothing
    extracted = String[]; used_choice = obs_choice; first_res = NaN
    if fits_file !== nothing
        isfile(fits_file) || error("fits_file not found: $(fits_file)")
        println("Using the local image ", fits_file, " (no archive query).")
        extracted = String[String(fits_file)]
    else
    results = query_target(target; public_only = public_only, proposal_id = proposal_id,
                           name_filter = name_filter, radius_deg = radius_deg)
    nrow(results) == 0 && return (target = target, results = results)
    if !target.name_match && hasproperty(results, :target_name)         # position-only query: check the names found
        seen = unique(string.(results.target_name))
        if !any(v -> any(k -> occursin(k, _norm(v)), target.keys), seen)
            println("WARNING: none of the archive target names in this cone (", join(first(seen, 6), ", "), length(seen) > 6 ? ", ..." : "",
                    ") matches the catalogue names ", target.keys, ". The catalogue position may belong to another star: check before trusting the result.")
        end
    end

    obs = summarize_observations(results)
    try
        # largest recoverable scale / the disc's 68%-flux diameter (catalogue R68): below ~0.5 the extended emission is resolved out
        di_t = _disc_info(target)
        if di_t !== nothing && isfinite(di_t.r68_au) && distance_pc > 0 && "las_arcsec" in names(obs)
            obs.las_over_diam = round.(obs.las_arcsec ./ (2 * di_t.r68_au / distance_pc); digits = 2)
        end
    catch
    end
    try
        _print_disc_hint(target, obs, distance_pc)
    catch
    end
    chosen = nothing; data_info = nothing; tarball_url = nothing
    extracted = String[]
    used_choice = obs_choice; first_res = NaN
    nsel = band === nothing ? nrow(obs) : count(b -> occursin(string(band), b), obs.band)
    last_choice = auto_fallback ? min(nsel, obs_choice + max_fallback) : obs_choice
    for choice in obs_choice:max(obs_choice, last_choice)
        more = auto_fallback && choice < last_choice          # another observation is left to try
        chosen = select_observation(obs; band = band, choice = choice, quiet = choice > obs_choice)
        chosen === nothing && return (target = target, results = results, observations = obs)
        choice == obs_choice && (first_res = Float64(chosen.best_res_arcsec))

        data_info = inspect_products(String(chosen.member_ous_uid); verbose = choice == obs_choice)
        tarball_url = pick_product_tarball(data_info)
        if tarball_url === nothing
            more && (println("No product tarball for this observation; trying the next one."); continue)
            return (target = target, results = results, observations = obs, data_info = data_info)
        end

        size_gb = tarball_size_gb(data_info, tarball_url)
        extracted = String[]
        used_direct = false
        direct_urls = (size_gb !== nothing && size_gb > max_tar_gb && !force_download) ?
                      direct_fits_urls(data_info, target; max_gb = max_tar_gb) : String[]
        if !isempty(direct_urls)
            println("\nThe full package is ", round(size_gb; digits = 1), " GB (above max_tar_gb = ", max_tar_gb,
                    "), but the archive lists ", length(direct_urls), " separate continuum image(s) for this target; downloading those.")
            used_direct = true
            extracted = download_direct_fits(direct_urls)
        elseif size_gb !== nothing && size_gb > max_tar_gb && !force_download
            _log_too_large(target, chosen, size_gb)
            @printf("\nSKIP: %s is %.1f GB, above max_tar_gb = %.1f.\n", basename(tarball_url), size_gb, max_tar_gb)
            if more
                println("Trying the next observation (auto_fallback = true) ...")
                continue
            end
            println("STOP: no smaller package found. Re-run with force_download=true, raise max_tar_gb, or pick an " *
                    "observation with obs_choice=N / band=N. print_too_large() lists the refused packages.")
            return (target = target, results = results, observations = obs, data_info = data_info, tarball_url = tarball_url)
        end

        used_direct || (extracted = download_and_extract_fits(tarball_url))
        if isempty(extracted) && !isfile(joinpath(CACHE_DIR, basename(tarball_url)))
            # download failed (all mirrors) or the package holds no FITS images
            if more
                println("Nothing usable from this package; trying the next observation ...")
                continue
            end
        end
        if !allow_cube && !isempty(extracted)
            cont = filter(f -> !_is_line_or_cube_file(f), extracted)
            if isempty(cont)
                println("This package holds only line or spectral-cube products (e.g. ", basename(first(extracted)),
                        "), no continuum image.")
                extracted = String[]
                more && println("Trying the next observation ...")
                continue
            end
            length(cont) < length(extracted) &&
                println("Ignoring $(length(extracted) - length(cont)) line / cube product(s) in this package.")
            extracted = cont
        end
        if !allow_object_mismatch && !isempty(extracted) && more && !any(f -> _object_matches(f, target.keys), extracted)
            println("None of the images in this package is labelled as ", target.name, " (FITS OBJECT: ",
                    join(unique(filter(!isempty, _fits_object.(extracted))), ", "), "); trying the next observation ...")
            extracted = String[]
            continue
        end
        if !isempty(extracted) && all(_is_calibrator_file, extracted)
            println("This package holds only calibrator images (e.g. ", basename(first(extracted)), "), none of the target.")
            extracted = String[]
            more && println("Trying the next observation ...")
            continue
        end
        isempty(extracted) || (used_choice = choice; break)
        println("No FITS images in this package (it may hold only measurement sets).")
        more && println("Trying the next observation ...")
    end
    end   # fits_file === nothing
    if isempty(extracted)
        println("No usable FITS images found (packages with only line cubes, calibrators or measurement sets are skipped; allow_cube = true keeps cubes). Try another observation with obs_choice=N, or the external_ari_l package.")
        return (target = target, results = results, observations = obs, data_info = data_info, tarball_url = tarball_url)
    end
    if used_choice != obs_choice && isfinite(first_res)
        res_used = Float64(chosen.best_res_arcsec)
        @printf("NOTE: fell back to observation #%d (%.3f\") because finer ones exceeded the size limit; that is %.1fx coarser than #%d (%.3f\"). Raise MAX_TAR_GB[] or use force_download = true for the finer one.\n",
                used_choice, res_used, res_used / first_res, obs_choice, first_res)
    end
    candidates = rank_fits_files(extracted; keys = target.keys)
    println("\nExtracted $(length(candidates)) FITS file(s), best candidate first:")
    foreach(f -> println("  ", f), candidates)
    local_path = candidates[1]
    _fits_score(local_path) >= 20 &&
        println("WARNING: the best candidate looks like a calibrator image (J....-.... / _bp / _phase), not the target.")
    10 <= _fits_score(local_path) < 20 &&
        println("WARNING: the best candidate looks like a PSF/model/cube-type product, not a continuum image.")
    println("\nUsing: ", local_path)

    img = load_fits_image(local_path)
    if target.name_match && hasproperty(img, :object) && !isempty(img.object)
        on = _norm(img.object)
        any(k -> occursin(k, on) || occursin(on, k), target.keys) ||
            println("WARNING: the FITS header OBJECT is '", img.object, "', which does not match the target '", target.name,
                    "'. This may be a calibrator or another source; try another observation (obs_choice = N).")
    end
    label = target.name
    profile = rings = gaps = rings_r = gaps_r = residual_data = nothing
    lit_features = literature_features_au(target, distance_pc)
    geometry_source = geom_known ? :catalogue : :assumed_face_on
    try
        profile = extract_radial_profile(img; incl_deg = incl_deg, pa_deg = pa_deg, distance_pc = distance_pc,
                                         max_radius_arcsec = max_radius_arcsec, n_bins = n_bins,
                                         center = center, center_mode = center_mode, fit_geom = fit_geom)
        if fit_geom && geom_known && profile.incl_deg !== nothing
            @printf("Geometry check: catalogue i = %.1f, PA = %.1f deg; fitted from the image i = %.1f, PA = %.1f deg.\n",
                    cat_incl, cat_pa, profile.incl_deg, profile.pa_deg)
        end
        if fit_geom && (profile.incl_deg != incl_deg || profile.pa_deg != pa_deg)
            geometry_source = :fitted
        end
        profile.bmaj_au !== nothing && profile.bmaj_au > BEAM_RESOLVED_LIMIT_AU &&
            @printf("WARNING: the beam is %.0f au across (> %.0f au): a typical disk (rings at tens of au) is NOT resolved in this image, so rings, gaps and asymmetries cannot be measured. Try a finer observation (obs_choice = N).\n",
                    profile.bmaj_au, BEAM_RESOLVED_LIMIT_AU)
        r90 = _disk_extent_au(profile)
        if profile.bmaj_au !== nothing && isfinite(r90)
            nb90 = r90 / profile.bmaj_au
            @printf("Disk extent (90%% of the flux): ~%.0f au = %.1f beams of %.1f au.\n", r90, nb90, profile.bmaj_au)
            nb90 < 1.5 && @printf("WARNING: 90%% of the flux lies within %.1f beams: the source is unresolved (a point source). If the target is a disk this is probably the wrong image (a calibrator?); check the FITS name and header OBJECT above.\n", nb90)
            nb90 < 8 && @printf("NOTE: the disk spans only ~%.1f beams, so rings/gaps narrower than ~%.0f au (2 beams) are not resolved. A finer-resolution observation would help (obs_choice = N; force_download = true for large packages).\n", nb90, 2 * profile.bmaj_au)
            min_radius_beams * profile.bmaj_au > 0.4 * r90 &&
                @printf("WARNING: the beam-artifact exclusion zone (< %.0f au) covers much of this disk (90%% flux radius %.0f au); detections inside it are dropped. Try min_radius_beams = 1.0.\n", min_radius_beams * profile.bmaj_au, r90)
        end

        display_fits_image(img; label = label, center = profile.center_pix,
                           cutout_arcsec = cutout_factor * max_radius_arcsec,
                           asinh_a = asinh_a, vmax_percentile = vmax_percentile,
                           filename = "$(target.slug)_image.png")

        rings, gaps = find_rings_and_gaps(profile)
        println("\n[raw profile] Detected $(length(rings)) ring(s), $(length(gaps)) gap(s) with plain find_peaks.")
        plot_radial_profile(profile; rings = rings, gaps = gaps,
                            title = "$(label) radial profile — $(basename(local_path))",
                            filename = "$(target.slug)_radial_profile.png")

        # Rings/gaps usually appear only as shoulders on a steep decline, so
        # fit and subtract a smooth baseline and search the residual instead.
        rings_r, gaps_r, residual_data = find_rings_and_gaps_residual(profile; prominence_sigma = prominence_sigma,
                                                                min_radius_beams = min_radius_beams)
        println("\n[baseline-subtracted] Detected $(length(rings_r)) ring(s):")
        for r in rings_r
            @printf("  r = %.1f au, residual = %.4e %s, prominence = %.2e\n", r.radius_au, r.residual, profile.bunit, r.prominence)
        end
        println("\n[baseline-subtracted] Detected $(length(gaps_r)) gap(s):")
        for g in gaps_r
            @printf("  r = %.1f au, residual = %.4e %s, prominence = %.2e\n", g.radius_au, g.residual, profile.bunit, g.prominence)
        end

        compare_with_literature(lit_features, gaps_r, residual_data; bmaj_au = profile.bmaj_au, rings = rings_r)

        plot_residual_profile(residual_data; rings = rings_r, gaps = gaps_r, lit_features = lit_features,
                              title = "$(label) continuum substructure — $(basename(local_path))",
                              filename = "$(target.slug)_substructure.png")
    catch err
        println("ERROR: analysis raised an exception:")
        showerror(stdout, err, catch_backtrace())
        println()
    end

    result = (target = target, results = results, observations = obs, chosen = chosen, data_info = data_info,
              tarball_url = tarball_url, fits_files = candidates, local_path = local_path,
              profile = profile, rings = rings, gaps = gaps,
              rings_residual = rings_r, gaps_residual = gaps_r, residual_data = residual_data,
              lit_features = lit_features,
              used_choice = used_choice, obs_choice = obs_choice, geometry_source = geometry_source,
              cat_incl = cat_incl, cat_pa = cat_pa,
              size_gb = (tarball_url === nothing || data_info === nothing) ? nothing : tarball_size_gb(data_info, tarball_url),
              settings = (geometry_mode = geometry_mode, min_radius_beams = min_radius_beams,
                          prominence_sigma = prominence_sigma, band = band, obs_choice = obs_choice,
                          center_mode = center_mode, max_radius_arcsec = max_radius_arcsec,
                          n_bins = n_bins, public_only = public_only))
    if SAVE_PLOTS[] && profile !== nothing && residual_data !== nothing
        try
            write_target_summary(result)
        catch err
            println("WARNING: could not write the target summary: ", err)
        end
    end
    return result
end

## Physical analysis (ring fits, geometry, flaring test, azimuthal asymmetry, round-beam test, brightness temperature)

Called as `phys = physical_analysis(out)` after a pipeline run (see the cell after *re-tune*). Cell order: ring fits → asymmetry (beam expectation, thickness scan) → ring-ridge geometry → ring-centre / core flaring test → round-beam test (smoothing to circular beams) → gap-opening planet masses → dust trapping (alpha / St) → driver and report. The flaring, round-beam and thickness diagnostics are opt-in (`do_flare`, `do_roundbeam`, `do_thickness`, all false by default).

In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (1/3): Gaussian ring fits
# ---------------------------------------------------------------------------
# Model of the azimuthally averaged profile I(r):
#     I(r) = tapered power law (smooth disk) + sum_k A_k exp(-(r - r_k)^2 / (2 s_k^2))
# fitted jointly with Levenberg-Marquardt (own implementation, no extra packages).
# s_k is the *observed* width: it contains the radial beam smearing, which is removed
# in quadrature to give the intrinsic width. Uncertainties come from the covariance
# matrix, scaled by sqrt(chi2_red) when chi2_red > 1 (so they are not under-stated when
# the Gaussians do not describe the profile perfectly).

const _FWHM_PER_SIGMA = 2 * sqrt(2 * log(2))

# Lynden-Bell & Pringle tapered power law; r is softened by the beam sigma `sb` so that r -> 0 stays finite.
function _lbp(r::Real, A::Real, rc::Real, gam::Real, sb::Real)
    x = sqrt(r * r + sb * sb) / rc
    return A * x^(-gam) * exp(-x^(2 - gam))
end

# Parameter vector: [A_bg, rc, gamma, (A_k, r_k, s_k) for each ring k]
function _ring_model(p::AbstractVector{<:Real}, r::AbstractVector{<:Real}, sb::Real)
    K = (length(p) - 3) ÷ 3
    m = Float64[_lbp(ri, p[1], p[2], p[3], sb) for ri in r]
    for k in 1:K
        A = p[4 + 3 * (k - 1)]
        r0 = p[5 + 3 * (k - 1)]
        s = p[6 + 3 * (k - 1)]
        for i in eachindex(r)
            m[i] += A * exp(-0.5 * ((r[i] - r0) / s)^2)
        end
    end
    return m
end

# d(model)/d(parameter) / sigma, by central differences
function _ring_jac(p::AbstractVector{<:Real}, r::AbstractVector{<:Real}, sig::AbstractVector{<:Real}, sb::Real)
    np = length(p)
    J = zeros(length(r), np)
    for j in 1:np
        h = 1e-6 * max(abs(p[j]), 1e-3)
        qp = collect(Float64, p);  qp[j] += h
        qm = collect(Float64, p);  qm[j] -= h
        J[:, j] = (_ring_model(qp, r, sb) .- _ring_model(qm, r, sb)) ./ (2 * h .* sig)
    end
    return J
end

# Bounded Levenberg-Marquardt (parameters are clamped to [lo, hi] after every step).
function _lm_fit(p0, lo, hi, r, y, sig, sb; maxit::Int = 300)
    p = clamp.(collect(Float64, p0), lo, hi)
    f = (y .- _ring_model(p, r, sb)) ./ sig
    c = dot(f, f)
    lam = 1e-3
    for _ in 1:maxit
        J = _ring_jac(p, r, sig, sb)
        A = J' * J
        gv = J' * f
        improved = false
        converged = false
        for _ in 1:15
            Aa = A + lam * Diagonal(diag(A) .+ 1e-12)
            dp = try
                Aa \ gv
            catch
                nothing
            end
            if dp === nothing
                lam *= 10
                continue
            end
            pn = clamp.(p .+ dp, lo, hi)
            fn = (y .- _ring_model(pn, r, sb)) ./ sig
            cn = dot(fn, fn)
            if isfinite(cn) && cn < c
                converged = abs(c - cn) < 1e-10 * max(c, 1.0)
                p = pn
                f = fn
                c = cn
                lam = max(lam / 3, 1e-10)
                improved = true
                break
            end
            lam *= 4
        end
        (improved && !converged) || break
    end
    return p, c
end

# independent beams in each annulus (counts are pixel counts; falls back to the annulus area)
function _profile_nbeam(profile)
    bp = (π / (4 * log(2))) * profile.bmaj_arcsec * profile.bmin_arcsec / profile.pix_scale_arcsec^2
    if hasproperty(profile, :pix_counts)
        return max.(Float64.(profile.pix_counts) ./ bp, 1.0)
    end
    dr = profile.radius_arcsec[2] - profile.radius_arcsec[1]
    ci = cosd(profile.incl_deg === nothing ? 0.0 : profile.incl_deg)
    area_pix = 2π .* profile.radius_arcsec .* dr .* ci ./ profile.pix_scale_arcsec^2
    return max.(area_pix ./ bp, 1.0)
end

# parameter bounds: ring centres within one beam of their anchor, widths between 0.8 beam-sigma and 0.35 r + 2 beam-sigma
function _ring_bounds(anchors, bmaj, sb, ymax, rc_hi)
    K = length(anchors)
    lo = vcat([0.0, 5.0, -0.5], reduce(vcat, [[0.0, anchors[k] - bmaj, 0.8 * sb] for k in 1:K]; init = Float64[]))
    hi = vcat([10 * ymax, rc_hi, 1.9],
              reduce(vcat, [[3 * ymax, anchors[k] + bmaj, 0.35 * anchors[k] + 2 * sb] for k in 1:K]; init = Float64[]))
    return lo, hi
end

"""
    fit_gaussian_rings(profile, seeds; frac_floor = 0.02, max_add = 3, verbose = true) -> NamedTuple

Joint fit of a smooth tapered-power-law disk plus one Gaussian per seed ring.
`seeds` is a vector of `(radius_au, origin::Symbol)`; origins `:detected`, `:user`,
`:literature` decide which seed survives when two lie within 0.6 beams.
Per-bin errors are image_rms / sqrt(independent beams) plus a `frac_floor` fraction of the
local flux (CLEAN / primary-beam ripple), which keeps the bright core from dominating.
After each fit the strongest unexplained residual (> 5 sigma, > 2% of the peak, farther than
0.75 beams from every ring) is added as a new ring, up to `max_add` times.
"""
function fit_gaussian_rings(profile, seeds; frac_floor::Real = 0.02, max_add::Int = 3, verbose::Bool = true)
    (profile.bmaj_au === nothing || profile.bmin_au === nothing) && error("no beam in the FITS header")
    bmaj = Float64(profile.bmaj_au)
    bmin = Float64(profile.bmin_au)
    sb = sqrt(bmaj * bmin) / _FWHM_PER_SIGMA
    r_all = Float64.(profile.radius_au)
    y_all = Float64.(profile.mean_flux)
    nb_all = _profile_nbeam(profile)
    ok = findall(isfinite, y_all)
    length(ok) < 12 && error("fewer than 12 valid profile bins")
    r = r_all[ok]
    y = y_all[ok]
    nbeam = nb_all[ok]
    ymax = maximum(y)
    ymax > 0 || error("the profile has no positive emission")
    rms = (isfinite(profile.image_rms) && profile.image_rms > 0) ? Float64(profile.image_rms) : 0.01 * ymax
    sig = sqrt.((rms ./ sqrt.(nbeam)) .^ 2 .+ (frac_floor .* abs.(y)) .^ 2)
    rmax = r[end]

    # clean the seed list: inside the profile, no two within 0.6 beams (detections win)
    prio = Dict(:detected => 1, :user => 2, :literature => 3)
    cand = sort([(Float64(s[1]), Symbol(s[2])) for s in seeds if r[1] <= s[1] <= r[end]];
                by = s -> (get(prio, s[2], 9), s[1]))
    kept = Tuple{Float64,Symbol}[]
    for s in cand
        any(k -> abs(k[1] - s[1]) <= (s[2] == :literature ? 1.0 : 0.6) * bmaj, kept) || push!(kept, s)
    end
    sort!(kept; by = first)
    maxk = max(length(r) ÷ 5, 1)
    length(kept) > maxk && (kept = kept[1:maxk])

    # smooth background first (no rings), from a small grid of starting points
    rc_hi = max(500.0, 3 * rmax)
    best_p = [ymax, 0.5 * rmax, 0.8]
    best_c = Inf
    for rc0 in (0.15, 0.3, 0.6, 1.2) .* rmax, g0 in (0.0, 0.8, 1.5)
        pp, cc = _lm_fit([ymax, rc0, g0], [0.0, 5.0, -0.5], [10 * ymax, rc_hi, 1.9], r, y, sig, sb)
        if cc < best_c
            best_c = cc
            best_p = pp
        end
    end

    bg = copy(best_p)
    comps = Vector{Vector{Float64}}()
    anchors = Float64[]
    origins = Symbol[]
    for s in kept
        i = argmin(abs.(r .- s[1]))
        amp0 = max(y[i] - _lbp(s[1], bg[1], bg[2], bg[3], sb), 0.02 * ymax)
        push!(comps, [amp0, s[1], 1.5 * sb])
        push!(anchors, s[1])
        push!(origins, s[2])
    end

    p = copy(bg)
    c = best_c
    lo = Float64[]
    hi = Float64[]
    for iter in 0:max_add
        K = length(comps)
        p0 = vcat(bg, reduce(vcat, comps; init = Float64[]))
        lo, hi = _ring_bounds(anchors, bmaj, sb, ymax, rc_hi)
        p, c = _lm_fit(p0, lo, hi, r, y, sig, sb)
        bg = p[1:3]
        comps = [p[(4 + 3 * (k - 1)):(6 + 3 * (k - 1))] for k in 1:K]
        iter == max_add && break
        mdl = _ring_model(p, r, sb)
        z = (y .- mdl) ./ sig
        mask = trues(length(r))
        for k in 1:K
            mask .&= abs.(r .- comps[k][2]) .> 0.75 * bmaj
        end
        zz = ifelse.(mask, z, -Inf)
        j = argmax(zz)
        if zz[j] > 5 && (y[j] - mdl[j]) > 0.02 * ymax
            push!(comps, [y[j] - mdl[j], r[j], 1.5 * sb])
            push!(anchors, r[j])
            push!(origins, :residual)
            verbose && @printf("  ring fit: unexplained residual (%.1f sigma) at r = %.1f au -> extra ring component\n", zz[j], r[j])
        else
            break
        end
    end

    # drop components whose amplitude is not significant (S/N < 2), weakest first, and refit
    for _ in 1:4
        isempty(comps) && break
        Jp = _ring_jac(p, r, sig, sb)
        ep = sqrt.(max.(diag(pinv(Jp' * Jp)), 0.0)) .* sqrt(max(1.0, c / max(length(r) - length(p), 1)))   # same scaling as the reported S/N
        snrs = [ep[4 + 3 * (k - 1)] > 0 ? comps[k][1] / ep[4 + 3 * (k - 1)] : Inf for k in eachindex(comps)]
        kmin = argmin(snrs)
        snrs[kmin] >= 2 && break
        # try removing it; the covariance error of a blended component is inflated by its degeneracy with the neighbours,
        # so the removal is kept only if the fit does not get significantly worse
        st = (p = copy(p), c = c, bg = copy(bg), comps = [copy(q) for q in comps], anchors = copy(anchors), origins = copy(origins), lo = copy(lo), hi = copy(hi))
        rdrop = comps[kmin][2]
        snr_drop = snrs[kmin]
        deleteat!(comps, kmin)
        deleteat!(anchors, kmin)
        deleteat!(origins, kmin)
        lo, hi = _ring_bounds(anchors, bmaj, sb, ymax, rc_hi)
        p, c = _lm_fit(vcat(bg, reduce(vcat, comps; init = Float64[])), lo, hi, r, y, sig, sb)
        dchi = (c - st.c) / max(1.0, st.c / max(length(r) - length(st.p), 1))
        if dchi > 9.0
            verbose && @printf("  ring fit: keeping the component at r = %.1f au (amplitude S/N %.1f, but removing it raises chi2 by %.0f in error units)\n", rdrop, snr_drop, dchi)
            p, c, bg, comps, anchors, origins, lo, hi = st.p, st.c, st.bg, st.comps, st.anchors, st.origins, st.lo, st.hi
            break
        end
        verbose && @printf("  ring fit: dropping the component at r = %.1f au (amplitude S/N %.1f) and refitting\n", rdrop, snr_drop)
        bg = p[1:3]
        comps = [p[(4 + 3 * (k - 1)):(6 + 3 * (k - 1))] for k in eachindex(comps)]
    end

    K = length(comps)
    npar = length(p)
    dof = max(length(r) - npar, 1)
    chi2red = c / dof
    J = _ring_jac(p, r, sig, sb)
    cov = pinv(J' * J) * max(1.0, chi2red)
    perr = sqrt.(max.(diag(cov), 0.0))

    ringfits = NamedTuple[]
    for k in 1:K
        i0 = 4 + 3 * (k - 1)
        A, r0, s = p[i0], p[i0 + 1], p[i0 + 2]
        eA, er, es = perr[i0], perr[i0 + 1], perr[i0 + 2]
        sint = sqrt(max(s^2 - sb^2, 0.0))
        resolved = (s - es) > sb
        fwhm_obs = _FWHM_PER_SIGMA * s
        fwhm_int = resolved ? _FWHM_PER_SIGMA * sint : NaN
        fwhm_int_err = (resolved && sint > 0) ? _FWHM_PER_SIGMA * s * es / sint : NaN
        fwhm_ub = _FWHM_PER_SIGMA * sqrt(max((s + es)^2 - sb^2, 0.0))
        bgr0 = _lbp(r0, p[1], p[2], p[3], sb)
        share = A / max(_ring_model(p, [r0], sb)[1], 1e-30)
        tol = 1e-6 * bmaj
        at_bound = r0 <= lo[i0 + 1] + tol || r0 >= hi[i0 + 1] - tol || s <= lo[i0 + 2] + tol || s >= hi[i0 + 2] - tol
        push!(ringfits, (r0 = r0, r0_err = er, amp = A, amp_err = eA, amp_snr = eA > 0 ? A / eA : Inf,
                         sigma_obs = s, sigma_err = es, fwhm_obs = fwhm_obs,
                         fwhm_int = fwhm_int, fwhm_int_err = fwhm_int_err, fwhm_ub = fwhm_ub, resolved = resolved,
                         bg_at_r0 = bgr0, share = share,
                         at_bound = at_bound, origin = origins[k]))
    end
    sort!(ringfits; by = rg -> rg.r0)

    model = _ring_model(p, r, sb)
    background = Float64[_lbp(ri, p[1], p[2], p[3], sb) for ri in r]
    return (rings = ringfits, bg = (A = p[1], rc_au = p[2], gamma = p[3]), bg_err = perr[1:3],
            r_au = r, y = y, sig = sig, model = model, background = background,
            chi2 = c, dof = dof, chi2_red = chi2red, sb_au = sb, bmaj_au = bmaj, bmin_au = bmin,
            rms = rms, frac_floor = frac_floor, bunit = profile.bunit)
end

"""
    plot_ring_fits(fit; lit_features = [], title = "", filename = "ring_fits.png")

Three panels: data + fit + smooth background; data minus background with the ring model;
standardised residuals.
"""
function plot_ring_fits(fit; lit_features = [], title = "", filename = "ring_fits.png")
    scale = lowercase(fit.bunit) == "jy/beam" ? 1e3 : 1.0
    ulab = scale == 1e3 ? "mJy/beam" : fit.bunit
    r = fit.r_au
    fig = Figure(size = (980, 860))
    ax1 = Axis(fig[1, 1]; ylabel = "Azimuthal mean [$ulab]", title = title, titlesize = 13)
    band!(ax1, r, scale .* (fit.y .- fit.sig), scale .* (fit.y .+ fit.sig); color = (C_FLUX, 0.25))
    lines!(ax1, r, scale .* fit.y; color = C_FLUX, linewidth = 1.6, label = "data (±1σ)")
    lines!(ax1, r, scale .* fit.model; color = "#FFFFFF", linewidth = 1.6, label = "fit")
    lines!(ax1, r, scale .* fit.background; color = C_RESID, linestyle = :dash, linewidth = 1.4, label = "smooth background")
    if !isempty(fit.rings)
        scatter!(ax1, Float64[rg.r0 for rg in fit.rings],
                 scale .* Float64[rg.bg_at_r0 + rg.amp for rg in fit.rings];
                 color = C_RING, marker = :diamond, markersize = 11, label = "fitted rings")
    end
    isempty(lit_features) || vlines!(ax1, first.(lit_features); color = (C_LIT, 0.35), linestyle = :dash, label = "literature")
    axislegend(ax1; position = :rt, labelsize = 11)

    ax2 = Axis(fig[2, 1]; ylabel = "Data − background [$ulab]")
    band!(ax2, r, scale .* (fit.y .- fit.background .- fit.sig), scale .* (fit.y .- fit.background .+ fit.sig); color = (C_FLUX, 0.25))
    lines!(ax2, r, scale .* (fit.y .- fit.background); color = C_FLUX, linewidth = 1.6)
    lines!(ax2, r, scale .* (fit.model .- fit.background); color = "#FFFFFF", linewidth = 1.6)
    hlines!(ax2, [0.0]; color = (FG_COLOR, 0.4), linewidth = 0.8)
    isempty(fit.rings) || vlines!(ax2, Float64[rg.r0 for rg in fit.rings]; color = (C_RING, 0.6), linestyle = :dot)

    ax3 = Axis(fig[3, 1]; xlabel = "Deprojected radius [au]", ylabel = "(data − fit) / σ")
    lines!(ax3, r, (fit.y .- fit.model) ./ fit.sig; color = C_RESID, linewidth = 1.2)
    hlines!(ax3, [-3.0, 0.0, 3.0]; color = (FG_COLOR, 0.4), linewidth = 0.8, linestyle = :dash)

    linkxaxes!(ax1, ax2, ax3)
    hidexdecorations!(ax1; grid = false)
    hidexdecorations!(ax2; grid = false)
    rowsize!(fig.layout, 3, Relative(0.18))
    display(fig)
    _save_fig(fig, filename)
    return fig
end

In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (2/3): azimuthal asymmetry (non-axisymmetric structure)
# ---------------------------------------------------------------------------
# The axisymmetric model is the azimuthal-mean profile itself, so the residual image
# (data - model) contains only non-axisymmetric emission. Three views of it:
#   * per ring: m = 1 and m = 2 Fourier amplitudes of the residual around the ring,
#               relative to the fitted ring amplitude, with a noise-based false-alarm probability;
#   * a beam-smoothed residual image with connected regions above `arc_sigma` x noise
#     (candidate arcs / vortices / clumps), each with radius, azimuthal extent and contrast;
#   * a deprojected (azimuth, radius) map of the residual.
# Azimuth phi is measured in the disk plane from the major-axis direction (PA east of north).

# linear interpolation of the (uniformly spaced) azimuthal-mean profile
function _interp_profile(xs::AbstractVector, ys::AbstractVector, x::Real)
    n = length(xs)
    x <= xs[1] && return Float64(ys[1])
    x >= xs[n] && return Float64(ys[n])
    k = clamp(floor(Int, (x - xs[1]) / (xs[2] - xs[1])) + 1, 1, n - 1)
    t = (x - xs[k]) / (xs[k + 1] - xs[k])
    a, b = ys[k], ys[k + 1]
    return (isfinite(a) && isfinite(b)) ? Float64(a + t * (b - a)) : NaN
end

# Gaussian smoothing that ignores NaN pixels (normalised convolution); NaN where < half the kernel is valid
function _smooth_nan(a::AbstractMatrix{Float32}, sigpx::Real)
    h = max(ceil(Int, 3 * sigpx), 1)
    k = Float32[exp(-0.5 * (t / sigpx)^2) for t in -h:h]
    k ./= sum(k)
    nx, ny = size(a)
    valid = Float32.(isfinite.(a))
    a0 = ifelse.(isfinite.(a), a, 0.0f0)
    function conv(m)
        tmp = zeros(Float32, nx, ny)
        for j in 1:ny, i in 1:nx
            s = 0.0f0
            for t in -h:h
                ii = i + t
                (1 <= ii <= nx) && (s += k[t + h + 1] * m[ii, j])
            end
            tmp[i, j] = s
        end
        res = zeros(Float32, nx, ny)
        for j in 1:ny, i in 1:nx
            s = 0.0f0
            for t in -h:h
                jj = j + t
                (1 <= jj <= ny) && (s += k[t + h + 1] * tmp[i, jj])
            end
            res[i, j] = s
        end
        return res
    end
    num = conv(a0)
    den = conv(valid)
    return ifelse.(den .> 0.5f0, num ./ max.(den, 1.0f-6), NaN32)
end

_angdiff_deg(a, b) = abs(mod(a - b + 180.0, 360.0) - 180.0)

"""
    _ring_m2_expectation(bmaj_au, bmin_au, bpa_deg, incl_deg, pa_deg, sigma_obs_au, w_au; sigz_au = 0.0) -> (amp, phi, sig_int)

The m = 2 term that an AXISYMMETRIC Gaussian ring shows in the quantity the Fourier step measures: the mean of (pixel - azimuthal mean) over the radial
window |r - r0| < `w_au`, divided by the ring amplitude. The observed width of the azimuthal-mean ring, `sigma_obs_au`, fixes the intrinsic width:
sigma_int^2 = sigma_obs^2 - <s_b^2> - (sigz tan i)^2 / 2 (floored at a quarter of <s_b^2>). The ring is locally a straight line, so after the convolution it is
a Gaussian of width s(phi) = sqrt(sigma_int^2 + s_b^2(phi)) and peak sigma_int / s(phi), where s_b^2 = n' Cd n is the disk-plane variance of the beam along the
radial direction n = (cos phi, sin phi). `sigz_au` is the vertical Gaussian scale of an optically thin dust layer: seen at inclination i it smears the ring along
the minor axis by sigz tan(i), which adds (sigz tan i)^2 sin^2(phi) to the radial variance. Returns the relative amplitude, the azimuth of the maximum
(phi = atan(xmin, xmaj), radians, same convention as the Fourier fits) and the intrinsic width. Checked against a pixel simulation (beam + 3-D ring, same
estimator) in five geometries: amplitude and orientation agree to a few percent. NOTE: the m = 2 of the ring PEAK is about 1.65x larger than this window-mean quantity.
"""
function _ring_m2_expectation(bmaj_au::Real, bmin_au::Real, bpa_deg::Real, incl_deg::Real, pa_deg::Real,
                              sigma_obs_au::Real, w_au::Real; sigz_au::Real = 0.0)
    cos_i = cosd(incl_deg)
    sin_pa, cos_pa = sind(pa_deg), cosd(pa_deg)
    sM = bmaj_au / _FWHM_PER_SIGMA
    sm = bmin_au / _FWHM_PER_SIGMA
    ux, uy = sind(bpa_deg), cosd(bpa_deg)          # beam major axis on the sky (east, north)
    vx, vy = cosd(bpa_deg), -sind(bpa_deg)
    Cxx = sM^2 * ux * ux + sm^2 * vx * vx
    Cxy = sM^2 * ux * uy + sm^2 * vx * vy
    Cyy = sM^2 * uy * uy + sm^2 * vy * vy
    # sky offsets -> disk-plane (major, minor) coordinates: x_maj = d11 E + d12 N, x_min = d21 E + d22 N; Cd = D C D'
    d11 = sin_pa
    d12 = cos_pa
    d21 = cos_pa / cos_i
    d22 = -sin_pa / cos_i
    Cd11 = d11^2 * Cxx + 2 * d11 * d12 * Cxy + d12^2 * Cyy
    Cd12 = d11 * d21 * Cxx + (d11 * d22 + d12 * d21) * Cxy + d12 * d22 * Cyy
    Cd22 = d21^2 * Cxx + 2 * d21 * d22 * Cxy + d22^2 * Cyy
    msb2 = 0.5 * (Cd11 + Cd22)                      # azimuthal mean of the beam variance along the radial direction
    sv2 = (sigz_au * tand(incl_deg))^2              # vertical smear, along the minor axis only
    sigma_int_au = sqrt(max(sigma_obs_au^2 - msb2 - 0.5 * sv2, 0.25 * msb2))
    Cd22 += sv2
    n = 72
    nint = 40                                       # Simpson intervals over the half window
    pk = zeros(n)                                   # peak of the convolved ring at each azimuth
    wm = zeros(n)                                   # mean of its unit-height profile over the window
    h = w_au / nint
    for t in 1:n
        phi = 2π * (t - 0.5) / n
        c, s = cos(phi), sin(phi)
        sb2 = c * c * Cd11 + 2 * c * s * Cd12 + s * s * Cd22
        sp = sqrt(sigma_int_au^2 + sb2)
        pk[t] = sigma_int_au / sp
        acc = 1.0 + exp(-0.5 * (w_au / sp)^2)
        for q in 1:(nint - 1)
            acc += (isodd(q) ? 4.0 : 2.0) * exp(-0.5 * (q * h / sp)^2)
        end
        wm[t] = acc * h / 3 / w_au
    end
    amp = sum(pk) / n
    res = pk .* wm
    res .-= sum(res) / n
    zc = 0.0
    zs = 0.0
    for t in 1:n
        phi = 2π * (t - 0.5) / n
        zc += res[t] * cos(2 * phi)
        zs += res[t] * sin(2 * phi)
    end
    return (amp = 2 * hypot(zc, zs) / n / amp, phi = atan(zs, zc) / 2, sig_int = sigma_int_au)
end

"""
    _m2_excess_with_thickness(x, beam, sigma_obs, zr) -> Float64

Size of the vector difference between the measured m = 2 term of the ring `x` (an element of `azimuthal_asymmetry(...).rings`) and the expectation for the
elliptical beam plus an optically thin dust layer of vertical Gaussian scale z = `zr` * r0, with the observed ring width `sigma_obs` [au] held fixed.
"""
function _m2_excess_with_thickness(x, beam, sigma_obs::Real, zr::Real)
    be = _ring_m2_expectation(beam.bmaj_au, beam.bmin_au, beam.bpa_deg, beam.incl, beam.pa, sigma_obs, x.w_au; sigz_au = zr * x.r0)
    mx = x.m2.amp * cos(2 * deg2rad(x.m2.phi_deg))
    my = x.m2.amp * sin(2 * deg2rad(x.m2.phi_deg))
    return hypot(mx - be.amp * cos(2 * be.phi), my - be.amp * sin(2 * be.phi))
end

"""
    azimuthal_asymmetry(img, profile, ringlist; arc_sigma = 4.0, nphi_max = 36) -> NamedTuple

`ringlist` is a vector of NamedTuples with fields `r0` [au], `sigma_obs` [au] and `amp`
[profile units] (the fitted rings, or the detected ones). `shift_arcsec = (dE, dN)` moves the disk centre by a sub-pixel
amount (east, north) relative to the integer centre pixel of `profile`.
"""
function azimuthal_asymmetry(img, profile, ringlist; arc_sigma::Real = 4.0, nphi_max::Int = 36, shift_arcsec = (0.0, 0.0))
    (profile.bmaj_au === nothing || profile.bmin_au === nothing) && error("no beam in the FITS header")
    dist = Float64(profile.distance_pc)
    pix = Float64(profile.pix_scale_arcsec)
    cx, cy = round(Int, profile.center_pix[1]), round(Int, profile.center_pix[2])
    incl = profile.incl_deg === nothing ? 0.0 : Float64(profile.incl_deg)
    pa = profile.pa_deg === nothing ? 0.0 : Float64(profile.pa_deg)
    g = _geometry(img, incl, pa)
    shE, shN = Float64(shift_arcsec[1]), Float64(shift_arcsec[2])
    rr = Float64.(profile.radius_arcsec)
    mf = Float64.(profile.mean_flux)
    binw = rr[2] - rr[1]
    nrb = length(rr)
    rlim = rr[end] + 0.5 * binw
    nx, ny = size(img.data)
    hw = ceil(Int, rlim / pix) + 2
    ilo, ihi = max(cx - hw, 1), min(cx + hw, nx)
    jlo, jhi = max(cy - hw, 1), min(cy + hw, ny)
    nci, ncj = ihi - ilo + 1, jhi - jlo + 1
    bmaj_au = Float64(profile.bmaj_au)
    bmin_au = Float64(profile.bmin_au)
    bp = (π / (4 * log(2))) * profile.bmaj_arcsec * profile.bmin_arcsec / pix^2     # pixels per beam
    rms = (isfinite(profile.image_rms) && profile.image_rms > 0) ? Float64(profile.image_rms) : NaN
    isfinite(rms) || error("the image noise is unknown")

    bpa_beam = (hasproperty(img, :bpa_deg) && img.bpa_deg isa Real && isfinite(img.bpa_deg)) ? Float64(img.bpa_deg) : NaN

    resid = fill(NaN32, nci, ncj)
    pr = Float32[]
    pphi = Float32[]
    pres = Float32[]
    for j in jlo:jhi, i in ilo:ihi
        v = img.data[i, j]
        isfinite(v) || continue
        E = (i - cx) * g.e - shE
        N = (j - cy) * g.n - shN
        xmaj = E * g.sin_pa + N * g.cos_pa
        xmin = (E * g.cos_pa - N * g.sin_pa) / g.cos_i
        r = sqrt(xmaj * xmaj + xmin * xmin)
        r < rlim || continue
        m = _interp_profile(rr, mf, r)
        isfinite(m) || continue
        res = Float64(v) - m
        resid[i - ilo + 1, j - jlo + 1] = res
        push!(pr, r)
        push!(pphi, atan(xmin, xmaj))
        push!(pres, res)
    end
    isempty(pr) && error("no valid pixels inside the profile radius")

    # ---- per-ring Fourier amplitudes -----------------------------------------------------
    rings_out = NamedTuple[]
    for rg in ringlist
        w_au = max(rg.sigma_obs, 0.5 * bmin_au)
        lo_a = (rg.r0 - w_au) / dist
        hi_a = (rg.r0 + w_au) / dist
        nphi = clamp(floor(Int, 2π * rg.r0 / bmaj_au), 8, nphi_max)      # ~1 beam of arc length per azimuth bin
        S = zeros(nphi)
        C = zeros(Int, nphi)
        for t in eachindex(pr)
            (pr[t] >= lo_a && pr[t] < hi_a) || continue
            b = clamp(floor(Int, (pphi[t] + π) / (2π) * nphi) + 1, 1, nphi)
            S[b] += pres[t]
            C[b] += 1
        end
        good = findall(>(0), C)
        if length(good) < 6 || !(rg.amp > 0)
            push!(rings_out, (r0 = rg.r0, w_au = w_au, nphi = nphi, ok = false))
            continue
        end
        phis = [-π + (b - 0.5) * 2π / nphi for b in good]
        mres = [S[b] / C[b] for b in good]
        sj = [rms / sqrt(max(C[b] / bp, 1.0)) for b in good]
        ng = length(good)
        harm = Dict{Int,NamedTuple}()
        for mm in (1, 2)
            cs = sum(mres .* cos.(mm .* phis))
            sn = sum(mres .* sin.(mm .* phis))
            amp_raw = 2 * hypot(cs, sn) / ng
            var_c = 4 / ng^2 * sum(sj .^ 2 .* cos.(mm .* phis) .^ 2)
            var_s = 4 / ng^2 * sum(sj .^ 2 .* sin.(mm .* phis) .^ 2)
            sa = sqrt((var_c + var_s) / 2)
            snr = sa > 0 ? amp_raw / sa : 0.0
            phi_max = atan(sn, cs) / mm
            xmaj = rg.r0 / dist * cos(phi_max)
            xmin = rg.r0 / dist * sin(phi_max) * g.cos_i
            Esky = xmaj * g.sin_pa + xmin * g.cos_pa
            Nsky = xmaj * g.cos_pa - xmin * g.sin_pa
            pa_sky = mod(rad2deg(atan(Esky, Nsky)), 360.0)
            harm[mm] = (amp = amp_raw / rg.amp, err = sa / rg.amp, snr = snr, fap = exp(-0.5 * snr^2),
                       phi_deg = rad2deg(phi_max), pa_sky = pa_sky)
        end
        # what the elliptical beam alone does to an axisymmetric ring of this width (peak higher where the ring runs along the beam's short axis)
        m2b = (amp = NaN, phi_deg = NaN, exc = NaN, exc_snr = NaN, sig_int = NaN)
        if isfinite(bpa_beam)
            be = _ring_m2_expectation(bmaj_au, bmin_au, bpa_beam, incl, pa, rg.sigma_obs, w_au)
            cs2 = sum(mres .* cos.(2 .* phis))
            sn2 = sum(mres .* sin.(2 .* phis))
            zx = 2 * cs2 / ng - rg.amp * be.amp * cos(2 * be.phi)
            zy = 2 * sn2 / ng - rg.amp * be.amp * sin(2 * be.phi)
            exc = hypot(zx, zy)
            m2b = (amp = be.amp, phi_deg = rad2deg(be.phi), exc = exc / rg.amp, exc_snr = harm[2].err > 0 ? exc / rg.amp / harm[2].err : NaN,
                   sig_int = be.sig_int)
        end
        pk = argmax(abs.(mres) ./ sj)
        push!(rings_out, (r0 = rg.r0, w_au = w_au, nphi = nphi, ok = true, nbins = ng,
                          m1 = harm[1], m2 = harm[2], m2b = m2b,
                          peak_pos = maximum(mres) / rg.amp, peak_neg = minimum(mres) / rg.amp,
                          peak_snr = abs(mres[pk]) / sj[pk],
                          peak_sign = mres[pk] >= 0 ? 1 : -1,
                          phis = phis, mres = mres, sj = sj))
    end

    # ---- beam-smoothed residual and candidate arcs ---------------------------------------
    sigpx = sqrt(profile.bmaj_arcsec * profile.bmin_arcsec) / _FWHM_PER_SIGMA / pix
    sm = _smooth_nan(resid, sigpx)
    svals = Float64[v for v in sm if isfinite(v)]
    isempty(svals) && error("the smoothed residual image has no valid pixels")
    smed = median(svals)
    ssig = 1.4826 * median(abs.(svals .- smed))
    thr = arc_sigma * ssig
    r0s = Float64[rg.r0 for rg in ringlist]
    nxs, nys = size(sm)
    min_pix = max(ceil(Int, bp), 4)
    arcs = NamedTuple[]
    lab = falses(nxs, nys)
    stack = Int[]
    for sgn in (1, -1)
        fill!(lab, false)
        for j0 in 1:nys, i0 in 1:nxs
            lab[i0, j0] && continue
            v0 = sm[i0, j0]
            (isfinite(v0) && sgn * v0 > thr) || continue
            members = Int[]
            empty!(stack)
            push!(stack, i0 + (j0 - 1) * nxs)
            lab[i0, j0] = true
            while !isempty(stack)
                idx = pop!(stack)
                push!(members, idx)
                j = (idx - 1) ÷ nxs + 1
                i = idx - (j - 1) * nxs
                for (di, dj) in ((1, 0), (-1, 0), (0, 1), (0, -1))
                    ii, jj = i + di, j + dj
                    (1 <= ii <= nxs && 1 <= jj <= nys) || continue
                    lab[ii, jj] && continue
                    w = sm[ii, jj]
                    (isfinite(w) && sgn * w > thr) || continue
                    lab[ii, jj] = true
                    push!(stack, ii + (jj - 1) * nxs)
                end
            end
            length(members) >= min_pix || continue
            rs = Float64[]
            phs = Float64[]
            sE = 0.0
            sN = 0.0
            peak = 0.0
            for idx in members
                j = (idx - 1) ÷ nxs + 1
                i = idx - (j - 1) * nxs
                E0 = (i + ilo - 1 - cx) * g.e              # sky offsets about the profile pixel (for the plot and the sky PA)
                N0 = (j + jlo - 1 - cy) * g.n
                E = E0 - shE                               # about the (sub-pixel shifted) disk centre (for r and phi)
                N = N0 - shN
                xmaj = E * g.sin_pa + N * g.cos_pa
                xmin = (E * g.cos_pa - N * g.sin_pa) / g.cos_i
                push!(rs, sqrt(xmaj * xmaj + xmin * xmin))
                push!(phs, atan(xmin, xmaj))
                sE += E0
                sN += N0
                peak = max(peak, sgn * Float64(sm[i, j]))
            end
            nm = length(members)
            r_mean = sum(rs) / nm
            r_au = r_mean * dist
            r_au < bmaj_au && continue                      # centre: residuals there are dominated by the centre fit
            phic = atan(sum(sin.(phs)), sum(cos.(phs)))
            maxdev = maximum(abs(mod(ph - phic + π, 2π) - π) for ph in phs)
            model_here = _interp_profile(rr, mf, r_mean)
            dsl = max(binw, Float64(profile.bmaj_arcsec))
            slope = abs(_interp_profile(rr, mf, r_mean + dsl) - _interp_profile(rr, mf, r_mean - dsl)) / (2 * dsl)   # |dI/dr| per arcsec
            kn = isempty(r0s) ? 0 : argmin(abs.(r0s .- r_au))
            at_peak = kn > 0 && abs(r_au - r0s[kn]) < 0.5 * Float64(ringlist[kn].sigma_obs)   # ring crest: the slope is ~0, so a centre shift changes nothing there
            shift_au = (!at_peak && isfinite(slope) && slope > 0) ? peak / slope * dist : NaN      # centre shift that would give a residual this large
            nearest = isempty(r0s) ? NaN : r0s[argmin(abs.(r0s .- r_au))]
            push!(arcs, (sign = sgn, r_au = r_au, phi_deg = rad2deg(phic), span_deg = min(rad2deg(2 * maxdev), 360.0),
                         pa_sky = mod(rad2deg(atan(sE, sN)), 360.0), snr = peak / ssig,
                         contrast = (isfinite(model_here) && model_here > 0) ? peak / model_here : NaN,
                         area_beams = nm / bp, nearest_ring = nearest, dipole = false, peak = peak, shift_au = shift_au, at_peak = at_peak,
                         E = sE / nm, N = sN / nm))
        end
    end
    # a + arc and a - arc at the same radius, ~180 deg apart AND of comparable amplitude (>= 0.7) and area (>= 0.5) look like a centre / geometry error, not a vortex;
    # a lopsided pair is a real arc plus the deficit the azimuthal mean leaves elsewhere
    for a in eachindex(arcs)
        for b in eachindex(arcs)
            (arcs[a].sign != arcs[b].sign && abs(arcs[a].r_au - arcs[b].r_au) < bmaj_au &&
             abs(_angdiff_deg(arcs[a].phi_deg, arcs[b].phi_deg) - 180.0) < 40.0 &&
             min(arcs[a].peak, arcs[b].peak) >= 0.7 * max(arcs[a].peak, arcs[b].peak) &&
             min(arcs[a].area_beams, arcs[b].area_beams) >= 0.5 * max(arcs[a].area_beams, arcs[b].area_beams)) || continue
            arcs[a] = merge(arcs[a], (dipole = true,))
            break
        end
    end
    sort!(arcs; by = a -> -a.snr)
    length(arcs) > 10 && (arcs = arcs[1:10])

    # ---- deprojected (azimuth, radius) map ------------------------------------------------
    nphi_map = 36
    Ssum = zeros(nphi_map, nrb)
    Cc = zeros(Int, nphi_map, nrb)
    for t in eachindex(pr)
        ir = min(floor(Int, pr[t] / binw) + 1, nrb)
        ip = clamp(floor(Int, (pphi[t] + π) / (2π) * nphi_map) + 1, 1, nphi_map)
        Ssum[ip, ir] += pres[t]
        Cc[ip, ir] += 1
    end
    pol = [Cc[a, b] >= 3 ? Ssum[a, b] / Cc[a, b] : NaN for a in 1:nphi_map, b in 1:nrb]

    return (rings = rings_out, arcs = arcs, smooth = sm, smooth_sigma = ssig, resid_mean = mean(pres),
            ilo = ilo, jlo = jlo, ihi = ihi, jhi = jhi, cx = cx, cy = cy, rlim_arcsec = rlim,
            polar = pol, polar_r_au = rr .* dist, polar_phi_deg = [-180.0 + (a - 0.5) * 360.0 / nphi_map for a in 1:nphi_map],
            geometry = g, cdelt1 = img.cdelt1_arcsec, cdelt2 = img.cdelt2_arcsec, dist = dist,
            beam = (bmaj_au = bmaj_au, bmin_au = bmin_au, bpa_deg = bpa_beam, incl = incl, pa = pa),
            arc_sigma = Float64(arc_sigma), bunit = profile.bunit)
end

"""
    plot_asymmetry(asym, ringlist; title = "", filename = "azimuthal.png")

Left: beam-smoothed residual image (units of its noise) with the ring ellipses and the candidate arcs numbered.
Top right: deprojected (azimuth, radius) residual map. Bottom right: m = 1 and m = 2 amplitudes per ring.
"""
function plot_asymmetry(asym, ringlist; title = "", filename = "azimuthal.png")
    g = asym.geometry
    scale = lowercase(asym.bunit) == "jy/beam" ? 1e3 : 1.0
    ulab = scale == 1e3 ? "mJy/beam" : asym.bunit
    xs = collect((asym.ilo - asym.cx):(asym.ihi - asym.cx)) .* asym.cdelt1
    ys = collect((asym.jlo - asym.cy):(asym.jhi - asym.cy)) .* asym.cdelt2
    M = Float64.(asym.smooth) ./ asym.smooth_sigma
    if asym.cdelt1 < 0
        xs = reverse(xs);  M = reverse(M; dims = 1)
    end
    if asym.cdelt2 < 0
        ys = reverse(ys);  M = reverse(M; dims = 2)
    end
    fig = Figure(size = (1280, 760))
    ax1 = Axis(fig[1:2, 1]; aspect = DataAspect(), xreversed = true, xlabel = "ΔRA [arcsec]", ylabel = "ΔDec [arcsec]",
               title = title, titlesize = 12)
    hm1 = heatmap!(ax1, xs, ys, M; colormap = :balance, colorrange = (-8, 8))
    th = range(0, 2π; length = 181)
    for rg in ringlist
        ra = rg.r0 / asym.dist
        xmaj = ra .* cos.(th)
        xmin = ra .* sin.(th) .* g.cos_i
        lines!(ax1, xmaj .* g.sin_pa .+ xmin .* g.cos_pa, xmaj .* g.cos_pa .- xmin .* g.sin_pa;
               color = (C_RING, 0.45), linewidth = 0.8, linestyle = :dash)
    end
    for (k, a) in enumerate(asym.arcs)
        scatter!(ax1, [a.E], [a.N]; marker = :circle, markersize = 22, color = :transparent,
                 strokecolor = a.dipole ? "#AAAAAA" : "#FFFFFF", strokewidth = 1.5)
        text!(ax1, Point2f(a.E, a.N); text = string(k), align = (:left, :bottom), color = FG_COLOR, fontsize = 13)
    end
    lim = asym.rlim_arcsec
    limits!(ax1, -lim, lim, -lim, lim)
    Colorbar(fig[3, 1], hm1; vertical = false, label = "beam-smoothed residual [σ of the smoothed residual]")

    ax2 = Axis(fig[1, 2]; xlabel = "Azimuth in the disk plane [deg from the major axis]", ylabel = "r [au]",
               title = "Residual (data − azimuthal mean) [$ulab]", titlesize = 12)
    fin = [abs(v) for v in asym.polar if isfinite(v)]
    cl = isempty(fin) ? 1.0 : max(quantile(fin, 0.99), 1e-12)
    hm2 = heatmap!(ax2, asym.polar_phi_deg, asym.polar_r_au, scale .* asym.polar; colormap = :balance,
                   colorrange = (-scale * cl, scale * cl))
    isempty(ringlist) || hlines!(ax2, Float64[rg.r0 for rg in ringlist]; color = (C_RING, 0.5), linestyle = :dash, linewidth = 0.8)
    Colorbar(fig[1, 3], hm2)

    ax3 = Axis(fig[2, 2]; xlabel = "Ring radius [au]", ylabel = "Amplitude / ring amplitude [%]",
               title = "Azimuthal Fourier amplitudes", titlesize = 12)
    okr = NamedTuple[x for x in asym.rings if x.ok]
    if !isempty(okr)
        xr = Float64[x.r0 for x in okr]
        off = 0.01 * (maximum(xr) - minimum(xr) + 1.0)
        scatter!(ax3, xr .- off, Float64[100 * x.m1.amp for x in okr]; color = C_RING, markersize = 10, label = "m = 1")
        errorbars!(ax3, xr .- off, Float64[100 * x.m1.amp for x in okr], Float64[100 * x.m1.err for x in okr]; color = C_RING)
        scatter!(ax3, xr .+ off, Float64[100 * x.m2.amp for x in okr]; color = C_GAP, marker = :utriangle, markersize = 10, label = "m = 2")
        errorbars!(ax3, xr .+ off, Float64[100 * x.m2.amp for x in okr], Float64[100 * x.m2.err for x in okr]; color = C_GAP)
        hlines!(ax3, [0.0]; color = (FG_COLOR, 0.4), linewidth = 0.8)
        axislegend(ax3; position = :rt, labelsize = 11)
    end
    display(fig)
    _save_fig(fig, filename)
    return fig
end

In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (2b/3): geometry check from the ring ridges
# ---------------------------------------------------------------------------
# The radial position of a bright ring's ridge is measured in azimuth wedges and all rings are fitted
# together with ONE shared geometry: sky centre offset (dE, dN), inclination i and position angle PA,
# plus a free radius per ring. A circular ring seen with the wrong geometry shows up as an m = 1 (centre)
# or m = 2 (i, PA) wobble of its radius with azimuth, which is exactly what the fit removes.

const _GEO_COLORS = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7"]

function _num_jac(resfun, p::AbstractVector{<:Real})
    f0 = resfun(p)
    J = zeros(length(f0), length(p))
    for j in eachindex(p)
        h = 1e-6 * max(abs(p[j]), 1.0)
        qp = collect(Float64, p);  qp[j] += h
        qm = collect(Float64, p);  qm[j] -= h
        J[:, j] = (resfun(qp) .- resfun(qm)) ./ (2 * h)
    end
    return J
end

# Bounded Levenberg-Marquardt for a residual function resfun(p) -> vector
function _lm_generic(resfun, p0, lo, hi; maxit::Int = 200)
    p = clamp.(collect(Float64, p0), lo, hi)
    f = resfun(p)
    c = dot(f, f)
    lam = 1e-3
    for _ in 1:maxit
        J = _num_jac(resfun, p)
        A = J' * J
        gv = J' * f
        improved = false
        converged = false
        for _ in 1:15
            Aa = A + lam * Diagonal(diag(A) .+ 1e-12)
            dp = try
                -(Aa \ gv)
            catch
                nothing
            end
            if dp === nothing
                lam *= 10
                continue
            end
            pn = clamp.(p .+ dp, lo, hi)
            fn = resfun(pn)
            cn = dot(fn, fn)
            if isfinite(cn) && cn < c
                converged = abs(c - cn) < 1e-12 * max(c, 1e-12)
                p = pn
                f = fn
                c = cn
                lam = max(lam / 3, 1e-10)
                improved = true
                break
            end
            lam *= 4
        end
        (improved && !converged) || break
    end
    return p, c
end

# deprojected radius and disk-plane azimuth (from the major axis) of a sky position (E, N) [arcsec]
# about the sky centre (E0, N0), for inclination `incl` and position angle `pa` [deg, east of north]
@inline function _sky_to_disk(E::Real, N::Real, E0::Real, N0::Real, incl::Real, pa::Real)
    s = sind(pa)
    c = cosd(pa)
    xm = (E - E0) * s + (N - N0) * c
    xn = ((E - E0) * c - (N - N0) * s) / cosd(incl)
    return sqrt(xm * xm + xn * xn), atan(xn, xm)
end

# residuals of the ridge points about the shared geometry: p = [E0, N0, incl, pa, R_1 .. R_K]
function _geom_resid(p::AbstractVector{<:Real}, Ep, Np, scv, kk)
    out = similar(Ep)
    for t in eachindex(Ep)
        r, _ = _sky_to_disk(Ep[t], Np[t], p[1], p[2], p[3], p[4])
        out[t] = (r - p[4 + kk[t]]) / scv[t]
    end
    return out
end

# Ridge radius of one ring in `nphi` azimuth wedges (flux-weighted centroid in a window that follows the ridge).
# Returns NamedTuples (E, N, sc, phi): sky position of the ridge point, its radial error [arcsec], wedge azimuth.
function _ring_ridge(img, cx::Int, cy::Int, E0::Real, N0::Real, incl::Real, pa::Real, r0::Real, sg::Real,
                     bmaj_arcsec::Real, bp::Real, rms::Real, nphi::Int, hw_cap::Real)
    pix = img.pix_scale_arcsec
    hw = min(max(4 * sg, 1.5 * bmaj_arcsec), hw_cap)   # annulus searched (never reaching the neighbouring rings)
    hw1 = min(max(1.5 * sg, 0.75 * bmaj_arcsec), 0.6 * hw)   # centroid window half-width
    nx, ny = size(img.data)
    half = ceil(Int, (r0 + hw + 0.05) / pix) + 2    # 0.05": room for the centre offset
    ilo, ihi = max(cx - half, 1), min(cx + half, nx)
    jlo, jhi = max(cy - half, 1), min(cy + half, ny)
    wr = [Float64[] for _ in 1:nphi]
    wv = [Float64[] for _ in 1:nphi]
    for j in jlo:jhi, i in ilo:ihi
        v = img.data[i, j]
        isfinite(v) || continue
        r, ph = _sky_to_disk((i - cx) * img.cdelt1_arcsec, (j - cy) * img.cdelt2_arcsec, E0, N0, incl, pa)
        abs(r - r0) < hw || continue
        b = clamp(floor(Int, (ph + π) / (2π) * nphi) + 1, 1, nphi)
        push!(wr[b], r)
        push!(wv[b], Float64(v))
    end
    pts = NamedTuple[]
    s = sind(pa)
    cp = cosd(pa)
    ci = cosd(incl)
    for q in 1:nphi
        rr = wr[q]
        vv = wv[q]
        length(rr) < 10 && continue
        c = r0
        fl = 0.0
        ncore = 0
        good = true
        for _ in 1:4
            core = abs.(rr .- c) .< hw1
            shell = .!core
            if count(core) < 4 || count(shell) < 4
                good = false
                break
            end
            fl = median(vv[shell])
            wt = max.(vv[core] .- fl, 0.0)
            if !(sum(wt) > 0)
                good = false
                break
            end
            c = sum(wt .* rr[core]) / sum(wt)
            ncore = count(core)
        end
        good || continue
        near = abs.(rr .- c) .< 0.5 * sg
        any(near) || continue
        amp = mean(vv[near]) - fl
        amp > 0 || continue
        snr = amp / (rms / sqrt(max(ncore / bp, 1.0)))
        snr >= 3 || continue
        phc = -π + (q - 0.5) * 2π / nphi
        xm = c * cos(phc)
        xn = c * sin(phc) * ci
        push!(pts, (E = E0 + xm * s + xn * cp, N = N0 + xm * cp - xn * s, sc = sg / snr, phi = phc))
    end
    return pts
end

"""
    fit_ring_geometry(img, profile, ringfits; min_snr = 10, nphi_max = 36, iters = 3) -> NamedTuple

Shared centre offset, inclination and PA from the ridges of the rings with amplitude S/N >= `min_snr`
(`ringfits` = `fit.rings` from `fit_gaussian_rings`). Only narrow, isolated rings are used: observed FWHM <= max(2.5 beams, 0.25 r) and no other
ring (S/N >= 3) closer than 2.5 beams; broad shoulders and blended neighbours give biased ridges. The ridges are re-measured after every fit with the updated
geometry. Errors are formal (covariance scaled by sqrt(chi2_red) when > 1); they do not include systematics such as
ring flaring or non-circular rings. Also returns, per ring, the residual m = 1 wobble of the ridge radius
(eccentricity e = amplitude / R) about the fitted geometry, and the centre of that ring (`cen_dx`, `cen_dy` [arcsec, deprojected major / minor axis, about the
shared centre] with its covariance `cen_cov`), which `fit_flaring` compares with the position of the central core.
"""
function fit_ring_geometry(img, profile, ringfits; min_snr::Real = 10.0, nphi_max::Int = 36, iters::Int = 3)
    dist = Float64(profile.distance_pc)
    cx, cy = round(Int, profile.center_pix[1]), round(Int, profile.center_pix[2])
    incl0 = profile.incl_deg === nothing ? 0.0 : Float64(profile.incl_deg)
    pa0 = profile.pa_deg === nothing ? 0.0 : Float64(profile.pa_deg)
    bmaj_arcsec = Float64(profile.bmaj_arcsec)
    bmin_arcsec = Float64(profile.bmin_arcsec)
    bp = (π / (4 * log(2))) * bmaj_arcsec * bmin_arcsec / img.pix_scale_arcsec^2
    rms = (isfinite(profile.image_rms) && profile.image_rms > 0) ? Float64(profile.image_rms) : error("the image noise is unknown")
    bmaj_au = Float64(profile.bmaj_au)
    others_r = Float64[rg.r0 for rg in ringfits if rg.amp_snr >= 3]
    nn_au(rg) = (d = [abs(x - rg.r0) for x in others_r if abs(x - rg.r0) > 1e-6]; isempty(d) ? Inf : minimum(d))
    use = [rg for rg in ringfits if rg.amp_snr >= min_snr && rg.fwhm_obs <= max(2.5 * bmaj_au, 0.25 * rg.r0) && nn_au(rg) >= 2.5 * bmaj_au]
    isempty(use) && error("no narrow, isolated ring with amplitude S/N >= $(min_snr) (narrow: observed FWHM <= max(2.5 beams, 0.25 r); isolated: no other ring within 2.5 beams)")
    hw_caps = Float64[0.8 * nn_au(rg) / dist for rg in use]
    K = length(use)
    r0_start = Float64[rg.r0 / dist for rg in use]
    r0s = copy(r0_start)
    sgs = Float64[max(rg.sigma_obs / dist, 0.5 * bmin_arcsec) for rg in use]
    E0 = 0.0
    N0 = 0.0
    incl = incl0
    pa = pa0
    p = vcat([0.0, 0.0, incl0, pa0], r0s)
    c = NaN
    Ep = Float64[]
    Np = Float64[]
    scv = Float64[]
    kk = Int[]
    for _ in 1:iters
        Ep = Float64[]
        Np = Float64[]
        scv = Float64[]
        kk = Int[]
        for k in 1:K
            nphi = clamp(floor(Int, 2π * r0s[k] / bmaj_arcsec), 12, nphi_max)
            for q in _ring_ridge(img, cx, cy, E0, N0, incl, pa, r0s[k], sgs[k], bmaj_arcsec, bp, rms, nphi, hw_caps[k])
                push!(Ep, q.E)
                push!(Np, q.N)
                push!(scv, q.sc)
                push!(kk, k)
            end
        end
        length(Ep) < max(8, 2 * K + 4) && error("too few ridge points ($(length(Ep))): the selected rings are not bright enough")
        lo = vcat([-0.05, -0.05, 1.0, pa0 - 30.0], 0.8 .* r0_start)
        hi = vcat([0.05, 0.05, 80.0, pa0 + 30.0], 1.2 .* r0_start)
        p, c = _lm_generic(q -> _geom_resid(q, Ep, Np, scv, kk), vcat([E0, N0, incl, pa], r0s), lo, hi)
        E0, N0, incl, pa = p[1], p[2], p[3], p[4]
        r0s = p[5:end]
    end
    npts = length(Ep)
    dof = max(npts - length(p), 1)
    chi2red = c / dof
    J = _num_jac(q -> _geom_resid(q, Ep, Np, scv, kk), p)
    cov = pinv(J' * J) * max(1.0, chi2red)
    perr = sqrt.(max.(diag(cov), 0.0))

    # per-ring residual wobble about the fitted geometry, and plot data for both geometries
    rings_out = NamedTuple[]
    phi_a = Float64[]
    dr_a = Float64[]
    phi_f = Float64[]
    dr_f = Float64[]
    for t in eachindex(Ep)
        ra, pha = _sky_to_disk(Ep[t], Np[t], 0.0, 0.0, incl0, pa0)
        rf, phf = _sky_to_disk(Ep[t], Np[t], E0, N0, incl, pa)
        push!(phi_a, rad2deg(pha))
        push!(dr_a, ra)
        push!(phi_f, rad2deg(phf))
        push!(dr_f, rf - p[4 + kk[t]])
    end
    for k in 1:K
        idx = findall(==(k), kk)
        ma = sum(dr_a[idx]) / length(idx)
        dr_a[idx] .-= ma
        w = 1 ./ scv[idx] .^ 2
        ph = deg2rad.(phi_f[idx])
        d = dr_f[idx]
        cs, sn = cos.(ph), sin.(ph)
        M = [sum(w .* cs .^ 2) sum(w .* cs .* sn); sum(w .* cs .* sn) sum(w .* sn .^ 2)]
        rhs = [sum(w .* d .* cs), sum(w .* d .* sn)]
        ecc = NaN
        ecc_err = NaN
        far = NaN
        cen_dx = NaN                                  # centre of this ring about the shared centre, in the deprojected plane [arcsec] (major, minor)
        cen_dy = NaN
        cen_cov = fill(NaN, 2, 2)
        if length(idx) >= 6 && abs(det(M)) > 0
            sol = M \ rhs
            chik = sum(w .* (d .- sol[1] .* cs .- sol[2] .* sn) .^ 2) / max(length(idx) - 2, 1)
            cv = inv(M) * max(1.0, chik)
            amp = hypot(sol[1], sol[2])
            err = amp > 0 ? sqrt(max((sol[1]^2 * cv[1, 1] + sol[2]^2 * cv[2, 2] + 2 * sol[1] * sol[2] * cv[1, 2]) / amp^2, 0.0)) :
                  sqrt(max(cv[1, 1], 0.0))
            ecc = amp / p[4 + k]
            ecc_err = err / p[4 + k]
            far = mod(rad2deg(atan(sol[2], sol[1])), 360.0)
            cen_dx, cen_dy, cen_cov = sol[1], sol[2], cv
        end
        push!(rings_out, (r_au_assumed = use[k].r0, r_au = p[4 + k] * dist, r_err_au = perr[4 + k] * dist, npts = length(idx),
                          rms_mas = 1000 * sqrt(sum(d .^ 2) / length(idx)), ecc = ecc, ecc_err = ecc_err,
                          ecc_snr = ecc_err > 0 ? ecc / ecc_err : NaN, far_deg = far,
                          cen_dx = cen_dx, cen_dy = cen_dy, cen_cov = cen_cov))
    end
    return (E0 = E0, N0 = N0, incl = incl, pa = pa, E0_err = perr[1], N0_err = perr[2], incl_err = perr[3], pa_err = perr[4],
            incl_assumed = incl0, pa_assumed = pa0, rings = rings_out, K = K, kk = kk, chi2_red = chi2red, dof = dof, npts = npts,
            new_center = (cx + E0 / img.cdelt1_arcsec, cy + N0 / img.cdelt2_arcsec),
            plot = (k = kk, phi_a = phi_a, dr_a = dr_a, phi_f = phi_f, dr_f = dr_f))
end

"""
    plot_geometry_fit(geo; title = "", filename = "geometry_fit.png")

Ridge radius minus the ring mean against disk azimuth, per ring, for the assumed and the fitted geometry.
A flat band means the geometry describes the ring; a sinusoid means a centre / inclination / PA error.
"""
function plot_geometry_fit(geo; title = "", filename = "geometry_fit.png")
    fig = Figure(size = (1150, 520))
    ax1 = Axis(fig[1, 1]; xlabel = "Disk-plane azimuth [deg from the major axis]", ylabel = "ridge radius − ring mean [mas]",
               title = @sprintf("assumed: i = %.1f°, PA = %.1f°, centre fixed", geo.incl_assumed, geo.pa_assumed), titlesize = 12)
    ax2 = Axis(fig[1, 2]; xlabel = "Disk-plane azimuth [deg from the major axis]",
               title = @sprintf("fitted: i = %.1f°, PA = %.1f°, centre shifted by (%+.1f, %+.1f) mas", geo.incl, geo.pa, 1000 * geo.E0, 1000 * geo.N0),
               titlesize = 12)
    for k in 1:geo.K
        idx = findall(==(k), geo.plot.k)
        col = _GEO_COLORS[mod1(k, length(_GEO_COLORS))]
        lab = @sprintf("%.0f au", geo.rings[k].r_au)
        scatter!(ax1, geo.plot.phi_a[idx], 1000 .* geo.plot.dr_a[idx]; color = col, markersize = 7, label = lab)
        scatter!(ax2, geo.plot.phi_f[idx], 1000 .* geo.plot.dr_f[idx]; color = col, markersize = 7, label = lab)
    end
    hlines!(ax1, [0.0]; color = (FG_COLOR, 0.4), linewidth = 0.8)
    hlines!(ax2, [0.0]; color = (FG_COLOR, 0.4), linewidth = 0.8)
    linkyaxes!(ax1, ax2)
    axislegend(ax2; position = :rt, labelsize = 11)
    isempty(title) || Label(fig[0, 1:2], title; fontsize = 14, color = FG_COLOR)
    display(fig)
    _save_fig(fig, filename)
    return fig
end

In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (2c/3): ring centres versus the star (dust-layer height from flaring)
# ---------------------------------------------------------------------------
# Dust that sits a height z above the midplane (the surface that faces us) is projected along the minor axis of the disk, toward the FAR side, by
# z sin(i) on the sky (z tan(i) in the deprojected plane); the star, i.e. the unresolved central core, does not move. A ring at radius R with z = eps R
# therefore has its centre displaced from the core by eps R sin(i) along the minor axis, and not at all along the major axis: a linear trend with radius.
# The ring centres come from the ridge fits (fit_ring_geometry), the star position from the core centroid fitted here.
# Assumptions: the core is centred on the star; the ring is partly optically thick along the line of sight (a fully transparent, vertically symmetric
# layer has no net shift), so eps is a LOWER limit on the height of the emitting dust layer relative to R.

"""
    core_centroid(img, profile; radius_beams = 1.0, iters = 3) -> NamedTuple (E, N, err, sx_mas, npix, chi2_red, peak)

Sky position (E, N [arcsec], relative to the centre pixel of `profile`) of the central core, from a fit to the pixels within `radius_beams` major axes:
the beam shape (CLEAN beam from the FITS header) plus an optional isotropic broadening, on a tilted-plane background that absorbs the gradient of the
neighbouring emission. The window is re-centred on the fitted position `iters` times (a window centred off the core biases the position by ~25% of the
offset; re-centred, the bias is ~5% in synthetic tests with the 14 au and 28 au rings). The error is formal, inflated for the correlated noise
(pixels per beam) and for chi2_red > 1; it does not include the ~0.5 mas systematic from the neighbouring rings.
"""
function core_centroid(img, profile; radius_beams::Real = 1.0, iters::Int = 3)
    bmaj = Float64(profile.bmaj_arcsec)
    bmin = Float64(profile.bmin_arcsec)
    bpa = (hasproperty(img, :bpa_deg) && img.bpa_deg isa Real && isfinite(img.bpa_deg)) ? Float64(img.bpa_deg) : NaN
    isfinite(bpa) || error("no beam position angle in the FITS header")
    rms = (isfinite(profile.image_rms) && profile.image_rms > 0) ? Float64(profile.image_rms) : NaN
    isfinite(rms) || error("the image noise is unknown")
    cx, cy = round(Int, profile.center_pix[1]), round(Int, profile.center_pix[2])
    nx, ny = size(img.data)
    d1, d2 = img.cdelt1_arcsec, img.cdelt2_arcsec
    rad = radius_beams * bmaj
    s0 = bmin / _FWHM_PER_SIGMA                     # length unit: the beam minor-axis sigma [arcsec]
    sM = bmaj / bmin
    ux, uy = sind(bpa), cosd(bpa)                   # beam major axis on the sky (east, north)
    vx, vy = cosd(bpa), -sind(bpa)
    Cxx = sM^2 * ux * ux + vx * vx
    Cxy = sM^2 * ux * uy + vx * vy
    Cyy = sM^2 * uy * uy + vy * vy

    function fit_window(cE::Float64, cN::Float64)
        hx = ceil(Int, (rad + abs(cE)) / abs(d1)) + 1
        hy = ceil(Int, (rad + abs(cN)) / abs(d2)) + 1
        e = Float64[]
        n = Float64[]
        Vp = Float64[]
        for j in max(cy - hy, 1):min(cy + hy, ny), i in max(cx - hx, 1):min(cx + hx, nx)
            v = img.data[i, j]
            isfinite(v) || continue
            dE = (i - cx) * d1 - cE
            dN = (j - cy) * d2 - cN
            hypot(dE, dN) <= rad || continue
            push!(e, dE / s0)
            push!(n, dN / s0)
            push!(Vp, Float64(v) / rms)
        end
        length(Vp) >= 30 || error("too few pixels around the centre")
        function resid(p)
            sx2 = p[7]^2
            qa = Cxx + sx2
            qb = Cxy
            qc = Cyy + sx2
            dt = qa * qc - qb * qb
            out = similar(Vp)
            for t in eachindex(Vp)
                du = e[t] - p[1]
                dv = n[t] - p[2]
                q = (qc * du * du - 2 * qb * du * dv + qa * dv * dv) / dt
                out[t] = Vp[t] - (p[3] * exp(-0.5 * q) + p[4] + p[5] * e[t] + p[6] * n[t])
            end
            return out
        end
        vmax = maximum(Vp)
        p0 = [0.0, 0.0, vmax, 0.0, 0.0, 0.0, 0.3]
        lo = [-1.5, -1.5, 0.0, -1e4, -1e4, -1e4, 0.0]
        hi = [1.5, 1.5, 10 * vmax, 1e4, 1e4, 1e4, 3.0]
        pf, cf = _lm_generic(resid, p0, lo, hi)
        return pf, cf, _num_jac(resid, pf), length(Vp)
    end

    cE = 0.0
    cN = 0.0
    pfit = zeros(7)
    cfit = 0.0
    Jm = zeros(1, 7)
    npix = 0
    for _ in 1:max(iters, 1)
        pfit, cfit, Jm, npix = fit_window(cE, cN)
        cE += pfit[1] * s0
        cN += pfit[2] * s0
    end
    chi2red = cfit / max(npix - length(pfit), 1)
    bp = (π / (4 * log(2))) * bmaj * bmin / (abs(d1) * abs(d2))
    cov = pinv(Jm' * Jm) * max(1.0, chi2red) * max(bp, 1.0)
    err = s0 * sqrt(max(0.5 * (cov[1, 1] + cov[2, 2]), 0.0))
    return (E = cE, N = cN, err = err, sx_mas = 1000 * pfit[7] * s0, npix = npix, chi2_red = chi2red, peak = pfit[3] * rms)
end

"""
    fit_flaring(geo, core, profile) -> NamedTuple

Offset of every ring centre from the core, along the disk major and minor axes (on the sky), from the ring centres of `fit_ring_geometry` (`geo`) and the
core position from `core_centroid`. For z = eps R the minor-axis offset is eps R sin(i) / distance; the major-axis offset should be zero. eps is fitted by
generalised least squares to all rings (the core error is shared by all rings). Positive eps: the rings are displaced toward the +minor-axis direction
(position angle `pa_plus`), which for a layer above the midplane is the FAR side of the disk. Errors are formal.
"""
function fit_flaring(geo, core, profile)
    isfinite(core.err) || error("the core position error is not finite")
    dist = Float64(profile.distance_pc)
    si, ci = sind(geo.incl), cosd(geo.incl)
    sp, cp = sind(geo.pa), cosd(geo.pa)
    rings = [rg for rg in geo.rings if isfinite(rg.cen_dx) && isfinite(rg.cen_cov[1, 1])]
    K = length(rings)
    K >= 1 || error("no ring with a measured centre")
    R = Float64[rg.r_au for rg in rings]
    ymaj = zeros(K)
    ymin = zeros(K)
    vmaj = zeros(K)
    vmin = zeros(K)
    for (k, rg) in enumerate(rings)
        Ek = geo.E0 + rg.cen_dx * sp + rg.cen_dy * ci * cp          # ring centre on the sky about the profile pixel
        Nk = geo.N0 + rg.cen_dx * cp - rg.cen_dy * ci * sp
        dE = Ek - core.E
        dN = Nk - core.N
        ymaj[k] = dE * sp + dN * cp                                   # along the major axis (position angle PA)
        ymin[k] = dE * cp - dN * sp                                   # along the minor axis, toward PA + 90 deg
        vmaj[k] = rg.cen_cov[1, 1]
        vmin[k] = ci^2 * rg.cen_cov[2, 2]
    end
    c2 = core.err^2
    Smin = Matrix(Diagonal(vmin)) .+ c2
    Smaj = Matrix(Diagonal(vmaj)) .+ c2
    a = R .* si ./ dist                                               # arcsec of minor-axis offset per unit z/R
    Si = inv(Smin)
    den = dot(a, Si * a)
    eps = dot(a, Si * ymin) / den
    eps_err = sqrt(1 / den)
    resv = ymin .- a .* eps
    chi2 = dot(resv, Si * resv)
    chi2_null = dot(ymin, Si * ymin)
    chi2_maj = dot(ymaj, inv(Smaj) * ymaj)
    rows = NamedTuple[]
    for k in 1:K
        push!(rows, (r_au = R[k], dmaj_mas = 1000 * ymaj[k], dmaj_err_mas = 1000 * sqrt(Smaj[k, k]),
                     dmin_mas = 1000 * ymin[k], dmin_err_mas = 1000 * sqrt(Smin[k, k]),
                     eps = ymin[k] / a[k], eps_err = sqrt(Smin[k, k]) / a[k]))
    end
    return (eps = eps, eps_err = eps_err, chi2 = chi2, dof = K - 1, chi2_null = chi2_null,
            signif = sqrt(max(chi2_null - chi2, 0.0)), chi2_maj = chi2_maj, K = K, rings = rows,
            pa_plus = mod(rad2deg(atan(cp, -sp)), 360.0), core = core, incl = geo.incl, pa = geo.pa, dist = dist)
end

"""
    plot_flaring(fl; title = "", filename = "flaring.png")

Ring-centre offset from the core along the minor axis (left) and the major axis (right) against ring radius. The line on the left is the fitted
z = eps R (zero offset at R = 0); the major-axis offsets should scatter about zero.
"""
function plot_flaring(fl; title = "", filename = "flaring.png")
    fig = Figure(size = (1100, 460))
    ax1 = Axis(fig[1, 1]; xlabel = "Ring radius [au]", ylabel = "centre − core, along the minor axis [mas]",
               title = @sprintf("toward PA %.0f°: z = (%.3f ± %.3f) R", fl.pa_plus, fl.eps, fl.eps_err), titlesize = 12)
    ax2 = Axis(fig[1, 2]; xlabel = "Ring radius [au]", ylabel = "centre − core, along the major axis [mas]",
               title = "should be consistent with zero", titlesize = 12)
    rr = Float64[x.r_au for x in fl.rings]
    xs = range(0.0, 1.15 * maximum(rr); length = 50)
    slope = 1000 * sind(fl.incl) / fl.dist                           # mas per au per unit z/R
    band_lo = (fl.eps - fl.eps_err) .* slope .* xs
    band_hi = (fl.eps + fl.eps_err) .* slope .* xs
    band!(ax1, xs, band_lo, band_hi; color = (C_RING, 0.2))
    lines!(ax1, xs, fl.eps .* slope .* xs; color = C_RING, linewidth = 1.5)
    errorbars!(ax1, rr, Float64[x.dmin_mas for x in fl.rings], Float64[x.dmin_err_mas for x in fl.rings]; color = FG_COLOR)
    scatter!(ax1, rr, Float64[x.dmin_mas for x in fl.rings]; color = FG_COLOR, markersize = 10)
    errorbars!(ax2, rr, Float64[x.dmaj_mas for x in fl.rings], Float64[x.dmaj_err_mas for x in fl.rings]; color = FG_COLOR)
    scatter!(ax2, rr, Float64[x.dmaj_mas for x in fl.rings]; color = FG_COLOR, markersize = 10)
    for ax in (ax1, ax2)
        hlines!(ax, [0.0]; color = (FG_COLOR, 0.4), linewidth = 0.8)
    end
    isempty(title) || Label(fig[0, 1:2], title; fontsize = 14, color = FG_COLOR)
    display(fig)
    _save_fig(fig, filename)
    return fig
end


In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (2b): does the ring m = 2 change when the image is smoothed to a round beam?
# ---------------------------------------------------------------------------
# The elliptical-beam check models the beam as a Gaussian. If the measured m = 2 is larger than that model predicts, the cause is either
#   (a) a modulation of the ring's AMPLITUDE (the ring really is brighter along one axis), or
#   (b) a modulation of its apparent WIDTH / resolution: a PSF that is not the fitted Gaussian (uv coverage, CLEAN), or a real width variation.
# Smoothing the image with a Gaussian kernel to a CIRCULAR beam (kernel covariance sigma_t^2 I - C_beam, which is the sum of a line kernel along the beam's
# minor axis and an isotropic kernel) removes the Gaussian part of the beam anisotropy exactly. A broader ring then dilutes a width-type modulation
# (relative m = 2 ~ 1 / sigma_obs^2 for a fixed absolute anisotropy), whereas an amplitude-type modulation keeps its relative size. Comparing the excess
# at several smoothing levels therefore separates (a) from (b), using only this image.

# Gaussian smoothing along ONE direction with bilinear sampling; taps on NaN pixels are skipped (normalised), NaN where < half the weight is valid.
# (di, dj): pixel displacement per arcsec along the line; sigt: kernel sigma [arcsec]; step: tap spacing [arcsec].
function _line_smooth(a::AbstractMatrix{Float32}, di::Real, dj::Real, sigt::Real, step::Real)
    K = max(ceil(Int, 3.5 * sigt / step), 1)
    ts = Float64[q * step for q in -K:K]
    ws = Float64[exp(-0.5 * (t / sigt)^2) for t in ts]
    wtot = sum(ws)
    nx, ny = size(a)
    res = fill(NaN32, nx, ny)
    for j in 1:ny, i in 1:nx
        num = 0.0
        den = 0.0
        for q in eachindex(ts)
            x = i + ts[q] * di
            y = j + ts[q] * dj
            i0 = floor(Int, x)
            j0 = floor(Int, y)
            (i0 >= 1 && i0 < nx && j0 >= 1 && j0 < ny) || continue
            fx = x - i0
            fy = y - j0
            v00 = a[i0, j0]
            v10 = a[i0 + 1, j0]
            v01 = a[i0, j0 + 1]
            v11 = a[i0 + 1, j0 + 1]
            (isfinite(v00) && isfinite(v10) && isfinite(v01) && isfinite(v11)) || continue
            v = (1 - fx) * (1 - fy) * v00 + fx * (1 - fy) * v10 + (1 - fx) * fy * v01 + fx * fy * v11
            num += ws[q] * v
            den += ws[q]
        end
        den > 0.5 * wtot && (res[i, j] = Float32(num / den))
    end
    return res
end

# one row of the comparison: the m = 2 of ring `x` (an element of azimuthal_asymmetry(...).rings) and its excess over the Gaussian-beam expectation
function _rb_row(x, sigma_obs::Real)
    (x.ok && isfinite(x.m2b.amp)) || return nothing
    return (r0 = x.r0, sigma_obs = Float64(sigma_obs), amp = x.m2.amp, phi_deg = mod(x.m2.phi_deg, 180.0), err = x.m2.err, snr = x.m2.snr,
            exp_amp = x.m2b.amp, exp_phi_deg = mod(x.m2b.phi_deg, 180.0), exc = x.m2b.exc, exc_snr = x.m2b.exc_snr)
end

"""
    round_beam_test(img, profile, ringlist, asym0; factors = (1.0, 1.5, 2.0), shift_arcsec = (0.0, 0.0)) -> NamedTuple

Smooth `img` to circular beams of `factors` x the native beam major axis, refit the rings of `ringlist` (same radial bins as `profile`), and
re-measure the m = 2 with `azimuthal_asymmetry`. `asym0` is the native-beam result for the same `ringlist`. Returns `(factors, native, sets)`, where
`native` and each `sets[i].rows` hold one entry per ring of `ringlist` (`nothing` where the ring could not be measured).
Jy/beam values are rescaled by the beam-area ratio, so ring amplitudes stay comparable; CLEAN residual (JvM) effects are not treated.
"""
function round_beam_test(img, profile, ringlist, asym0; factors = (1.0, 1.5, 2.0), shift_arcsec = (0.0, 0.0))
    bpa = hasproperty(img, :bpa_deg) ? img.bpa_deg : nothing
    (bpa isa Real && isfinite(bpa)) || error("the FITS header has no beam position angle")
    (profile.bmaj_arcsec === nothing || profile.bmin_arcsec === nothing) && error("no beam in the FITS header")
    length(asym0.rings) == length(ringlist) || error("the native asymmetry does not match the ring list")
    pix = Float64(img.pix_scale_arcsec)
    dist = Float64(profile.distance_pc)
    bmaj_as = Float64(profile.bmaj_arcsec)
    bmin_as = Float64(profile.bmin_arcsec)
    sM = bmaj_as / _FWHM_PER_SIGMA
    sn = bmin_as / _FWHM_PER_SIGMA
    nbin = length(profile.radius_arcsec)
    binw = profile.radius_arcsec[2] - profile.radius_arcsec[1]
    rmax = profile.radius_arcsec[end] + 0.5 * binw
    cx, cy = round(Int, profile.center_pix[1]), round(Int, profile.center_pix[2])
    nx, ny = size(img.data)
    hw = ceil(Int, (rmax + 0.3) / pix)
    ilo, ihi = max(cx - hw, 1), min(cx + hw, nx)
    jlo, jhi = max(cy - hw, 1), min(cy + hw, ny)
    a0 = Float32.(img.data[ilo:ihi, jlo:jhi])
    cen = (cx - ilo + 1, cy - jlo + 1)

    # line kernel along the beam's minor axis (sky direction (cos bpa, -sin bpa) in (east, north)): sigma^2 = sM^2 - sn^2 turns the beam into a round one of sigma sM
    a_line = a0
    vl = sM^2 - sn^2
    if vl > 1e-6 * sM^2
        tb = deg2rad(Float64(bpa))
        a_line = _line_smooth(a0, cos(tb) / img.cdelt1_arcsec, -sin(tb) / img.cdelt2_arcsec, sqrt(vl), 0.5 * pix)
    end

    native = Any[]
    for k in eachindex(ringlist)
        push!(native, _rb_row(asym0.rings[k], ringlist[k].sigma_obs))
    end

    sets = NamedTuple[]
    for f in factors
        st = f * sM
        vi = st^2 - sM^2
        am = a_line
        if vi > 1e-9 * sM^2
            sp = sqrt(vi) / pix
            sp > 0.3 && (am = _smooth_nan(a_line, sp))
        end
        am = am .* Float32(st^2 / (sM * sn))                    # Jy/beam of the new (larger) beam
        fw = f * bmaj_as
        img2 = (path = img.path, data = am, cdelt1_arcsec = img.cdelt1_arcsec, cdelt2_arcsec = img.cdelt2_arcsec,
                pix_scale_arcsec = img.pix_scale_arcsec, bmaj_deg = fw / 3600, bmin_deg = fw / 3600, bpa_deg = 0.0,
                bunit = img.bunit, object = img.object, freq_hz = img.freq_hz)
        prof_c = extract_radial_profile(img2; incl_deg = profile.incl_deg, pa_deg = profile.pa_deg, distance_pc = dist,
                                        max_radius_arcsec = rmax, n_bins = nbin, center = cen, fit_geom = false)
        seeds = Tuple{Float64,Symbol}[(Float64(rg.r0), :detected) for rg in ringlist]
        fit_c = fit_gaussian_rings(prof_c, seeds; max_add = 0, verbose = false)
        rl_c = NamedTuple[(r0 = rg.r0, sigma_obs = rg.sigma_obs, amp = rg.amp) for rg in fit_c.rings if rg.amp_snr >= 2]
        rows = Any[]
        if isempty(rl_c)
            for k in eachindex(ringlist)
                push!(rows, nothing)
            end
        else
            asym_c = azimuthal_asymmetry(img2, prof_c, rl_c; shift_arcsec = shift_arcsec)
            for rg in ringlist
                kk = 0
                dbest = 0.5 * Float64(profile.bmaj_au)
                for q in eachindex(rl_c)
                    dq = abs(rl_c[q].r0 - rg.r0)
                    if dq < dbest
                        dbest = dq
                        kk = q
                    end
                end
                push!(rows, kk == 0 ? nothing : _rb_row(asym_c.rings[kk], rl_c[kk].sigma_obs))
            end
        end
        push!(sets, (factor = Float64(f), fwhm_au = fw * dist, rows = rows))
    end
    return (factors = collect(Float64, factors), native = native, sets = sets)
end

"""
    round_beam_summary(rb; min_snr = 3.0) -> Vector{NamedTuple}

For every ring whose native excess is significant: the excess relative to the native one (`rho`) at each smoothing level, the value expected for a
width-type (resolution) modulation, `pred = (sigma_obs_native / sigma_obs)^2`, and a verdict from the strongest smoothing:
`:flat` (consistent with no change and not with dilution), `:falls` (the reverse), `:weak` (consistent with both) or `:between` (neither). The tests use 2 sigma.
"""
function round_beam_summary(rb; min_snr::Real = 3.0)
    out = NamedTuple[]
    for k in eachindex(rb.native)
        n0 = rb.native[k]
        (n0 !== nothing && isfinite(n0.exc_snr) && n0.exc_snr >= min_snr && n0.exc > 0) || continue
        pts = NamedTuple[]
        for (si, s) in enumerate(rb.sets)
            r = s.rows[k]
            r === nothing && continue
            push!(pts, (idx = si, factor = s.factor, fwhm_au = s.fwhm_au, exc = r.exc, err = r.err, snr = r.exc_snr,
                        rho = r.exc / n0.exc, rho_err = hypot(r.err, n0.err) / n0.exc, pred = (n0.sigma_obs / r.sigma_obs)^2))
        end
        isempty(pts) && continue
        pl = pts[end]
        zf = (pl.rho - 1) / pl.rho_err                 # distance from an unchanged excess (amplitude-type), in sigma
        zr = (pl.rho - pl.pred) / pl.rho_err           # distance from the diluted excess (resolution-type), in sigma
        verdict = if abs(zf) <= 2 && abs(zr) > 2
            :flat
        elseif abs(zr) <= 2 && abs(zf) > 2
            :falls
        elseif abs(zf) <= 2 && abs(zr) <= 2
            :weak
        else
            :between
        end
        push!(out, (k = k, r0 = n0.r0, exc0 = n0.exc, err0 = n0.err, snr0 = n0.exc_snr, sigma0 = n0.sigma_obs, pts = pts, verdict = verdict, zf = zf, zr = zr))
    end
    return out
end

function _round_beam_report(io, rb, summ, bmaj_arcsec::Real)
    println(io, "\n  Round-beam test for m=2. The image is smoothed with a Gaussian kernel to a circular beam of ", join([@sprintf("%.1f", f) for f in rb.factors], ", "),
            " x the native major axis (", @sprintf("%.4f", bmaj_arcsec), "\"), the rings are refitted, and the m=2 is re-measured with the same estimator.")
    println(io, "  Excess [%] over the Gaussian-beam expectation (S/N); second line: what a width-type modulation (a resolution effect: the excess diluted as 1 / sigma_obs^2) would give:")
    heads = vcat(["native"], [@sprintf("%.1fx", f) for f in rb.factors])
    @printf(io, "  %8s | %s\n", "r [au]", join([@sprintf("%-13s", h) for h in heads], " "))
    for x in summ
        cells = String[@sprintf("%.1f (%.1f s)", 100 * x.exc0, x.snr0)]
        pcell = String["-"]
        for si in eachindex(rb.sets)
            q = findfirst(p -> p.idx == si, x.pts)
            push!(cells, q === nothing ? "n/a" : @sprintf("%.1f (%.1f s)", 100 * x.pts[q].exc, x.pts[q].snr))
            push!(pcell, q === nothing ? "n/a" : @sprintf("%.1f", 100 * x.exc0 * x.pts[q].pred))
        end
        @printf(io, "  %8.1f | %s\n", x.r0, join([@sprintf("%-13s", c) for c in cells], " "))
        @printf(io, "  %8s | %s\n", "(resol.)", join([@sprintf("%-13s", c) for c in pcell], " "))
    end
    for x in summ
        pl = x.pts[end]
        msg = if x.verdict == :flat
            "the excess survives the smoothing (x" * @sprintf("%.2f +/- %.2f", pl.rho, pl.rho_err) * " of the native value, a resolution effect would give x" * @sprintf("%.2f", pl.pred) *
            "): an amplitude-type modulation, i.e. the ring itself is brighter along one axis, not a resolution effect"
        elseif x.verdict == :falls
            "the excess shrinks like a resolution effect (x" * @sprintf("%.2f +/- %.2f", pl.rho, pl.rho_err) * " of the native value, expected x" * @sprintf("%.2f", pl.pred) *
            "): a PSF / uv-coverage shape that is not the fitted Gaussian beam, or a real azimuthal variation of the ring width"
        elseif x.verdict == :between
            "the excess is inconsistent with both an unchanged one and a resolution-type dilution (x" * @sprintf("%.2f +/- %.2f", pl.rho, pl.rho_err) * " vs x1 and x" * @sprintf("%.2f", pl.pred) *
            "): a mixture, or underestimated errors"
        else
            "not decisive: x" * @sprintf("%.2f +/- %.2f", pl.rho, pl.rho_err) * " of the native value is consistent with both an unchanged excess (x1, " * @sprintf("%.1f", abs(x.zf)) *
            " sigma away) and a resolution effect (x" * @sprintf("%.2f", pl.pred) * ", " * @sprintf("%.1f", abs(x.zr)) * " sigma away)"
        end
        @printf(io, "    r = %6.1f au: %s.\n", x.r0, msg)
    end
    if !isempty(summ)
        c_flat = sum(x.zf^2 for x in summ)
        c_res = sum(x.zr^2 for x in summ)
        @printf(io, "  Taken together (%d ring(s), strongest smoothing; the rings share one image, so this is indicative): chi2 = %.1f for an unchanged excess, %.1f for a resolution-type one.\n", length(summ), c_flat, c_res)
    end
    println(io, "  Caveats: the kernel removes the Gaussian part of the beam only (a PSF with sidelobes or uv-hole structure is not made round). The columns use the same data, so their S/N are not independent.")
    println(io, "  A width-type signal needs the PSF or a real width variation to explain it; an amplitude-type signal is what a surface-density or temperature variation along the ring would look like (check with a second observation).")
end

"""
    plot_round_beam(rb, summ; title = "", filename = "round_beam.png")

The m=2 excess against the smoothing level for each ring with a significant native excess, with the resolution-effect prediction (dashed) and the
no-change expectation (dotted).
"""
function plot_round_beam(rb, summ; title = "", filename = "round_beam.png")
    fig = Figure(size = (1300, 560))
    labs = vcat(["native"], [@sprintf("%.1f×", f) for f in rb.factors])
    ax = Axis(fig[1, 1]; xlabel = "smoothing: circular beam FWHM / native major axis", ylabel = "m = 2 excess over the Gaussian-beam expectation [%]",
              xticks = (collect(0:length(rb.sets)), labs), title = "solid: measured; dashed: resolution effect; dotted: amplitude-type (no change)", titlesize = 12,
              limits = (-0.25, length(rb.sets) + 0.25, 0, nothing))
    cols = [C_RING, C_GAP, C_FLUX, C_RESID, C_LIT]
    for (q, x) in enumerate(summ)
        col = cols[mod1(q, length(cols))]
        xs = Float64[0.0]
        ys = Float64[100 * x.exc0]
        es = Float64[100 * x.err0]
        yp = Float64[100 * x.exc0]
        for p in x.pts
            push!(xs, Float64(p.idx))
            push!(ys, 100 * p.exc)
            push!(es, 100 * p.err)
            push!(yp, 100 * x.exc0 * p.pred)
        end
        lines!(ax, xs, ys; color = col, linewidth = 1.5, label = @sprintf("%.0f au", x.r0))
        errorbars!(ax, xs, ys, es; color = col)
        scatter!(ax, xs, ys; color = col, markersize = 9)
        lines!(ax, xs, yp; color = col, linewidth = 1.0, linestyle = :dash)
        lines!(ax, [xs[1], xs[end]], [ys[1], ys[1]]; color = col, linewidth = 1.0, linestyle = :dot)
    end
    hlines!(ax, [0.0]; color = (FG_COLOR, 0.4), linewidth = 0.8)
    axislegend(ax; position = :rb)
    isempty(title) || Label(fig[0, 1], title; fontsize = 14, color = FG_COLOR, tellwidth = false)
    display(fig)
    _save_fig(fig, filename)
    return fig
end

In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (item 3): gap-opening planet masses from gap depth and width
# ---------------------------------------------------------------------------

const PHYS_MJUP = 1.89813e27      # kg
const PHYS_MNEP = 1.02413e26      # kg

# Mp/M* from the Kanagawa et al. (2016) width relation, Delta_gap/Rp = 0.41 (Mp/M*)^(1/2) (h/r)^(-3/4) alpha^(-1/4)
_q_from_width(wr::Real, hr::Real, alpha::Real) = 2.1e-3 * wr^2 * (hr / 0.05)^1.5 * sqrt(alpha / 1e-3)

# Mp/M* from the Kanagawa et al. (2015) depth relation, Sigma_min/Sigma_0 = 1/(1 + 0.04 K), K = q^2 (h/r)^-5 alpha^-1
# (delta = Sigma_0 / Sigma_min >= 1)
_q_from_depth(delta::Real, hr::Real, alpha::Real) = sqrt(25 * max(delta - 1, 0.0) * alpha * hr^5)

# radius where the profile first rises through `level`, walking from bin i0 in direction `step` but no further than bin `ilim`
# (the neighbouring ring peak, so two gaps cannot merge); NaN if it never does
function _edge_crossing(r, y, i0::Int, step::Int, level::Real, ilim::Int)
    n = length(r)
    j = i0
    while true
        k = j + step
        (1 <= k <= n && isfinite(y[k])) || return NaN
        (step > 0 ? k <= ilim : k >= ilim) || return NaN        # never walk past the neighbouring ring peak
        if y[k] >= level
            f = (level - y[j]) / (y[k] - y[j])
            return r[j] + f * (r[k] - r[j])
        end
        j = k
    end
end

# index of the local maximum reached by climbing from bin i in direction `step` (looks two bins ahead to ride over noise)
function _climb(y, i::Int, step::Int)
    n = length(y)
    j = i
    while true
        a = j + step
        b = j + 2 * step
        (1 <= a <= n && isfinite(y[a])) || break
        bok = 1 <= b <= n && isfinite(y[b])
        nxt = bok ? max(y[a], y[b]) : y[a]
        nxt > y[j] || break
        j = (bok && y[b] > y[a]) ? b : a
    end
    return j
end

"""
    measure_gap(profile, r_gap_au) -> NamedTuple or nothing

Depth and width of one gap on the azimuthal-mean profile, following the definitions of Zhang et al. (2018): the gap minimum,
the next ring peak outside it, the edge level (peak + minimum) / 2, and the width between the two crossings of that level.
Each edge is searched only up to the neighbouring ring peak, so two adjacent gaps keep their own edges. If the inner side never reaches
the edge level before its ring peak (the inner ring is fainter), the width is twice the outer half-width.
Returns `nothing` when the gap cannot be measured; the reason is left in `_GAP_FAIL[]`.
"""
const _GAP_FAIL = Ref("")
_gapfail(msg::AbstractString) = (_GAP_FAIL[] = String(msg); nothing)

function measure_gap(profile, r_gap_au::Real)
    r = Float64.(profile.radius_au)
    y = Float64.(profile.mean_flux)
    n = length(r)
    rms = (isfinite(profile.image_rms) && profile.image_rms > 0) ? Float64(profile.image_rms) : 0.0
    nb = _profile_nbeam(profile)
    half = 0.5 * Float64(profile.bmaj_au)
    i0 = 0
    best = Inf
    for i in 1:n
        (isfinite(y[i]) && abs(r[i] - r_gap_au) <= half) || continue
        if y[i] < best
            best = y[i]
            i0 = i
        end
    end
    i0 == 0 && return _gapfail("no profile bin within half a beam of the gap radius")
    # the window minimum can sit on a slope at the edge of the window: slide down to the true local minimum, but at most 1.5 beams away (a baseline-subtracted gap can sit about a beam inside the raw minimum)
    negy = -y
    j0 = _climb(negy, _climb(negy, i0, 1), -1)
    abs(r[j0] - r_gap_au) <= 1.5 * Float64(profile.bmaj_au) && (i0 = j0)
    ipk = _climb(y, i0, 1)
    ipk == i0 && return _gapfail(@sprintf("no local maximum outward of the gap minimum at %.1f au (the profile does not rise there)", r[i0]))
    imin = y[i0]
    ipeak = y[ipk]
    ipeak > imin || return _gapfail("the outer peak is not above the gap minimum")
    s_min = rms / sqrt(max(Float64(nb[i0]), 1.0))
    s_pk = rms / sqrt(max(Float64(nb[ipk]), 1.0))
    sig = rms > 0 ? (ipeak - imin) / hypot(s_min, s_pk) : Inf
    imin_eff = max(imin, s_min)                    # an empty gap gives only a lower limit on the contrast
    delta = ipeak / imin_eff
    limit = imin < 3 * s_min                       # gap floor within 3 sigma of zero: the depth is only a lower limit
    level = 0.5 * (ipeak + imin)
    ipk_in = _climb(y, i0, -1)
    r_out = _edge_crossing(r, y, i0, 1, level, ipk)
    isfinite(r_out) || return _gapfail(@sprintf("outer edge not found: the edge level (%.3g mJy/beam) is not reached before the outer peak at %.1f au", 1e3 * level, r[ipk]))
    r_in = _edge_crossing(r, y, i0, -1, level, ipk_in)
    missing_in = !isfinite(r_in)
    w_obs = missing_in ? 2 * (r_out - r[i0]) : (r_out - r_in)
    r_c = missing_in ? r[i0] : 0.5 * (r_in + r_out)
    return (r_min = r[i0], r_c = r_c, r_in = r_in, r_out = r_out, r_peak = r[ipk], delta = delta, delta_limit = limit,
            sig = sig, w_obs = w_obs, inner_missing = missing_in)
end

"""
    gap_planet_masses(profile, gaps_au, lstar, mstar; phi = 0.02, alphas = (1e-3, 1e-4), min_snr = 3.0) -> Vector

For each gap radius: measure the depth and width (`measure_gap`), take h/r from the passive-disk temperature at the gap, and
convert to a planet mass for each alpha. Two estimates:
  width: Kanagawa+16 relation, range from the beam-deconvolved width (low) to the observed width (high)
  depth: Kanagawa+15 relation applied to the dust-intensity contrast (a cross-check only; dust gaps are deeper than gas gaps)
Masses are returned as Mp/M* (`qw_lo`, `qw_hi`, `qd`) for each alpha.
"""
function gap_planet_masses(profile, gaps_au::AbstractVector{<:Real}, lstar::Real, mstar::Real; phi::Real = 0.02,
                           alphas = (1e-3, 1e-4), min_snr::Real = 3.0)
    bf = sqrt(Float64(profile.bmaj_au) * Float64(profile.bmin_au))
    res = NamedTuple[]
    for rg in sort(collect(Float64.(gaps_au)))
        g = measure_gap(profile, rg)
        if g === nothing
            push!(res, (r_gap = rg, ok = false, note = _GAP_FAIL[]))
            continue
        end
        if g.sig < min_snr
            push!(res, (r_gap = rg, ok = false, note = @sprintf("gap contrast only %.1f sigma (< %.1f)", g.sig, min_snr)))
            continue
        end
        Td = _dust_temperature(g.r_c, lstar; phi = phi)
        hr = _gas_scale_height_au(g.r_c, Td, mstar) / g.r_c
        w_int = sqrt(max(g.w_obs^2 - bf^2, 0.0))
        wr_obs = g.w_obs / g.r_c
        wr_int = w_int / g.r_c
        ms = NamedTuple[]
        for a in alphas
            push!(ms, (alpha = Float64(a), qw_lo = _q_from_width(wr_int, hr, a), qw_hi = _q_from_width(wr_obs, hr, a),
                       qd = _q_from_depth(g.delta, hr, a)))
        end
        push!(res, (r_gap = rg, ok = true, note = "", g = g, hr = hr, w_int = w_int, wr_obs = wr_obs, wr_int = wr_int,
                    beam_au = bf, masses = ms))
    end
    return res
end

# "lo-hi Mnep" or "lo-hi Mjup" (a single value when lo == hi; "< hi" when lo <= 0)
function _mass_txt(q_lo::Real, q_hi::Real, mstar::Real)
    mlo = q_lo * mstar * PHYS_MSUN / PHYS_MJUP
    mhi = q_hi * mstar * PHYS_MSUN / PHYS_MJUP
    if mhi < 0.3
        lo = mlo * PHYS_MJUP / PHYS_MNEP
        hi = mhi * PHYS_MJUP / PHYS_MNEP
        q_lo <= 0 && return @sprintf("< %.2g Mnep", hi)
        q_lo == q_hi && return @sprintf("%.2g Mnep", hi)
        return @sprintf("%.2g-%.2g Mnep", lo, hi)
    end
    q_lo <= 0 && return @sprintf("< %.2g Mjup", mhi)
    q_lo == q_hi && return @sprintf("%.2g Mjup", mhi)
    return @sprintf("%.2g-%.2g Mjup", mlo, mhi)
end

function _planet_report(io::IO, pm, sp; props = LitProp[])
    println(io, "  Gap depth and width are measured on the azimuthal-mean profile (Zhang et al. 2018 definitions: next ring peak outside the gap, edge level = (peak + minimum)/2).")
    println(io, "  Width estimate: Kanagawa et al. (2016) Delta_gap/Rp = 0.41 (Mp/M*)^(1/2) (h/r)^(-3/4) alpha^(-1/4), i.e. Mp/M* = 2.1e-3 (Delta_gap/Rp)^2 (h/r / 0.05)^(3/2) (alpha/1e-3)^(1/2).")
    println(io, "  Depth estimate: Kanagawa et al. (2015) Sigma_min/Sigma_0 = 1/(1 + 0.04 K), K = (Mp/M*)^2 (h/r)^(-5) alpha^(-1), with Sigma_0/Sigma_min taken as the dust-intensity contrast.")
    println(io, "  (Zhang et al. 2018 give dust-specific calibrations that also depend on grain size; they are not applied here, so the dust gaps are read with the gas relations.)")
    @printf(io, "  M* = %.2f Msun; h/r from the passive-disk T_d at each gap (H = c_s/v_K r). Both alpha values are shown because the mass scales roughly as alpha^(1/2) (width) and alpha^(1/2) (depth).\n\n", sp.mstar)
    @printf(io, "  %6s %6s %6s %6s | %5s %5s %6s | %-18s %-18s | %-18s %-18s  %s\n", "gap", "r_c", "h/r", "depth", "W_obs", "W_int", "W/r",
            "width a=1e-3", "width a=1e-4", "depth a=1e-3", "depth a=1e-4", "notes")
    skipped = String[]
    for x in pm
        if !x.ok
            push!(skipped, @sprintf("%.0f au: %s", x.r_gap, x.note))
            continue
        end
        g = x.g
        dtxt = g.delta_limit ? @sprintf(">%.0f", g.delta) : @sprintf("%.2f", g.delta)
        m1 = x.masses[1]
        m2 = x.masses[end]
        flags = String[]
        g.inner_missing && push!(flags, "inner ring peak below the edge level (width = 2 x outer half-width)")
        if !g.inner_missing && isfinite(g.r_in)
            hin = g.r_min - g.r_in
            hout = g.r_out - g.r_min
            (min(hin, hout) > 0 && max(hin, hout) / min(hin, hout) >= 1.5) &&
                push!(flags, @sprintf("asymmetric gap: half-widths %.1f au inside, %.1f au outside the minimum (%.1f au); a sloping baseline or a neighbouring arc may bias the width", hin, hout, g.r_min))
        end
        g.delta < 2.2 && push!(flags, "shallow (< 2.2): width relation formally invalid")
        x.w_int <= 0 && push!(flags, "unresolved: width is an upper limit")
        g.delta_limit && push!(flags, "gap floor at the noise: depth is a lower limit, no depth mass")
        dm1 = g.delta_limit ? "n/a (noise floor)" : _mass_txt(m1.qd, m1.qd, sp.mstar)
        dm2 = g.delta_limit ? "n/a (noise floor)" : _mass_txt(m2.qd, m2.qd, sp.mstar)
        @printf(io, "  %6.1f %6.1f %6.3f %6s | %5.1f %5.1f %6.3f | %-18s %-18s | %-18s %-18s  %s\n", x.r_gap, g.r_c, x.hr, dtxt, g.w_obs, x.w_int, x.wr_obs,
                _mass_txt(m1.qw_lo, m1.qw_hi, sp.mstar), _mass_txt(m2.qw_lo, m2.qw_hi, sp.mstar), dm1, dm2, join(flags, "; "))
    end
    isempty(skipped) || println(io, "\n  Not evaluated: ", join(skipped, "; "))
    okx = [x for x in pm if x.ok]
    bfs = isempty(okx) ? NaN : okx[1].beam_au
    bftxt = isfinite(bfs) ? @sprintf("geometric-mean FWHM %.1f au", bfs) : "the beam"
    println(io, "\n  Columns: gap = radius you asked for, r_c = centre of the measured gap; depth = I(ring peak outside) / I(gap minimum); W_obs / W_int = gap width at the edge level, observed and with the beam (", bftxt,
            ") removed in quadrature; W/r = observed width over the gap centre.")
    println(io, "  The width range runs from the deconvolved to the observed width. Treat the numbers as order-of-magnitude: alpha is unknown (the two values bracket the usual range),")
    println(io, "  h/r comes from an assumed passive-disk temperature, the width relation was calibrated for gas gaps (dust gaps from large grains are wider, so masses are upper estimates),")
    println(io, "  the depth is diluted by the beam (lower bound on the contrast) but dust gaps are deeper than gas gaps (upper bound on the mass), and several gaps may be carved by one planet or by other mechanisms.")
    if !isempty(props)
        rows = String[]
        for x in okx
            g = x.g
            p = _nearest_prop(props, g.r_c, :gap, max(isfinite(bfs) ? bfs : 5.0, 0.08 * g.r_c))
            p === nothing && continue
            pw = @sprintf("%.1f%s", p[2], p[5] ? " (<beam)" : "")
            pd = isfinite(p[3]) ? @sprintf("%.3g (x%.3g)", p[3], 1 / p[3]) : "-"
            od = g.delta_limit ? @sprintf(">x%.0f", g.delta) : @sprintf("x%.3g", g.delta)
            push!(rows, @sprintf("  %6.1f | %6.1f %-15s %-16s | %6.1f %-9s %s", g.r_c, p[1], pw, pd, g.w_obs, od,
                                 isfinite(p[2]) && p[2] > 0 ? @sprintf("%.2f", g.w_obs / p[2]) : "-"))
        end
        if !isempty(rows)
            println(io, "\n  PUBLISHED WIDTHS AND DEPTHS of the matched gaps (Huang+18, DSHARP Table 1; radii and widths rescaled to the adopted distance):")
            @printf(io, "  %6s | %6s %-15s %-16s | %6s %-9s %s\n", "r_c", "pub r", "pub width", "pub I_gap/I_ring", "W_obs", "depth", "W_obs/W_pub")
            foreach(r -> println(io, r), rows)
            println(io, "  Published width = outer minus inner edge of the gap, with the beam NOT removed, so W_obs is the comparable number. Published depth = mean intensity in the radial bin at the gap over that at the exterior ring;")
            println(io, "  'x' is its inverse, to compare with the depth column above, which uses the gap minimum and the next ring peak (so it is the larger of the two for the same gap). (<beam) = narrower than the beam, so diluted.")
        end
    end
end


In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (item 4): ring widths, dust trapping (alpha / St) and ring spacing
# ---------------------------------------------------------------------------
# A dust ring held in a Gaussian gas pressure bump of width w has the width
#     w_d = w * sqrt(alpha / (alpha + St))          (Dullemond et al. 2018, DSHARP VI, Eqs. 15-16, Schmidt number 1)
# so alpha / St = 1 / ((w / w_d)^2 - 1)             (their Eq. 21).
# w_d is the Gaussian sigma of the ring with the beam removed. A long-lived gas bump cannot be narrower than about one
# pressure scale height H, so w >= max(H, w_d): the bump width w = H gives the LARGEST alpha / St the ring allows, and a
# wider bump (up to the distance to the neighbouring gap) gives smaller values. A ring wider than H gives no limit.

const _RESONANCES = ((6, 5), (5, 4), (4, 3), (3, 2), (5, 3), (2, 1), (5, 2), (3, 1))

_alpha_over_st(wr::Real) = wr > 1 ? 1 / (wr^2 - 1) : Inf          # wr = w / w_d

"""
    ring_trapping(fit, gaps_au, lstar, mstar; phi = 0.02, min_snr = 3.0, alphas = (1e-3, 1e-4)) -> Vector{NamedTuple}

For every fitted ring with amplitude S/N >= `min_snr`: the deconvolved Gaussian width sigma_d (an upper limit when the ring is
unresolved), the gas scale height H at the ring (passive-disk temperature), sigma_d / H, and the dust-trapping limits on
alpha / St: `a_max` for a bump as narrow as H (with the +/- 1 sigma range from the width error, `a_max_lo`, `a_max_hi`) and
`a_min` for a bump as wide as the distance to the nearest gap (NaN if that is not wider than H). `st` holds the Stokes
number range for each alpha. `status` is `:limit` or `:no_limit` (ring wider than H).
"""
function ring_trapping(fit, gaps_au::AbstractVector{<:Real}, lstar::Real, mstar::Real; phi::Real = 0.02,
                       min_snr::Real = 3.0, alphas = (1e-3, 1e-4))
    res = NamedTuple[]
    for rg in fit.rings
        rg.amp_snr >= min_snr || continue
        H = _gas_scale_height_au(rg.r0, _dust_temperature(rg.r0, lstar; phi = phi), mstar)
        resolved = rg.resolved
        sd = (resolved ? rg.fwhm_int : rg.fwhm_ub) / _FWHM_PER_SIGMA
        sd_err = resolved ? rg.fwhm_int_err / _FWHM_PER_SIGMA : 0.0
        wmax = isempty(gaps_au) ? NaN : minimum(abs.(Float64.(gaps_au) .- rg.r0))
        if !(sd < H)
            push!(res, (r0 = rg.r0, H = H, sd = sd, sd_err = sd_err, limit_width = !resolved, wd_h = sd / H, wmax = wmax,
                        status = :no_limit, a_max = NaN, a_max_lo = NaN, a_max_hi = NaN, a_min = NaN, st = NamedTuple[]))
            continue
        end
        a_max = _alpha_over_st(H / sd)
        a_lo = _alpha_over_st(H / max(sd - sd_err, 1e-6))
        a_hi = sd + sd_err < H ? _alpha_over_st(H / (sd + sd_err)) : Inf
        a_min = (isfinite(wmax) && wmax > H) ? _alpha_over_st(wmax / sd) : NaN
        st = NamedTuple[]
        for a in alphas
            push!(st, (alpha = Float64(a), st_lo = a / a_max, st_hi = isfinite(a_min) && a_min > 0 ? a / a_min : NaN))
        end
        push!(res, (r0 = rg.r0, H = H, sd = sd, sd_err = sd_err, limit_width = !resolved, wd_h = sd / H, wmax = wmax,
                    status = :limit, a_max = a_max, a_max_lo = a_lo, a_max_hi = a_hi, a_min = a_min, st = st))
    end
    return res
end

# period ratios of neighbouring radii and the nearest low-order commensurability
function ring_period_ratios(r_au::AbstractVector{<:Real})
    r = sort(collect(Float64.(r_au)))
    out = NamedTuple[]
    for i in 1:(length(r) - 1)
        pr = (r[i + 1] / r[i])^1.5
        best = (1, 1)
        bd = Inf
        for (p, q) in _RESONANCES
            d = abs(pr / (p / q) - 1)
            if d < bd
                bd = d
                best = (p, q)
            end
        end
        push!(out, (r1 = r[i], r2 = r[i + 1], rratio = r[i + 1] / r[i], pratio = pr, p = best[1], q = best[2], dev = 100 * (pr / (best[1] / best[2]) - 1)))
    end
    return out
end

function _range_txt(lo::Real, hi::Real)
    isfinite(lo) || return "-"
    isfinite(hi) || return @sprintf(">= %.1e", lo)
    return @sprintf("%.1e-%.1e", lo, hi)
end

function _trapping_report(io::IO, tr, gaps_au, sp; pm = nothing)
    println(io, "  A dust ring in a Gaussian gas pressure bump of width w has width w_d = w sqrt(alpha / (alpha + St)), i.e. alpha / St = 1 / ((w / w_d)^2 - 1) (Dullemond et al. 2018, DSHARP VI).")
    println(io, "  w_d = Gaussian sigma of the fitted ring with the beam removed. A long-lived bump cannot be narrower than about one gas scale height H (passive-disk temperature, M* = ",
            @sprintf("%.2f", sp.mstar), " Msun), so w = H gives the LARGEST alpha / St the ring allows;")
    println(io, "  a bump as wide as the distance to the nearest gap gives the smallest. A ring wider than H gives no limit: the bump could be as wide as the ring.")
    println(io)
    @printf(io, "  %8s %9s %7s %6s | %-20s %-12s | %-19s %-19s  %s\n", "r [au]", "sigma_d", "H [au]", "sd/H", "alpha/St max (w=H)", "min (w=w_max)",
            "St (alpha=1e-3)", "St (alpha=1e-4)", "notes")
    for x in tr
        sdtxt = x.limit_width ? @sprintf("<%.1f", x.sd) : @sprintf("%.1f+/-%.1f", x.sd, x.sd_err)
        if x.status === :no_limit
            @printf(io, "  %8.1f %9s %7.1f %6.2f | %-20s %-12s | %-19s %-19s  %s\n", x.r0, sdtxt, x.H, x.wd_h, "no limit", "-", "-", "-",
                    x.limit_width ? "unresolved and not narrower than H" : "ring at least as wide as H")
            continue
        end
        amax = x.limit_width ? @sprintf("< %.2f", x.a_max) :
               (isfinite(x.a_max_hi) ? @sprintf("%.2f (%.2f-%.2f)", x.a_max, x.a_max_lo, x.a_max_hi) : @sprintf("%.2f (>%.2f)", x.a_max, x.a_max_lo))
        amin = isfinite(x.a_min) ? @sprintf("%.3f", x.a_min) : "-"
        s1 = x.st[1]
        s2 = x.st[end]
        note = x.limit_width ? "width is an upper limit" : ""
        @printf(io, "  %8.1f %9s %7.1f %6.2f | %-20s %-12s | %-19s %-19s  %s\n", x.r0, sdtxt, x.H, x.wd_h, amax, amin,
                _range_txt(s1.st_lo, s1.st_hi), _range_txt(s2.st_lo, s2.st_hi), note)
    end
    println(io, "\n  alpha/St: ratio of the turbulent diffusion parameter to the Stokes number of the trapped grains (smaller = stronger trapping). The range in the 'St' columns is alpha divided by the two limits,")
    println(io, "  for the two alpha values of section 4; '>=' means no upper limit (the nearest gap is not wider than H). The ring width is the intensity width: for optically thick rings it overestimates the dust-density width,")
    println(io, "  so the true rings are, if anything, narrower (stronger trapping). Blended components (see the warning in section 1) and rings with large errors are unreliable; trust the isolated rings.")
    # ring spacing
    gp = pm === nothing ? sort(collect(Float64.(gaps_au))) : sort(Float64[x.g.r_c for x in pm if x.ok])
    println(io, "\n  Spacing of the gaps (planets sit in the gaps): radius ratio, orbital period ratio (r2/r1)^(3/2) and the nearest low-order commensurability",
            pm === nothing ? ":" : " (measured gaps only, at their measured centres r_c):")
    if length(gp) < 2
        println(io, "  fewer than two gaps: nothing to compare.")
    else
        for x in ring_period_ratios(gp)
            @printf(io, "    %6.1f -> %6.1f au: radius ratio %.2f, period ratio %.2f, nearest %d:%d (%+.1f%%)%s\n", x.r1, x.r2, x.rratio, x.pratio, x.p, x.q, x.dev,
                    abs(x.dev) <= 3 ? "  within 3%" : "")
        end
        println(io, "  Suggestive only: the ratios above are of the gap radii, not of measured planet orbits, and a +/-3% match to one of the eight ratios 6:5, 5:4, 4:3, 3:2, 5:3, 2:1, 5:2, 3:1")
        println(io, "  happens by chance for about 40% of all values between 1.1 and 3.2, so a near-resonance is weak evidence on its own.")
    end
end


In [ ]:
# ---------------------------------------------------------------------------
# Physical analysis (3/3): brightness temperature, optical depth, and the driver
# ---------------------------------------------------------------------------

const PHYS_H        = 6.62607015e-34      # J s
const PHYS_KB       = 1.380649e-23        # J / K
const PHYS_C        = 2.99792458e8        # m / s
const PHYS_SIGMA_SB = 5.670374419e-8      # W m^-2 K^-4
const PHYS_LSUN     = 3.828e26            # W
const PHYS_AU       = 1.495978707e11      # m
const PHYS_MSUN     = 1.98847e30          # kg
const PHYS_G        = 6.6743e-11          # m^3 kg^-1 s^-2
const PHYS_MH       = 1.6735575e-27       # kg

# Stellar luminosity [Lsun] and mass [Msun]. The first ten entries are APPROXIMATE values entered from memory
# (mostly Andrews et al. 2018, DSHARP I); the later ones come from Andrews+18 Table 1 (MIST masses), Oberg+21 (MAPS I,
# dynamical masses) and Long+18 Table 2 (Taurus). Masses from different methods differ (e.g. AS 209: 0.83 MIST, 1.2 dynamical).
# VERIFY before quoting. Pass `lstar=` / `mstar=` to
# `physical_analysis` for any other target (the temperature and scale-height numbers need them).
const STELLAR_PARAMS = Dict{String,NamedTuple{(:lstar, :mstar, :ref),Tuple{Float64,Float64,String}}}(
    "as209"    => (lstar = 1.41, mstar = 0.83, ref = "Andrews+18 (approx.)"),
    "hd163296" => (lstar = 17.0, mstar = 2.0,  ref = "Andrews+18 (approx.)"),
    "twhya"    => (lstar = 0.28, mstar = 0.8,  ref = "Andrews+12 / Teague (approx.)"),
    "imlup"    => (lstar = 2.57, mstar = 1.1,  ref = "Andrews+18 (approx.)"),
    "gwlup"    => (lstar = 0.33, mstar = 0.46, ref = "Andrews+18 (approx.)"),
    "elias24"  => (lstar = 6.0,  mstar = 0.78, ref = "Andrews+18 (approx.)"),
    "elias27"  => (lstar = 0.91, mstar = 0.49, ref = "Andrews+18 (approx.)"),
    "elias20"  => (lstar = 2.24, mstar = 0.48, ref = "Andrews+18 (approx.)"),
    "doar25"   => (lstar = 0.95, mstar = 0.95, ref = "Andrews+18 (approx.)"),
    "hd143006" => (lstar = 3.8,  mstar = 1.8,  ref = "Andrews+18 (approx.)"),
    # added from the published tables (values read from the papers through a text summary; VERIFY before quoting):
    "htlup"   => (lstar = 5.5, mstar = 1.7, ref = "Andrews+18 Table 1 (MIST mass); triple system"),
    "mylup"   => (lstar = 0.871, mstar = 1.23, ref = "Andrews+18 Table 1 (MIST mass); L* may be underestimated"),
    "rulup"   => (lstar = 1.45, mstar = 0.63, ref = "Andrews+18 Table 1 (MIST mass)"),
    "sz114"   => (lstar = 0.204, mstar = 0.17, ref = "Andrews+18 Table 1 (MIST mass)"),
    "sz129"   => (lstar = 0.437, mstar = 0.83, ref = "Andrews+18 Table 1 (MIST mass)"),
    "hd142666"=> (lstar = 9.12, mstar = 1.58, ref = "Andrews+18 Table 1 (MIST mass)"),
    "as205"   => (lstar = 2.14, mstar = 0.87, ref = "Andrews+18 Table 1 (MIST mass); triple system"),
    "sr4"     => (lstar = 1.17, mstar = 0.68, ref = "Andrews+18 Table 1 (MIST mass)"),
    "doar33"  => (lstar = 1.51, mstar = 1.1, ref = "Andrews+18 Table 1 (MIST mass)"),
    "wsb52"   => (lstar = 0.708, mstar = 0.48, ref = "Andrews+18 Table 1 (MIST mass)"),
    "gmaur"   => (lstar = 1.2, mstar = 1.1, ref = "Oberg+21 MAPS I (dynamical mass)"),
    "mwc480"  => (lstar = 21.9, mstar = 2.1, ref = "Oberg+21 MAPS I (dynamical mass)"),
    "iptau"   => (lstar = 0.339, mstar = 0.52, ref = "Long+18 Table 2"),
    "rytau"   => (lstar = 12.303, mstar = 2.04, ref = "Long+18 Table 2"),
    "uztau"   => (lstar = 0.347, mstar = 0.39, ref = "Long+18 Table 2 (disk E component)"),
    "dstau"   => (lstar = 0.245, mstar = 0.58, ref = "Long+18 Table 2"),
    "fttau"   => (lstar = 0.148, mstar = 0.34, ref = "Long+18 Table 2"),
    "dntau"   => (lstar = 0.692, mstar = 0.52, ref = "Long+18 Table 2"),
    "gotau"   => (lstar = 0.214, mstar = 0.36, ref = "Long+18 Table 2"),
    "iqtau"   => (lstar = 0.214, mstar = 0.5, ref = "Long+18 Table 2"),
    "dltau"   => (lstar = 0.646, mstar = 0.98, ref = "Long+18 Table 2"),
    "citau"   => (lstar = 0.813, mstar = 0.89, ref = "Long+18 Table 2"),
    # added from papers read through a text summary (VERIFY before quoting):
    "xztau"   => (lstar = 0.17, mstar = 0.37, ref = "XZ Tau B only (A: 0.31 Lsun, 0.29 Msun); arXiv:2106.11924 Table 5 (Hartigan & Kenyon 2003, Herczeg & Hillenbrand 2014), mass also Osorio+16"),
    "zztau"   => (lstar = 0.652, mstar = 0.86, ref = "ZZ Tau A+B binary: L = 0.411 + 0.241 (Schaefer+14); M = dynamical total 0.86 +/- 0.09 (arXiv:2206.14154; the H-R diagram sum is 0.59)"),
    "tmc1a"   => (lstar = 2.7, mstar = 0.68, ref = "Aso+15 (dynamical mass, inclination-corrected; Lbol is the total, accretion included)"),
)

# ---------------------------------------------------------------------------
# Published disc properties from Bosschaart+26 (VizieR J/A+A/708/A143/tabled1, 730 discs): class, 68% effective dust
# radius R68, 1.33 mm flux, dust mass, stellar mass, distance. Matched to the targets by prefill_disc_info! (menu cell);
# keyed by AlmaTarget.slug, empty until that has run.
# ---------------------------------------------------------------------------
const DISC_INFO = Dict{String,NamedTuple}()
const ESTNOTE = "  L* ESTIMATED from the stellar mass (L = M^2, a rough pre-main-sequence relation; the true value can differ by a factor of several). It sets the dust temperature, h/r and so the masses below; a factor-3 error in L* changes the planet masses by roughly 25%."
const DISC_CLASS = Dict("CD" => "compact disc", "ED" => "extended disc", "RD" => "ring disc",
                        "TD" => "transition disc", "HID" => "highly inclined disc", "ND" => "non-detection")
_disc_info(t) = get(DISC_INFO, t.slug, nothing)

"One-line summary of the catalogue entry for `t` (empty if none). `r68_measured` = the 68%-flux radius of the image used."
function disc_info_line(t; r68_measured::Real = NaN)
    di = _disc_info(t)
    di === nothing && return ""
    parts = String[]
    isempty(di.class) || push!(parts, "class " * di.class * " (" * get(DISC_CLASS, di.class, "?") * ")")
    isfinite(di.r68_au) && push!(parts, @sprintf("R68 %s%.0f au", di.r68_lim, di.r68_au))
    isfinite(di.fcont_mjy) && push!(parts, @sprintf("F(1.33 mm) %s%.3g mJy", di.fcont_lim, di.fcont_mjy))
    isfinite(di.mdust_earth) && push!(parts, @sprintf("M_dust %s%.3g Earth masses", di.mdust_lim, di.mdust_earth))
    isfinite(di.mstar) && push!(parts, @sprintf("M* %.2f Msun", di.mstar))
    if isfinite(di.dist_pc)
        d = @sprintf("distance %.0f pc", di.dist_pc)
        isempty(di.dist_flag) || (d *= " (flag " * di.dist_flag * ")")
        if t.distance_known && abs(t.distance_pc / di.dist_pc - 1) > 0.2
            d *= @sprintf(", differs from the %.0f pc used", t.distance_pc)
        end
        push!(parts, d)
    end
    if isfinite(r68_measured) && isfinite(di.r68_au)
        push!(parts, @sprintf("this image: 68%%-flux radius %.0f au", r68_measured))
    end
    return "Bosschaart+26 (" * di.name * "): " * join(parts, "; ")
end

# console hint printed after the archive query, before any download
function _print_disc_hint(t, obs, distance_pc)
    di = _disc_info(t)
    di === nothing && return nothing
    println("Catalogue: ", disc_info_line(t))
    di.class == "ND" && println("NOTE: the catalogue lists this disc as a NON-DETECTION; expect nothing in the image.")
    di.class == "HID" && println("NOTE: highly inclined disc: radial profiles, deprojection and ring widths are less reliable.")
    if isfinite(di.r68_au) && obs !== nothing && nrow(obs) > 0 && t.distance_known
        beam_au = Float64(obs.best_res_arcsec[1]) * Float64(distance_pc)
        n = di.r68_au / beam_au
        verdict = n < 1.5 ? "UNRESOLVED at the finest resolution" : (n < 4 ? "marginally resolved" : "resolved")
        @printf("Finest archive beam ~ %.0f au; catalogue R68 = %.0f au = %.1f beams: %s.\n", beam_au, di.r68_au, n, verdict)
    end
    return nothing
end

# (lstar, mstar, source); either may be `nothing`
function _stellar_params(t; lstar = nothing, mstar = nothing)
    sp = nothing
    for k in vcat(_norm(t.name), t.keys)
        if haskey(STELLAR_PARAMS, k)
            sp = STELLAR_PARAMS[k]
            break
        end
    end
    L = lstar !== nothing ? Float64(lstar) : (sp === nothing ? nothing : sp.lstar)
    M = mstar !== nothing ? Float64(mstar) : (sp === nothing ? nothing : sp.mstar)
    src = (lstar !== nothing || mstar !== nothing) ? "user-supplied" : (sp === nothing ? "none" : sp.ref)
    di = _disc_info(t)
    if M === nothing && mstar === nothing && di !== nothing && isfinite(di.mstar)      # catalogue stellar mass (no L*)
        M = di.mstar
        sp === nothing && lstar === nothing && (src = "Bosschaart+26 catalogue stellar mass")
    end
    return (lstar = L, mstar = M, source = src)
end

# Planck-inverted brightness temperature from specific intensity I [W m^-2 Hz^-1 sr^-1]
_tb_planck(I::Real, nu::Real) = I > 0 ? (PHYS_H * nu / PHYS_KB) / log1p(2 * PHYS_H * nu^3 / (PHYS_C^2 * I)) : NaN

# Passive-disk dust temperature (Dullemond et al. 2018): T = (phi L / (8 pi r^2 sigma_SB))^(1/4), floored
_dust_temperature(r_au::Real, lstar::Real; phi::Real = 0.02, tfloor::Real = 10.0) =
    max((phi * lstar * PHYS_LSUN / (8π * (r_au * PHYS_AU)^2 * PHYS_SIGMA_SB))^0.25, tfloor)

# Gas pressure scale height H = (c_s / v_K) r, with mean molecular weight 2.3
function _gas_scale_height_au(r_au::Real, Td::Real, mstar::Real)
    cs = sqrt(PHYS_KB * Td / (2.3 * PHYS_MH))
    vk = sqrt(PHYS_G * mstar * PHYS_MSUN / (r_au * PHYS_AU))
    return (cs / vk) * r_au
end

"""
    brightness_temperature_profile(profile, freq_hz, lstar; phi = 0.02, tfloor = 10.0) -> NamedTuple

Per radial bin: Planck brightness temperature T_b (with its statistical error), the passive-disk dust
temperature T_d, the ratio T_b / T_d and the optical depth tau = -ln(1 - T_b / T_d) (NaN where T_b >= T_d).
Assumes absorption only (no scattering) and a beam-averaged intensity, so T_b is a lower bound for
features narrower than the beam.
"""
function brightness_temperature_profile(profile, freq_hz::Real, lstar::Real; phi::Real = 0.02, tfloor::Real = 10.0)
    lowercase(profile.bunit) == "jy/beam" || error("brightness temperature needs a Jy/beam image (BUNIT = $(profile.bunit))")
    (profile.bmaj_arcsec === nothing || profile.bmin_arcsec === nothing) && error("no beam in the FITS header")
    omega = (π / (4 * log(2))) * (profile.bmaj_arcsec * π / 648000) * (profile.bmin_arcsec * π / 648000)     # sr
    nbeam = _profile_nbeam(profile)
    r = Float64.(profile.radius_au)
    y = Float64.(profile.mean_flux)
    rms = (isfinite(profile.image_rms) && profile.image_rms > 0) ? Float64(profile.image_rms) : 0.0
    n = length(r)
    tb = fill(NaN, n)
    tberr = fill(NaN, n)
    td = fill(NaN, n)
    ratio = fill(NaN, n)
    tau = fill(NaN, n)
    for i in 1:n
        td[i] = _dust_temperature(r[i], lstar; phi = phi, tfloor = tfloor)
        isfinite(y[i]) || continue
        I = y[i] / omega * 1e-26
        sI = rms / sqrt(nbeam[i]) / omega * 1e-26
        tb[i] = _tb_planck(I, freq_hz)
        thi = _tb_planck(I + sI, freq_hz)
        tlo = I > sI ? _tb_planck(I - sI, freq_hz) : 0.0
        tberr[i] = (isfinite(thi) && isfinite(tlo)) ? 0.5 * (thi - tlo) : NaN
        ratio[i] = tb[i] / td[i]
        tau[i] = ratio[i] < 1 ? -log1p(-ratio[i]) : NaN
    end
    return (r_au = r, tb = tb, tb_err = tberr, td = td, ratio = ratio, tau = tau, freq_hz = Float64(freq_hz),
            lstar = Float64(lstar), phi = Float64(phi), tfloor = Float64(tfloor))
end

# nearest valid bin
function _nearest_bin(tb, r_au)
    best = 0
    bd = Inf
    for i in eachindex(tb.r_au)
        isfinite(tb.tb[i]) || continue
        d = abs(tb.r_au[i] - r_au)
        if d < bd
            bd = d
            best = i
        end
    end
    return best
end

function plot_brightness_temperature(tb; ring_au = Float64[], gap_au = Float64[], title = "", filename = "brightness_temperature.png")
    r = tb.r_au
    fig = Figure(size = (980, 700))
    ax1 = Axis(fig[1, 1]; ylabel = "Temperature [K]", title = title, titlesize = 13)
    lo = tb.tb .- tb.tb_err
    hi = tb.tb .+ tb.tb_err
    ok = findall(i -> isfinite(tb.tb[i]) && isfinite(tb.tb_err[i]), eachindex(r))
    isempty(ok) || band!(ax1, r[ok], lo[ok], hi[ok]; color = (C_FLUX, 0.3))
    lines!(ax1, r, tb.tb; color = C_FLUX, linewidth = 1.8, label = "brightness temperature T_b")
    lines!(ax1, r, tb.td; color = C_LIT, linestyle = :dash, linewidth = 1.6, label = "dust temperature T_d (phi = $(tb.phi))")
    isempty(ring_au) || vlines!(ax1, ring_au; color = (C_RING, 0.5), linestyle = :dot, label = "rings")
    isempty(gap_au) || vlines!(ax1, gap_au; color = (C_GAP, 0.5), linestyle = :dot, label = "gaps")
    ymax = maximum(filter(isfinite, tb.tb); init = 1.0)
    ylims!(ax1, 0, 1.5 * ymax)
    axislegend(ax1; position = :rt, labelsize = 11)

    ax2 = Axis(fig[2, 1]; xlabel = "Deprojected radius [au]", ylabel = "τ = −ln(1 − T_b/T_d)  (capped at 4.6)")
    tau_plot = [isfinite(tb.ratio[i]) ? (tb.ratio[i] < 0.99 ? -log1p(-tb.ratio[i]) : 4.6) : NaN for i in eachindex(r)]
    lines!(ax2, r, tau_plot; color = C_RING, linewidth = 1.8)
    hlines!(ax2, [1.0]; color = (FG_COLOR, 0.4), linestyle = :dash, linewidth = 0.8)
    isempty(ring_au) || vlines!(ax2, ring_au; color = (C_RING, 0.5), linestyle = :dot)
    isempty(gap_au) || vlines!(ax2, gap_au; color = (C_GAP, 0.5), linestyle = :dot)
    linkxaxes!(ax1, ax2)
    hidexdecorations!(ax1; grid = false)
    rowsize!(fig.layout, 2, Relative(0.35))
    display(fig)
    _save_fig(fig, filename)
    return fig
end

_pm(v, e, d::Int) = d == 1 ? @sprintf("%.1f +/- %.1f", v, e) : (d == 2 ? @sprintf("%.2f +/- %.2f", v, e) : @sprintf("%.3f +/- %.3f", v, e))

# circular mean [deg] and resultant length R (1 = all angles identical) of angles in degrees
function _circ_stats(angles_deg::AbstractVector{<:Real})
    isempty(angles_deg) && return (R = NaN, mean = NaN)
    cs = sum(cosd.(angles_deg))
    sn = sum(sind.(angles_deg))
    return (R = hypot(cs, sn) / length(angles_deg), mean = mod(rad2deg(atan(sn, cs)), 360.0))
end

function _asym_verdict(x, pa_disk)
    flags = String[]
    x.m1.snr >= 3 && push!(flags, @sprintf("m=1 at %.1f sigma", x.m1.snr))
    x.m2.snr >= 3 && push!(flags, @sprintf("m=2 at %.1f sigma", x.m2.snr))
    isempty(flags) && return "none (m=1 and m=2 both < 3 sigma)"
    txt = join(flags, ", ")
    if x.m1.snr >= 3
        d = min(_angdiff_deg(x.m1.pa_sky, pa_disk + 90.0), _angdiff_deg(x.m1.pa_sky, pa_disk - 90.0))
        d < 25 && (txt *= "; maximum within 25 deg of the minor axis (near/far-side brightness asymmetry?)")
    end
    return txt
end

"""
    physical_analysis(out; kwargs...) -> NamedTuple

Run the three physical analyses on the result of `run_alma_pipeline` and write
`<slug>_physical_analysis.txt` plus PNGs next to the other outputs:

  1. Gaussian ring fits  (radius, amplitude, observed and intrinsic width, with errors)
  2. azimuthal asymmetry (m = 1, 2 amplitudes per ring; candidate arcs / vortices in the residual image)
  3. brightness temperature and optical depth at 1.3 mm-type wavelengths (needs L*; uses M* for scale heights)
  4. gap-opening planet masses (Kanagawa+15/16 depth and width relations; needs L* and M*, skipped otherwise)
  5. ring widths and dust trapping: alpha / St from the ring width against the gas scale height (Dullemond+18), ring-spacing period ratios

Options
  rings, gaps            detections to use (default out.rings_residual / out.gaps_residual; pass rings2, gaps2 after retuning)
  extra_seeds_au         extra ring radii to start the fit from (e.g. [15.0, 28.0] for shoulders the detector missed)
  use_literature_seeds   also start from published ring radii (default true; marked 'literature' in the table)
  lstar, mstar           stellar luminosity / mass in solar units (override or supply STELLAR_PARAMS)
  phi_dust               flaring angle factor in T_d = (phi L / 8 pi r^2 sigma)^(1/4) (0.02)
  arc_sigma              threshold of the arc finder in units of the smoothed-residual noise (4)
  max_added_rings        extra ring components allowed for unexplained residuals (3)
  do_geom                fit centre offset, inclination and PA from the ring ridges (uses narrow (FWHM <= max(2.5 beams, 0.25 r)), isolated rings with S/N >= geom_min_snr, 10)
  do_flare               OPT-IN (default false): flaring test, ring centres versus the central core, z/R of the emitting dust layer (needs do_geom)
  do_roundbeam           OPT-IN (default false): re-measure the ring m=2 after smoothing to circular beams of 1.0, 1.5, 2.0 x the beam major axis (needs do_asym; takes minutes)
  do_thickness           OPT-IN (default false): vertical-thickness scan of the m=2 beam expectation (part of the asymmetry report)
  do_trapping            alpha / St from the deconvolved ring widths and the scale height, and gap-spacing period ratios (needs the ring fit, M* and L*)
  do_planets             gap-opening planet masses from gap depth and width (needs M* and L*; skipped when the target has no stored stellar parameters)
  use_fitted_geometry    recompute the profile with the fitted geometry and use it for the asymmetry and temperature steps
                         (default false: the asymmetry is measured with both geometries and compared)
  do_fit, do_asym, do_temp   switch individual analyses off
Each analysis is isolated: if one fails, the others still run.
"""
function physical_analysis(out; rings = nothing, gaps = nothing, extra_seeds_au = Float64[],
                           use_literature_seeds::Bool = true, lstar = nothing, mstar = nothing,
                           phi_dust::Real = 0.02, arc_sigma::Real = 4.0,
                           seed_prominence_sigma::Real = 0.6, seed_min_abs_frac::Real = 0.1,
                           max_added_rings::Int = 3, do_fit::Bool = true, do_geom::Bool = true, use_fitted_geometry::Bool = false,
                           geom_min_snr::Real = 10.0, do_flare::Bool = false, do_asym::Bool = true, do_roundbeam::Bool = false, do_thickness::Bool = false,
                           do_temp::Bool = true, do_planets::Bool = true, do_trapping::Bool = true,
                           min_detect_snr::Real = 5.0, estimate_lstar::Bool = false)
    if !(hasproperty(out, :profile) && out.profile !== nothing && hasproperty(out, :local_path))
        println("Run the pipeline first: `out` has no radial profile.")
        return nothing
    end
    t = out.target
    prof = out.profile
    slug = t.slug
    rings === nothing && (rings = out.rings_residual)
    gaps === nothing && (gaps = out.gaps_residual)
    rings === nothing && (rings = NamedTuple[])
    gaps === nothing && (gaps = NamedTuple[])
    lit = hasproperty(out, :lit_features) ? out.lit_features : Tuple{Float64,String,Symbol}[]
    println("="^80)
    println("Physical analysis: ", t.name)
    println("="^80)

    img = load_fits_image(out.local_path)
    freq = hasproperty(img, :freq_hz) ? img.freq_hz : nothing
    sp = _stellar_params(t; lstar = lstar, mstar = mstar)
    lstar_estimated = false
    if estimate_lstar && sp.lstar === nothing && sp.mstar !== nothing
        sp = (lstar = Float64(sp.mstar)^2, mstar = sp.mstar, source = sp.source * "; L* ESTIMATED as M*^2")
        lstar_estimated = true
        println("NOTE: L* is not known for ", t.name, "; estimated as M*^2 = ", round(sp.lstar; digits = 2),
                " Lsun (estimate_lstar = true). Temperatures and masses inherit this uncertainty.")
    end
    if prof.bmaj_au === nothing
        println("No beam information: the physical analysis needs BMAJ/BMIN in the FITS header.")
        return nothing
    end
    unit_scale = lowercase(prof.bunit) == "jy/beam" ? 1e3 : 1.0
    unit_lab = unit_scale == 1e3 ? "mJy/beam" : prof.bunit
    bmaj_au = Float64(prof.bmaj_au)
    bmin_au = Float64(prof.bmin_au)

    # ---- detection gate: no point fitting rings, temperatures or masses to noise -------------------------
    # Same quantity as the target summary: peak of the azimuthal mean / robust image rms. min_detect_snr = 0 disables it.
    okb = findall(isfinite, prof.mean_flux)
    rms_img = hasproperty(prof, :image_rms) ? Float64(prof.image_rms) : NaN
    if !isempty(okb) && isfinite(rms_img) && rms_img > 0 && min_detect_snr > 0
        det_snr = maximum(prof.mean_flux[okb]) / rms_img          # peak / per-beam noise (for the message)
        det_sig = _detection_sigma(prof)                          # strongest bin in its own statistical sigma (the test)
        if isfinite(det_sig) ? det_sig < min_detect_snr : det_snr < min_detect_snr
            lim = 3 * rms_img * unit_scale
            msg = @sprintf("NOT DETECTED: the strongest bin of the azimuthal mean is %.1f sigma (< %.1f; the peak is %.1f x the per-beam image noise). Nothing below is meaningful, so the physical analysis was skipped. Upper limit: 3 sigma = %.3g %s (one beam; image rms %.3g %s). Try a deeper or different observation, or set min_detect_snr = 0 to force the analysis.",
                           det_sig, Float64(min_detect_snr), det_snr, lim, unit_lab, rms_img * unit_scale, unit_lab)
            println("\n", msg)
            try
                open(joinpath(OUTPUT_DIR, "$(slug)_physical_analysis.txt"), "w") do io
                    println(io, "="^88); println(io, "PHYSICAL ANALYSIS: ", t.name); println(io, "="^88)
                    println(io, "Generated ", Libc.strftime("%Y-%m-%d %H:%M", time()), " by alma_target_explorer_julia (v7)")
                    println(io, "Image: ", basename(out.local_path))
                    println(io, "\n", msg)
                end
            catch err
                println("(could not write the report file: ", first(sprint(showerror, err), 120), ")")
            end
            return nothing
        end
    end

    # ---- seeds --------------------------------------------------------------------------
    seeds = Tuple{Float64,Symbol}[]
    old_dbg = DEBUG[]
    DEBUG[] = false
    try
        rs, _, _ = find_rings_and_gaps_residual(prof; prominence_sigma = seed_prominence_sigma,
                                                min_abs_residual_frac = seed_min_abs_frac, min_radius_beams = 1.0)
        for x in rs
            push!(seeds, (Float64(x.radius_au), :detected))
        end
    catch err
        println("WARNING: seed detection failed: ", err)
    finally
        DEBUG[] = old_dbg
    end
    for x in rings
        push!(seeds, (Float64(x.radius_au), :detected))
    end
    for x in extra_seeds_au
        push!(seeds, (Float64(x), :user))
    end
    if use_literature_seeds
        for (ra, _, kind) in lit
            kind === :ring && push!(seeds, (Float64(ra), :literature))
        end
    end

    # ---- 1. ring fits ---------------------------------------------------------------------
    fit = nothing
    if do_fit
        println("\n[1/3] Gaussian ring fits (", length(seeds), " candidate ring position(s) before merging)")
        try
            fit = fit_gaussian_rings(prof, seeds; max_add = max_added_rings)
            @printf("  %d ring component(s), chi2_red = %.2f (dof %d)\n", length(fit.rings), fit.chi2_red, fit.dof)
            plot_ring_fits(fit; lit_features = lit, title = "$(t.name) ring fits",
                           filename = "$(slug)_ring_fits.png")
        catch err
            println("WARNING: the ring fit failed:")
            showerror(stdout, err, catch_backtrace())
            println()
        end
    end

    ringlist = NamedTuple[]
    if fit !== nothing
        for rg in fit.rings
            rg.amp_snr >= 2 && push!(ringlist, (r0 = rg.r0, sigma_obs = rg.sigma_obs, amp = rg.amp))
        end
    else
        for x in rings
            x.residual > 0 && push!(ringlist, (r0 = Float64(x.radius_au), sigma_obs = bmaj_au / _FWHM_PER_SIGMA, amp = Float64(x.residual)))
        end
    end

    # ---- 1b. geometry check from the ring ridges ------------------------------------------------
    geo = nothing
    prof_g = prof              # profile used by the asymmetry and temperature steps
    ringlist_g = ringlist
    asym_alt = nothing         # asymmetry with the other geometry, for the comparison table
    prof2 = nothing
    ringlist2 = ringlist
    sh2 = (0.0, 0.0)           # sub-pixel part of the fitted centre shift [arcsec east, north]
    if do_geom && fit !== nothing
        println("\n[1b] Geometry check from the ring ridges")
        try
            geo = fit_ring_geometry(img, prof, fit.rings; min_snr = geom_min_snr)
            @printf("  %d ring(s), %d ridge points: centre shift (%+.1f, %+.1f) mas, i = %.1f deg, PA = %.1f deg (assumed %.1f, %.1f); chi2_red = %.2f\n",
                    geo.K, geo.npts, 1000 * geo.E0, 1000 * geo.N0, geo.incl, geo.pa, geo.incl_assumed, geo.pa_assumed, geo.chi2_red)
            plot_geometry_fit(geo; title = "$(t.name) ring-ridge geometry", filename = "$(slug)_geometry_fit.png")
            DEBUG[] = false
            try
                prof2 = extract_radial_profile(img; incl_deg = geo.incl, pa_deg = geo.pa, distance_pc = prof.distance_pc,
                                               max_radius_arcsec = out.settings.max_radius_arcsec, n_bins = out.settings.n_bins,
                                               center = (round(Int, geo.new_center[1]), round(Int, geo.new_center[2])), fit_geom = false)
            finally
                DEBUG[] = old_dbg
            end
            sh2 = ((geo.new_center[1] - round(Int, geo.new_center[1])) * img.cdelt1_arcsec,
                   (geo.new_center[2] - round(Int, geo.new_center[2])) * img.cdelt2_arcsec)
            ringlist2 = NamedTuple[]
            for rg in ringlist
                kfit = findfirst(x -> abs(x.r_au_assumed - rg.r0) < 1e-6, geo.rings)
                push!(ringlist2, kfit === nothing ? rg : (r0 = geo.rings[kfit].r_au, sigma_obs = rg.sigma_obs, amp = rg.amp))
            end
            if use_fitted_geometry
                prof_g = prof2
                ringlist_g = ringlist2
            end
        catch err
            println("WARNING: the geometry check failed (catalogue geometry is used): ", sprint(showerror, err))
        end
    end

    # ---- 1c. flaring test: ring centres versus the central core ---------------------------------
    flare = nothing
    if do_flare && geo !== nothing
        println("\n[1c] Ring centres versus the central core (height of the emitting dust layer)")
        try
            core = core_centroid(img, prof)
            @printf("  core centroid (%+.1f, %+.1f) mas from the profile centre (+/- %.1f), extra width %.1f mas, chi2_red = %.1f\n",
                    1000 * core.E, 1000 * core.N, 1000 * core.err, core.sx_mas, core.chi2_red)
            flare = fit_flaring(geo, core, prof)
            @printf("  z = (%+.3f +/- %.3f) R (%.1f sigma); chi2 = %.1f for %d dof; major-axis offsets: chi2 = %.1f for %d dof\n",
                    flare.eps, flare.eps_err, abs(flare.eps) / max(flare.eps_err, 1e-12), flare.chi2, flare.dof, flare.chi2_maj, flare.K)
            plot_flaring(flare; title = "$(t.name) ring centres versus the central core", filename = "$(slug)_flaring.png")
        catch err
            println("WARNING: the flaring test failed:")
            showerror(stdout, err, catch_backtrace())
            println()
        end
    end

    # ---- 2. azimuthal asymmetry --------------------------------------------------------------
    asym = nothing
    rb = nothing
    rb_summ = NamedTuple[]
    if do_asym
        println("\n[2/3] Azimuthal asymmetry (", length(ringlist_g), " ring(s); geometry ", prof_g === prof ? "as assumed" : "fitted from the ring ridges", ")")
        try
            asym = azimuthal_asymmetry(img, prof_g, ringlist_g; arc_sigma = arc_sigma, shift_arcsec = prof_g === prof ? (0.0, 0.0) : sh2)
            if geo !== nothing && prof2 !== nothing
                try
                    asym_alt = azimuthal_asymmetry(img, use_fitted_geometry ? prof : prof2, use_fitted_geometry ? ringlist : ringlist2; arc_sigma = arc_sigma,
                                                   shift_arcsec = use_fitted_geometry ? (0.0, 0.0) : sh2)
                catch err2
                    println("WARNING: the comparison asymmetry failed: ", err2)
                end
            end
            @printf("  %d candidate arc/clump region(s) above %.1f sigma; mean residual %.2e %s\n",
                    length(asym.arcs), arc_sigma, asym.resid_mean * unit_scale, unit_lab)
            plot_asymmetry(asym, ringlist_g; title = "$(t.name) non-axisymmetric residual",
                           filename = "$(slug)_azimuthal.png")
            if do_roundbeam
                println("\n[2b] Round-beam test: smoothing to circular beams (a minute or so per level)")
                try
                    DEBUG[] = false
                    try
                        rb = round_beam_test(img, prof_g, ringlist_g, asym; shift_arcsec = prof_g === prof ? (0.0, 0.0) : sh2)
                    finally
                        DEBUG[] = old_dbg
                    end
                    rb_summ = round_beam_summary(rb)
                    for x in rb_summ
                        @printf("  r = %.1f au: native excess %.1f%%; at %.1fx the beam %.1f%% (a resolution effect would give %.1f%%) -> %s\n", x.r0, 100 * x.exc0,
                                x.pts[end].factor, 100 * x.pts[end].exc, 100 * x.exc0 * x.pts[end].pred, String(x.verdict))
                    end
                    isempty(rb_summ) && println("  no ring has a significant m=2 excess to test")
                    isempty(rb_summ) || plot_round_beam(rb, rb_summ; title = "$(t.name) m=2 versus smoothing", filename = "$(slug)_round_beam.png")
                catch err
                    println("WARNING: the round-beam test failed:")
                    showerror(stdout, err, catch_backtrace())
                    println()
                end
            end
        catch err
            println("WARNING: the asymmetry analysis failed:")
            showerror(stdout, err, catch_backtrace())
            println()
        end
    end

    # ---- 3. brightness temperature ---------------------------------------------------------
    tbp = nothing
    # features marked on the temperature table and plot: only those at 3 sigma or above (noise "gaps" are not tabulated)
    sig3(x) = begin
        hasproperty(x, :residual) || return true
        s = _stat_sigma(prof, Float64(x.radius_au))
        !(isfinite(s) && s > 0) || abs(Float64(x.residual)) / s >= 3
    end
    ring_au = fit !== nothing ? Float64[rg.r0 for rg in fit.rings if rg.amp_snr >= 3] : Float64[x.radius_au for x in rings if sig3(x)]
    gap_au = Float64[x.radius_au for x in gaps if sig3(x)]
    lstar_used = sp.lstar === nothing ? 1.0 : sp.lstar
    if do_temp
        println("\n[3/3] Brightness temperature and optical depth")
        if freq === nothing
            println("  The FITS header has no frequency (CTYPEn = FREQ / RESTFRQ): skipped.")
        else
            sp.lstar === nothing && println("  WARNING: no stellar luminosity for ", t.name,
                                            ": T_d assumes L* = 1 Lsun. Pass lstar= to physical_analysis.")
            try
                tbp = brightness_temperature_profile(prof_g, freq, lstar_used; phi = phi_dust)
                @printf("  frequency %.1f GHz; peak T_b = %.1f K\n", freq / 1e9, maximum(filter(isfinite, tbp.tb); init = 0.0))
                if sp.lstar === nothing
                    println("  No figure: T_d (and with it T_b/T_d and tau) needs L*; pass lstar= to physical_analysis.")
                else
                    plot_brightness_temperature(tbp; ring_au = ring_au, gap_au = gap_au, title = "$(t.name) brightness temperature",
                                                filename = "$(slug)_brightness_temperature.png")
                end
            catch err
                println("WARNING: the brightness-temperature analysis failed:")
                showerror(stdout, err, catch_backtrace())
                println()
            end
        end
    end

    # ---- 3b. gap-opening planet masses --------------------------------------------------------
    pm = nothing
    planets_note = "switched off (do_planets = false)"
    if do_planets
        if sp.lstar === nothing || sp.mstar === nothing
            planets_note = "needs L* and M*, which are not stored for this target; pass lstar= and mstar="
            println("\n[3b] Gap-opening planet masses: skipped (", planets_note, ")")
        elseif isempty(gap_au)
            planets_note = "no gaps were detected"
            println("\n[3b] Gap-opening planet masses: skipped (no gaps detected)")
        else
            println("\n[3b] Gap-opening planet masses (", length(gap_au), " gap(s))")
            try
                pm = gap_planet_masses(prof_g, gap_au, sp.lstar, sp.mstar; phi = phi_dust)
                @printf("  %d of %d gap(s) usable\n", count(x -> x.ok, pm), length(pm))
            catch err
                planets_note = "the calculation failed; see the console"
                println("WARNING: the planet-mass estimate failed:")
                showerror(stdout, err, catch_backtrace())
                println()
            end
        end
    end

    # ---- 3c. ring widths and dust trapping -------------------------------------------------------
    tr = nothing
    trap_note = "switched off (do_trapping = false)"
    if do_trapping
        if fit === nothing
            trap_note = "needs the ring fit (section 1)"
            println("\n[3c] Dust trapping: skipped (", trap_note, ")")
        elseif sp.lstar === nothing || sp.mstar === nothing
            trap_note = "needs L* and M*, which are not stored for this target; pass lstar= and mstar="
            println("\n[3c] Dust trapping: skipped (", trap_note, ")")
        else
            println("\n[3c] Ring widths and dust trapping")
            try
                tr = ring_trapping(fit, gap_au, sp.lstar, sp.mstar; phi = phi_dust)
                @printf("  %d ring(s) with amplitude S/N >= 3; %d give a limit on alpha/St\n", length(tr), count(x -> x.status === :limit, tr))
            catch err
                trap_note = "the calculation failed; see the console"
                println("WARNING: the dust-trapping estimate failed:")
                showerror(stdout, err, catch_backtrace())
                println()
            end
        end
    end

    # ---- report ---------------------------------------------------------------------------
    io = IOBuffer()
    println(io, "="^88)
    println(io, "PHYSICAL ANALYSIS: ", t.name)
    println(io, "="^88)
    println(io, "Generated ", Libc.strftime("%Y-%m-%d %H:%M", time()), " by alma_target_explorer_julia (v7)")
    println(io, "Image: ", basename(out.local_path))
    @printf(io, "Distance %.1f pc; inclination %.1f deg, PA %.1f deg; beam %.4f x %.4f arcsec = %.1f x %.1f au (radial beam sigma %.1f au)\n",
            prof.distance_pc, prof.incl_deg === nothing ? 0.0 : prof.incl_deg, prof.pa_deg === nothing ? 0.0 : prof.pa_deg,
            prof.bmaj_arcsec, prof.bmin_arcsec, bmaj_au, bmin_au, sqrt(bmaj_au * bmin_au) / _FWHM_PER_SIGMA)
    if max(bmaj_au, bmin_au) > BEAM_RESOLVED_LIMIT_AU
        msg = @sprintf("RESOLUTION WARNING: the beam is %.0f x %.0f au (> %.0f au). A typical disk (rings at tens of au) is not resolved, so the rings, asymmetries, temperatures and masses below are not meaningful. A finer observation is needed.",
                       bmaj_au, bmin_au, BEAM_RESOLVED_LIMIT_AU)
        println(io, msg)
        println(msg)
    end
    r90_u = _disk_extent_au(prof; frac = 0.9)
    bm_u = sqrt(bmaj_au * bmin_au)
    if isfinite(r90_u) && bm_u > 0 && r90_u < 1.5 * bm_u
        msg = @sprintf("UNRESOLVED: 90%% of the flux lies within %.1f au = %.1f beams of %.1f au, so this is a point source at this resolution. Ring and gap structure cannot be measured from this image, and the temperatures and optical depths describe the beam, not the disk.",
                       r90_u, r90_u / bm_u, bm_u)
        println(io, msg)
        println(msg)
    end
    freq === nothing || @printf(io, "Observing frequency %.1f GHz (%.2f mm)\n", freq / 1e9, 299.792458 / (freq / 1e9))
    let dl = disc_info_line(t; r68_measured = _disk_extent_au(prof; frac = 0.68))
        isempty(dl) || println(io, dl)
    end
    let frt = flux_recovery_text(flux_recovery(t, _integrated_flux_mjy(prof), freq === nothing ? NaN : freq / 1e9))
        isempty(frt) || (println(io, frt); occursin("FILTERED", frt) && println("WARNING: ", frt))
    end
    if sp.lstar === nothing && sp.mstar === nothing
        println(io, "Stellar parameters: none stored for this target (pass lstar= / mstar=).")
    else
        @printf(io, "Stellar parameters (%s; verify): L* = %s Lsun, M* = %s Msun\n", sp.source,
                sp.lstar === nothing ? "?" : @sprintf("%.2f", sp.lstar), sp.mstar === nothing ? "?" : @sprintf("%.2f", sp.mstar))
    end

    # 1. ring fits
    println(io, "\n1. GAUSSIAN RING FITS")
    println(io, "-"^88)
    if fit === nothing
        println(io, "  Not available (see the console for the reason).")
    else
        @printf(io, "  Model: tapered power law + %d Gaussian ring(s); chi2_red = %.2f (dof %d). Errors are 1 sigma, scaled by sqrt(chi2_red) when > 1.\n",
                length(fit.rings), fit.chi2_red, fit.dof)
        @printf(io, "  Per-bin errors: image noise / sqrt(independent beams) plus %.0f%% of the local flux.\n", 100 * fit.frac_floor)
        bg = fit.bg
        @printf(io, "  Smooth disk: amplitude %.3g %s, characteristic radius %.0f au, power-law index %.2f\n",
                bg.A * unit_scale, unit_lab, bg.rc_au, bg.gamma)
        have_h = sp.lstar !== nothing && sp.mstar !== nothing
        println(io)
        @printf(io, "  %2s  %-15s %-17s %5s  %-17s %7s %8s %9s  %-10s\n", "#", "r [au]", "amp [" * unit_lab * "]", "S/N",
                "FWHM_int [au]", "FWHM/r", have_h ? "FWHM/H" : "", "share", "seed")
        for (k, rg) in enumerate(fit.rings)
            fw = rg.resolved ? _pm(rg.fwhm_int, rg.fwhm_int_err, 1) : @sprintf("< %.1f (unresolved)", rg.fwhm_ub)
            fwr = rg.resolved ? @sprintf("%.2f", rg.fwhm_int / rg.r0) : @sprintf("< %.2f", rg.fwhm_ub / rg.r0)
            fwh = ""
            if have_h
                H = _gas_scale_height_au(rg.r0, _dust_temperature(rg.r0, sp.lstar; phi = phi_dust), sp.mstar)
                fwh = rg.resolved ? @sprintf("%.2f", rg.fwhm_int / H) : @sprintf("< %.2f", rg.fwhm_ub / H)
            end
            contrast = @sprintf("%.0f%%", 100 * rg.share)
            amp_txt = @sprintf("%.3g +/- %.2g", rg.amp * unit_scale, rg.amp_err * unit_scale)
            flag = (rg.at_bound ? " [parameter at its bound]" : "") * (rg.amp_snr < 3 ? " [amplitude < 3 sigma]" : "")
            @printf(io, "  %2d  %-15s %-17s %5.1f  %-17s %7s %8s %9s  %-10s%s\n", k, _pm(rg.r0, rg.r0_err, 1), amp_txt, rg.amp_snr,
                    fw, fwr, fwh, contrast, string(rg.origin), flag)
        end
        println(io, "\n  FWHM_int: observed Gaussian FWHM with the radial beam (sigma ", @sprintf("%.1f", fit.sb_au),
                " au) subtracted in quadrature; '<' marks widths consistent with the beam (upper limit at +1 sigma).")
        println(io, "  share: fraction of the local azimuthal-mean flux (at the ring radius) that this ring component supplies.")
        if bg.rc_au < 3 * bmaj_au
            println(io, "  WARNING: the smooth disk collapsed to the unresolved core (characteristic radius ", @sprintf("%.0f", bg.rc_au), " au < 3 beams): the extended emission is shared among the ring components,",
                    " so overlapping neighbours (shoulders) have degenerate widths and amplitudes; trust the isolated rings, not the blended ones.")
        end
        have_h && println(io, "  FWHM/H: width over the gas pressure scale height H = (c_s/v_K) r from the passive-disk T_d. A ring narrower than H",
                          " cannot be a gas density bump alone: it needs dust trapping in a pressure maximum (a gap-opening planet or similar).")
        props_r = literature_props_au(t, prof.distance_pc)
        if !isempty(props_r)
            rows = String[]
            for rg in fit.rings
                rg.amp_snr >= 3 || continue
                p = _nearest_prop(props_r, rg.r0, :ring, max(bmaj_au, 0.1 * rg.r0))
                p === nothing && continue
                obs = rg.resolved ? sqrt(rg.fwhm_int^2 + (_FWHM_PER_SIGMA * fit.sb_au)^2) : NaN
                push!(rows, @sprintf("    %6.1f | %6.1f %-15s | %s", rg.r0, p[1], @sprintf("%.1f%s", p[2], p[5] ? " (<beam)" : ""),
                                     isfinite(obs) ? @sprintf("%.1f", obs) : "unresolved"))
            end
            if !isempty(rows)
                println(io, "\n  PUBLISHED RING WIDTHS (Huang+18, DSHARP Table 1; rescaled to the adopted distance) against the observed Gaussian FWHM (beam NOT removed):")
                @printf(io, "    %6s | %6s %-15s | %s\n", "r [au]", "pub r", "pub width", "observed FWHM")
                foreach(r -> println(io, r), rows)
                println(io, "  Published ring width = outer minus inner edge, essentially the FWHM for an isolated Gaussian ring; (<beam) = narrower than the beam.")
            end
        end
        rr = Float64[rg.r0 for rg in fit.rings if rg.amp_snr >= 3]
        if length(rr) >= 2
            println(io, "\n  Radius ratios of neighbouring rings (3 sigma and above): ",
                    join([@sprintf("%.0f->%.0f au: %.2f", rr[i], rr[i + 1], rr[i + 1] / rr[i]) for i in 1:(length(rr) - 1)], ";  "))
        end
        nweak = count(rg -> rg.amp_snr < 3, fit.rings)
        nweak > 0 && println(io, "  ", nweak, " component(s) have amplitude below 3 sigma: treat them as unconfirmed (blended components have inflated errors: one is kept only if removing it makes the fit much worse).")
        if fit.chi2_red > 3
            println(io, "  NOTE: chi2_red is ", @sprintf("%.1f", fit.chi2_red), ": the Gaussians do not capture all the structure (unseeded rings, asymmetric ring profiles or",
                    " an unresolved core). Radii are reliable to about a bin; widths and amplitudes less so. Try extra_seeds_au = [...].")
        end
    end

    # 1b. geometry
    println(io, "\n1b. GEOMETRY CHECK FROM THE RING RIDGES")
    println(io, "-"^88)
    if geo === nothing
        println(io, do_geom ? "  Not available (see the console for the reason)." : "  Switched off (do_geom = false).")
    else
        @printf(io, "  %d narrow, isolated ring(s) with amplitude S/N >= %.0f, %d ridge points (azimuth wedges ~1 beam wide), chi2_red = %.2f. Errors are formal.\n",
                geo.K, geom_min_snr, geo.npts, geo.chi2_red)
        println(io, "  One shared geometry is fitted to all ridges; each ring has its own free radius.\n")
        @printf(io, "  %-22s %10s %18s %10s\n", "parameter", "assumed", "fitted +/- err", "shift")
        @printf(io, "  %-22s %10s %18s %10s\n", "centre dE [mas]", "0", @sprintf("%+.1f +/- %.1f", 1000 * geo.E0, 1000 * geo.E0_err), "")
        @printf(io, "  %-22s %10s %18s %10s\n", "centre dN [mas]", "0", @sprintf("%+.1f +/- %.1f", 1000 * geo.N0, 1000 * geo.N0_err), "")
        @printf(io, "  %-22s %10.1f %18s %+9.1f\n", "inclination [deg]", geo.incl_assumed, @sprintf("%.1f +/- %.1f", geo.incl, geo.incl_err), geo.incl - geo.incl_assumed)
        @printf(io, "  %-22s %10.1f %18s %+9.1f\n", "position angle [deg]", geo.pa_assumed, @sprintf("%.1f +/- %.1f", geo.pa, geo.pa_err), geo.pa - geo.pa_assumed)
        shift_mas = 1000 * hypot(geo.E0, geo.N0)
        @printf(io, "\n  Centre shift %.1f mas = %.1f pixels (%.0f%% of the beam minor axis).\n", shift_mas, shift_mas / (1000 * prof.pix_scale_arcsec),
                100 * shift_mas / (1000 * prof.bmin_arcsec))
        println(io)
        @printf(io, "  %8s %10s %7s %9s %-18s %s\n", "R [au]", "R err [au]", "points", "rms [mas]", "ellipticity e", "ridge radius max at [deg]")
        for rg in geo.rings
            @printf(io, "  %8.1f %10.2f %7d %9.1f %-18s %s\n", rg.r_au, rg.r_err_au, rg.npts, rg.rms_mas,
                    isfinite(rg.ecc) ? @sprintf("%.3f +/- %.3f (%.1f s)", rg.ecc, rg.ecc_err, rg.ecc_snr) : "n/a",
                    isfinite(rg.far_deg) ? @sprintf("%.0f", rg.far_deg) : "")
        end
        println(io, "\n  rms: scatter of the ridge radius about the fitted circle (the ridge position is good to a few mas where it is bright). ellipticity: residual m=1 wobble of the radius, amplitude / R;")
        println(io, "  a significant e after the shared geometry is removed means that ring is eccentric on its own (or the geometry differs between rings, e.g. a warp).")
        if geo.incl < 15
            println(io, "  NOTE: i < 15 deg: inclination and PA are only weakly constrained by a nearly circular ring; trust the centre shift, not i and PA.")
        end
        println(io, "  Caveats: a flared, optically thick ring appears shifted along the minor axis (near side vs far side), which mimics a centre offset along that axis (section 1c tests it);")
        println(io, "  formal errors exclude this. The asymmetry step applies the full (sub-pixel) centre shift to the residual image; the radial profile is recomputed about the nearest pixel (", @sprintf("%.1f", 1000 * prof.pix_scale_arcsec / 2), " mas rounding).")
        println(io, "  To adopt these values: physical_analysis(out; use_fitted_geometry = true), or pass incl_deg / pa_deg to run_alma_pipeline.")
    end

    # 1c. flaring test
    println(io, "\n1c. FLARING TEST: ARE THE RING CENTRES LIFTED OFF THE STAR?")
    println(io, "-"^88)
    if flare === nothing
        println(io, !do_flare ? "  Switched off (do_flare = false)." : (geo === nothing ? "  Needs the geometry fit of section 1b." : "  Not available (see the console for the reason)."))
    else
        core = flare.core
        println(io, "  Dust that sits a height z above the midplane (the surface facing us) is projected away from the star along the minor axis, toward the far side, by z sin(i);")
        println(io, "  the star (the unresolved central core) does not move. For z = eps R the centre of a ring is therefore displaced from the core by eps R sin(i) along the minor axis")
        println(io, "  and not at all along the major axis. Ring centres: ridge fits of section 1b (each ring its own centre); star: centroid of the central core.\n")
        @printf(io, "  Core centroid: (%+.1f, %+.1f) mas east, north of the profile centre, +/- %.1f mas (formal); fitted extra width %.1f mas; chi2_red = %.1f.\n",
                1000 * core.E, 1000 * core.N, 1000 * core.err, core.sx_mas, core.chi2_red)
        have_hf = sp.lstar !== nothing && sp.mstar !== nothing
        if have_hf
            @printf(io, "\n  %8s | %-20s | %-20s | %-17s | %s\n", "R [au]", "along major [mas]", "along minor [mas]", "z/R implied", "z / H_gas")
        else
            @printf(io, "\n  %8s | %-20s | %-20s | %s\n", "R [au]", "along major [mas]", "along minor [mas]", "z/R implied")
        end
        for rw in flare.rings
            a1 = @sprintf("%+.1f +/- %.1f", rw.dmaj_mas, rw.dmaj_err_mas)
            a2 = @sprintf("%+.1f +/- %.1f", rw.dmin_mas, rw.dmin_err_mas)
            a3 = @sprintf("%+.3f +/- %.3f", rw.eps, rw.eps_err)
            if have_hf
                Hf = _gas_scale_height_au(rw.r_au, _dust_temperature(rw.r_au, sp.lstar; phi = phi_dust), sp.mstar)
                @printf(io, "  %8.1f | %-20s | %-20s | %-17s | %+.2f\n", rw.r_au, a1, a2, a3, rw.eps * rw.r_au / Hf)
            else
                @printf(io, "  %8.1f | %-20s | %-20s | %s\n", rw.r_au, a1, a2, a3)
            end
        end
        sigf = abs(flare.eps) / max(flare.eps_err, 1e-12)
        @printf(io, "\n  Joint fit z = eps R through all %d ring(s): eps = %+.3f +/- %.3f (%.1f sigma from zero); chi2 = %.1f for %d dof.\n",
                flare.K, flare.eps, flare.eps_err, sigf, flare.chi2, flare.dof)
        @printf(io, "  Along the major axis (expected zero for a flared layer): chi2 = %.1f for %d dof.\n", flare.chi2_maj, flare.K)
        @printf(io, "  Sign: eps > 0 means the ring centres lie toward position angle %.0f deg from the core (the +minor-axis direction); if the emission comes from above the midplane,\n", flare.pa_plus)
        println(io, "  that side is the FAR side of the disk (check against the kinematic near side from CO).")
        clean = flare.dof == 0 || flare.chi2 / flare.dof <= 3
        majok = flare.chi2_maj / flare.K <= 3
        if flare.K < 2
            println(io, "  Only one ring: the trend with radius cannot be tested; eps is from that ring alone.")
        elseif sigf >= 3 && clean && majok
            @printf(io, "  -> Consistent with a lifted dust layer: z ~ %.3f R (%.1f au at %.0f au), a LOWER limit on the height of the emitting layer.\n", abs(flare.eps), abs(flare.eps) * flare.rings[end].r_au, flare.rings[end].r_au)
        elseif sigf >= 3
            println(io, "  -> A significant offset, but it does not follow the flaring pattern (a linear trend with radius and none along the major axis): centre errors, a warp, an eccentric ring or an")
            println(io, "     asymmetric core are as likely as flaring.")
        else
            @printf(io, "  -> No significant minor-axis trend: z/R < %.3f (3 sigma) for the emitting layer of these rings.\n", abs(flare.eps) + 3 * flare.eps_err)
        end
        println(io, "  Caveats: a vertically symmetric, optically thin layer shows no net shift (it only smears the ring; see the thickness check in section 2), so eps is a lower limit on the layer height;")
        println(io, "  the core is assumed to sit on the star; ring warps or eccentricity also move centres; errors are formal (the shared core error is propagated).")
    end

    # 2. asymmetry
    println(io, "\n2. AZIMUTHAL ASYMMETRY (non-axisymmetric structure)")
    println(io, "-"^88)
    if asym === nothing
        println(io, "  Not available (see the console for the reason).")
    else
        @printf(io, "  Residual = pixel value minus the azimuthal mean at the same deprojected radius. Noise: image rms %.3g %s per beam.\n",
                (prof.image_rms) * unit_scale, unit_lab)
        println(io, "  Fourier amplitudes around each ring (window = +/- one observed sigma, at least half a beam), relative to the fitted ring amplitude.")
        println(io, "  FAP = false-alarm probability exp(-S/N^2/2) for one ring (floored at 1e-6), for pure noise only; with N rings expect N x FAP chance detections. Geometry errors are NOT in the noise.\n")
        @printf(io, "  %8s %5s | %-14s %5s %7s %8s | %-14s %5s %7s | %-18s\n", "r [au]", "bins", "m=1 [%]", "S/N", "FAP", "max @PA", "m=2 [%]", "S/N", "FAP", "peak +/- [%] (S/N)")
        for x in asym.rings
            if !x.ok
                @printf(io, "  %8.1f %5s | not enough azimuth coverage or non-positive ring amplitude\n", x.r0, "-")
                continue
            end
            @printf(io, "  %8.1f %5d | %-14s %5.1f %7.1e %7.0f° | %-14s %5.1f %7.1e | %+.0f / %+.0f (%.1f)\n", x.r0, x.nbins,
                    @sprintf("%.1f +/- %.1f", 100 * x.m1.amp, 100 * x.m1.err), x.m1.snr, max(x.m1.fap, 1e-6), x.m1.pa_sky,
                    @sprintf("%.1f +/- %.1f", 100 * x.m2.amp, 100 * x.m2.err), x.m2.snr, max(x.m2.fap, 1e-6),
                    100 * x.peak_pos, 100 * x.peak_neg, x.peak_snr)
        end
        pa_disk = prof.pa_deg === nothing ? 0.0 : Float64(prof.pa_deg)
        println(io)
        for x in asym.rings
            x.ok || continue
            println(io, @sprintf("  r = %6.1f au: ", x.r0), _asym_verdict(x, pa_disk))
        end
        if any(x -> x.ok && isfinite(x.m2b.amp), asym.rings)
            println(io, "\n  Elliptical-beam check for m=2: an axisymmetric ring convolved with a beam that is not round has a higher peak where the ring runs along the")
            println(io, "  beam's short axis, i.e. a pure m=2 pattern with no real structure. Expectation for each ring (fitted intrinsic width) versus what is measured;")
            println(io, "  excess = size of the vector difference between the measured and the expected m=2 term (formal S/N, noise only):")
            @printf(io, "  %8s | %-18s | %-18s | %s\n", "r [au]", "beam alone: % @ az", "measured: % @ az", "excess [%] (S/N)")
            for x in asym.rings
                (x.ok && isfinite(x.m2b.amp)) || continue
                @printf(io, "  %8.1f | %-18s | %-18s | %.1f (%.1f s)\n", x.r0,
                        @sprintf("%.1f @ %.0f deg", 100 * x.m2b.amp, mod(x.m2b.phi_deg, 180.0)),
                        @sprintf("%.1f @ %.0f deg", 100 * x.m2.amp, mod(x.m2.phi_deg, 180.0)),
                        100 * x.m2b.exc, x.m2b.exc_snr)
            end
            println(io, "  Azimuths are disk-plane angles from the major axis (mod 180). The expectation uses the same window-mean estimator as the measurement (the m=2 of the ring PEAK is ~1.65x larger).")
            println(io, "  Not included: ring curvature at small r, non-Gaussian ring profiles, CLEAN / uv-coverage effects (the vertical-thickness scan is opt-in: do_thickness = true).")
            if do_thickness
            # vertical thickness: the same expectation with an optically thin layer of Gaussian scale z = (z/R) R that smears the ring along the minor axis
            println(io, "\n  Vertical-thickness check for m=2. An optically thin, geometrically thick ring has a lower peak on the minor axis than on the major axis (Doi & Kataoka 2021, ApJ,")
            println(io, "  HD 163296), i.e. a layer of Gaussian scale z smears the ring along the minor axis by z tan(i). Excess [%] over beam + such a layer, observed ring width held fixed:")
            @printf(io, "  %8s | %-17s | %7s %7s %7s %7s | %s\n", "r [au]", "measured: % @ az", "z/R=0", "0.01", "0.02", "0.04", "best z/R (0 - 0.08)")
            zgrid = collect(0.0:0.005:0.08)
            tot = zeros(length(zgrid))
            nuse = 0
            for (k, x) in enumerate(asym.rings)
                (x.ok && isfinite(x.m2b.amp) && k <= length(ringlist_g)) || continue
                sgk = ringlist_g[k].sigma_obs
                ex = Float64[_m2_excess_with_thickness(x, asym.beam, sgk, zr) for zr in zgrid]
                @printf(io, "  %8.1f | %-17s | %7.1f %7.1f %7.1f %7.1f | %.3f\n", x.r0, @sprintf("%.1f @ %.0f deg", 100 * x.m2.amp, mod(x.m2.phi_deg, 180.0)),
                        100 * ex[1], 100 * ex[3], 100 * ex[5], 100 * ex[9], zgrid[argmin(ex)])
                if x.m2.snr >= 3 && x.m2.err > 0
                    tot .+= (ex ./ x.m2.err) .^ 2
                    nuse += 1
                end
            end
            if nuse >= 1
                ibz = argmin(tot)
                @printf(io, "  Rings with a significant m=2 (%d): chi2 of the excess = %.0f at z/R = 0, %.0f at the best z/R = %.3f.\n", nuse, tot[1], tot[ibz], zgrid[ibz])
                println(io, "  A thicker layer rotates the expected pattern from the beam's orientation toward the major axis (0 deg) and reduces power along the beam's orientation first, so it can")
                println(io, "  only explain a measured m=2 that is oriented toward the major axis; ", (ibz == 1 || tot[1] - tot[ibz] < 4) ?
                        "here the data prefer no thickness (improvement < 4 in chi2): vertical thickness does not explain the excess." :
                        string("here a layer of z/R ~ ", @sprintf("%.3f", zgrid[ibz]), " lowers the chi2 by ", @sprintf("%.0f", tot[1] - tot[ibz]), "; compare with section 1c."))
            end
            end
            if rb !== nothing && !isempty(rb_summ)
                _round_beam_report(io, rb, rb_summ, Float64(prof.bmaj_arcsec))
            elseif !do_roundbeam
                println(io, "\n  Round-beam test: switched off (do_roundbeam = false).")
            end
        end
        if asym_alt !== nothing
            println(io, "\n  Before / after the geometry correction (m=1 and m=2 amplitude [%] and S/N; first = ", use_fitted_geometry ? "fitted" : "assumed",
                    " geometry as in the table above, second = ", use_fitted_geometry ? "assumed" : "fitted", "):")
            @printf(io, "  %8s | %-22s | %-22s\n", "r [au]", "m=1: this -> other", "m=2: this -> other")
            for x in asym.rings
                x.ok || continue
                y = nothing
                for z in asym_alt.rings
                    z.ok && abs(z.r0 - x.r0) < 0.25 * bmaj_au && (y = z; break)
                end
                y === nothing && continue
                @printf(io, "  %8.1f | %-22s | %-22s\n", x.r0,
                        @sprintf("%.1f (%.1f s) -> %.1f (%.1f s)", 100 * x.m1.amp, x.m1.snr, 100 * y.m1.amp, y.m1.snr),
                        @sprintf("%.1f (%.1f s) -> %.1f (%.1f s)", 100 * x.m2.amp, x.m2.snr, 100 * y.m2.amp, y.m2.snr))
            end
            println(io, "  Note: this window-mean estimator is blind to small geometry errors at first order (they move the ridge, not the ring's brightness), so amplitudes are expected to survive;")
            println(io, "  the ridge-based geometry fit (section 1b) is the sensitive test of centre, inclination and PA.")
        end
        okr = [x for x in asym.rings if x.ok]
        s1 = [x for x in okr if x.m1.snr >= 3]
        s2 = [x for x in okr if x.m2.snr >= 3]
        println(io, "\n  Coherence across rings (a vortex sits at one radius; a centre / inclination / PA error shows at every ring with the same orientation):")
        if length(s1) >= 2
            cs = _circ_stats(Float64[x.m1.pa_sky for x in s1])
            println(io, "    m=1 maxima (sky PA, deg) by ring radius: ", join([@sprintf("%.0f au: %.0f", x.r0, x.m1.pa_sky) for x in s1], ";  "))
            @printf(io, "    m=1: %d rings above 3 sigma, maxima at sky PA %.0f deg, resultant length R = %.2f (random orientations give ~%.2f)%s\n",
                    length(s1), cs.mean, cs.R, 1 / sqrt(length(s1)),
                    cs.R >= 0.8 ? " -> a common direction: one-sided brightness (e.g. near/far-side flaring, a large-scale asymmetry) or a calibration/phase effect; a small centre offset gives almost no ring-averaged m=1" : "")
        end
        if length(s2) >= 2
            cs = _circ_stats(Float64[2 * x.m2.phi_deg for x in s2])
            println(io, "    m=2 orientation (disk azimuth of a maximum, deg from the major axis, mod 180) by ring radius: ", join([@sprintf("%.0f au: %.0f", x.r0, mod(x.m2.phi_deg, 180.0)) for x in s2], ";  "))
            @printf(io, "    m=2: %d rings above 3 sigma, orientation %.0f deg from the major axis (mod 180), R = %.2f (random ~%.2f)%s\n",
                    length(s2), mod(cs.mean / 2, 180.0), cs.R, 1 / sqrt(length(s2)),
                    cs.R >= 0.8 ? " -> a common orientation: a brightness (peak / width) modulation tied to the disk axes or to the beam: the elliptical beam, CLEAN / uv-coverage effects, ring thickness (see the checks above) or real structure" : "")
            if all(x -> isfinite(x.m2b.amp), s2)
                nal = count(x -> cos(2 * deg2rad(x.m2.phi_deg - x.m2b.phi_deg)) > 0.7, s2)
                nal > 0 && println(io, "    ", nal, " of ", length(s2), " of these orientations lie within ~20 deg of the one the beam alone predicts (modulo the excess in the beam table).")
            end
        end
        (length(s1) < 2 && length(s2) < 2) && println(io, "    fewer than 2 rings with a significant amplitude: no coherence test.")
        println(io, "\n  Candidate arcs / clumps (beam-smoothed residual above ", @sprintf("%.1f", asym.arc_sigma),
                " sigma of the smoothed residual, >= 1 beam of area, outside 1 beam from the centre):")
        if isempty(asym.arcs)
            println(io, "    none")
        else
            @printf(io, "  %3s %5s %8s %9s %9s %8s %8s %8s %11s  %s\n", "#", "sign", "r [au]", "az [deg]", "span[deg]", "sky PA", "S/N", "contrast", "area[beams]", "note")
            for (k, a) in enumerate(asym.arcs)
                note = a.dipole ? "paired with a comparable opposite-sign region ~180 deg away: likely centre / geometry error" :
                       (isfinite(a.nearest_ring) ? @sprintf("nearest ring %.0f au (%+.1f au)", a.nearest_ring, a.r_au - a.nearest_ring) : "")
                note *= isfinite(a.shift_au) ? @sprintf("; a centre error would need %.1f au", a.shift_au) : (a.at_peak ? "; on a ring crest, so a centre error has no first-order effect (slope ~0): not informative" : "")
                @printf(io, "  %3d %5s %8.1f %9.0f %9.0f %7.0f° %8.1f %7s %11.1f  %s\n", k, a.sign > 0 ? "+" : "-", a.r_au, a.phi_deg, a.span_deg,
                        a.pa_sky, a.snr, isfinite(a.contrast) ? @sprintf("%+.0f%%", 100 * a.sign * a.contrast) : "n/a", a.area_beams, note)
            end
            nreal = count(a -> !a.dipole, asym.arcs)
            println(io, "    ", nreal == 0 ? "None of the regions survives as an isolated (non-dipole) arc." : string(nreal, " region(s) are not part of a comparable +/- pair."))
            println(io, "    'a centre error would need X au': the centre shift whose dipole (shift x local profile slope) equals this residual; compare with the fitted centre uncertainty (~1 au or less).")
            println(io, "    az: disk-plane azimuth from the major axis; span: full azimuthal extent; contrast: peak smoothed residual / local azimuthal mean.")
        end
        println(io, "\n  How to read this: a single significant positive arc covering a limited azimuth range at a ring radius (span of tens of degrees,")
        println(io, "  contrast tens of percent) is the signature of a dust trap such as a vortex. A +/- pair 180 deg apart (in the residual image, not the ring-averaged amplitudes)")
        println(io, "  is what a small centre or inclination error produces.")
        println(io, "  The noise estimate for the arcs is the robust scatter of the smoothed residual itself, so real structure inflates it: the threshold is conservative.")
    end

    # 3. temperature and optical depth
    println(io, "\n3. BRIGHTNESS TEMPERATURE AND OPTICAL DEPTH")
    println(io, "-"^88)
    if tbp === nothing
        println(io, "  Not available (", freq === nothing ? "no observing frequency in the FITS header" : "see the console for the reason", ").")
    else
        lstar_missing = sp.lstar === nothing
        if lstar_missing
            @printf(io, "  Planck brightness temperature at %.1f GHz from the azimuthal mean. T_d, T_b/T_d and tau are NOT given: no stellar luminosity is known for this target, and a placeholder L* would make them misleading (pass lstar= to physical_analysis).\n", tbp.freq_hz / 1e9)
            println(io, "  T_b is beam-averaged, so it is a lower bound for features narrower than the beam.\n")
            @printf(io, "  %-6s %8s | %-16s\n", "type", "r [au]", "T_b [K]")
        else
            @printf(io, "  Planck brightness temperature at %.1f GHz from the azimuthal mean; T_d = (phi L* / 8 pi r^2 sigma_SB)^(1/4) with phi = %.3f, L* = %.2f Lsun%s, floor %.0f K.\n",
                    tbp.freq_hz / 1e9, tbp.phi, tbp.lstar, lstar_estimated ? " (ESTIMATED from M*)" : "", tbp.tfloor)
            println(io, "  tau = -ln(1 - T_b/T_d), absorption only. T_b is beam-averaged, so it is a lower bound for features narrower than the beam.\n")
        end
        have_h = sp.lstar !== nothing && sp.mstar !== nothing
        lstar_missing || @printf(io, "  %-6s %8s | %-16s %8s %7s %7s %8s\n", "type", "r [au]", "T_b [K]", "T_d [K]", "T_b/T_d", "tau", have_h ? "H/r" : "")
        feats = vcat([("ring", x) for x in ring_au], [("gap", x) for x in gap_au])
        sort!(feats; by = f -> f[2])
        for (kind, rau) in feats
            i = _nearest_bin(tbp, rau)
            i == 0 && continue
            if lstar_missing
                @printf(io, "  %-6s %8.1f | %-16s\n", kind, rau, _pm(tbp.tb[i], isfinite(tbp.tb_err[i]) ? tbp.tb_err[i] : 0.0, 1))
                continue
            end
            hr = have_h ? @sprintf("%.3f", _gas_scale_height_au(tbp.r_au[i], tbp.td[i], sp.mstar) / tbp.r_au[i]) : ""
            tau_txt = isfinite(tbp.tau[i]) ? @sprintf("%.2f", tbp.tau[i]) : ">~3 (T_b >= T_d)"
            @printf(io, "  %-6s %8.1f | %-16s %8.1f %7.2f %7s %8s\n", kind, rau, _pm(tbp.tb[i], isfinite(tbp.tb_err[i]) ? tbp.tb_err[i] : 0.0, 1),
                    tbp.td[i], tbp.ratio[i], tau_txt, hr)
        end
        sat = lstar_missing ? Float64[] : [tbp.r_au[i] for i in eachindex(tbp.r_au) if isfinite(tbp.ratio[i]) && tbp.ratio[i] >= 1]
        isempty(sat) || println(io, "\n  T_b >= T_d at ", length(sat), " radial bin(s) (r = ", @sprintf("%.0f", minimum(sat)), "-", @sprintf("%.0f", maximum(sat)),
                                " au): the disk is optically thick there OR the adopted T_d is too low (L*, phi) OR the flux scale / distance is off.")
        if !lstar_missing
            println(io, "\n  Interpretation: tau well below 1 means the emission is optically thin and the flux traces the dust surface density directly;")
            println(io, "  tau of order 1 or more means a column that is partly or fully optically thick. Scattering lowers T_b below the")
            println(io, "  absorption-only value, so the true optical depth of strongly scattering grains is higher than tau here.")
        end
    end
    # 4. planet masses
    println(io, "\n4. GAP-OPENING PLANET MASSES")
    println(io, "-"^88)
    lstar_estimated && println(io, ESTNOTE)
    if pm === nothing
        println(io, "  Not available (", planets_note, ").")
    else
        _planet_report(io, pm, sp; props = literature_props_au(t, prof.distance_pc))
    end
    # 5. ring widths and dust trapping
    println(io, "\n5. RING WIDTHS, DUST TRAPPING AND RING SPACING")
    println(io, "-"^88)
    lstar_estimated && println(io, ESTNOTE)
    if tr === nothing
        println(io, "  Not available (", trap_note, ").")
    else
        _trapping_report(io, tr, gap_au, sp; pm = pm)
    end
    println(io, "\nCAVEATS: L*, M* are approximate and the passive-disk T_d ignores accretion heating and the surface-layer/midplane difference; ")
    println(io, "the ring fits depend on the seed positions (see the 'seed' column); all errors are statistical only.")

    txt = String(take!(io))
    println("\n", txt)
    path = joinpath(OUTPUT_DIR, "$(slug)_physical_analysis.txt")
    try
        write(path, txt)
        println("Saved: ", path)
    catch err
        println("WARNING: could not write ", path, ": ", err)
    end
    return (fit = fit, geometry = geo, flaring = flare, asym = asym, roundbeam = rb, asym_alt = asym_alt, tb = tbp, planets = pm, trapping = tr, ringlist = ringlist_g, stellar = sp, report = path)
end

# ---------------------------------------------------------------------------
# Is the flux of the image all there? (missing short spacings resolve out extended emission)
# ---------------------------------------------------------------------------
"""
    flux_recovery(t, total_mjy, freq_ghz) -> Union{Nothing,NamedTuple}

Compares the flux integrated over the radial profile with the catalogue 1.33 mm flux (Bosschaart+26) scaled to the
observing frequency with a spectral index of 2 to 3 (optically thick to typical dust). Returns `nothing` when there is no
catalogue flux (or only an upper limit). `status`: :filtered (the image holds less than half of the expected flux, even for the most
favourable index), :partial (less than 80%), :excess (more than 1.3 times the largest expected flux) or :ok. The catalogue flux is a
total over the whole disc, the profile covers only the area inside the profile radius, and a band 3 extrapolation is uncertain
(free-free emission, spectral index), so the ratio is an indicator, not a measurement.
"""
function flux_recovery(t, total_mjy::Real, freq_ghz::Real)
    di = _disc_info(t)
    (di === nothing || !isfinite(di.fcont_mjy) || startswith(String(di.fcont_lim), "<")) && return nothing
    (isfinite(total_mjy) && isfinite(freq_ghz) && freq_ghz > 0) || return nothing
    e2 = di.fcont_mjy * (freq_ghz / 225.4)^2.0
    e3 = di.fcont_mjy * (freq_ghz / 225.4)^3.0
    elo, ehi = minmax(e2, e3)
    rlo, rhi = total_mjy / ehi, total_mjy / elo          # ratio image / expected: lowest and highest
    status = rhi < 0.5 ? :filtered : (rhi < 0.8 ? :partial : (rlo > 1.3 ? :excess : :ok))
    return (status = status, ratio_lo = rlo, ratio_hi = rhi, expected_lo = elo, expected_hi = ehi, total_mjy = Float64(total_mjy),
            freq_ghz = Float64(freq_ghz), fcont_mjy = di.fcont_mjy)
end

function flux_recovery_text(fr)
    fr === nothing && return ""
    pct(x) = 100 * x < 10 ? @sprintf("%.1f", 100 * x) : @sprintf("%.0f", 100 * x)
    exp_txt = fr.expected_hi / fr.expected_lo < 1.03 ? @sprintf("%.3g mJy", fr.expected_lo) : @sprintf("%.3g-%.3g mJy", fr.expected_lo, fr.expected_hi)
    ratio_txt = pct(fr.ratio_lo) == pct(fr.ratio_hi) ? pct(fr.ratio_lo) : pct(fr.ratio_lo) * "-" * pct(fr.ratio_hi)
    head = @sprintf("Flux check: the image integrates to %.3g mJy inside the profile radius; the catalogue flux (%.3g mJy at 1.33 mm) scales to %s at %.0f GHz (spectral index 2-3), so the image holds %s%% of it",
                    fr.total_mjy, fr.fcont_mjy, exp_txt, fr.freq_ghz, ratio_txt)
    if fr.status === :filtered
        return head * ". FILTERED: the extended emission is resolved out by missing short spacings, so the radii enclosing 50-90% of the flux, R68, the contrast of arcs and asymmetries, the brightness temperatures and the ring amplitudes describe only what survived, not the disc. Use an observation with a larger recoverable angular scale (obs_choice = N)."
    elseif fr.status === :partial
        return head * ". PARTLY FILTERED: some extended emission is missing; sizes are biased low and the smooth baseline is underestimated."
    elseif fr.status === :excess
        return head * ". ABOVE the expectation: check for an offset in the outer bins (primary-beam correction, CLEAN bias), other sources in the profile area or flux variability; the radii enclosing 90% of the flux are then biased high."
    end
    return head * ": consistent."
end


## Extended menu (archive disk targets)

The menu below grows from the 100 curated disks to **every disk target in the ALMA archive** (`MENU_SIZE = 100_000`; set `MENU_SIZE = 500` for the old short menu or `100` for the curated list only), alphabetically. Survey designations such as `J1604...` or `2MASS J...` are left out unless `MENU_INCLUDE_CATALOGUE_NAMES = true`. The archive list is cached; the first run queries the archive (about a minute). The menu is printed in four cells of 500 entries; comment out the ones you do not need.


In [ ]:
# ---------------------------------------------------------------------------
# Extended menu: the most-observed protoplanetary-disk targets in the ALMA archive
# ---------------------------------------------------------------------------
# The 100 curated disks above are always included (with their distance, inclination, PA and
# literature features). The rest of the menu is built from the archive: targets whose science
# category is "Disks and planet formation", ranked by number of public observations, name
# variants merged, listed alphabetically. Archive entries are not curated: the distance comes from
# a SIMBAD parallax when you select one, and the inclination/PA are fitted from the image.
# Archive entries also get a Bailer-Jones+21 Gaia distance (SIMBAD parallax as fallback) for the whole menu (prefill_distances!, cached; PREFETCH_DISTANCES = false skips it).
# Set MENU_SIZE = 100 for the curated list only; the default lists every archive disk target (the menu cells
# below print it in blocks of 500). The first call queries the archive (~1 min) and caches the list in
# ./alma_cache; refresh = true re-queries.

MENU_SIZE = 100_000                  # edit and re-run this cell; 100_000 = every archive disk target; 500 = the old short menu; 100 = curated list only
MENU_INCLUDE_CATALOGUE_NAMES = false  # true also lists survey designations (J1604..., 2MASS J..., ODISEA ..., Cl star ...): many more, mostly unreadable names
const DISK_CATALOGUE_DIR = CACHE_DIR
const _ARCHIVE_EXCLUDE = r"(_OFF_?\d*$|^Sun(_\d+)?$|^(Mars|Venus|Mercury|Jupiter|Saturn|Uranus|Neptune|Pluto|Moon|Io|Europa|Ganymede|Callisto|Titan|Enceladus|Ceres|Vesta)(_\d+)?$|ATLAS|PANSTARRS|^\d+P|(^|_)C\d{4}_[A-Z]\d)"i
# survey/catalogue designations (unreadable, mostly faint survey targets) and non-disk stars: left out of the menu
# unless include_catalogue_names = true
const _CATALOGUE_NAME = r"^(RA\d|J\d{6}|J\d{2}[\s:]?\d{2}|2MASS|Cl star|ODISEA|SSTc2d|SSTtau|source \d|f\d+$|Tau\d{6}|TYC |rrs\d|d\d+-\d+|G\d{2,3}\.\d|V star |Proxima|Alpha ?Cen|PSR |trappist|Betelgeuse|serp smm|sigOri|LupusI|HIP \d|VOS \d|CrA-\d|G09-CrA|EPIC \d|ISO-CHAII|ISO-CrA|eta crv|gamma lep|WD\d|p105-|NGC 2024|ONC$|S255)"i
const _DISK_FILTERS = ("scientific_category LIKE '%Disks and planet formation%'",
                       "science_keyword LIKE '%Disks around%'",
                       "science_keyword LIKE '%isk%'")

"""
    _with(t::AlmaTarget; kwargs...) -> AlmaTarget

Copy of `t` with some fields replaced (AlmaTarget is immutable).
"""
function _with(t::AlmaTarget; kw...)
    d = Dict{Symbol,Any}(f => getfield(t, f) for f in fieldnames(AlmaTarget))
    for (k, v) in kw
        d[k] = v
    end
    return AlmaTarget(; d...)
end

"""
    simbad_distance(ra_deg, dec_deg; radius_arcsec=5, min_snr=5) -> Union{Nothing,NamedTuple}

Distance (pc) from the SIMBAD parallax of the nearest object within `radius_arcsec`, if its
parallax is significant (parallax / error > `min_snr`). `nothing` if there is none.
"""
function simbad_distance(ra_deg::Real, dec_deg::Real; radius_arcsec::Real = 5.0, min_snr::Real = 5.0, strict::Bool = false)
    for rad in (radius_arcsec, 3 * radius_arcsec)
        # plain cone search; the nearest object with a significant parallax is picked on this side
        adql = "SELECT main_id, ra, dec, plx_value, plx_err FROM basic WHERE " *
               "CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', $ra_deg, $dec_deg, $(rad / 3600))) = 1 " *
               "AND plx_value IS NOT NULL AND plx_err IS NOT NULL"
        resp = HTTP.get("https://simbad.cds.unistra.fr/simbad/sim-tap/sync";
                        query = Dict("REQUEST" => "doQuery", "LANG" => "ADQL", "FORMAT" => "csv", "QUERY" => adql),
                        status_exception = false, retry = false, read_idle_timeout = 30)
        if resp.status != 200
            strict && error("SIMBAD TAP returned HTTP $(resp.status)")
            println("  SIMBAD TAP returned HTTP $(resp.status): ", first(String(resp.body), 200))
            return nothing
        end
        df = CSV.read(IOBuffer(resp.body), DataFrame)
        _dbg("simbad_distance: radius =", rad, "arcsec | objects with a parallax =", nrow(df))
        best = nothing; bd = Inf
        for row in eachrow(df)
            plx = tryparse(Float64, string(row.plx_value)); e = tryparse(Float64, string(row.plx_err))
            ra = tryparse(Float64, string(row.ra)); dec = tryparse(Float64, string(row.dec))
            any(isnothing, (plx, e, ra, dec)) && continue
            (plx > 0 && e > 0 && plx / e > min_snr) || continue
            d = hypot((ra - ra_deg) * cosd(dec_deg), dec - dec_deg) * 3600
            if d < bd
                bd = d; best = (distance_pc = 1000 / plx, id = String(row.main_id), plx = plx)
            end
        end
        best === nothing || return best
    end
    return nothing
end

"""
    _ensure_distance(t) -> AlmaTarget

Targets without a known distance (archive entries, custom targets) get one from Bailer-Jones+21 (Gaia),
else the SIMBAD parallax; if there is none, the 100 pc placeholder stays and radii in au are flagged as assumed.
"""
function _ensure_distance(t::AlmaTarget)
    t.distance_known && return t
    r = nothing; method = "none"
    try
        r, method, _ = best_distance(t.ra_deg, t.dec_deg)
    catch err
        println("Distance lookup failed: ", first(sprint(showerror, err), 160))
    end
    if r === nothing
        println("No Gaia or SIMBAD distance found near $(t.name); radii in au assume 100 pc (pass distance_pc = ... to override).")
        return t
    end
    getproperty(r, :multimodal) && println("  NOTE: ", _MULTIMODAL_NOTE)
    @printf("Distance from %s for %s: %.1f pc.\n", method == "BJ21" ? "Bailer-Jones+21 (" * r.id * ")" : "the SIMBAD parallax of " * r.id, t.name, r.distance_pc)
    return _with(t; distance_pc = r.distance_pc, distance_known = true,
                 note = r.multimodal ? strip(t.note * " " * _MULTIMODAL_NOTE) : t.note)
end

_prepare(t::AlmaTarget) = _ensure_distance(_ensure_coords(t))

function _fetch_disk_names(; public_only::Bool = true, max_rows::Int = 20000)
    base = "science_observation = 'T'" * (public_only ? " AND data_rights = 'Public'" : "")
    for f in _DISK_FILTERS
        adql = "SELECT TOP $max_rows target_name, COUNT(*) AS n_rows, AVG(s_ra) AS ra_deg, AVG(s_dec) AS dec_deg, " *
               "MIN(s_ra) AS ra_min, MAX(s_ra) AS ra_max, MIN(s_dec) AS dec_min, MAX(s_dec) AS dec_max, " *
               "MIN(spatial_resolution) AS best_res FROM ivoa.obscore WHERE $base AND $f " *
               "GROUP BY target_name ORDER BY n_rows DESC"
        try
            df = tap_query(adql)
            if nrow(df) > 0
                println("Disk target list from the ALMA archive using: ", f)
                return df
            end
        catch err
            println("  This filter failed (", first(sprint(showerror, err), 160), "); trying the next one ...")
        end
    end
    error("The ALMA archive returned no disk targets.")
end

"""
    build_disk_catalogue(n=500; refresh=false, public_only=true) -> Vector{AlmaTarget}

The 100 curated disks plus the most-observed other disk targets from the archive (science
category "Disks and planet formation"), `n` in all, alphabetical. Name variants and same-position
duplicates are merged; OFF positions, the Sun, planets, comets, survey/catalogue designations
(RA16 21 31.9, J1625..., 2MASS J..., sigOri 341 ...) and non-disk stars are dropped. Archive disks are taken
most-observed first among those with a best resolution <= `max_res_arcsec` (0.15"), relaxing that limit
only if fewer than `n` qualify. The archive list is cached.
"""
function build_disk_catalogue(n::Integer = 500; refresh::Bool = false, public_only::Bool = true,
                              exclude::Regex = _ARCHIVE_EXCLUDE, max_res_arcsec::Real = 0.15,
                              include_catalogue_names::Bool = false)
    file = joinpath(DISK_CATALOGUE_DIR, "alma_disk_catalogue_v3_" * (public_only ? "public" : "all") * ".csv")
    tbl = DataFrame()
    if !refresh && isfile(file)
        tbl = CSV.read(file, DataFrame; types = Dict(:key => String, :name => String))
        println("Disk target list read from the cache ($file); refresh = true re-queries the archive.")
    else
        raw = _fetch_disk_names(; public_only = public_only)
        merged = Dict{String,NamedTuple}()
        order = String[]
        for row in eachrow(raw)
            name = String(row.target_name)
            occursin(exclude, name) && continue
            cnt = tryparse(Int, string(row.n_rows)); cnt === nothing && continue
            ra = tryparse(Float64, string(row.ra_deg)); dec = tryparse(Float64, string(row.dec_deg))
            (ra === nothing || dec === nothing) && continue
            # a real single source has one position: generic names ("1", "A", "Field") used by many projects
            # average to meaningless coordinates, so drop names whose pointings spread over > 0.1 deg
            rlo, rhi = tryparse(Float64, string(row.ra_min)), tryparse(Float64, string(row.ra_max))
            dlo, dhi = tryparse(Float64, string(row.dec_min)), tryparse(Float64, string(row.dec_max))
            (rlo === nothing || rhi === nothing || dlo === nothing || dhi === nothing) && continue
            ((rhi - rlo) * cosd(dec) > 0.1 || (dhi - dlo) > 0.1) && continue
            res = tryparse(Float64, string(row.best_res)); res === nothing && (res = NaN)
            key = _norm(name)
            (length(key) < 3 || !occursin(r"[a-z]", key)) && continue      # "1", "A", "42" are not source names
            if haskey(merged, key)
                m = merged[key]
                r2 = isnan(m.res) ? res : (isnan(res) ? m.res : min(m.res, res))
                merged[key] = (name = m.name, n = m.n + cnt, ra = m.ra, dec = m.dec, res = r2)   # keep the biggest variant's name and position
            else
                merged[key] = (name = name, n = cnt, ra = ra, dec = dec, res = res)
                push!(order, key)
            end
        end
        sort!(order; by = k -> -merged[k].n)
        tbl = DataFrame(key = order, name = [merged[k].name for k in order], n_rows = [merged[k].n for k in order],
                        ra_deg = [merged[k].ra for k in order], dec_deg = [merged[k].dec for k in order],
                        best_res = [merged[k].res for k in order])
        try
            CSV.write(file, tbl)
            println("Saved the disk target list to $file ($(nrow(tbl)) targets).")
        catch err
            println("Could not cache the disk target list: ", first(sprint(showerror, err), 120))
        end
    end

    curated_keys = Set(k for t in ALMA_TARGETS for k in t.keys)
    # positions of the curated disks (their coordinates, or the archive position of the name that matched):
    # archive entries within 5 arcsec of one of them, or of an entry already accepted, are the same object
    # under another name ("AB Aur" / "AB Auriga" / "AB Aurigae", "49 Cet" / "49 Ceti")
    pos = Tuple{Float64,Float64}[(t.ra_deg, t.dec_deg) for t in ALMA_TARGETS if !isnan(t.ra_deg) && !isnan(t.dec_deg)]
    for row in eachrow(tbl)
        String(row.key) in curated_keys && push!(pos, (Float64(row.ra_deg), Float64(row.dec_deg)))
    end
    tol = 5.0 / 3600
    same_object(ra, dec) = any(p -> abs(p[2] - dec) < tol && abs(p[1] - ra) * cosd(dec) < tol, pos)
    extra = AlmaTarget[]
    for res_lim in (max_res_arcsec, 3 * max_res_arcsec, Inf)     # finest-resolution targets first; relax only if the list is short
    for row in eachrow(tbl)                      # most-observed first
        length(ALMA_TARGETS) + length(extra) >= n && break
        key = String(row.key)
        key in curated_keys && continue
        rres = Float64(coalesce(row.best_res, NaN))
        (isfinite(rres) ? rres <= res_lim : res_lim == Inf) || continue
        include_catalogue_names || !occursin(_CATALOGUE_NAME, replace(String(row.name), '_' => ' ')) || continue
        same_object(Float64(row.ra_deg), Float64(row.dec_deg)) && continue
        push!(pos, (Float64(row.ra_deg), Float64(row.dec_deg)))
        name = strip(replace(String(row.name), '_' => ' '))
        res = Float64(coalesce(row.best_res, NaN))
        nrows = Int(row.n_rows)
        differs = isnan(res) ? @sprintf("Not curated; %d archive rows; i/PA fitted, d from Gaia", nrows) :
                  @sprintf("Not curated; %d archive rows, finest %.3f\"; i/PA fitted, d from Gaia", nrows, res)
        push!(extra, AlmaTarget(; name = String(name), keys = [key], ra_deg = Float64(row.ra_deg),
                                dec_deg = Float64(row.dec_deg), distance_pc = 100.0, distance_known = false,
                                rmax_arcsec = 2.0, group = :archive, differs = differs,
                                note = "ALMA archive: $nrows rows"))
    end
    end      # res_lim
    all = vcat(ALMA_TARGETS, extra)
    sort!(all; by = t -> _sortkey(t.name))
    return all
end

"""
    use_extended_catalogue!(n=MENU_SIZE; refresh=false)

Make the menu (and `select_target`) use the `n`-target archive-based catalogue; `n <= 100`
restores the curated list. Falls back to the curated list if the archive query fails.
"""
function use_extended_catalogue!(n::Integer = MENU_SIZE; kwargs...)
    if n <= length(ALMA_TARGETS)
        ACTIVE_TARGETS[] = ALMA_TARGETS
        println("Menu: the curated catalogue ($(length(ALMA_TARGETS)) disks).")
        return ACTIVE_TARGETS[]
    end
    try
        ACTIVE_TARGETS[] = build_disk_catalogue(n; kwargs...)
        nex = count(t -> t.group === :archive, ACTIVE_TARGETS[])
        println("Menu: $(length(ACTIVE_TARGETS[])) disk targets ($(length(ALMA_TARGETS)) curated + $nex from the ALMA archive).")
    catch err
        println("WARNING: could not build the archive-based menu (", first(sprint(showerror, err), 200),
                "); using the curated catalogue.")
        ACTIVE_TARGETS[] = ALMA_TARGETS
    end
    return ACTIVE_TARGETS[]
end

use_extended_catalogue!(MENU_SIZE; include_catalogue_names = MENU_INCLUDE_CATALOGUE_NAMES);

# ---------------------------------------------------------------------------
# Distances for the whole menu: Bailer-Jones+21 (Gaia) for every archive entry, SIMBAD parallax as fallback, cached
# ---------------------------------------------------------------------------
const _MULTIMODAL_NOTE = "Gaia distance posterior possibly multimodal (Bailer-Jones Flag): treat the distance with caution."
PREFETCH_DISTANCES = true     # false skips the lookup (archive entries then keep the 100 pc placeholder until selected)
const DISTANCE_CACHE = joinpath(CACHE_DIR, "archive_distances.csv")

"""
    bailerjones_distance(ra_deg, dec_deg; radius_arcsec=5) -> Union{Nothing,NamedTuple}

Geometric distance (pc) from Bailer-Jones+21 (Gaia EDR3/DR3; VizieR table I/352/gedr3dis, column `rgeo`)
for the Gaia source nearest to the position, via the VizieR TAP service. The table positions are at
epoch 2016, so a 5 arcsec cone is used. Returns (distance_pc, lo, hi, id, multimodal) -- lo/hi are the 16th/84th
percentile limits when the reply has them, else NaN; `multimodal` is true when the Flag says the posterior may be multimodal -- or `nothing` if no source is in the cone.
Throws on a network/HTTP/column error so the caller can fall back to the SIMBAD parallax. The position
columns are read from the reply (RA_ICRS/DE_ICRS, else RAJ2000/DEJ2000), not assumed.
"""
function bailerjones_distance(ra_deg::Real, dec_deg::Real; radius_arcsec::Real = 5.0)
    rad = radius_arcsec / 3600
    last_err = "VizieR TAP query failed"
    for (rac, decc) in (("RA_ICRS", "DE_ICRS"), ("RAJ2000", "DEJ2000"))
        adql = "SELECT * FROM \"I/352/gedr3dis\" WHERE 1 = CONTAINS(POINT('ICRS', $rac, $decc), " *
               "CIRCLE('ICRS', $ra_deg, $dec_deg, $rad))"
        resp = HTTP.get("https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync";
                        query = Dict("REQUEST" => "doQuery", "LANG" => "ADQL", "FORMAT" => "csv", "QUERY" => adql),
                        status_exception = false, retry = false, read_idle_timeout = 30)
        if resp.status != 200
            last_err = "VizieR TAP HTTP $(resp.status): " * first(String(resp.body), 120)
            continue                    # possibly the other pair of position-column names
        end
        df = CSV.read(IOBuffer(resp.body), DataFrame)
        cols = Set(String.(propertynames(df)))
        for need in ("rgeo", "Source", rac, decc)
            need in cols || error("VizieR I/352 reply lacks column $need (has: " * join(sort(collect(cols)), ", ") * ")")
        end
        getf(row, c) = c in cols ? tryparse(Float64, string(row[Symbol(c)])) : nothing
        best = nothing; bd = Inf
        for row in eachrow(df)
            r, ra, dec = getf(row, "rgeo"), getf(row, rac), getf(row, decc)
            any(isnothing, (r, ra, dec)) && continue
            d = hypot((ra - ra_deg) * cosd(dec_deg), dec - dec_deg) * 3600
            if d < bd
                lo, hi = getf(row, "b_rgeo"), getf(row, "B_rgeo")
                # Flag = ABBCC (five digits, leading zeros may have been dropped); the 2nd digit (left B digit) = 1 means
                # the geometric distance posterior may be multimodal, so its median can be misleading
                fl = "Flag" in cols ? lpad(string(row[:Flag]), 5, '0') : "00000"
                multi = length(fl) >= 2 && fl[2] == '1'
                bd = d; best = (distance_pc = r, lo = something(lo, NaN), hi = something(hi, NaN),
                                id = "Gaia DR3 " * string(row[:Source]), multimodal = multi)
            end
        end
        return best
    end
    error(last_err)
end

"""
    best_distance(ra_deg, dec_deg) -> (result, method, cacheable)

Bailer-Jones+21 geometric distance first; if there is no Gaia source in the cone, or the VizieR lookup
fails, the SIMBAD parallax (1/parallax, S/N > 5). `method` is "BJ21" or "SIMBAD" (or "none"). A SIMBAD
result after a failed (not empty) Bailer-Jones lookup is not cacheable, so it is retried later.
SIMBAD HTTP errors throw, so the caller counts them as failures.
"""
function best_distance(ra_deg::Real, dec_deg::Real)
    bj_failed = false
    try
        r = bailerjones_distance(ra_deg, dec_deg)
        r === nothing || return (r, "BJ21", true)
    catch
        bj_failed = true
    end
    s = simbad_distance(ra_deg, dec_deg; strict = true)          # throws on error: the caller counts it as a failure
    s === nothing && return (nothing, "none", !bj_failed)
    return ((distance_pc = s.distance_pc, lo = NaN, hi = NaN, id = s.id, multimodal = false), "SIMBAD", !bj_failed)
end

"""
    prefill_distances!(targets = ACTIVE_TARGETS[]; refresh=false, ntasks=8)

Look up a distance for every archive (not curated) menu entry: the Bailer-Jones+21 Gaia geometric
distance for the nearest Gaia source within 5 arcsec, else the SIMBAD parallax (1/parallax, S/N > 5).
Results, including "no distance found", are cached in ./alma_cache/archive_distances.csv (columns
key, distance_pc, id, method), so later runs are instant; `refresh = true` re-queries everything.
Entries with a distance show it in the menu, and radii in au use it; entries with none keep 100 pc and
are flagged ASSUMED. Failed lookups (network, HTTP error) are not cached and are retried next time.
The first run makes one or two requests per entry and takes several minutes.

Caveats: the nearest Gaia source within 5 arcsec is taken, so a bright neighbour or an unresolved
binary can give the wrong star; embedded sources often have no Gaia counterpart; Bailer-Jones
distances for sources with poor astrometry (large RUWE) are less reliable.
"""
function prefill_distances!(targets = ACTIVE_TARGETS[]; refresh::Bool = false, ntasks::Int = 8)
    cache = Dict{String,NamedTuple}()
    if !refresh && isfile(DISTANCE_CACHE)
        try
            df = CSV.read(DISTANCE_CACHE, DataFrame)
            if :method in propertynames(df) && :multimodal in propertynames(df)   # an older cache layout is discarded and re-queried
                for r in eachrow(df)
                    cache[String(r.key)] = (distance_pc = Float64(coalesce(r.distance_pc, NaN)),
                                            id = String(coalesce(r.id, "")), method = String(coalesce(r.method, "none")),
                                            multimodal = coalesce(r.multimodal, false) == true)
                end
            end
        catch err
            println("Distance cache unreadable (", first(sprint(showerror, err), 120), "); re-querying.")
            empty!(cache)
        end
    end
    isarch(t) = t.group === :archive && !t.distance_known && isfinite(t.ra_deg) && isfinite(t.dec_deg)
    todo = [k for k in eachindex(targets) if isarch(targets[k]) && !haskey(cache, _norm(targets[k].name))]
    session = Dict{String,NamedTuple}()              # results used now but not cached (Bailer-Jones lookup failed)
    if !isempty(todo)
        println("Looking up Gaia (Bailer-Jones+21) distances, SIMBAD parallax as fallback, for $(length(todo)) archive entries ($(ntasks) at a time) ...")
        dbg = DEBUG[]; DEBUG[] = false
        done = Threads.Atomic{Int}(0); nfail = Threads.Atomic{Int}(0)
        try
            res = asyncmap(todo; ntasks = ntasks) do k
                t = targets[k]
                out = nothing
                for attempt in 1:2
                    try
                        r, method, cacheable = best_distance(t.ra_deg, t.dec_deg)
                        out = (key = _norm(t.name), distance_pc = r === nothing ? NaN : r.distance_pc,
                               id = r === nothing ? "" : r.id, method = method, cacheable = cacheable,
                               multimodal = r !== nothing && r.multimodal)
                        break
                    catch
                        sleep(1.5 * attempt)
                    end
                end
                out === nothing && Threads.atomic_add!(nfail, 1)
                n = Threads.atomic_add!(done, 1) + 1
                n % 100 == 0 && println("  ", n, " / ", length(todo))
                out
            end
            for r in res
                r === nothing && continue
                v = (distance_pc = r.distance_pc, id = r.id, method = r.method, multimodal = r.multimodal)
                r.cacheable ? (cache[r.key] = v) : (session[r.key] = v)
            end
        finally
            DEBUG[] = dbg
        end
        try
            mkpath(dirname(DISTANCE_CACHE))
            ks = sort(collect(keys(cache)))
            CSV.write(DISTANCE_CACHE, DataFrame(key = ks, distance_pc = [cache[k].distance_pc for k in ks],
                                                id = [cache[k].id for k in ks], method = [cache[k].method for k in ks],
                                                multimodal = [cache[k].multimodal for k in ks]))
        catch err
            println("Could not write the distance cache: ", first(sprint(showerror, err), 120))
        end
        nfail[] > 0 && println("  $(nfail[]) lookup(s) failed (network or server error); they are retried next time.")
    end
    nbj = 0; nsim = 0; nnone = 0; nmulti = 0
    for k in eachindex(targets)
        t = targets[k]
        isarch(t) || continue
        c = get(cache, _norm(t.name), get(session, _norm(t.name), nothing))
        c === nothing && continue
        if isfinite(c.distance_pc)
            note = c.multimodal ? strip(t.note * " " * _MULTIMODAL_NOTE) : t.note
            targets[k] = _with(t; distance_pc = c.distance_pc, distance_known = true, note = note)
            c.method == "BJ21" ? (nbj += 1) : (nsim += 1)
            c.multimodal && (nmulti += 1)
        else
            nnone += 1
        end
    end
    narch = count(t -> t.group === :archive, targets)
    nhave = count(t -> t.group === :archive && t.distance_known, targets)
    println("Archive distances: $nhave of $narch archive entries have one ($(narch - nhave) have none: the cloud average or 100 pc is assumed, flagged in the menu and the summary); ",
            "this call set $nbj from Bailer-Jones+21 and $nsim from the SIMBAD parallax.")
    nmulti > 0 && println("  $nmulti of them have a possibly multimodal Bailer-Jones distance posterior (Flag); this is noted in their catalogue note.")
    return targets
end

PREFETCH_DISTANCES && prefill_distances!();

# ---------------------------------------------------------------------------
# Published disc properties (Bosschaart+26, VizieR J/A+A/708/A143/tabled1, 730 discs) matched to the menu
# ---------------------------------------------------------------------------
# Downloaded once through VizieR TAP and cached in ./alma_cache; matched by position (3 arcsec) or by name; the
# class, R68 and stellar mass then appear in the menu text, the target summary and the physical-analysis report,
# and the catalogue stellar mass is used when no curated one exists (L* is not in this table).
# PREFETCH_DISC_INFO = false skips it; prefill_disc_info!(refresh = true) re-downloads the table.
PREFETCH_DISC_INFO = true
const DISC_CATALOGUE_FILE = joinpath(CACHE_DIR, "bosschaart26_tabled1.csv")

function _vz_col(df, name)
    for c in propertynames(df)
        String(c) == name && return df[!, c]
    end
    for c in propertynames(df)
        lowercase(String(c)) == lowercase(name) && return df[!, c]
    end
    return nothing
end
_vz_num(x) = x isa Real ? Float64(x) : (x isa AbstractString ? something(tryparse(Float64, strip(x)), NaN) : NaN)
_vz_str(x) = (x === missing || x === nothing) ? "" : String(strip(string(x)))

function fetch_disc_catalogue(; refresh::Bool = false)
    if !refresh && isfile(DISC_CATALOGUE_FILE)
        return CSV.read(DISC_CATALOGUE_FILE, DataFrame)
    end
    adql = "SELECT * FROM \"J/A+A/708/A143/tabled1\""
    resp = HTTP.get("https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync";
                    query = Dict("REQUEST" => "doQuery", "LANG" => "ADQL", "FORMAT" => "csv", "QUERY" => adql),
                    status_exception = false, retry = false, read_idle_timeout = 60)
    resp.status == 200 || error("VizieR TAP HTTP $(resp.status): " * first(String(resp.body), 160))
    df = CSV.read(IOBuffer(resp.body), DataFrame)
    nrow(df) > 0 || error("the VizieR table came back empty")
    try
        CSV.write(DISC_CATALOGUE_FILE, df)
    catch err
        println("Could not cache the Bosschaart+26 table: ", first(sprint(showerror, err), 120))
    end
    return df
end

function prefill_disc_info!(targets = ACTIVE_TARGETS[]; refresh::Bool = false, tol_arcsec::Real = 3.0,
                            add_missing::Bool = true)
    df = try
        fetch_disc_catalogue(; refresh = refresh)
    catch err
        println("Bosschaart+26 table not available (", first(sprint(showerror, err), 200), "); no published disc properties.")
        return targets
    end
    ra = _vz_col(df, "RA_ICRS"); de = _vz_col(df, "DE_ICRS")
    if ra === nothing || de === nothing
        println("Bosschaart+26 table lacks RA_ICRS / DE_ICRS (has: ", join(String.(propertynames(df)), ", "), "); skipped.")
        return targets
    end
    n = nrow(df)
    sc(name) = (c = _vz_col(df, name); c === nothing ? fill("", n) : _vz_str.(c))
    nc(name) = (c = _vz_col(df, name); c === nothing ? fill(NaN, n) : _vz_num.(c))
    cra = _vz_num.(ra); cde = _vz_num.(de)
    nm1, nm2, nm3 = sc("Name"), sc("SName"), sc("SimbadName")
    cls, sfr = sc("Class"), sc("SFR")
    dist, fdist, fc, lfc = nc("Dist"), sc("f_Dist"), nc("Fcont"), sc("l_Fcont")
    md, lmd, ms = nc("Mdust"), sc("l_Mdust"), nc("Mass")
    r68, lr68 = nc("R68"), sc("l_R68")
    # A catalogue position can be wrong (AB Aur is listed at RA 68.2, Dec +22.9 instead of 73.9, +30.6). Names of the form
    # [2MASS]JHHMMSSss+DDMMSSs encode the J2000 position; where it is more than 20 arcsec from the listed RA/Dec, the name's
    # position is used instead (a neighbouring component would differ by a few arcsec at most).
    refixed = Tuple{String,Float64}[]
    for i in 1:n
        for s in (nm3[i], nm2[i], nm1[i])
            m = match(r"^(?:2MASS)?J(\d{2})(\d{2})(\d{4})([+-])(\d{2})(\d{2})(\d{3})$", replace(String(s), r"\s+" => ""))
            m === nothing && continue
            rad = 15 * (parse(Float64, m[1]) + parse(Float64, m[2]) / 60 + parse(Float64, m[3]) / 100 / 3600)
            ded = (m[4] == "-" ? -1 : 1) * (parse(Float64, m[5]) + parse(Float64, m[6]) / 60 + parse(Float64, m[7]) / 10 / 3600)
            sepn = (isfinite(cra[i]) && isfinite(cde[i])) ? hypot((rad - cra[i]) * cosd(ded), ded - cde[i]) * 3600 : Inf
            if sepn > 20
                push!(refixed, (isempty(nm1[i]) ? String(s) : nm1[i], sepn))
                cra[i] = rad; cde[i] = ded
            end
            break
        end
    end
    if !isempty(refixed)
        println("  Position check: ", length(refixed), " catalogue row(s) are more than 20 arcsec from the position encoded in their 2MASS-style name; the name's position is used: ",
                join([@sprintf("%s (%.0f arcsec)", a, b) for (a, b) in first(refixed, 10)], ", "), length(refixed) > 10 ? ", ..." : "")
    end
    nameidx = Dict{String,Int}()
    for i in 1:n, s in (nm1[i], nm2[i], nm3[i])
        k = _norm(s)
        isempty(k) || haskey(nameidx, k) || (nameidx[k] = i)
    end
    tol = tol_arcsec / 3600
    nmatch = 0; npos = 0; nname = 0
    counts = Dict{String,Int}()
    matched_rows = Set{Int}()
    far = Tuple{String,Float64}[]                  # menu entries whose name is in the catalogue but whose position is > 30 arcsec away
    mkinfo(row, how, sep) = (name = isempty(nm1[row]) ? nm3[row] : nm1[row], class = uppercase(cls[row]), sfr = sfr[row],
                             matched_by = how, sep_arcsec = sep, r68_au = r68[row], r68_lim = lr68[row], fcont_mjy = fc[row],
                             fcont_lim = lfc[row], mdust_earth = md[row], mdust_lim = lmd[row], mstar = ms[row],
                             dist_pc = dist[row], dist_flag = fdist[row])
    empty!(DISC_INFO)
    for k in eachindex(targets)
        t = targets[k]
        row = 0; how = ""; sep = NaN
        if isfinite(t.ra_deg) && isfinite(t.dec_deg)
            bd = Inf
            for i in 1:n
                (isfinite(cra[i]) && isfinite(cde[i])) || continue
                abs(cde[i] - t.dec_deg) < tol || continue
                d = hypot((cra[i] - t.ra_deg) * cosd(t.dec_deg), cde[i] - t.dec_deg)
                d < tol && d < bd && (bd = d; row = i)
            end
            row > 0 && (how = "position"; sep = bd * 3600)
        end
        if row == 0
            for kk in unique(vcat(_norm(t.name), t.keys))
                i = get(nameidx, kk, 0)
                i == 0 && continue
                if isfinite(t.ra_deg) && isfinite(t.dec_deg) && isfinite(cra[i]) && isfinite(cde[i])
                    sepn = hypot((cra[i] - t.ra_deg) * cosd(t.dec_deg), cde[i] - t.dec_deg) * 3600
                    sepn > 30 && (push!(far, (t.name, sepn)); continue)   # same name, different place
                end
                row = i; how = "name"; break
            end
        end
        row == 0 && continue
        nmatch += 1
        push!(matched_rows, row)
        how == "position" ? (npos += 1) : (nname += 1)
        c = uppercase(cls[row])
        counts[c] = get(counts, c, 0) + 1
        DISC_INFO[t.slug] = mkinfo(row, how, sep)
        if !occursin("Bosschaart+26", t.differs)
            tag = "Bosschaart+26: " * (isempty(c) ? "listed" : c) * (isfinite(r68[row]) ? @sprintf(", R68 %s%.0f au", lr68[row], r68[row]) : "")
            targets[k] = _with(t; differs = isempty(t.differs) ? tag : t.differs * "; " * tag)
        end
    end
    println("Bosschaart+26 (", n, " discs): ", nmatch, " of ", length(targets), " menu entries matched (", npos, " by position, ", nname, " by name).")
    isempty(counts) || println("  classes among the matches: ", join(["$k = $(counts[k])" for k in sort(collect(keys(counts)))], ", "))
    if !isempty(far)
        println("  Coordinate check: ", length(far), " menu entries share a name with a catalogue disc but lie more than 30 arcsec from it ",
                "(the catalogue disc is appended as a separate entry; one of the two positions is wrong or they are different stars):")
        for (nm, sp) in first(sort(far; by = x -> -x[2]), 12)
            println("    ", @sprintf("%s (%.0f arcsec)", nm, sp))
        end
    end
    # catalogue discs with no menu entry are appended at the end of the menu (extended menu only)
    if add_missing && targets === ACTIVE_TARGETS[] && any(t -> t.group === :archive, targets)
        slugs = Set(t.slug for t in targets)
        nadd = 0
        for i in 1:n
            i in matched_rows && continue
            (isfinite(cra[i]) && isfinite(cde[i])) || continue
            nm = !isempty(nm1[i]) ? nm1[i] : (!isempty(nm3[i]) ? nm3[i] : nm2[i])
            isempty(nm) && continue
            slug = replace(nm, r"[^A-Za-z0-9]+" => "_")
            slug in slugs && (slug = slug * "_" * replace(sfr[i], r"[^A-Za-z0-9]+" => ""))
            slug in slugs && continue
            ks = unique(filter(!isempty, [_norm(nm1[i]), _norm(nm2[i]), _norm(nm3[i])]))
            c = uppercase(cls[i])
            tag = (isempty(c) ? "listed" : c) * (isfinite(r68[i]) ? @sprintf(", R68 %s%.0f au", lr68[i], r68[i]) : "")
            tnew = AlmaTarget(; name = String(nm), slug = slug, keys = ks, ra_deg = cra[i], dec_deg = cde[i],
                              distance_pc = isfinite(dist[i]) ? dist[i] : 100.0,
                              distance_known = isfinite(dist[i]) && !occursin("*", fdist[i]),   # "*" = cloud average, no Gaia parallax (Note 2): look up Gaia, keep the cloud value if none
                              rmax_arcsec = 2.0, group = :archive, name_match = false,
                              differs = "Bosschaart+26 only (" * sfr[i] * "): " * tag * "; archive rows by position, i/PA fitted",
                              note = "Bosschaart+26 catalogue entry (" * sfr[i] * "); position-based archive query" *
                                     (occursin("*", fdist[i]) && isfinite(dist[i]) ? "; distance = cloud average (no Gaia parallax)" : ""))
            push!(targets, tnew)
            push!(slugs, slug)
            DISC_INFO[slug] = mkinfo(i, "added", 0.0)
            counts[c] = get(counts, c, 0) + 1
            nadd += 1
        end
        println("  ", nadd, " catalogue discs without a menu entry were appended at the end of the menu (numbers ",
                length(targets) - nadd + 1, "-", length(targets), "); they are queried by position, without name filtering.")
        nadd > 0 && PREFETCH_DISTANCES && prefill_distances!(targets)
    end
    return targets
end

PREFETCH_DISC_INFO && prefill_disc_info!();

# ---------------------------------------------------------------------------
# Which discs can actually be downloaded? Package-size scan
# ---------------------------------------------------------------------------
# Many archive packages are 100-300 GB (calibrated measurement sets), far above max_tar_gb. This asks the archive,
# for each Bosschaart+26-matched disc (default: ring / transition / extended discs with R68 >= 40 au), how big the
# product package of each of its observations is, and whether single FITS files are offered instead, and reports the
# finest observation that fits under the limit. One TAP query plus one datalink request per observation, so it runs in
# batches (limit) and caches each disc in ./alma_cache/package_scan.csv. Nothing is downloaded.
const PACKAGE_SCAN_FILE = joinpath(CACHE_DIR, "package_scan.csv")

function _scan_one_disc(t::AlmaTarget, di, max_gb::Real, max_obs::Integer, min_las_ratio::Real)
    dbg = DEBUG[]; DEBUG[] = false
    nobs = 0; nchk = 0; smallest = NaN; best = nothing; nlasfail = 0
    diam_arcsec = 2 * di.r68_au / t.distance_pc               # 68%-flux diameter of the disc
    try
        redirect_stdout(devnull) do
            results = query_target(t; name_filter = t.name_match)
            obs = summarize_observations(results)
            nobs = nrow(obs)
            for i in 1:min(nobs, max_obs)
                r = obs[i, :]
                info = try
                    robust_get_data_info(r.member_ous_uid)
                catch
                    continue
                end
                nchk += 1
                url = pick_product_tarball(info)
                sz = url === nothing ? NaN : something(tarball_size_gb(info, url), NaN)
                dfits = direct_fits_urls(info, t; max_gb = max_gb)
                isfinite(sz) && (smallest = isfinite(smallest) ? min(smallest, sz) : sz)
                size_ok = (isfinite(sz) && sz <= max_gb) || !isempty(dfits)
                las = hasproperty(r, :las_arcsec) ? Float64(r.las_arcsec) : NaN
                lasr = (isfinite(las) && isfinite(diam_arcsec) && diam_arcsec > 0) ? las / diam_arcsec : NaN
                las_ok = !isfinite(lasr) || lasr >= min_las_ratio           # unknown scale: not excluded
                size_ok && !las_ok && (nlasfail += 1)
                if size_ok && las_ok && best === nothing                    # observations are sorted finest first
                    best = (via = (isfinite(sz) && sz <= max_gb) ? "package" : "single FITS",
                            size_gb = (isfinite(sz) && sz <= max_gb) ? sz : NaN,
                            member = String(r.member_ous_uid), band = String(r.band), res = Float64(r.best_res_arcsec),
                            las = las, lasr = lasr)
                end
            end
        end
    catch err
        DEBUG[] = dbg
        return (n_obs = nobs, n_checked = nchk, usable = false, via = "error: " * first(sprint(showerror, err), 60),
                size_gb = NaN, member = "", band = "", res_arcsec = NaN, beam_au = NaN, smallest_gb = smallest, las_arcsec = NaN, las_ratio = NaN)
    end
    DEBUG[] = dbg
    best === nothing && return (n_obs = nobs, n_checked = nchk, usable = false,
                                via = nobs == 0 ? "no public rows" : (nlasfail > 0 ? "too filtered (small enough packages recover only small scales)" : "all too large"),
                                size_gb = NaN, member = "", band = "", res_arcsec = NaN, beam_au = NaN, smallest_gb = smallest, las_arcsec = NaN, las_ratio = NaN)
    return (n_obs = nobs, n_checked = nchk, usable = true, via = best.via, size_gb = best.size_gb, member = best.member,
            band = best.band, res_arcsec = best.res, beam_au = best.res * t.distance_pc, smallest_gb = smallest,
            las_arcsec = best.las, las_ratio = best.lasr)
end

"""
    scan_small_packages(targets = ACTIVE_TARGETS[]; classes = ("RD","TD","ED"), min_r68_au = 40, max_gb = MAX_TAR_GB[],
                        limit = 20, max_obs = 12, max_beam_au = 15, min_las_ratio = 0.5, refresh = false) -> DataFrame

Scan up to `limit` not-yet-scanned Bosschaart+26-matched discs of the given classes with R68 >= `min_r68_au` (largest
first) and print a table of the ones with an observation that (1) fits under `max_gb` (as a package or as single FITS
files), (2) has an archive beam of at most `max_beam_au` au and (3) recovers angular scales of at least `min_las_ratio` times
the disc's 68%-flux diameter (the archive's largest angular scale; if the archive does not give it, the observation is not
excluded). The finest such observation is listed, with the menu number to run, the beam, the beams across R68 and the ratio of
the largest scale to the disc diameter. The beam is the archive's best case: the image delivered is typically 1.5-3.6 times
coarser (HD 100546, ODISEA_C4_062, AB Aur). Run it again for the next batch; `refresh = true` rescans;
`max_beam_au = Inf` and `min_las_ratio = 0` list everything.
"""
function scan_small_packages(targets = ACTIVE_TARGETS[]; classes = ("RD", "TD", "ED"), min_r68_au::Real = 40.0,
                             max_gb::Real = MAX_TAR_GB[], limit::Integer = 20, max_obs::Integer = 12,
                             max_beam_au::Real = 15.0, min_las_ratio::Real = 0.5, refresh::Bool = false)
    cache = Dict{String,NamedTuple}()
    if !refresh && isfile(PACKAGE_SCAN_FILE)
        try
            df = CSV.read(PACKAGE_SCAN_FILE, DataFrame)
            if :las_ratio in propertynames(df) && :min_las_ratio in propertynames(df)
                for r in eachrow(df)
                    cache[String(r.slug)] = (n_obs = Int(r.n_obs), n_checked = Int(r.n_checked), usable = r.usable == true,
                                             via = String(coalesce(r.via, "")), size_gb = Float64(coalesce(r.size_gb, NaN)),
                                             member = String(coalesce(r.member, "")), band = String(coalesce(r.band, "")),
                                             res_arcsec = Float64(coalesce(r.res_arcsec, NaN)), beam_au = Float64(coalesce(r.beam_au, NaN)),
                                             smallest_gb = Float64(coalesce(r.smallest_gb, NaN)), max_gb = Float64(coalesce(r.max_gb, NaN)),
                                             las_arcsec = Float64(coalesce(r.las_arcsec, NaN)), las_ratio = Float64(coalesce(r.las_ratio, NaN)),
                                             min_las_ratio = Float64(coalesce(r.min_las_ratio, NaN)))
                end
            else
                println("Package-scan cache from an older version (no largest-angular-scale column); rescanning.")
            end
        catch err
            println("Package-scan cache unreadable (", first(sprint(showerror, err), 100), "); rescanning.")
            empty!(cache)
        end
    end
    cand = Int[]
    for (k, t) in enumerate(targets)
        di = get(DISC_INFO, t.slug, nothing)
        di === nothing && continue
        di.class in classes || continue
        isfinite(di.r68_au) && di.r68_au >= min_r68_au && !startswith(String(di.r68_lim), "<") || continue
        (isfinite(t.ra_deg) && isfinite(t.dec_deg)) || continue
        push!(cand, k)
    end
    sort!(cand; by = k -> -DISC_INFO[targets[k].slug].r68_au)
    stale(k) = !haskey(cache, targets[k].slug) || cache[targets[k].slug].max_gb != Float64(max_gb) ||
               cache[targets[k].slug].min_las_ratio != Float64(min_las_ratio)
    todo = first([k for k in cand if refresh || stale(k)], limit)
    println(length(cand), " matched discs of class ", join(classes, "/"), " with R68 >= ", min_r68_au, " au; ",
            length(todo), " to scan now (", count(k -> !stale(k), cand), " already in the cache), limit ", max_gb, " GB, largest scale >= ",
            min_las_ratio, " x the disc diameter.")
    for (j, k) in enumerate(todo)
        t = targets[k]; di = DISC_INFO[t.slug]
        r = _scan_one_disc(t, di, max_gb, max_obs, min_las_ratio)
        cache[t.slug] = merge(r, (max_gb = Float64(max_gb), min_las_ratio = Float64(min_las_ratio)))
        if r.usable
            @printf("  [%d/%d] %-26s R68 %3.0f au: OK, %s, band %s, archive beam %.1f au%s\n", j, length(todo), t.name, di.r68_au, r.via, r.band, r.beam_au,
                    isfinite(r.las_ratio) ? @sprintf(", largest scale %.1f x the disc diameter", r.las_ratio) : ", largest scale not given")
        else
            @printf("  [%d/%d] %-26s R68 %3.0f au: %s (smallest package %s GB)\n", j, length(todo), t.name, di.r68_au, r.via,
                    isfinite(r.smallest_gb) ? @sprintf("%.0f", r.smallest_gb) : "n/a")
        end
    end
    try
        mkpath(dirname(PACKAGE_SCAN_FILE))
        ks = sort(collect(keys(cache)))
        CSV.write(PACKAGE_SCAN_FILE, DataFrame(slug = ks, n_obs = [cache[s].n_obs for s in ks], n_checked = [cache[s].n_checked for s in ks],
                  usable = [cache[s].usable for s in ks], via = [cache[s].via for s in ks], size_gb = [cache[s].size_gb for s in ks],
                  member = [cache[s].member for s in ks], band = [cache[s].band for s in ks], res_arcsec = [cache[s].res_arcsec for s in ks],
                  beam_au = [cache[s].beam_au for s in ks], smallest_gb = [cache[s].smallest_gb for s in ks],
                  max_gb = [cache[s].max_gb for s in ks], las_arcsec = [cache[s].las_arcsec for s in ks],
                  las_ratio = [cache[s].las_ratio for s in ks], min_las_ratio = [cache[s].min_las_ratio for s in ks]))
    catch err
        println("Could not write the package-scan cache: ", first(sprint(showerror, err), 100))
    end
    rows = NamedTuple[]
    ncoarse = 0
    for k in cand
        t = targets[k]; c = get(cache, t.slug, nothing)
        (c !== nothing && c.usable && c.max_gb == Float64(max_gb) && c.min_las_ratio == Float64(min_las_ratio)) || continue
        isfinite(c.beam_au) && c.beam_au > max_beam_au && (ncoarse += 1; continue)
        di = DISC_INFO[t.slug]
        push!(rows, (menu = k, name = t.name, cls = di.class, r68_au = di.r68_au, dist_pc = t.distance_pc,
                     distance_known = t.distance_known, via = c.via, size_gb = c.size_gb, band = c.band,
                     archive_beam_au = c.beam_au, beams_across_r68 = di.r68_au / c.beam_au, las_over_diameter = c.las_ratio))
    end
    if isempty(rows)
        println("\nNo scanned disc has a package under ", max_gb, " GB with an archive beam of at most ", max_beam_au, " au and enough large-scale sensitivity yet.")
        return DataFrame()
    end
    out = sort!(DataFrame(rows), :beams_across_r68; rev = true)
    println("\nDiscs with a downloadable observation, an archive beam of at most ", max_beam_au, " au and a largest scale of at least ", min_las_ratio,
            " x the disc diameter (most beams across R68 first); run the menu number:")
    show(stdout, out; allrows = true, allcols = true, eltypes = false)
    println()
    println("archive_beam_au is the archive's best case; the delivered image is typically 1.5-3.6 times coarser. las_over_diameter = largest recoverable scale / 68%-flux diameter (NaN: not given by the archive).")
    ncoarse > 0 && println(ncoarse, " more have a usable observation but a beam above ", max_beam_au, " au (max_beam_au = Inf lists them).")
    any(!, out.distance_known) && println("distance_known = false: the distance is a cloud average or 100 pc, so the beam in au is approximate.")
    return out
end

# ---------------------------------------------------------------------------
# Are the curated positions right? Check against CDS Sesame (SIMBAD)
# ---------------------------------------------------------------------------
# A wrong curated position makes the archive cone miss the star (HD 100546 and DM Tau were each off by exactly one minute
# of RA, 0.25 deg, and returned either no rows or a different star). This resolves every curated entry's name (the
# `simbad` aliases first, then the name) with Sesame and lists the entries that are more than `tol_arcsec` from the
# stored position, with the corrected RA/Dec to paste into the catalogue. It reads nothing else and changes nothing.
function check_curated_coordinates(targets = ACTIVE_TARGETS[]; tol_arcsec::Real = 10.0, ntasks::Int = 4)
    idx = [k for k in eachindex(targets) if targets[k].group !== :archive && isfinite(targets[k].ra_deg) && isfinite(targets[k].dec_deg)]
    println("Checking ", length(idx), " curated entries against CDS Sesame (SIMBAD) ...")
    res = asyncmap(idx; ntasks = ntasks) do k
        t = targets[k]
        names = unique(filter(!isempty, strip.(vcat(split(string(t.simbad), '|'), [t.name]))))
        for nm in names
            try
                ra, de = resolve_name(nm)
                return (k = k, via = nm, ra = ra, dec = de)
            catch
            end
        end
        return (k = k, via = "", ra = NaN, dec = NaN)
    end
    bad = NamedTuple[]
    nres = 0
    unresolved = String[]
    for r in res
        t = targets[r.k]
        if !isfinite(r.ra)
            push!(unresolved, t.name)
            continue
        end
        nres += 1
        sep = hypot((r.ra - t.ra_deg) * cosd(t.dec_deg), r.dec - t.dec_deg) * 3600
        sep > tol_arcsec && push!(bad, (menu = r.k, name = t.name, stored_ra = t.ra_deg, stored_dec = t.dec_deg,
                                         sesame_ra = r.ra, sesame_dec = r.dec, sep_arcsec = sep, resolved_as = r.via))
    end
    println(nres, " resolved; ", length(bad), " more than ", tol_arcsec, " arcsec from the stored position; ", length(unresolved), " could not be resolved",
            isempty(unresolved) ? "." : ": " * join(first(unresolved, 10), ", ") * (length(unresolved) > 10 ? ", ..." : "."))
    if isempty(bad)
        println("All resolved curated positions agree with SIMBAD.")
        return DataFrame()
    end
    out = sort!(DataFrame(bad), :sep_arcsec; rev = true)
    for r in eachrow(out)
        dra = (r.sesame_ra - r.stored_ra) * cosd(r.stored_dec) * 3600
        note = abs(abs(dra) - 900 * cosd(r.stored_dec)) < 30 && abs(r.sesame_dec - r.stored_dec) * 3600 < 30 ? "  <- one minute of RA" : ""
        @printf("  %4d  %-22s stored RA %.5f Dec %+.5f  ->  SIMBAD RA %.5f Dec %+.5f  (%.0f arcsec)%s\n",
                r.menu, r.name, r.stored_ra, r.stored_dec, r.sesame_ra, r.sesame_dec, r.sep_arcsec, note)
    end
    println("The Sesame position of a name can itself be a neighbour or a binary component; check the name before pasting it into the catalogue.")
    return out
end


In [ ]:
# Menu numbers 1-500 (comment this line out to hide the block)
print_target_menu(from = 1, to = 500, legend = false)

In [ ]:
# Menu numbers 501-1000
print_target_menu(from = 501, to = 1000, legend = false)

In [ ]:
# Menu numbers 1001-1500
print_target_menu(from = 1001, to = 1500, legend = false)

In [ ]:
# Menu numbers 1501 to the end (with the notes on groups and columns)
print_target_menu(from = 1501)

In [ ]:
# Check of the Bosschaart+26 catalogue cache: file present, its size, and the first 25 rows
# ALMA star forming regions observations (Bosschaart+, 2026)
# J/A+A/708/A143/tabled1    Full disc sample (730 rows)
# https://vizier.cds.unistra.fr/viz-bin/VizieR-3?-source=J/A%2bA/708/A143/tabled1

println("\nJ/A+A/708/A143/tabled1   Full disc sample (730 rows)\n")
isfile(DISC_CATALOGUE_FILE), filesize(DISC_CATALOGUE_FILE)
first(CSV.read(DISC_CATALOGUE_FILE, DataFrame), 25)

In [ ]:
# Which catalogue discs can be downloaded at all? Scans package sizes (no download) for ring/transition/extended discs
# with R68 >= 40 au, 20 at a time; results are cached, so run it again for the next batch.
# Lists only observations that fit the size limit, have an archive beam <= 15 au (max_beam_au) and recover large enough
# angular scales (min_las_ratio = 0.5 x the disc diameter); the delivered image is typically 1.5-3.6x coarser than the archive beam.
# Uncomment to run:
# scan_small_packages(limit = 20);   # the trailing ; stops the table printing twice

In [ ]:
# Are the curated target positions right? Compares each curated entry with CDS Sesame (SIMBAD) and lists the ones that
# are more than 10 arcsec off, with the corrected RA/Dec. Uncomment to run (about a minute):
# check_curated_coordinates();

## Select and run
The first run for a target downloads its product tarball (several GB possible; only FITS files are kept). Packages larger than `MAX_TAR_GB` are refused unless `force_download = true`. Plots are saved to `./alma_plots/`.

In [ ]:
# Choose the target:
#   nothing            -> print the menu and prompt for a number or name
#   <number> or  "MWC 480"   -> pick directly (menu numbers are alphabetical; names are easier) (any spelling works: "mwc480")
#   custom_target("V883 Ori"; distance_pc = 388)   -> any other source (name resolved via Sesame)
TARGET_CHOICE = nothing

target = select_target(TARGET_CHOICE)
out = run_alma_pipeline(target);
# Useful options:  band = 7,  obs_choice = 2,  proposal_id = "2016.1.00484.L",
#                  center_mode = :peak,  incl_deg = 40, pa_deg = 90,  force_download = true

In [ ]:
# ALMA archive link for the selected target, in its own cell so it is not buried in the pipeline output.
# Also saves a PNG of the archive page next to the other plots (<slug>_archive_snapshot.png; snapshot = false to skip).
# Run it after the target is selected (it also works before the pipeline has finished or if the run fails).
# Headless-browser snapshot of the archive page (needs Chrome, Chromium or Edge installed; nothing else).
function _find_chrome()
    for c in ["/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
              "/Applications/Chromium.app/Contents/MacOS/Chromium",
              "/Applications/Microsoft Edge.app/Contents/MacOS/Microsoft Edge",
              "/usr/bin/google-chrome", "/usr/bin/chromium", "/usr/bin/chromium-browser",
              "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe",
              "C:\\Program Files (x86)\\Google\\Chrome\\Application\\chrome.exe",
              "C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe"]
        isfile(c) && return c
    end
    return nothing
end

"""
    snapshot_archive_page(url, png; width = 1600, height = 1400, wait_ms = 25000, timeout_s = 120) -> Bool

Save a PNG of `url` using headless Chrome (a fresh temporary profile, so a running Chrome is not disturbed).
`wait_ms` is the virtual time given to the page to load its table. Returns true if the file was written.
"""
function snapshot_archive_page(url::AbstractString, png::AbstractString; width::Integer = 1600,
                               height::Integer = 1400, wait_ms::Integer = 25000, timeout_s::Real = 120)
    chrome = _find_chrome()
    if chrome === nothing
        println("No Chrome / Chromium / Edge found: the archive snapshot was skipped.")
        return false
    end
    profile = mktempdir()
    cmd = Cmd([chrome, "--headless=new", "--disable-gpu", "--hide-scrollbars", "--user-data-dir=$profile",
               "--window-size=$(width),$(height)", "--virtual-time-budget=$(wait_ms)", "--screenshot=$png", String(url)])
    println("Taking an archive snapshot with headless ", basename(chrome), " (can take ~30 s) ...")
    proc = nothing
    try
        proc = run(pipeline(cmd; stdout = devnull, stderr = devnull); wait = false)
        t0 = time()
        while process_running(proc) && time() - t0 < timeout_s
            sleep(0.5)
        end
        if process_running(proc)
            println("The browser did not finish within ", timeout_s, " s; stopping it.")
            kill(proc, Base.SIGKILL)
        end
    catch err
        err isa InterruptException && rethrow()
        println("Archive snapshot failed: ", first(sprint(showerror, err), 160))
    finally
        proc !== nothing && process_running(proc) && kill(proc, Base.SIGKILL)
        rm(profile; recursive = true, force = true)
    end
    ok = isfile(png) && filesize(png) > 0
    println(ok ? "Saved archive snapshot: $png" : "The snapshot file was not written.")
    return ok
end

function show_archive_link(t = (@isdefined(target) ? target : nothing); snapshot::Bool = true)
    t === nothing && (println("Select a target first (run the 'Select and run' cell)."); return nothing)
    t = _ensure_coords(t)
    url = archive_url(t)
    println("="^80)
    @printf("Target: %s   (RA %.4f, Dec %+.4f; %s)\n", t.name, t.ra_deg, t.dec_deg, t.note)
    println("="^80)
    println("ALMA archive web portal: ", url)
    try                                           # clickable link where the notebook renders Markdown
        display("text/markdown", "**ALMA archive web portal:** [" * t.name * " in the ALMA Science Archive](" * url * ")")
    catch
    end
    snapshot && snapshot_archive_page(url, joinpath(OUTPUT_DIR, t.slug * "_archive_snapshot.png"))
    return url
end

show_archive_link();


In [ ]:
# Packages refused by the size guard so far (target, project, band, size in GB).
# Use obs_choice = N, band = N, force_download = true, or raise MAX_TAR_GB[] to get one anyway.
print_too_large()


## Optional: re-tune the substructure detection

In [ ]:
# Re-run just the substructure detection with different settings -- no
# re-download needed. Note `min_abs_residual_frac` (default 0.3) can drop
# candidates that pass the prominence test; try 0.1 to see weak features.
if out !== nothing && hasproperty(out, :profile) && out.profile !== nothing
    rings2, gaps2, rd2 = find_rings_and_gaps_residual(out.profile; prominence_sigma = 0.6,
                                                       min_abs_residual_frac = 0.1)
    compare_with_literature(out.lit_features, gaps2, rd2; bmaj_au = out.profile.bmaj_au, rings = rings2)
    plot_residual_profile(rd2; rings = rings2, gaps = gaps2, lit_features = out.lit_features,
        title = "$(out.target.name) substructure (retuned)",
        filename = "$(out.target.slug)_substructure_retuned.png");
    write_target_summary(out; rings = rings2, gaps = gaps2, residual_data = rd2,
        note = "re-tuned detection (prominence_sigma = 0.6, min_abs_residual_frac = 0.1)")
end

## Optional: physical analysis of the current target

In [ ]:
# Physical analysis of the target in `out`: Gaussian ring fits, azimuthal asymmetry, brightness temperature.
# Takes about a minute on a large image (it re-reads the FITS file). Writes <slug>_physical_analysis.txt and PNGs.
# Options (all optional):
#   rings = rings2, gaps = gaps2        use the re-tuned detections from the cell above
#   extra_seeds_au = [15.0, 28.0]       start the fit from ring radii the detector missed
#   lstar = 1.4, mstar = 0.83           stellar luminosity / mass in solar units for targets not in STELLAR_PARAMS
#   use_literature_seeds = false        start only from the detected rings
phys = physical_analysis(out);

## Optional: rank targets by archive observation count

In [ ]:
# OPTIONAL: the 100 most-observed science targets in the ALMA archive, listed alphabetically.
# Name variants are merged, OFF positions/Sun dropped, and curated targets (marked *) keep their
# distance / inclination / PA. Uncurated ones assume 100 pc unless you pass distance_pc.
# top_archive_targets(100; sort_by = :rows) lists by archive rows instead.
archive_top = top_archive_targets(100)
# out2 = run_alma_pipeline(archive_top[3]; distance_pc = 150.0)
